# Full CheXpert VLM Benchmarking & Relabeling Pipeline (Kaggle Ready)

## 1. Install Dependencies

In [ ]:
!pip install -q hi-ml-multimodal
!pip install -q open_clip_torch   # needed for whyxrayclip
!pip install -q "transformers>=4.36.0,<5" accelerate

## 2. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from PIL import Image
from pathlib import Path
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer

## 3. Dataset Paths & Loading

In [ ]:

BASE_PATH        = "/kaggle/input/chexpert-dataset-ml-project"
TEST_BASE_PATH   = "/kaggle/input/chexpert-test-dataset-ml-project/cheXpert-test-set-labels-main"

train_df        = pd.read_csv(f"{BASE_PATH}/train.csv")
valid_df        = pd.read_csv(f"{BASE_PATH}/valid.csv")
rule_labeler_df = pd.read_csv(f"{TEST_BASE_PATH}/labeler/chexpert_labels.csv")
ground_truth_df = pd.read_csv(f"{TEST_BASE_PATH}/groundtruth.csv")

print(train_df.shape, valid_df.shape, rule_labeler_df.shape, ground_truth_df.shape)


In [ ]:
# ================================================================
# Count of 0/1/-1 per feature (with total)
# ================================================================

# Choose which dataframe to inspect (default: train_df)
_df = train_df

features = [c for c in _df.columns if c not in ["Path", "Study", "Sex", "Age", "Frontal/Lateral", "AP/PA"]]
rows = []

for col in features:
    if col not in _df.columns:
        continue
    counts = _df[col].value_counts(dropna=False)
    rows.append({
        "feature": col,
        "count_-1": int(counts.get(-1, 0)),
        "count_0": int(counts.get(0, 0)),
        "count_1": int(counts.get(1, 0)),
        "count_nan": int(counts.get(np.nan, 0)) if np.nan in counts.index else 0,
        "total": int(counts.sum())
    })

counts_df = pd.DataFrame(rows).sort_values("feature")
print(counts_df.to_string(index=False))

# Optional: save to CSV
counts_df.to_csv("label_value_counts.csv", index=False)
print("\nSaved: label_value_counts.csv")

## 4. Path Fixing Logic (Your Structure)

In [ ]:
TEST_IMG_BASE = f"{TEST_BASE_PATH}/test"

def fix_test_path(path):
    """
    Handles both cases:
    - path points to a directory (patient/study) → find jpg inside
    - path points to a file directly → use as is
    """
    if 'test/' in str(path):
        relative = path.split('test/')[-1]
    else:
        relative = path
    
    full_path = os.path.join(TEST_IMG_BASE, relative)
    
    # If it's a directory, find the first jpg inside
    if os.path.isdir(full_path):
        for fname in os.listdir(full_path):
            if fname.endswith('.jpg') or fname.endswith('.png'):
                return os.path.join(full_path, fname)
        # Try one level deeper
        for sub in os.listdir(full_path):
            sub_path = os.path.join(full_path, sub)
            if os.path.isdir(sub_path):
                for fname in os.listdir(sub_path):
                    if fname.endswith('.jpg') or fname.endswith('.png'):
                        return os.path.join(sub_path, fname)
    
    return full_path

## 5. Pathology Selection

In [ ]:
# DIAGNOSTIC — find best replacement labels
print("Distribution of uncertain samples per label in rule_labeler_df vs ground_truth_df")
print("="*70)

results = []
for col in rule_labeler_df.columns:
    if col in ["Path", "Study", "Sex", "Age", "Frontal/Lateral", 
               "AP/PA", "No Finding", "Support Devices"]:
        continue
    if col not in ground_truth_df.columns:
        continue
    
    # Uncertain in rule_labeler
    uncertain_idx = rule_labeler_df[rule_labeler_df[col] == -1].index
    if len(uncertain_idx) == 0:
        continue
    
    # Ground truth for those uncertain samples
    gt_vals = ground_truth_df.loc[uncertain_idx, col]
    gt_vals = gt_vals[gt_vals.isin([0, 1])]
    
    pos = (gt_vals == 1).sum()
    neg = (gt_vals == 0).sum()
    total = pos + neg
    
    if total == 0:
        continue
    
    ratio = min(pos, neg) / max(pos, neg) if max(pos, neg) > 0 else 0
    
    results.append({
        "label": col,
        "n_uncertain": len(uncertain_idx),
        "gt_pos": pos,
        "gt_neg": neg,
        "total_matched": total,
        "balance_ratio": round(ratio, 3)  # 1.0 = perfect, 0.0 = worst
    })
    
results_df = pd.DataFrame(results).sort_values("balance_ratio", ascending=False)
print(results_df.to_string(index=False))

In [ ]:

TARGET_LABELS = [
    "Edema",
    "Atelectasis",
    "Pleural Effusion",
    "Enlarged Cardiomediastinum",
    "Consolidation", 
]


# Feature Correlation Analysis

In [ ]:
# Cell 5.5: Feature Correlation Analysis
import numpy as np
import pandas as pd
from scipy.stats import pointbiserialr

def find_top_correlated_features(train_df, target_label, n_top=5):
    """
    Find top N features correlated with target label using point-biserial correlation
    (since target is binary and features are binary/continuous)
    """
    # Make sure target is binary (convert -1 to 0 or NaN as needed)
    train_df_clean = train_df.copy()
    
    # Get all pathology columns (excluding non-numeric columns)
    # First, identify which columns are numeric
    numeric_cols = []
    for col in train_df_clean.columns:
        try:
            # Try to convert first few values to see if column is numeric
            pd.to_numeric(train_df_clean[col].iloc[:5], errors='raise')
            numeric_cols.append(col)
        except:
            continue
    
    # Filter to only pathology-related columns (you can adjust this list)
    pathology_cols = [
        'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
        'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
        'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
    ]
    
    # Only use columns that exist in the dataframe and are numeric
    feature_cols = [col for col in pathology_cols if col in train_df_clean.columns]
    
    print(f"  Analyzing correlations with {len(feature_cols)} potential features")
    
    correlations = []
    
    # Clean target data
    train_df_clean[target_label] = pd.to_numeric(train_df_clean[target_label], errors='coerce')
    target_data = train_df_clean[target_label].dropna()
    
    if len(target_data) < 10:
        print(f"  Warning: Not enough samples for {target_label}")
        return [], pd.DataFrame()
    
    for feature in feature_cols:
        if feature == target_label:
            continue
        
        # Clean feature data
        train_df_clean[feature] = pd.to_numeric(train_df_clean[feature], errors='coerce')
        
        # Get non-null pairs for correlation
        mask = train_df_clean[[target_label, feature]].dropna().index
        if len(mask) >= 10:  # Minimum samples for correlation
            y = train_df_clean.loc[mask, target_label]
            x = train_df_clean.loc[mask, feature]
            
            try:
                # For binary features, use point-biserial correlation
                if len(x.unique()) <= 2:  # Binary feature
                    corr, p_value = pointbiserialr(x, y)
                else:  # Continuous feature
                    corr = x.corr(y)
                    p_value = None
                
                if not np.isnan(corr):
                    correlations.append({
                        'feature': feature,
                        'correlation': abs(corr),
                        'direction': corr,
                        'p_value': p_value,
                        'n_samples': len(mask)
                    })
            except Exception as e:
                # Skip if correlation calculation fails
                continue
    
    if not correlations:
        print(f"  Warning: No correlations found for {target_label}")
        return [], pd.DataFrame()
    
    # Sort by absolute correlation and get top N
    correlations_df = pd.DataFrame(correlations)
    top_features = correlations_df.nlargest(min(n_top, len(correlations_df)), 'correlation')['feature'].tolist()
    
    print(f"\n  Top {len(top_features)} features correlated with {target_label}:")
    for feat in top_features:
        feat_corr = correlations_df[correlations_df['feature'] == feat]['direction'].values[0]
        feat_samples = correlations_df[correlations_df['feature'] == feat]['n_samples'].values[0]
        print(f"    • {feat}: {feat_corr:.3f} (n={feat_samples})")
    
    return top_features, correlations_df

# Find top features for each target label
top_features_per_label = {}
all_correlations = {}

print("📊 Starting correlation analysis...")
print("="*60)

for label in TARGET_LABELS:
    print(f"\n{'='*50}")
    print(f"Analyzing correlations for: {label}")
    print('='*50)
    top_features, correlations = find_top_correlated_features(train_df, label, n_top=5)
    top_features_per_label[label] = top_features
    all_correlations[label] = correlations

# Save correlations for reference
if all_correlations:
    correlation_summary = pd.concat([
        correlations.assign(target_label=label) 
        for label, correlations in all_correlations.items() 
        if not correlations.empty
    ])
    correlation_summary.to_csv('feature_correlations.csv', index=False)
    print("\n✅ Correlation analysis complete. Saved to 'feature_correlations.csv'")
else:
    print("\n⚠️ No correlations found. Using default features.")
    
    # Provide default features if correlation analysis fails
    default_features = {
        'Atelectasis': ['Lung Opacity', 'Consolidation', 'Pneumonia', 'Edema', 'Pleural Effusion'],
        'Consolidation': ['Lung Opacity', 'Atelectasis', 'Pneumonia', 'Edema', 'Pleural Effusion'],
        'Pneumonia': ['Lung Opacity', 'Consolidation', 'Atelectasis', 'Edema', 'Pleural Effusion'],
        'Pleural Effusion': ['Lung Opacity', 'Edema', 'Atelectasis', 'Consolidation', 'Cardiomegaly'],
        'Enlarged Cardiomediastinum': ['Cardiomegaly', 'Edema', 'Pleural Effusion', 'Lung Opacity', 'Atelectasis']
    }
    
    for label in TARGET_LABELS:
        if label in default_features:
            top_features_per_label[label] = default_features[label]
            print(f"Using default features for {label}: {default_features[label]}")

# Display Top Features

In [ ]:
# Cell 5.6: Display Top Features for Each Label
print("\n📊 Top 5 Features for Each Target Label:")
print("="*60)
for label, features in top_features_per_label.items():
    print(f"\n{label}:")
    for i, feat in enumerate(features, 1):
        corr_value = all_correlations[label][all_correlations[label]['feature'] == feat]['direction'].values[0]
        print(f"  {i}. {feat} (corr: {corr_value:.3f})")

# Function to Get Feature Values for Test Sample

In [ ]:
# REPLACE get_sample_features with this clean version

def get_sample_features(study_path, label, top_features, rule_labeler_df):
    feature_values = []
    if not top_features:
        return feature_values
    try:
        # rule_labeler_df is already the 500 test samples
        # just get the row by matching study path directly
        path_col = "Study" if "Study" in rule_labeler_df.columns else "Path"
        mask = rule_labeler_df[path_col] == study_path
        matched = rule_labeler_df[mask]
        
        if len(matched) == 0:
            return feature_values
            
        labeler_row = matched.iloc[0]  # pandas Series
        
        for feat in top_features:
            if feat in labeler_row.keys():   # .keys() not .index on Series
                val = labeler_row[feat]
                if pd.notna(val):
                    # Skip uncertain (-1) context features — they add noise.
                    # Only pass through clear 0/1 values.
                    if float(val) == -1:
                        continue
                    feature_values.append(f"{feat}={val}")
    except Exception as e:
        pass  # return empty list silently
    return feature_values

## 6. Extract Uncertain Subsets

In [ ]:

def extract_uncertain_cases(rule_df, gt_df, label):
    subset = rule_df[rule_df[label] == -1].copy()
    subset["gt"] = gt_df.loc[subset.index, label]
    subset = subset.dropna(subset=["gt"])
    return subset

benchmark_sets = {
    label: extract_uncertain_cases(rule_labeler_df, ground_truth_df, label)
    for label in TARGET_LABELS
}

for k,v in benchmark_sets.items():
    print(k, v.shape)


# Load WhyXrayCLIP

In [ ]:
# ================================================================
# CELL 6: Load WhyXrayCLIP
# - Trained on 243,334 MIMIC-CXR chest X-rays + GPT-4 cleaned reports
# - NOT trained on CheXpert — no data leakage
# - Outperforms PubMedCLIP and BioMedCLIP on zero-shot chest X-ray tasks
# - Uses open_clip (not transformers CLIPModel)
# ================================================================
import open_clip
import torch
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading WhyXrayCLIP (MIMIC-CXR trained)...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "hf-hub:yyupenn/whyxrayclip"
)
clip_model = clip_model.to(device)
clip_model.eval()

clip_tokenizer = open_clip.get_tokenizer("ViT-L-14")
print(f"✅ WhyXrayCLIP loaded on {device}")

## 9. Inference Functions

In [ ]:
# ================================================================
# CELL 7: Clinical radiology prompts
# These match the language style WhyXrayCLIP was trained on
# (MIMIC-CXR radiology reports use specific clinical terminology)
# ================================================================

POSITIVE_PROMPTS = {
    "Edema":
        "pulmonary edema with bilateral interstitial opacities and vascular congestion",
    "Atelectasis":
        "bibasilar atelectasis with opacification and airspace disease in bilateral lower lobes",
    "Pleural Effusion":
        "moderate to large pleural effusion with blunting of costophrenic angle and layering fluid",
    "Enlarged Cardiomediastinum":
        "enlarged cardiac silhouette with widened mediastinum",
    "Consolidation": (
        "dense lobar airspace consolidation with complete opacification "
        "of lung segment, air bronchograms and silhouette sign"
    ),
}

NEGATIVE_PROMPTS = {
    "Edema":
        "no pulmonary edema with clear lung fields and normal vascularity",
    "Atelectasis":
        "no atelectasis or subsegmental opacities, clear lower lobes with normal diaphragm position",
    "Pleural Effusion":
        "no pleural effusion, sharp costophrenic angles bilaterally, no layering fluid",
    "Enlarged Cardiomediastinum":
        "normal cardiac and mediastinal contours",
    "Consolidation": (
        "normal chest radiograph with clear lung fields, visible pulmonary "
        "vascular markings throughout, sharp costophrenic angles, "
        "no opacity no infiltrate no consolidation"
    ),
}



In [ ]:
# ================================================================
# Shared Feature-Context Infrastructure
#
# WHAT IS SHARED:
#   FEATURE_TO_RADIOLOGY_LANGUAGE — sentence templates for each finding
#   RELEVANT_NEGATIVES            — clinically contrastive negatives
#   _build_old_format_context()   — "X is present, Y is absent" builder
#   _build_radiology_context()    — natural radiology-report sentence builder
#   build_context()               — generic dispatcher (all models use this)
#
# WHAT IS MODEL-SPECIFIC (defined in each model's own section):
#   WXRC_CONTEXT_STRATEGY  — defined below (WhyXrayCLIP, defaults only)
#   CZERO_CONTEXT_STRATEGY — defined in CheXZero section
#   LABEL_CONTEXT_STRATEGY — defined in BioViL-T section
#
# EXECUTION ORDER (WhyXrayCLIP):
#   1. This cell  — builders + WXRC defaults
#   2. infer_clip fn definition
#   3. WhyXrayCLIP ablation → updates WXRC_CONTEXT_STRATEGY with winner
#   4. Threshold calibration RUN  → uses winning strategy (no mismatch)
#   5. Benchmark
# ================================================================

FEATURE_TO_RADIOLOGY_LANGUAGE = {
    "Enlarged Cardiomediastinum": {"positive":"The cardiomediastinal silhouette is enlarged.","negative":"The cardiomediastinal silhouette is within normal limits.","uncertain":"The cardiomediastinal silhouette may be mildly prominent."},
    "Cardiomegaly": {"positive":"Cardiomegaly is present with an increased cardiothoracic ratio.","negative":"The cardiac size is normal.","uncertain":"The cardiac size is borderline."},
    "Lung Opacity": {"positive":"There is lung opacity present.","negative":"The lung fields are clear without opacity.","uncertain":"There is possible subtle lung opacity."},
    "Lung Lesion": {"positive":"A lung lesion is identified.","negative":"No lung lesion is identified.","uncertain":"A possible lung lesion cannot be excluded."},
    "Edema": {"positive":"Pulmonary edema is present with perihilar hazy infiltrates.","negative":"No pulmonary edema is identified.","uncertain":"Mild pulmonary edema cannot be excluded."},
    "Consolidation": {"positive":"There is airspace consolidation present.","negative":"No consolidation is identified.","uncertain":"Possible early consolidation cannot be excluded."},
    "Atelectasis": {"positive":"Atelectasis is present at the lung base.","negative":"No atelectasis is identified.","uncertain":"Subsegmental atelectasis cannot be excluded."},
    "Pneumothorax": {"positive":"Pneumothorax is present.","negative":"No pneumothorax is identified.","uncertain":"A small pneumothorax cannot be excluded."},
    "Pleural Effusion": {"positive":"There is pleural effusion with blunting of the costophrenic angle.","negative":"No pleural effusion is identified. The costophrenic angles are sharp.","uncertain":"A small pleural effusion cannot be excluded."},
    "Pleural Other": {"positive":"There is pleural thickening or other pleural abnormality.","negative":"No pleural abnormality is identified.","uncertain":"Pleural abnormality cannot be excluded."},
    "Fracture": {"positive":"A rib or bony fracture is identified.","negative":"No acute fracture is identified.","uncertain":"A fracture cannot be excluded."},
    "Support Devices": {"positive":"Support devices are present including lines and tubes.","negative":"No support devices are identified.","uncertain":"Support device positioning is difficult to assess."},
    "No Finding": {"positive":"No acute cardiopulmonary finding is identified.","negative":"Findings are present on this examination.","uncertain":"Findings are equivocal on this examination."},
}

RELEVANT_NEGATIVES = {
    "Pleural Effusion":           ["Consolidation", "Lung Opacity"],
    "Atelectasis":                ["Pleural Effusion", "Consolidation"],
    "Edema":                      ["Consolidation", "Pleural Effusion"],
    "Enlarged Cardiomediastinum": ["Cardiomegaly", "Pleural Effusion"],
    "Consolidation":              ["Atelectasis", "Pleural Effusion"],
}

# WhyXrayCLIP context strategy — safe defaults, overwritten by ablation cell.
# DO NOT edit manually. Run the WhyXrayCLIP ablation cell to update.
WXRC_CONTEXT_STRATEGY = {
    "Edema":                      "radiology_format",
    "Atelectasis":                "old_format",
    "Pleural Effusion":           "old_format",
    "Enlarged Cardiomediastinum": "no_context",
    "Consolidation":              "no_context",
}


def _build_old_format_context(feature_values: list, target_label: str) -> str:
    parts = []
    for fv in feature_values:
        try:
            feat, val = fv.split("="); val = float(val); feat = feat.strip()
            if feat.lower() == target_label.lower() or val == -1 or (0 < val < 1):
                continue
            parts.append(f"{feat} is present" if val >= 1.0 else f"{feat} is absent")
        except Exception:
            continue
    return ", ".join(parts) + ". " if parts else ""


def _build_radiology_context(feature_values: list, target_label: str, cap: int = 2) -> str:
    pos_parts, neg_parts, unc_parts = [], [], []
    relevant_neg = RELEVANT_NEGATIVES.get(target_label, [])
    for fv in feature_values:
        try:
            feat, val = fv.split("="); val = float(val); feat = feat.strip()
            if feat.lower() == target_label.lower() or (0 < val < 1):
                continue
            lang = FEATURE_TO_RADIOLOGY_LANGUAGE.get(feat)
            if lang is None:
                continue
            if val >= 1.0:
                pos_parts.append(lang["positive"])
            elif val == 0.0 and feat in relevant_neg:
                neg_parts.append(lang["negative"])
            elif val == -1 and feat in relevant_neg:
                unc_parts.append(lang["uncertain"])
        except Exception:
            continue
    parts = pos_parts + neg_parts + unc_parts
    return (" ".join(parts[:cap]) + " ") if parts else ""


def build_context(label: str, feature_values: list, strategy_map: dict) -> str:
    """Generic dispatcher. Pass model-specific strategy map:
      WhyXrayCLIP → WXRC_CONTEXT_STRATEGY
      CheXZero    → CZERO_CONTEXT_STRATEGY  (CheXZero section)
      BioViL-T    → LABEL_CONTEXT_STRATEGY  (BioViL-T section)
    """
    if not feature_values:
        return ""
    strategy = strategy_map.get(label, "no_context")
    if strategy == "old_format":
        return _build_old_format_context(feature_values, label)
    if strategy == "radiology_format":
        return _build_radiology_context(feature_values, label)
    return ""


def build_wxrc_context(label: str, feature_values: list) -> str:
    """WhyXrayCLIP context — uses WXRC_CONTEXT_STRATEGY."""
    return build_context(label, feature_values, WXRC_CONTEXT_STRATEGY)


print("✅ Shared feature-context infrastructure loaded.")
print(f"   WXRC defaults: {WXRC_CONTEXT_STRATEGY}")
print("   Run WhyXrayCLIP ablation next to pick winning strategy per label.")


In [ ]:
# ================================================================
# CELL 8: infer function — updated for open_clip API
# Prompts stay exactly the same as before
# ================================================================

def infer_clip(img_path: str, label: str, feature_values: list = None) -> int:
    try:
        # Build feature context
        feature_context = build_wxrc_context(label, feature_values or [])
        pos_text = feature_context + POSITIVE_PROMPTS[label]
        neg_text = feature_context + NEGATIVE_PROMPTS[label]

        # open_clip image preprocessing (different from CLIPProcessor)
        image  = clip_preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)

        # open_clip text tokenization
        texts  = clip_tokenizer([pos_text, neg_text]).to(device)

        with torch.no_grad():
            image_features = clip_model.encode_image(image)
            text_features  = clip_model.encode_text(texts)

            # Normalize
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            text_features  = text_features  / text_features.norm(dim=-1, keepdim=True)

            # Cosine similarity: [1, 2]
            sims = (image_features @ text_features.T).squeeze(0)

        # delta: same score space as calibration and ablation
        delta     = sims[0].item() - sims[1].item()
        threshold = THRESHOLDS.get(label, 0.0)
        return 1 if delta > threshold else 0

    except Exception as e:
        print(f"  [infer_clip] Error — {label}: {e}")
        return 0



# WhyXrayCLIP Threshold Calibration

> Runs **before** the ablation. Calibrates `THRESHOLDS` using the current
> `WXRC_CONTEXT_STRATEGY` defaults (delta score = pos_sim − neg_sim,
> same space as the ablation and `infer_clip`). After the ablation updates
> `WXRC_CONTEXT_STRATEGY` with the winning strategy, the
> **Post-Ablation Recalibration** cell re-runs this to align thresholds
> exactly with inference. This two-cell pattern is identical to BioViLT.

In [ ]:
# ================================================================
# WhyXrayCLIP Threshold Calibration
#
# Score space: delta = sims[0] - sims[1]  (pos_sim − neg_sim)
#   ← identical to what the ablation cell scores
#   ← identical to what infer_clip() decides on
#   All three (calibration / ablation / inference) use the same
#   score space. No mismatch.
#
# Strategy: identical half-split cross-validated approach as BioViLT
#   (calibrate_biovilt_threshold) and CheXZero for full consistency.
#
# Steps:
#   1. Sample balanced pos/neg from train_df certain (0/1) labels
#   2. Compute delta scores using current WXRC_CONTEXT_STRATEGY
#   3. Half-split CV: find candidates on first half, evaluate on second
#   4. Return candidate with highest bal_acc on held-out half
# ================================================================
from sklearn.metrics import roc_curve, balanced_accuracy_score

def calibrate_threshold(label, n_samples=1000):
    """
    Calibrate WhyXrayCLIP decision threshold on train_df certain samples.
    Score = sims[0] - sims[1]  (delta, same space as ablation + infer_clip).
    Identical half-split CV strategy as BioViLT / CheXZero.
    """
    df = train_df[train_df[label].isin([0, 1])].copy()
    n_each = min(n_samples // 2, (df[label]==1).sum(), (df[label]==0).sum())
    if n_each < 10:
        print(f"  ⚠️  Not enough images for {label}, using 0.0")
        return 0.0

    pos    = df[df[label]==1].sample(n_each, random_state=42).reset_index(drop=True)
    neg    = df[df[label]==0].sample(n_each, random_state=42).reset_index(drop=True)
    df_cal = pd.concat([pos, neg], ignore_index=True).sample(
        frac=1, random_state=42
    ).reset_index(drop=True)

    print(f"  {label}: calibrating on {len(df_cal)} samples (pos={n_each}, neg={n_each})")

    scores, true_labels = [], []

    for _, row in tqdm(df_cal.iterrows(), total=len(df_cal),
                       desc=f"  Calibrating {label}", leave=False):
        try:
            clean    = row["Path"].replace("CheXpert-v1.0-small/", "").replace("CheXpert-v1.0/", "")
            img_path = os.path.join(BASE_PATH, clean)

            image = clip_preprocess(
                Image.open(img_path).convert("RGB")
            ).unsqueeze(0).to(device)

            fv = (get_sample_features(
                row["Path"], label, top_features_per_label.get(label, []), train_df
            ) if "top_features_per_label" in globals() else [])
            ctx      = build_wxrc_context(label, fv)
            pos_text = ctx + POSITIVE_PROMPTS[label]
            neg_text = ctx + NEGATIVE_PROMPTS[label]

            texts = clip_tokenizer([pos_text, neg_text]).to(device)

            with torch.no_grad():
                img_f = clip_model.encode_image(image)
                txt_f = clip_model.encode_text(texts)
                img_f = img_f / img_f.norm(dim=-1, keepdim=True)
                txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
                sims  = (img_f @ txt_f.T).squeeze(0)

            # ── delta: same score space as ablation and infer_clip ──
            scores.append(sims[0].item() - sims[1].item())
            true_labels.append(int(row[label]))

        except Exception:
            continue

    if len(scores) < 40:
        print(f"  ⚠️  Too few valid samples, using 0.0")
        return 0.0

    scores_arr = np.array(scores)
    labels_arr = np.array(true_labels)

    print(f"  Score stats: min={scores_arr.min():.4f}, max={scores_arr.max():.4f}, "
          f"mean={scores_arr.mean():.4f}")
    print(f"  Score for positives : mean={scores_arr[labels_arr==1].mean():.4f}")
    print(f"  Score for negatives : mean={scores_arr[labels_arr==0].mean():.4f}")
    gap = scores_arr[labels_arr==1].mean() - scores_arr[labels_arr==0].mean()
    print(f"  Gap                 : {gap:.4f}  {'✅ good' if gap > 0.01 else '⚠️ small'}")

    # ── Half-split CV (identical to BioViLT / CheXZero) ──────────
    idx          = np.random.RandomState(42).permutation(len(scores_arr))
    mid          = len(idx) // 2
    s_tr, l_tr   = scores_arr[idx[:mid]], labels_arr[idx[:mid]]
    s_val, l_val = scores_arr[idx[mid:]], labels_arr[idx[mid:]]

    fpr, tpr, thresholds_roc = roc_curve(l_tr, s_tr)
    bal_acc    = (tpr + (1 - fpr)) / 2
    thresh_bal = float(thresholds_roc[np.argmax(bal_acc)])
    thresh_mid = float((s_tr[l_tr==1].mean() + s_tr[l_tr==0].mean()) / 2)
    thresh_med = float(np.median(s_tr[l_tr==1]))

    candidates = {k: v for k, v in {
        "balanced_acc": thresh_bal,
        "midpoint":     thresh_mid,
        "median_pos":   thresh_med,
    }.items() if not np.isinf(v) and not np.isnan(v)}

    print(f"  Threshold candidates (evaluated on held-out half):")
    best_name, best_thresh, best_score = None, None, -1
    for name, thresh in candidates.items():
        preds = (s_val > thresh).astype(int)
        score = balanced_accuracy_score(l_val, preds)
        print(f"    {name:15s}: {thresh:.4f} → val bal_acc={score:.3f}")
        if score > best_score:
            best_score, best_thresh, best_name = score, thresh, name

    print(f"  ✅ Chosen: {best_name} = {best_thresh:.4f} "
          f"(val balanced acc = {best_score:.3f})")
    return best_thresh


print("🔧 Calibrating WhyXrayCLIP thresholds from train_df...")
print(f"   Strategy: {WXRC_CONTEXT_STRATEGY}")
print("=" * 60)
THRESHOLDS = {}
for label in TARGET_LABELS:
    THRESHOLDS[label] = calibrate_threshold(label, n_samples=1000)
print("\n📊 WhyXrayCLIP Thresholds:")
for label, thresh in THRESHOLDS.items():
    print(f"  {label}: {thresh:.4f}")
print("\n✅ Calibration done. Run the ablation cell next.")
print("   After ablation updates WXRC_CONTEXT_STRATEGY, run the")
print("   Post-Ablation Recalibration cell to re-align thresholds")
print("   with the winning strategy before benchmarking.")


# Context Format Ablation — WhyXrayCLIP

> **Run after threshold calibration.** `THRESHOLDS` must exist.
> After this cell updates `WXRC_CONTEXT_STRATEGY`, run the
> **Post-Ablation Recalibration** cell.

In [ ]:
# ================================================================
# Context Format Ablation — WhyXrayCLIP (Zero-Shot)
#
# PURPOSE:
#   For each target label, compare 3 context strategies:
#     no_context        : bare prompt only
#     old_format        : "X is present, Y is absent."
#     radiology_format  : "X is present with... No Y is identified."
#
# SELECTION METRIC:
#   Composite score = 0.5 × AUC + 0.5 × Balanced Accuracy.
#   AUC alone is unreliable for severely imbalanced labels (e.g.
#   Consolidation: 3 pos / 60 neg). Balanced accuracy penalises
#   strategies that win AUC by predicting everything as negative.
#   The composite gives a robust signal across both balanced and
#   imbalanced label distributions in our GT-uncertain eval set.
#
# PREREQUISITE: Run the WhyXrayCLIP Threshold Calibration cell
#   (immediately before this one) to populate THRESHOLDS.
#   Score space: delta = sims[0] - sims[1], same as calibration
#   and infer_clip — no mismatch.
#
# DESIGN:
#   Image embedding computed once per sample and reused across
#   strategies. Balanced accuracy uses THRESHOLDS[label] (train_df
#   calibrated, no leakage). AUC is threshold-free.
#   Winner per label updates WXRC_CONTEXT_STRATEGY.
# ================================================================
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

STRATEGIES_TO_TEST = ["no_context", "old_format", "radiology_format"]
_WXRC_ABL_RESULTS  = {}   # label → {strategy: {"auc", "bal_acc", "composite"}}

def _run_wxrc_context_ablation():
    """
    Runs context format ablation for WhyXrayCLIP.
    Returns dict: {label: {strategy: {auc, bal_acc, composite}}}
    """
    encode_img  = lambda img: clip_model.encode_image(img)
    tokenize    = lambda texts: clip_tokenizer(texts).to(device)
    encode_txt  = lambda tok:  clip_model.encode_text(tok)
    preprocess  = clip_preprocess
    pos_prompts = POSITIVE_PROMPTS
    neg_prompts = NEGATIVE_PROMPTS

    results = {label: {} for label in TARGET_LABELS}

    labeler_df = rule_labeler_df.copy()
    gt_df      = ground_truth_df.copy()
    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True)
                break

    for label in TARGET_LABELS:
        n_uncertain = int((rule_labeler_df.get(label, pd.Series([])) == -1).sum()
                          if label in rule_labeler_df.columns else 0)
        uncertain_idx    = labeler_df[labeler_df[label] == -1].index
        gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
        gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
        gt_for_uncertain = (gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]
                            .reset_index(drop=True))

        n_pos = int(gt_for_uncertain["gt"].sum())
        n_neg = int((gt_for_uncertain["gt"] == 0).sum())
        balance = n_pos / (n_pos + n_neg) if (n_pos + n_neg) > 0 else 0
        print(f"\n  [WXRC] {label}  "
              f"(n={len(gt_for_uncertain)}, pos={n_pos}, neg={n_neg}, "
              f"balance={balance:.2f})")

        if len(gt_for_uncertain) < 5:
            print(f"    ⚠️  Too few samples — skipping")
            continue

        thresh = THRESHOLDS.get(label, 0.0)
        print(f"    Calibrated threshold: {thresh:.4f}")
        top_feats = top_features_per_label.get(label, [])

        sample_data = []
        for _, row in tqdm(gt_for_uncertain.iterrows(),
                           total=len(gt_for_uncertain),
                           desc=f"    Embedding [{label}]", leave=False):
            try:
                img_path = fix_test_path(row["Study"])
                img = preprocess(
                    Image.open(img_path).convert("RGB")
                ).unsqueeze(0).to(device)
                fv = get_sample_features(row["Study"], label, top_feats, rule_labeler_df)
                with torch.no_grad():
                    img_f = encode_img(img)
                    img_f = img_f / img_f.norm(dim=-1, keepdim=True)
                sample_data.append({"y_true": int(row["gt"]), "img_f": img_f, "fv": fv})
            except Exception:
                continue

        if len(sample_data) < 5:
            continue

        y_true = np.array([sd["y_true"] for sd in sample_data])

        print(f"    {'Strategy':<22} {'AUC':>7} {'Bal-Acc':>9} {'Composite':>10}")
        print(f"    {'─'*52}")

        for strategy in STRATEGIES_TO_TEST:
            scores = []
            for sd in sample_data:
                try:
                    if strategy == "no_context":
                        ctx = ""
                    elif strategy == "old_format":
                        ctx = _build_old_format_context(sd["fv"], label)
                    else:
                        ctx = _build_radiology_context(sd["fv"], label)

                    pos_text = ctx + pos_prompts[label]
                    neg_text = ctx + neg_prompts[label]

                    with torch.no_grad():
                        tok   = tokenize([pos_text, neg_text])
                        txt_f = encode_txt(tok)
                        txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
                        sims  = (sd["img_f"] @ txt_f.T).squeeze(0)
                    scores.append(sims[0].item() - sims[1].item())  # delta — matches calibration + infer_clip
                except Exception:
                    scores.append(0.0)

            scores_arr = np.array(scores)
            try:
                auc = roc_auc_score(y_true, scores_arr) if len(np.unique(y_true)) > 1 else 0.5
            except Exception:
                auc = 0.5

            preds    = (scores_arr > thresh).astype(int)
            bal_acc  = balanced_accuracy_score(y_true, preds)
            composite = 0.5 * auc + 0.5 * bal_acc

            results[label][strategy] = {
                "auc":       round(auc, 3),
                "bal_acc":   round(bal_acc, 3),
                "composite": round(composite, 3),
            }
            print(f"    {strategy:<22} {auc:>7.3f} {bal_acc:>9.3f} {composite:>10.3f}")

        # Pick winner by composite score
        if results[label]:
            winner    = max(results[label], key=lambda s: results[label][s]["composite"])
            best      = results[label][winner]
            baseline  = results[label].get("no_context", {}).get("composite", 0.0)
            delta     = best["composite"] - baseline
            sign      = "✅" if delta > 0.01 else ("🔁" if delta > -0.01 else "❌")
            print(f"\n    Winner: {winner}  "
                  f"(composite={best['composite']:.3f}, "
                  f"AUC={best['auc']:.3f}, "
                  f"Bal-Acc={best['bal_acc']:.3f}, "
                  f"Δ composite vs no_context={delta:+.3f}) {sign}")

    return results


print("="*70)
print("  CONTEXT FORMAT ABLATION — WhyXrayCLIP (Zero-Shot)")
print("="*70)
_WXRC_ABL_RESULTS = _run_wxrc_context_ablation()

# ── Auto-update WXRC_CONTEXT_STRATEGY ────────────────────────────
print("\n" + "="*70)
print("  UPDATING WXRC_CONTEXT_STRATEGY FROM ABLATION RESULTS")
print("="*70)
for label in TARGET_LABELS:
    if _WXRC_ABL_RESULTS.get(label):
        winner = max(_WXRC_ABL_RESULTS[label],
                     key=lambda s: _WXRC_ABL_RESULTS[label][s]["composite"])
        old    = WXRC_CONTEXT_STRATEGY.get(label, "not set")
        WXRC_CONTEXT_STRATEGY[label] = winner
        marker = "✅ (unchanged)" if old == winner else f"🔄 updated: {old} → {winner}"
        best   = _WXRC_ABL_RESULTS[label][winner]
        print(f"  [{label}]: {winner}  "
              f"(composite={best['composite']:.3f}, "
              f"AUC={best['auc']:.3f}, "
              f"Bal-Acc={best['bal_acc']:.3f})  {marker}")

print(f"\n✅ WXRC_CONTEXT_STRATEGY = {WXRC_CONTEXT_STRATEGY}")
print("\n✅ WXRC_CONTEXT_STRATEGY updated. Calibration runs next with this strategy.")


# WhyXrayCLIP Post-Ablation Recalibration

> Runs **after** ablation. `WXRC_CONTEXT_STRATEGY` now holds the winning
> strategy. Re-runs `calibrate_threshold()` so `THRESHOLDS` is calibrated
> with the exact same context used at inference — no train/test mismatch.
> This is the same pattern BioViLT uses (cell 97 re-runs
> `calibrate_biovilt_threshold` after ablation).

In [ ]:
# ================================================================
# WhyXrayCLIP Post-Ablation Recalibration
#
# Runs AFTER ablation — WXRC_CONTEXT_STRATEGY holds the winner.
# calibrate_threshold() reads WXRC_CONTEXT_STRATEGY via
# build_wxrc_context() — thresholds calibrated with the same
# context used at inference. Score space is delta throughout.
# ================================================================
print("🔧 Recalibrating WhyXrayCLIP thresholds (post-ablation, winning strategy)...")
print(f"   Strategy: {WXRC_CONTEXT_STRATEGY}")
print("=" * 60)
THRESHOLDS = {}
for label in TARGET_LABELS:
    THRESHOLDS[label] = calibrate_threshold(label, n_samples=1000)
print("\n📊 WhyXrayCLIP Final Thresholds (calibrated with winning strategy):")
for label, thresh in THRESHOLDS.items():
    print(f"  {label}: {thresh:.4f}")
print(f"\n  Strategy used: {WXRC_CONTEXT_STRATEGY}")


## 10. Benchmark Loop

In [ ]:
def benchmark_uncertain_relabeling():
    all_results = []

    labeler_df = rule_labeler_df.copy()
    gt_df      = ground_truth_df.copy()

    # Normalise path column to "Study" in both
    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True)
                break

    for label in TARGET_LABELS:
        print(f"\n{'='*60}")
        print(f"  Label: {label}")
        print(f"{'='*60}")

        if label not in labeler_df.columns or label not in gt_df.columns:
            print(f"  ⚠️  '{label}' column missing — skipping")
            continue

        # Step 1: from the 500 test samples, get only uncertain rows
        uncertain_idx = labeler_df[labeler_df[label] == -1].index
        if len(uncertain_idx) == 0:
            print(f"  ℹ️  No uncertain samples — skipping")
            continue
        print(f"  Uncertain samples: {len(uncertain_idx)} out of {len(labeler_df)}")

        # Step 2: get ground truth for THOSE SAME rows by index
        # Both dataframes are the same 500 test samples — same index
        gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
        gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})

        # Keep only rows where ground truth is 0 or 1 (drop any NaN)
        gt_for_uncertain = gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]
        print(f"  With valid ground truth (0/1): {len(gt_for_uncertain)}  "
              f"(pos={gt_for_uncertain['gt'].sum()}, "
              f"neg={(gt_for_uncertain['gt']==0).sum()})")

        if len(gt_for_uncertain) == 0:
            print(f"  ⚠️  No valid ground truth for uncertain samples — skipping")
            continue

        top_features = top_features_per_label.get(label, [])

        for idx, row in tqdm(gt_for_uncertain.iterrows(),
                             total=len(gt_for_uncertain),
                             desc=f"rad-dino · {label}"):
            try:
                img_path       = fix_test_path(row["Study"])
                feature_values = get_sample_features(
                    row["Study"], label, top_features, rule_labeler_df
                )
                pred = infer_clip(img_path, label, feature_values)

                all_results.append({
                    "label":           label,
                    "study":           row["Study"],
                    "y_true":          int(row["gt"]),
                    "y_pred":          pred,
                    "feature_context": " | ".join(feature_values),
                })
            except Exception as e:
                print(f"  Error: {e}")
                all_results.append({
                    "label":           label,
                    "study":           row["Study"],
                    "y_true":          int(row["gt"]) if pd.notna(row["gt"]) else -1,
                    "y_pred":          0,
                    "feature_context": f"ERROR: {e}",
                })

    return pd.DataFrame(all_results)



## 11. Metrics

In [ ]:
# Cell 11.5: Enhanced Metrics with Confusion Matrix Details
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, average_precision_score

def evaluate_results_with_details(df):
    """
    Enhanced evaluation with detailed confusion matrix information.
    NOTE: AUC and AUPRC use y_pred (binary) as scores here because this
    function is used for zero-shot results where only binary predictions
    are stored. For few-shot evaluation with continuous scores, use the
    model-specific evaluate_* functions that receive score arrays.
    """
    metrics = []
    
    for label in TARGET_LABELS:
        sub = df[df["label"] == label]
        
        if len(sub) == 0:
            continue
            
        y_true = sub.y_true.values
        y_pred = sub.y_pred.values
        
        # Use continuous score if available (few-shot results), else binary pred
        score_col = None
        for col in ["combined_score", "score", "text_score"]:
            if col in sub.columns and sub[col].notna().all():
                score_col = col
                break
        scores = sub[score_col].values if score_col else y_pred.astype(float)
        
        # Calculate confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
        else:
            if len(np.unique(y_true)) == 1:
                if y_true[0] == 0:
                    tn, fp, fn, tp = len(y_true), 0, 0, 0
                else:
                    tn, fp, fn, tp = 0, 0, len(y_true), 0
            else:
                tn, fp, fn, tp = 0, 0, 0, 0
        
        acc = accuracy_score(y_true, y_pred)
        
        try:
            f1 = f1_score(y_true, y_pred, zero_division=0)
        except:
            f1 = 0
            
        try:
            auc = roc_auc_score(y_true, scores) if len(np.unique(y_true)) > 1 else 0.5
        except:
            auc = 0.5

        try:
            auprc = average_precision_score(y_true, scores) if len(np.unique(y_true)) > 1 else float(y_true.mean())
        except:
            auprc = float(y_true.mean())
        
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal_acc = (sens + spec) / 2
        
        metrics.append({
            "label": label,
            "n_samples": len(sub),
            "n_true_0": np.sum(y_true == 0),
            "n_true_1": np.sum(y_true == 1),
            "accuracy": acc,
            "f1_score": f1,
            "auc": auc,
            "auprc": round(float(auprc), 3),
            "bal_acc": bal_acc,
            "sens": sens,
            "spec": spec,
            "true_negative": tn,
            "false_positive": fp,
            "false_negative": fn,
            "true_positive": tp,
            "precision_0": tn / (tn + fn) if (tn + fn) > 0 else 0,
            "recall_0": tn / (tn + fp) if (tn + fp) > 0 else 0,
            "precision_1": tp / (tp + fp) if (tp + fp) > 0 else 0,
            "recall_1": tp / (tp + fn) if (tp + fn) > 0 else 0
        })
    
    return pd.DataFrame(metrics)

def print_detailed_results(metrics_df):
    """
    Print detailed results in a readable format
    """
    print("\n" + "="*80)
    print("📊 DETAILED BENCHMARK RESULTS")
    print("="*80)
    
    for _, row in metrics_df.iterrows():
        print(f"\n🔬 {row['label']} (n={int(row['n_samples'])})")
        print(f"   Ground Truth Distribution: 0={int(row['n_true_0'])}, 1={int(row['n_true_1'])}")
        print("-" * 50)
        print("   Performance Metrics:")
        print(f"   • Accuracy:  {row['accuracy']:.3f}")
        print(f"   • F1-Score:  {row['f1_score']:.3f}")
        print(f"   • AUC:       {row['auc']:.3f}")
        auprc_val = row.get('auprc', None)
        if auprc_val is not None:
            print(f"   • AUPRC:     {auprc_val:.3f}")
        print(f"   • Bal. Acc:  {row['bal_acc']:.3f}")
        print(f"   • Sens:      {row['sens']:.3f}")
        print(f"   • Spec:      {row['spec']:.3f}")
        print("\n   Confusion Matrix:")
        print(f"   • True Negatives  (GT=0, Pred=0): {int(row['true_negative'])}")
        print(f"   • False Positives (GT=0, Pred=1): {int(row['false_positive'])}")
        print(f"   • False Negatives (GT=1, Pred=0): {int(row['false_negative'])}")
        print(f"   • True Positives  (GT=1, Pred=1): {int(row['true_positive'])}")
        print("\n   Per-Class Metrics:")
        print(f"   • Class 0 (Negative): Precision={row['precision_0']:.3f}, Recall={row['recall_0']:.3f}")
        print(f"   • Class 1 (Positive): Precision={row['precision_1']:.3f}, Recall={row['recall_1']:.3f}")
        print("=" * 50)


## 12. Run Full Benchmark

In [ ]:
print("🚀 Running uncertain label relabeling with rad-dino + feature context...")
print("="*80)

uncertain_results = benchmark_uncertain_relabeling()

# Save raw predictions
uncertain_results.to_csv("rad_dino_uncertain_relabeling_results.csv", index=False)
print("\n✅ Raw results saved to 'rad_dino_uncertain_relabeling_results.csv'")

# Evaluate
uncertain_metrics = evaluate_results_with_details(uncertain_results)
print_detailed_results(uncertain_metrics)

# Summary
print("\n" + "="*80)
print("📈 SUMMARY")
print("="*80)
if len(uncertain_metrics) > 0:
    summary = uncertain_metrics[["label", "n_samples", "accuracy", "f1_score", "auc",
                                  "true_negative", "false_positive",
                                  "false_negative", "true_positive"]]
    print(summary.to_string(index=False))

    print("\nBalanced Accuracy per label:")
    for _, row in uncertain_metrics.iterrows():
        sens = row["true_positive"] / (row["true_positive"] + row["false_negative"]) \
               if (row["true_positive"] + row["false_negative"]) > 0 else 0
        spec = row["true_negative"] / (row["true_negative"] + row["false_positive"]) \
               if (row["true_negative"] + row["false_positive"]) > 0 else 0
        bal  = (sens + spec) / 2
        print(f"  {row['label']}: Sens={sens:.3f}, Spec={spec:.3f}, Balanced Acc={bal:.3f}")
else:
    print("No results to summarize — check uncertain sample counts in Cell 10.5")


In [ ]:
# # ================================================================
# # BENCHMARK WhyXrayCLIP on 3 New Labels — GT Evaluation Only
# # No relabeling. Just adding prompts + running existing benchmark.
# # ================================================================

# NEW_TARGET_LABELS = ["Consolidation"]

# # ── Consolidation ─────────────────────────────────────────────────
# # Key: emphasize DENSE HOMOGENEOUS lobar pattern + air bronchograms
# # Avoid: anything that sounds like infection (that's Pneumonia)
# POSITIVE_PROMPTS["Consolidation"] = (
#     "dense lobar airspace consolidation with complete opacification "
#     "of lung segment, air bronchograms and silhouette sign"
# )

# NEGATIVE_PROMPTS["Consolidation"] = (
#     "normal chest radiograph with clear lung fields, visible pulmonary "
#     "vascular markings throughout, sharp costophrenic angles, "
#     "no opacity no infiltrate no consolidation"
# )

# # ── Calibrate thresholds using existing function ─────────────────
# print("Calibrating thresholds for new labels...")

# THRESHOLDS = {}
# for label in NEW_TARGET_LABELS:
#     print(f"\n  [{label}]")
#     THRESHOLDS[label] = calibrate_threshold(label, n_samples=1000)

# print("\n✅ Thresholds:")
# for label in NEW_TARGET_LABELS:
#     print(f"  {label:<20}: {THRESHOLDS[label]:.4f}")

# # ── Temporarily extend TARGET_LABELS and run existing benchmark ─

# results_new = benchmark_uncertain_relabeling()

# print(results_new)

# new_results = results_new[results_new["label"].isin(["Consolidation"])]

# for label in ["Consolidation"]:
#     subset = new_results[new_results["label"] == label]
#     y_true = subset["y_true"].values
#     y_pred = subset["y_pred"].values

#     print(f"\n  [{label}]")
#     print(f"  Samples : {len(subset)}  (pos={int(y_true.sum())}, neg={int((y_true==0).sum())})")

#     if len(np.unique(y_true)) < 2:
#         print(f"  ⚠️  Only one class in GT — AUC not computable")
#         print(f"  Accuracy : {accuracy_score(y_true, y_pred):.4f}")
#         continue

#     tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
#     print(f"  AUC          : {roc_auc_score(y_true, y_pred):.4f}")
#     print(f"  Balanced Acc : {balanced_accuracy_score(y_true, y_pred):.4f}")
#     print(f"  F1           : {f1_score(y_true, y_pred, zero_division=0):.4f}")
#     print(f"  Sensitivity  : {tp/(tp+fn) if (tp+fn)>0 else 0:.4f}  (TP={tp}, FN={fn})")
#     print(f"  Specificity  : {tn/(tn+fp) if (tn+fp)>0 else 0:.4f}  (TN={tn}, FP={fp})")

# WhyXrayCLIP Few-Shot: N-Shot Sweep

In [ ]:
# ================================================================
# WhyXrayCLIP Few-Shot — Core Helpers
#
# build_wxrc_prototype : builds mean-pooled pos/neg image prototype
#                        embeddings from train_df certain samples.
#                        Uses 2x buffer to handle missing files.
#
# collect_wxrc_scores  : for each GT-uncertain sample computes
#                        text_score  = sim(img, pos_txt) - sim(img, neg_txt)
#                        proto_score = sim(img, pos_proto) - sim(img, neg_proto)
#                        Returns a DataFrame — no thresholding done here.
#
# Both functions are pure helpers; the sweep loop calls them.
# No GT labels are used for building prototypes — only train_df.
# ================================================================

WXRC_N_SHOTS_LIST = [4, 8, 16, 32, 64]
WXRC_ALPHA_SWEEP  = [0.0, 0.25, 0.5, 0.75, 1.0]
WXRC_RANDOM_SEED  = 42


def build_wxrc_prototype(label: str, n_shots: int, seed: int = WXRC_RANDOM_SEED):
    """
    Build positive and negative prototype vectors from train_df.
    Returns (pos_proto [1,D], neg_proto [1,D], meta_dict).
    Samples a 2x buffer to survive missing / corrupt files.
    """
    df      = train_df[train_df[label].isin([0, 1])].copy()
    n_pos_a = int((df[label] == 1).sum())
    n_neg_a = int((df[label] == 0).sum())
    n_pos   = min(n_shots, n_pos_a)
    n_neg   = min(n_shots, n_neg_a)

    if n_pos < 2 or n_neg < 2:
        print(f"    ⚠️  {label}: not enough train samples "
              f"(pos={n_pos_a}, neg={n_neg_a})")
        return None, None, {}

    buf      = 2
    pos_pool = df[df[label] == 1].sample(min(n_pos * buf, n_pos_a), random_state=seed)
    neg_pool = df[df[label] == 0].sample(min(n_neg * buf, n_neg_a), random_state=seed)

    def load_embs(pool, target_n):
        embs, skipped = [], 0
        for _, row in pool.iterrows():
            if len(embs) >= target_n:
                break
            try:
                clean = (row["Path"]
                         .replace("CheXpert-v1.0-small/", "")
                         .replace("CheXpert-v1.0/", ""))
                img = clip_preprocess(
                    Image.open(os.path.join(BASE_PATH, clean)).convert("RGB")
                ).unsqueeze(0).to(device)
                with torch.no_grad():
                    e = clip_model.encode_image(img)
                    embs.append(e / e.norm(dim=-1, keepdim=True))
            except Exception:
                skipped += 1
        return embs, skipped

    pos_embs, sk_pos = load_embs(pos_pool, n_pos)
    neg_embs, sk_neg = load_embs(neg_pool, n_neg)

    if len(pos_embs) < 2 or len(neg_embs) < 2:
        print(f"    ⚠️  {label}: too few valid embeddings "
              f"(pos={len(pos_embs)}, neg={len(neg_embs)})")
        return None, None, {}

    pos_proto = torch.cat(pos_embs).mean(0, keepdim=True)
    neg_proto = torch.cat(neg_embs).mean(0, keepdim=True)
    pos_proto = pos_proto / pos_proto.norm(dim=-1, keepdim=True)
    neg_proto = neg_proto / neg_proto.norm(dim=-1, keepdim=True)

    return pos_proto, neg_proto, {
        "n_pos": len(pos_embs), "n_neg": len(neg_embs),
        "skipped": sk_pos + sk_neg,
    }


def collect_wxrc_scores(label: str, pos_proto, neg_proto) -> pd.DataFrame:
    """
    For every GT-uncertain sample of `label`, compute:
      text_score  = sim(img, pos_txt_emb) - sim(img, neg_txt_emb)
      proto_score = sim(img, pos_proto)   - sim(img, neg_proto)
    Both are pure cosine similarities — no threshold applied here.
    GT labels are stored for evaluation only, never used in scoring.
    """
    labeler_df = rule_labeler_df.copy()
    gt_df      = ground_truth_df.copy()

    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True)
                break

    if label not in labeler_df.columns or label not in gt_df.columns:
        return pd.DataFrame()

    uncertain_idx    = labeler_df[labeler_df[label] == -1].index
    gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
    gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
    gt_for_uncertain = (gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]
                        .reset_index(drop=True))

    if len(gt_for_uncertain) == 0:
        return pd.DataFrame()

    # Text embeddings are computed PER SAMPLE with feature context.
    # This matches inference (infer_clip uses build_wxrc_context per sample).
    top_feats = top_features_per_label.get(label, [])

    rows = []
    for _, row in tqdm(gt_for_uncertain.iterrows(),
                       total=len(gt_for_uncertain),
                       desc=f"    [{label}]", leave=False):
        try:
            img_path = fix_test_path(row["Study"])
            img = clip_preprocess(
                Image.open(img_path).convert("RGB")
            ).unsqueeze(0).to(device)

            # Per-sample feature context
            fv  = get_sample_features(row["Study"], label, top_feats, rule_labeler_df)
            ctx = build_wxrc_context(label, fv)
            pos_text = ctx + POSITIVE_PROMPTS[label]
            neg_text = ctx + NEGATIVE_PROMPTS[label]

            with torch.no_grad():
                img_f   = clip_model.encode_image(img)
                img_f   = img_f / img_f.norm(dim=-1, keepdim=True)
                pos_tok = clip_tokenizer([pos_text]).to(device)
                neg_tok = clip_tokenizer([neg_text]).to(device)
                pos_txt = clip_model.encode_text(pos_tok)
                neg_txt = clip_model.encode_text(neg_tok)
                pos_txt = pos_txt / pos_txt.norm(dim=-1, keepdim=True)
                neg_txt = neg_txt / neg_txt.norm(dim=-1, keepdim=True)

            text_score  = ((img_f @ pos_txt.T) - (img_f @ neg_txt.T)).item()
            proto_score = ((img_f @ pos_proto.T) - (img_f @ neg_proto.T)).item()

            rows.append({
                "study":       row["Study"],
                "y_true":      int(row["gt"]),
                "text_score":  text_score,
                "proto_score": proto_score,
            })
        except Exception as e:
            continue

    return pd.DataFrame(rows)


In [ ]:
# ================================================================
# WhyXrayCLIP Few-Shot — Evaluation Helper
#
# evaluate_wxrc_scores: given a combined score array and ground truth,
#   returns AUC (primary, threshold-free) plus Acc/F1/Sens/Spec.
#
# THRESHOLD POLICY (paper-ready):
#   - Zero-shot  → THRESHOLDS[label]  (calibrated on pos_text_score from train_df)
#   - Few-shot   → WXRC_FS_THRESHOLDS[label][n_shots][alpha]
#                  calibrated on the combined score from train_df
#                  (separate from prototype samples)
#   AUC is always threshold-free and is the primary metric.
# ================================================================
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             balanced_accuracy_score, confusion_matrix, f1_score)

def evaluate_wxrc_scores(scores_arr: np.ndarray,
                         y_true_arr: np.ndarray,
                         label: str,
                         method_name: str,
                         threshold: float = None) -> dict:
    """
    Evaluate a score array against ground truth.

    Args:
        scores_arr  : raw combined scores (NOT binary predictions)
        y_true_arr  : ground truth labels (0/1)
        label       : pathology name (used for threshold lookup if threshold is None)
        method_name : string identifier for this strategy
        threshold   : if provided, use this directly; otherwise look up in THRESHOLDS
    """
    if len(np.unique(y_true_arr)) < 2 or len(y_true_arr) < 4:
        return {}

    auc = roc_auc_score(y_true_arr, scores_arr)

    try:
        auprc = average_precision_score(y_true_arr, scores_arr)
    except Exception:
        auprc = float(y_true_arr.mean())

    if threshold is not None:
        thresh     = threshold
        thresh_src = "fs_calibrated"
    elif label in THRESHOLDS:
        thresh     = THRESHOLDS[label]
        thresh_src = "zs_calibrated"
    else:
        thresh     = ((scores_arr[y_true_arr == 1].mean() +
                       scores_arr[y_true_arr == 0].mean()) / 2.0)
        thresh_src = "midpoint_fallback"

    preds = (scores_arr > thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_arr, preds, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "method":     method_name,
        "auc":        round(float(auc),  3),
        "auprc":      round(float(auprc), 3),
        "bal_acc":    round((sens + spec) / 2, 3),
        "f1":         round(float(f1_score(y_true_arr, preds, zero_division=0)), 3),
        "sens":       round(float(sens),  3),
        "spec":       round(float(spec),  3),
        "threshold":  round(float(thresh), 4),
        "thresh_src": thresh_src,
        "tp": int(tp), "tn": int(tn),
        "fp": int(fp), "fn": int(fn),
        "n":  len(y_true_arr),
    }


def calibrate_wxrc_combined_threshold(label: str,
                                       n_shots: int,
                                       alpha: float,
                                       n_cal: int = 300,
                                       seed: int = 99) -> float:
    """
    Calibrate a threshold for WhyXrayCLIP combined score on train_df.
    Uses samples NOT in the prototype set (seed=42) to avoid leakage.

    combined = alpha * text_score + (1-alpha) * proto_score
    Both computed on train_df certain (0/1) samples.

    Returns the best threshold (float).
    """
    from sklearn.metrics import roc_curve, balanced_accuracy_score

    df = train_df[train_df[label].isin([0, 1])].copy()

    # Build prototypes (same seed as sweep — seed=42)
    pos_proto, neg_proto, _ = build_wxrc_prototype(label, n_shots, seed=42)
    if pos_proto is None:
        print(f"    ⚠️  {label}: prototype failed, using midpoint")
        return 0.0

    # Prototype indices to exclude from calibration set
    buf = 2
    n_pos_avail = int((df[label] == 1).sum())
    n_neg_avail = int((df[label] == 0).sum())
    proto_pos_pool = df[df[label] == 1].sample(
        min(n_shots * buf, n_pos_avail), random_state=42)
    proto_neg_pool = df[df[label] == 0].sample(
        min(n_shots * buf, n_neg_avail), random_state=42)
    proto_idx = set(proto_pos_pool.index) | set(proto_neg_pool.index)

    cal_df = df[~df.index.isin(proto_idx)]
    n_each = min(n_cal // 2, (cal_df[label]==1).sum(), (cal_df[label]==0).sum())

    if n_each < 10:
        print(f"    ⚠️  {label}: too few cal samples, using midpoint")
        return 0.0

    cal_pos = cal_df[cal_df[label]==1].sample(n_each, random_state=seed)
    cal_neg = cal_df[cal_df[label]==0].sample(n_each, random_state=seed)
    cal_df  = pd.concat([cal_pos, cal_neg], ignore_index=True)

    top_feats = top_features_per_label.get(label, [])
    scores, labels_list = [], []
    for _, row in cal_df.iterrows():
        try:
            clean = (row["Path"].replace("CheXpert-v1.0-small/", "")
                                .replace("CheXpert-v1.0/", ""))
            img = clip_preprocess(
                Image.open(os.path.join(BASE_PATH, clean)).convert("RGB")
            ).unsqueeze(0).to(device)
            # Per-sample context (train_df rows have Path not Study)
            fv  = get_sample_features(row["Path"], label, top_feats, train_df)
            ctx = build_wxrc_context(label, fv)
            pos_text = ctx + POSITIVE_PROMPTS[label]
            neg_text = ctx + NEGATIVE_PROMPTS[label]
            with torch.no_grad():
                img_f   = clip_model.encode_image(img)
                img_f   = img_f / img_f.norm(dim=-1, keepdim=True)
                pos_tok = clip_tokenizer([pos_text]).to(device)
                neg_tok = clip_tokenizer([neg_text]).to(device)
                pos_txt = clip_model.encode_text(pos_tok)
                neg_txt = clip_model.encode_text(neg_tok)
                pos_txt = pos_txt / pos_txt.norm(dim=-1, keepdim=True)
                neg_txt = neg_txt / neg_txt.norm(dim=-1, keepdim=True)
            text_s  = ((img_f @ pos_txt.T) - (img_f @ neg_txt.T)).item()
            proto_s = ((img_f @ pos_proto.T) - (img_f @ neg_proto.T)).item()
            combined = alpha * text_s + (1 - alpha) * proto_s
            scores.append(combined)
            labels_list.append(int(row[label]))
        except Exception:
            continue

    if len(scores) < 20:
        return 0.0

    scores_arr = np.array(scores)
    labels_arr = np.array(labels_list)

    # Cross-validated threshold selection
    idx = np.random.RandomState(42).permutation(len(scores_arr))
    mid = len(idx) // 2
    s_tr, l_tr = scores_arr[idx[:mid]], labels_arr[idx[:mid]]
    s_val, l_val = scores_arr[idx[mid:]], labels_arr[idx[mid:]]

    try:
        fpr, tpr, thrs = roc_curve(l_tr, s_tr)
        ba              = (tpr + (1 - fpr)) / 2
        thresh_bal      = float(thrs[np.argmax(ba)])
    except Exception:
        thresh_bal = 0.0

    thresh_mid = float((s_tr[l_tr==1].mean() + s_tr[l_tr==0].mean()) / 2)

    best_t, best_s = thresh_mid, -1
    for t in [thresh_bal, thresh_mid]:
        if np.isinf(t) or np.isnan(t):
            continue
        preds = (s_val > t).astype(int)
        sc    = balanced_accuracy_score(l_val, preds)
        if sc > best_s:
            best_s, best_t = sc, t

    return float(best_t)


# Storage: WXRC_FS_THRESHOLDS[label][(n_shots, alpha)] = threshold
WXRC_FS_THRESHOLDS = {}


In [ ]:
# ================================================================
# WhyXrayCLIP Few-Shot — N-Shot × Alpha Sweep
#
# THRESHOLD POLICY (paper-ready):
#   For each (label, n_shots, alpha) combination:
#     1. Build prototypes from train_df (seed=42)
#     2. Collect combined scores on GT-uncertain test samples
#     3. Calibrate threshold on train_df combined scores
#        (seed=99, proto samples excluded — no leakage)
#     4. Report AUC (primary) + binary metrics using calibrated threshold
#
# No GT labels are used for threshold or prototype construction.
# ================================================================

def run_wxrc_fewshot_sweep():

    print("\n" + "=" * 80)
    print("🔬 WhyXrayCLIP FEW-SHOT N-SHOT SWEEP")
    print(f"   Shot counts : {WXRC_N_SHOTS_LIST}")
    print(f"   Alpha sweep : {WXRC_ALPHA_SWEEP}")
    print(f"   Labels      : {TARGET_LABELS}")
    print("=" * 80)

    all_rows       = []
    best_per_label = {}

    for label in TARGET_LABELS:
        print(f"\n{'━' * 70}")
        print(f"  Label: {label}")
        print(f"{'━' * 70}")
        WXRC_FS_THRESHOLDS[label] = {}

        for n_shots in WXRC_N_SHOTS_LIST:
            print(f"\n  ── {n_shots}-shot ──────────────────────────────────────")

            # Step 1: build prototypes from train_df
            pos_proto, neg_proto, meta = build_wxrc_prototype(label, n_shots)
            if pos_proto is None:
                print(f"    ⚠️  Prototype build failed — skipping {n_shots}-shot")
                continue
            print(f"    ✅ Prototypes: pos={meta['n_pos']} | "
                  f"neg={meta['n_neg']} | skipped={meta['skipped']}")

            # Step 2: collect scores on GT-uncertain samples
            scores_df = collect_wxrc_scores(label, pos_proto, neg_proto)
            if len(scores_df) < 4:
                print(f"    ⚠️  Too few samples ({len(scores_df)}) — skipping")
                continue

            y_true       = scores_df["y_true"].values
            text_scores  = scores_df["text_score"].values
            proto_scores = scores_df["proto_score"].values

            t_gap = (text_scores[y_true == 1].mean() - text_scores[y_true == 0].mean())
            p_gap = (proto_scores[y_true == 1].mean() - proto_scores[y_true == 0].mean())
            print(f"    n={len(scores_df)} | pos={int(y_true.sum())} | neg={int((y_true == 0).sum())}")
            print(f"    text_score  gap : {t_gap:>+.4f}  {'✅' if t_gap > 0.02 else '⚠️  weak'}")
            print(f"    proto_score gap : {p_gap:>+.4f}  {'✅' if p_gap > 0.02 else '⚠️  weak'}")

            # Step 3: alpha sweep — calibrate threshold per (n_shots, alpha)
            print(f"\n    Calibrating few-shot thresholds on train_df...")
            for alpha in WXRC_ALPHA_SWEEP:
                if alpha == 1.0:
                    # text-only → use existing ZS calibrated threshold
                    fs_thresh = THRESHOLDS.get(label, 0.0)
                elif alpha == 0.0:
                    # proto-only → calibrate on proto score
                    fs_thresh = calibrate_wxrc_combined_threshold(
                        label, n_shots, alpha=0.0, n_cal=300)
                else:
                    # hybrid → calibrate on combined score
                    fs_thresh = calibrate_wxrc_combined_threshold(
                        label, n_shots, alpha=alpha, n_cal=300)

                WXRC_FS_THRESHOLDS[label][(n_shots, alpha)] = fs_thresh
                print(f"    [{label}] n={n_shots} α={alpha:.2f} → "
                      f"fs_thresh={fs_thresh:.4f}")

            print(f"\n    {'Method':<24} "
                  f"{'AUC':>7} {'AUPRC':>7} {'BA':>7} "
                  f"{'Sens':>7} {'Spec':>7} {'F1':>7} "
                  f"{'Thresh':>8} {'Src':>15}")
            print(f"    {'─' * 88}")

            best_score_this_shot = 0.0

            for alpha in WXRC_ALPHA_SWEEP:
                combined = alpha * text_scores + (1 - alpha) * proto_scores

                mname = ("text_only"    if alpha == 1.0 else
                         "proto_only"   if alpha == 0.0 else
                         f"hybrid_a{alpha:.2f}")

                fs_thresh = WXRC_FS_THRESHOLDS[label].get((n_shots, alpha), None)
                res = evaluate_wxrc_scores(combined, y_true, label, mname,
                                           threshold=fs_thresh)
                if not res:
                    continue

                res.update({"label": label, "n_shots": n_shots, "alpha": alpha})
                res["score"] = (res["auc"] + res["bal_acc"]) / 2
                all_rows.append(res)

                marker = ""
                if res["score"] > best_score_this_shot:
                    best_score_this_shot = res["score"]
                    marker = " ←"

                if (label not in best_per_label or
                        res["score"] > best_per_label[label]["score"]):
                    best_per_label[label] = res

                print(f"    {mname:<24} "
                      f"{res['auc']:>7.3f} "
                      f"{res.get('auprc', 0.0):>7.3f} "
                      f"{res['bal_acc']:>7.3f} "
                      f"{res['sens']:>7.3f} "
                      f"{res['spec']:>7.3f} "
                      f"{res['f1']:>7.3f} "
                      f"{res['threshold']:>8.4f} "
                      f"{res['thresh_src']:>15}{marker}")

    if not all_rows:
        print("\n⚠️  No results — check errors above.")
        return pd.DataFrame()

    results_df = pd.DataFrame(all_rows)

    # ── TABLE 1: text-only baseline ───────────────────────────────
    print("\n\n" + "=" * 80)
    print("📊 TABLE 1: TEXT-ONLY BASELINE  (alpha=1.0, ZS-calibrated threshold)")
    print("=" * 80)
    text_only = (results_df[results_df["alpha"] == 1.0]
                 .groupby("label").first().reset_index())
    print(text_only[["label", "n", "auc", "bal_acc",
                      "sens", "spec", "f1", "threshold"]].to_string(index=False))
    mean_text_auc = text_only["auc"].mean()
    mean_text_ba = text_only["bal_acc"].mean()
    mean_text_auprc = text_only["auprc"].mean() if "auprc" in text_only.columns else 0.0
    mean_text_score = (mean_text_auc + mean_text_ba) / 2
    print(f"\n  Mean AUC   (text-only): {mean_text_auc:.3f}")
    print(f"  Mean AUPRC (text-only): {mean_text_auprc:.3f}")
    print(f"  Mean BA    (text-only): {mean_text_ba:.3f}")
    print(f"  Mean Score (AUC+BA)/2:  {mean_text_score:.3f}")

    # ── TABLE 2: best few-shot per label ──────────────────────────
    print("\n\n" + "=" * 80)
    print("📊 TABLE 2: BEST FEW-SHOT RESULT PER LABEL")
    print("=" * 80)
    print(f"  {'Label':<35} {'Method':<22} "
          f"{'Shots':>6} {'Score':>7} {'AUC':>7} {'BA':>7} "
          f"{'Δ Score':>8} {'Thresh':>8}")
    print("  " + "─" * 106)

    best_scores = []
    for label in TARGET_LABELS:
        if label not in best_per_label:
            print(f"  {label:<35} no results")
            continue
        best   = best_per_label[label]
        to_row = text_only[text_only["label"] == label]
        if len(to_row) > 0:
            to_auc = float(to_row["auc"].values[0])
            to_ba = float(to_row["bal_acc"].values[0])
            to_score = (to_auc + to_ba) / 2
        else:
            to_score = 0.0
        delta  = best["score"] - to_score
        sign   = ("✅" if delta > 0.01 else "🔁" if delta > -0.01 else "❌")
        best_scores.append(best["score"])
        print(f"  {label:<35} {best['method']:<22} {best['n_shots']:>6} "
              f"{best['score']:>7.3f} {best['auc']:>7.3f} {best['bal_acc']:>7.3f} "
              f"{delta:>+8.3f}  {sign}  "
              f"{best['threshold']:>8.4f}")

    mean_best_score = float(np.mean(best_scores)) if best_scores else 0.0
    print("  " + "─" * 106)
    print(f"  Mean Score (best few-shot) : {mean_best_score:.3f}")
    print(f"  Δ vs text-only             : {mean_best_score - mean_text_score:+.3f}")

    # ── TABLE 3: mean score by alpha ──────────────────────────────
    print("\n\n" + "=" * 80)
    print("📊 TABLE 3: MEAN SCORE BY ALPHA")
    print("=" * 80)
    alpha_grp = (results_df.groupby("alpha")
                 .agg(mean_score=("score", "mean"),
                      mean_auc=("auc", "mean"),
                      mean_ba=("bal_acc", "mean"))
                 .reset_index())
    best_alpha_score = alpha_grp["mean_score"].max()
    print(f"  {'Alpha':<8} {'Method':<22} {'Mean Score':>11} {'Mean AUC':>10} {'Mean BA':>10}")
    print("  " + "─" * 65)
    for _, r in alpha_grp.sort_values("alpha").iterrows():
        mname  = ("text_only" if r["alpha"] == 1.0 else
                  "proto_only" if r["alpha"] == 0.0 else
                  f"hybrid_a{r['alpha']:.2f}")
        marker = " ← best" if r["mean_score"] == best_alpha_score else ""
        print(f"  {r['alpha']:<8.2f} {mname:<22} "
              f"{r['mean_score']:>11.3f} {r['mean_auc']:>10.3f} {r['mean_ba']:>10.3f}{marker}")

    # ── TABLE 4: mean score by n_shots ────────────────────────────
    print("\n\n" + "=" * 80)
    print("📊 TABLE 4: MEAN SCORE BY N_SHOTS")
    print("=" * 80)
    shots_grp = (results_df.groupby("n_shots")
                 .agg(mean_score=("score", "mean"),
                      mean_auc=("auc", "mean"),
                      mean_ba=("bal_acc", "mean"))
                 .reset_index())
    best_shots_score = shots_grp["mean_score"].max()
    print(f"  {'N-Shots':<10} {'Mean Score':>11} {'Mean AUC':>10} {'Mean BA':>10}")
    print("  " + "─" * 47)
    for _, r in shots_grp.sort_values("n_shots").iterrows():
        marker = " ← best" if r["mean_score"] == best_shots_score else ""
        print(f"  {int(r['n_shots']):<10} {r['mean_score']:>11.3f} {r['mean_auc']:>10.3f} "
              f"{r['mean_ba']:>10.3f}{marker}")

    # ── TABLE 5: vs zero-shot baseline ───────────────────────────
    print("\n\n" + "=" * 80)
    print("📊 TABLE 5: BEST FEW-SHOT vs ZERO-SHOT  (uncertain_metrics AUC)")
    print("=" * 80)
    print(f"  {'Label':<35} {'ZeroShot':>10} {'BestFS':>10} {'Δ AUC':>8}")
    print("  " + "─" * 68)
    zs_aucs, fs_aucs = [], []
    for label in TARGET_LABELS:
        z_row = uncertain_metrics[uncertain_metrics["label"] == label]
        if len(z_row) == 0 or label not in best_per_label:
            continue
        z_auc = float(z_row["auc"].values[0])
        f_auc = best_per_label[label]["auc"]
        delta = f_auc - z_auc
        icon  = "✅" if delta > 0.01 else ("🔁" if delta > -0.01 else "❌")
        zs_aucs.append(z_auc); fs_aucs.append(f_auc)
        print(f"  {label:<35} {z_auc:>10.3f} {f_auc:>10.3f} {delta:>+8.3f}  {icon}")
    if zs_aucs:
        print("  " + "─" * 68)
        avg_z = float(np.mean(zs_aucs)); avg_f = float(np.mean(fs_aucs))
        print(f"  {'Average':<35} {avg_z:>10.3f} {avg_f:>10.3f} {avg_f-avg_z:>+8.3f}")

    # ── Verdict ───────────────────────────────────────────────────
    best_alpha_val = float(alpha_grp.loc[alpha_grp["mean_score"].idxmax(), "alpha"])
    best_shots_val = int(shots_grp.loc[shots_grp["mean_score"].idxmax(), "n_shots"])
    delta_vs_text  = mean_best_score - mean_text_score

    print("\n\n" + "=" * 80)
    print("🏁 VERDICT")
    print("=" * 80)
    if delta_vs_text > 0.01:
        print(f"  ✅ Few-shot IMPROVES over text-only (Δ Score = {delta_vs_text:+.3f})")
        print(f"     → Best alpha  : {best_alpha_val}")
        print(f"     → Best n_shots: {best_shots_val}")
        print(f"     → Use these settings in the final benchmark cell.")
    elif delta_vs_text > -0.01:
        print(f"  🔁 Few-shot comparable to text-only (Δ Score = {delta_vs_text:+.3f})")
        print(f"     → Report both; text-only is simpler to reproduce.")
    else:
        print(f"  ❌ Few-shot HURTS performance (Δ Score = {delta_vs_text:+.3f})")
        print(f"     → Stick with zero-shot.")

    results_df.to_csv("wxrc_fewshot_sweep_results.csv", index=False)
    print(f"\n  💾 Saved: wxrc_fewshot_sweep_results.csv")
    return results_df


WXRC_SWEEP_RESULTS = run_wxrc_fewshot_sweep()

In [ ]:
# # ================================================================
# # WhyXrayCLIP Few-Shot — Final Benchmark with Best Config
# #
# # THRESHOLD POLICY:
# #   Uses WXRC_FS_THRESHOLDS[label][(WXRC_FINAL_N_SHOTS, WXRC_FINAL_ALPHA)]
# #   calibrated on train_df combined scores (not the ZS text-score threshold).
# #   AUC is threshold-free and is the primary metric.
# #
# # Set WXRC_FINAL_N_SHOTS and WXRC_FINAL_ALPHA from sweep TABLE 4 / VERDICT.
# # ================================================================

# WXRC_FINAL_N_SHOTS = 64    # ← update from sweep TABLE 4 verdict
# WXRC_FINAL_ALPHA   = 0.75  # ← update from sweep TABLE 3 verdict

# print(f"Building final prototypes: {WXRC_FINAL_N_SHOTS}-shot, alpha={WXRC_FINAL_ALPHA}")
# print("=" * 60)

# WXRC_FINAL_PROTO = {}
# for label in TARGET_LABELS:
#     pos_p, neg_p, meta = build_wxrc_prototype(
#         label, WXRC_FINAL_N_SHOTS, seed=WXRC_RANDOM_SEED
#     )
#     WXRC_FINAL_PROTO[label] = (pos_p, neg_p)
#     print(f"  {label}: pos={meta.get('n_pos','?')} | "
#           f"neg={meta.get('n_neg','?')} | skipped={meta.get('skipped','?')}")

# # ── Calibrate final FS threshold if not already done by sweep ────
# print("\n🔧 Ensuring final FS thresholds are calibrated...")
# for label in TARGET_LABELS:
#     key = (WXRC_FINAL_N_SHOTS, WXRC_FINAL_ALPHA)
#     if label not in WXRC_FS_THRESHOLDS or key not in WXRC_FS_THRESHOLDS.get(label, {}):
#         print(f"  Calibrating {label} (n={WXRC_FINAL_N_SHOTS}, α={WXRC_FINAL_ALPHA})...")
#         thresh = calibrate_wxrc_combined_threshold(
#             label, WXRC_FINAL_N_SHOTS, alpha=WXRC_FINAL_ALPHA, n_cal=1000)
#         if label not in WXRC_FS_THRESHOLDS:
#             WXRC_FS_THRESHOLDS[label] = {}
#         WXRC_FS_THRESHOLDS[label][key] = thresh
#         print(f"    → threshold = {thresh:.4f}")
#     else:
#         print(f"  {label}: threshold = "
#               f"{WXRC_FS_THRESHOLDS[label][key]:.4f} (already calibrated)")

# print("\n🚀 Running final WhyXrayCLIP few-shot benchmark...")
# print("=" * 80)

# final_rows = []
# labeler_df = rule_labeler_df.copy()
# gt_df      = ground_truth_df.copy()
# for df_ in [gt_df, labeler_df]:
#     for c in ["Path", "path", "study"]:
#         if c in df_.columns and "Study" not in df_.columns:
#             df_.rename(columns={c: "Study"}, inplace=True)
#             break

# for label in TARGET_LABELS:
#     print(f"\n{'='*60}\n  Label: {label}\n{'='*60}")
#     pos_proto, neg_proto = WXRC_FINAL_PROTO[label]
#     fs_thresh = WXRC_FS_THRESHOLDS.get(label, {}).get(
#         (WXRC_FINAL_N_SHOTS, WXRC_FINAL_ALPHA), 0.0)
#     print(f"  FS threshold (train_df calibrated): {fs_thresh:.4f}")

#     uncertain_idx    = labeler_df[labeler_df[label] == -1].index
#     gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
#     gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
#     gt_for_uncertain = gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]

#     print(f"  Samples: {len(gt_for_uncertain)} | "
#           f"pos={int(gt_for_uncertain['gt'].sum())} | "
#           f"neg={int((gt_for_uncertain['gt']==0).sum())}")

#     with torch.no_grad():
#         pos_tok = clip_tokenizer([POSITIVE_PROMPTS[label]]).to(device)
#         neg_tok = clip_tokenizer([NEGATIVE_PROMPTS[label]]).to(device)
#         pos_txt = clip_model.encode_text(pos_tok)
#         neg_txt = clip_model.encode_text(neg_tok)
#         pos_txt = pos_txt / pos_txt.norm(dim=-1, keepdim=True)
#         neg_txt = neg_txt / neg_txt.norm(dim=-1, keepdim=True)

#     for _, row in tqdm(gt_for_uncertain.iterrows(),
#                        total=len(gt_for_uncertain),
#                        desc=f"WXRC-final · {label}"):
#         try:
#             img_path = fix_test_path(row["Study"])
#             img = clip_preprocess(
#                 Image.open(img_path).convert("RGB")
#             ).unsqueeze(0).to(device)
#             with torch.no_grad():
#                 img_f = clip_model.encode_image(img)
#                 img_f = img_f / img_f.norm(dim=-1, keepdim=True)

#             text_score  = ((img_f @ pos_txt.T) - (img_f @ neg_txt.T)).item()
#             proto_score = ((img_f @ pos_proto.T) - (img_f @ neg_proto.T)).item()
#             combined    = (WXRC_FINAL_ALPHA * text_score
#                            + (1 - WXRC_FINAL_ALPHA) * proto_score)
#             # Use calibrated FS threshold (NOT 0.0 and NOT ZS threshold)
#             pred = 1 if combined > fs_thresh else 0

#             final_rows.append({
#                 "label":           label,
#                 "study":           row["Study"],
#                 "y_true":          int(row["gt"]),
#                 "y_pred":          pred,
#                 "combined_score":  combined,
#                 "threshold_used":  fs_thresh,
#             })
#         except Exception:
#             final_rows.append({
#                 "label":  label, "study": row["Study"],
#                 "y_true": int(row["gt"]) if pd.notna(row.get("gt")) else -1,
#                 "y_pred": 0, "combined_score": None, "threshold_used": fs_thresh,
#             })

# wxrc_final_results = pd.DataFrame(final_rows)
# wxrc_final_results.to_csv("whyxrayclip_fewshot_final_results.csv", index=False)
# print("\n✅ Saved: whyxrayclip_fewshot_final_results.csv")
# print("   Note: y_pred uses train_df-calibrated FS threshold, not 0.0")

# wxrc_final_metrics = evaluate_results_with_details(wxrc_final_results)
# print_detailed_results(wxrc_final_metrics)

# # ── Zero-shot vs best few-shot comparison ─────────────────────────
# print("\n" + "=" * 80)
# print(f"📊 WhyXrayCLIP: Zero-Shot vs Few-Shot "
#       f"({WXRC_FINAL_N_SHOTS}-shot, alpha={WXRC_FINAL_ALPHA})")
# print(f"   ZS threshold: train_df ZS-calibrated | "
#       f"FS threshold: train_df FS-calibrated (combined score)")
# print("=" * 80)
# print(f"  {'Label':<30} "
#       f"{'ZS AUC':>7} {'FS AUC':>7} {'ΔAUC':>7} "
#       f"{'ZS BA':>7} {'FS BA':>7} {'ΔBA':>7}")
# print(f"  {'':<30} "
#       f"{'ZS AUPRC':>8} {'FS AUPRC':>8} {'ΔAUPRC':>7} "
#       f"{'ZS F1':>7} {'FS F1':>7} {'ΔF1':>7} "
#       f"{'ZS Se':>7} {'FS Se':>7} {'ΔSe':>7} "
#       f"{'ZS Sp':>7} {'FS Sp':>7} {'ΔSp':>7}")
# print("  " + "-" * 125)

# def _metric(row, key, fallback_key=None):
#     if key in row:
#         return float(row[key])
#     if fallback_key and fallback_key in row:
#         return float(row[fallback_key])
#     return 0.0

# zs_auc, fs_auc = [], []
# zs_ba, fs_ba = [], []
# zs_auprc, fs_auprc = [], []
# zs_f1, fs_f1 = [], []
# zs_sens, fs_sens = [], []
# zs_spec, fs_spec = [], []
# for label in TARGET_LABELS:
#     z_row = uncertain_metrics[uncertain_metrics["label"] == label]
#     f_row = wxrc_final_metrics[wxrc_final_metrics["label"] == label]
#     if len(z_row) == 0 or len(f_row) == 0:
#         continue
#     z = z_row.iloc[0]
#     f = f_row.iloc[0]

#     z_auc   = _metric(z, "auc");                    f_auc   = _metric(f, "auc")
#     z_ba    = _metric(z, "bal_acc", "balanced_acc"); f_ba    = _metric(f, "bal_acc", "balanced_acc")
#     z_auprc = _metric(z, "auprc");                  f_auprc = _metric(f, "auprc")
#     z_f1    = _metric(z, "f1", "f1_score");          f_f1    = _metric(f, "f1", "f1_score")
#     z_se    = _metric(z, "sens", "sensitivity");     f_se    = _metric(f, "sens", "sensitivity")
#     z_sp    = _metric(z, "spec", "specificity");     f_sp    = _metric(f, "spec", "specificity")

#     zs_auc.append(z_auc);     fs_auc.append(f_auc)
#     zs_ba.append(z_ba);       fs_ba.append(f_ba)
#     zs_auprc.append(z_auprc); fs_auprc.append(f_auprc)
#     zs_f1.append(z_f1);       fs_f1.append(f_f1)
#     zs_sens.append(z_se);     fs_sens.append(f_se)
#     zs_spec.append(z_sp);     fs_spec.append(f_sp)

#     print(f"  {label:<30} "
#           f"{z_auc:>7.3f} {f_auc:>7.3f} {f_auc - z_auc:>+7.3f} "
#           f"{z_ba:>7.3f} {f_ba:>7.3f} {f_ba - z_ba:>+7.3f}")
#     print(f"  {'':<30} "
#           f"{z_auprc:>8.3f} {f_auprc:>8.3f} {f_auprc - z_auprc:>+7.3f} "
#           f"{z_f1:>7.3f} {f_f1:>7.3f} {f_f1 - z_f1:>+7.3f} "
#           f"{z_se:>7.3f} {f_se:>7.3f} {f_se - z_se:>+7.3f} "
#           f"{z_sp:>7.3f} {f_sp:>7.3f} {f_sp - z_sp:>+7.3f}")
# print("  " + "-" * 125)
# if zs_auc:
#     avg_z_auc   = float(np.mean(zs_auc));   avg_f_auc   = float(np.mean(fs_auc))
#     avg_z_ba    = float(np.mean(zs_ba));    avg_f_ba    = float(np.mean(fs_ba))
#     avg_z_auprc = float(np.mean(zs_auprc)); avg_f_auprc = float(np.mean(fs_auprc))
#     avg_z_f1    = float(np.mean(zs_f1));    avg_f_f1    = float(np.mean(fs_f1))
#     avg_z_se    = float(np.mean(zs_sens));  avg_f_se    = float(np.mean(fs_sens))
#     avg_z_sp    = float(np.mean(zs_spec));  avg_f_sp    = float(np.mean(fs_spec))
#     print(f"  {'Average':<30} "
#           f"{avg_z_auc:>7.3f} {avg_f_auc:>7.3f} {avg_f_auc - avg_z_auc:>+7.3f} "
#           f"{avg_z_ba:>7.3f} {avg_f_ba:>7.3f} {avg_f_ba - avg_z_ba:>+7.3f}")
#     print(f"  {'':<30} "
#           f"{avg_z_auprc:>8.3f} {avg_f_auprc:>8.3f} {avg_f_auprc - avg_z_auprc:>+7.3f} "
#           f"{avg_z_f1:>7.3f} {avg_f_f1:>7.3f} {avg_f_f1 - avg_z_f1:>+7.3f} "
#           f"{avg_z_se:>7.3f} {avg_f_se:>7.3f} {avg_f_se - avg_z_se:>+7.3f} "
#           f"{avg_z_sp:>7.3f} {avg_f_sp:>7.3f} {avg_f_sp - avg_z_sp:>+7.3f}")
#     delta = avg_f_auc - avg_z_auc
#     print(f"\n  Verdict: {'Few-Shot better ✅' if delta > 0 else 'Zero-Shot better 🔴'}")

# ================================================================
# WhyXrayCLIP Few-Shot — Final Benchmark with Best Config
#
# THRESHOLD POLICY:
#   Uses WXRC_FS_THRESHOLDS[label][(WXRC_FINAL_N_SHOTS, WXRC_FINAL_ALPHA)]
#   calibrated on train_df combined scores (not the ZS text-score threshold).
#   AUC is threshold-free and is the primary metric.
#
# Set WXRC_FINAL_N_SHOTS and WXRC_FINAL_ALPHA from sweep TABLE 4 / VERDICT.
# ================================================================

WXRC_FINAL_N_SHOTS = 64    # ← update from sweep TABLE 4 verdict
WXRC_FINAL_ALPHA   = 0.75  # ← update from sweep TABLE 3 verdict

print(f"Building final prototypes: {WXRC_FINAL_N_SHOTS}-shot, alpha={WXRC_FINAL_ALPHA}")
print("=" * 60)

WXRC_FINAL_PROTO = {}
for label in TARGET_LABELS:
    pos_p, neg_p, meta = build_wxrc_prototype(
        label, WXRC_FINAL_N_SHOTS, seed=WXRC_RANDOM_SEED
    )
    WXRC_FINAL_PROTO[label] = (pos_p, neg_p)
    print(f"  {label}: pos={meta.get('n_pos','?')} | "
          f"neg={meta.get('n_neg','?')} | skipped={meta.get('skipped','?')}")

# ── Calibrate final FS threshold if not already done by sweep ────
print("\n🔧 Ensuring final FS thresholds are calibrated...")
for label in TARGET_LABELS:
    key = (WXRC_FINAL_N_SHOTS, WXRC_FINAL_ALPHA)
    if label not in WXRC_FS_THRESHOLDS or key not in WXRC_FS_THRESHOLDS.get(label, {}):
        print(f"  Calibrating {label} (n={WXRC_FINAL_N_SHOTS}, α={WXRC_FINAL_ALPHA})...")
        thresh = calibrate_wxrc_combined_threshold(
            label, WXRC_FINAL_N_SHOTS, alpha=WXRC_FINAL_ALPHA, n_cal=1000)
        if label not in WXRC_FS_THRESHOLDS:
            WXRC_FS_THRESHOLDS[label] = {}
        WXRC_FS_THRESHOLDS[label][key] = thresh
        print(f"    → threshold = {thresh:.4f}")
    else:
        print(f"  {label}: threshold = "
              f"{WXRC_FS_THRESHOLDS[label][key]:.4f} (already calibrated)")

print("\n🚀 Running final WhyXrayCLIP few-shot benchmark...")
print("=" * 80)

final_rows = []
labeler_df = rule_labeler_df.copy()
gt_df      = ground_truth_df.copy()
for df_ in [gt_df, labeler_df]:
    for c in ["Path", "path", "study"]:
        if c in df_.columns and "Study" not in df_.columns:
            df_.rename(columns={c: "Study"}, inplace=True)
            break

for label in TARGET_LABELS:
    print(f"\n{'='*60}\n  Label: {label}\n{'='*60}")
    pos_proto, neg_proto = WXRC_FINAL_PROTO[label]
    fs_thresh = WXRC_FS_THRESHOLDS.get(label, {}).get(
        (WXRC_FINAL_N_SHOTS, WXRC_FINAL_ALPHA), 0.0)
    print(f"  FS threshold (train_df calibrated): {fs_thresh:.4f}")

    uncertain_idx    = labeler_df[labeler_df[label] == -1].index
    gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
    gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
    gt_for_uncertain = gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]

    print(f"  Samples: {len(gt_for_uncertain)} | "
          f"pos={int(gt_for_uncertain['gt'].sum())} | "
          f"neg={int((gt_for_uncertain['gt']==0).sum())}")

    # Fetch top features for the current label
    top_features = top_features_per_label.get(label, [])

    for _, row in tqdm(gt_for_uncertain.iterrows(),
                       total=len(gt_for_uncertain),
                       desc=f"WXRC-final · {label}"):
        try:
            img_path = fix_test_path(row["Study"])
            img = clip_preprocess(
                Image.open(img_path).convert("RGB")
            ).unsqueeze(0).to(device)
            
            # ── NEW: Per-sample feature context ──
            fv = get_sample_features(row["Study"], label, top_features, rule_labeler_df)
            ctx = build_wxrc_context(label, fv)
            pos_text = ctx + POSITIVE_PROMPTS[label]
            neg_text = ctx + NEGATIVE_PROMPTS[label]

            with torch.no_grad():
                img_f = clip_model.encode_image(img)
                img_f = img_f / img_f.norm(dim=-1, keepdim=True)
                
                # Encode text dynamically per sample
                pos_tok = clip_tokenizer([pos_text]).to(device)
                neg_tok = clip_tokenizer([neg_text]).to(device)
                pos_txt = clip_model.encode_text(pos_tok)
                neg_txt = clip_model.encode_text(neg_tok)
                pos_txt = pos_txt / pos_txt.norm(dim=-1, keepdim=True)
                neg_txt = neg_txt / neg_txt.norm(dim=-1, keepdim=True)

            text_score  = ((img_f @ pos_txt.T) - (img_f @ neg_txt.T)).item()
            proto_score = ((img_f @ pos_proto.T) - (img_f @ neg_proto.T)).item()
            combined    = (WXRC_FINAL_ALPHA * text_score
                           + (1 - WXRC_FINAL_ALPHA) * proto_score)
            
            # Use calibrated FS threshold (NOT 0.0 and NOT ZS threshold)
            pred = 1 if combined > fs_thresh else 0

            final_rows.append({
                "label":           label,
                "study":           row["Study"],
                "y_true":          int(row["gt"]),
                "y_pred":          pred,
                "combined_score":  combined,
                "threshold_used":  fs_thresh,
            })
        except Exception:
            final_rows.append({
                "label":  label, "study": row["Study"],
                "y_true": int(row["gt"]) if pd.notna(row.get("gt")) else -1,
                "y_pred": 0, "combined_score": None, "threshold_used": fs_thresh,
            })

wxrc_final_results = pd.DataFrame(final_rows)
wxrc_final_results.to_csv("whyxrayclip_fewshot_final_results.csv", index=False)
print("\n✅ Saved: whyxrayclip_fewshot_final_results.csv")
print("   Note: y_pred uses train_df-calibrated FS threshold, not 0.0")

wxrc_final_metrics = evaluate_results_with_details(wxrc_final_results)
print_detailed_results(wxrc_final_metrics)

# ── Zero-shot vs best few-shot comparison ─────────────────────────
print("\n" + "=" * 80)
print(f"📊 WhyXrayCLIP: Zero-Shot vs Few-Shot "
      f"({WXRC_FINAL_N_SHOTS}-shot, alpha={WXRC_FINAL_ALPHA})")
print(f"   ZS threshold: train_df ZS-calibrated | "
      f"FS threshold: train_df FS-calibrated (combined score)")
print("=" * 80)
print(f"  {'Label':<30} "
      f"{'ZS AUC':>7} {'FS AUC':>7} {'ΔAUC':>7} "
      f"{'ZS BA':>7} {'FS BA':>7} {'ΔBA':>7}")
print(f"  {'':<30} "
      f"{'ZS AUPRC':>8} {'FS AUPRC':>8} {'ΔAUPRC':>7} "
      f"{'ZS F1':>7} {'FS F1':>7} {'ΔF1':>7} "
      f"{'ZS Se':>7} {'FS Se':>7} {'ΔSe':>7} "
      f"{'ZS Sp':>7} {'FS Sp':>7} {'ΔSp':>7}")
print("  " + "-" * 125)

def _metric(row, key, fallback_key=None):
    if key in row:
        return float(row[key])
    if fallback_key and fallback_key in row:
        return float(row[fallback_key])
    return 0.0

zs_auc, fs_auc = [], []
zs_ba, fs_ba = [], []
zs_auprc, fs_auprc = [], []
zs_f1, fs_f1 = [], []
zs_sens, fs_sens = [], []
zs_spec, fs_spec = [], []
for label in TARGET_LABELS:
    z_row = uncertain_metrics[uncertain_metrics["label"] == label]
    f_row = wxrc_final_metrics[wxrc_final_metrics["label"] == label]
    if len(z_row) == 0 or len(f_row) == 0:
        continue
    z = z_row.iloc[0]
    f = f_row.iloc[0]

    z_auc   = _metric(z, "auc");                    f_auc   = _metric(f, "auc")
    z_ba    = _metric(z, "bal_acc", "balanced_acc"); f_ba    = _metric(f, "bal_acc", "balanced_acc")
    z_auprc = _metric(z, "auprc");                  f_auprc = _metric(f, "auprc")
    z_f1    = _metric(z, "f1", "f1_score");          f_f1    = _metric(f, "f1", "f1_score")
    z_se    = _metric(z, "sens", "sensitivity");     f_se    = _metric(f, "sens", "sensitivity")
    z_sp    = _metric(z, "spec", "specificity");     f_sp    = _metric(f, "spec", "specificity")

    zs_auc.append(z_auc);     fs_auc.append(f_auc)
    zs_ba.append(z_ba);       fs_ba.append(f_ba)
    zs_auprc.append(z_auprc); fs_auprc.append(f_auprc)
    zs_f1.append(z_f1);       fs_f1.append(f_f1)
    zs_sens.append(z_se);     fs_sens.append(f_se)
    zs_spec.append(z_sp);     fs_spec.append(f_sp)

    print(f"  {label:<30} "
          f"{z_auc:>7.3f} {f_auc:>7.3f} {f_auc - z_auc:>+7.3f} "
          f"{z_ba:>7.3f} {f_ba:>7.3f} {f_ba - z_ba:>+7.3f}")
    print(f"  {'':<30} "
          f"{z_auprc:>8.3f} {f_auprc:>8.3f} {f_auprc - z_auprc:>+7.3f} "
          f"{z_f1:>7.3f} {f_f1:>7.3f} {f_f1 - z_f1:>+7.3f} "
          f"{z_se:>7.3f} {f_se:>7.3f} {f_se - z_se:>+7.3f} "
          f"{z_sp:>7.3f} {f_sp:>7.3f} {f_sp - z_sp:>+7.3f}")
print("  " + "-" * 125)
if zs_auc:
    avg_z_auc   = float(np.mean(zs_auc));   avg_f_auc   = float(np.mean(fs_auc))
    avg_z_ba    = float(np.mean(zs_ba));    avg_f_ba    = float(np.mean(fs_ba))
    avg_z_auprc = float(np.mean(zs_auprc)); avg_f_auprc = float(np.mean(fs_auprc))
    avg_z_f1    = float(np.mean(zs_f1));    avg_f_f1    = float(np.mean(fs_f1))
    avg_z_se    = float(np.mean(zs_sens));  avg_f_se    = float(np.mean(fs_sens))
    avg_z_sp    = float(np.mean(zs_spec));  avg_f_sp    = float(np.mean(fs_spec))
    print(f"  {'Average':<30} "
          f"{avg_z_auc:>7.3f} {avg_f_auc:>7.3f} {avg_f_auc - avg_z_auc:>+7.3f} "
          f"{avg_z_ba:>7.3f} {avg_f_ba:>7.3f} {avg_f_ba - avg_z_ba:>+7.3f}")
    print(f"  {'':<30} "
          f"{avg_z_auprc:>8.3f} {avg_f_auprc:>8.3f} {avg_f_auprc - avg_z_auprc:>+7.3f} "
          f"{avg_z_f1:>7.3f} {avg_f_f1:>7.3f} {avg_f_f1 - avg_z_f1:>+7.3f} "
          f"{avg_z_se:>7.3f} {avg_f_se:>7.3f} {avg_f_se - avg_z_se:>+7.3f} "
          f"{avg_z_sp:>7.3f} {avg_f_sp:>7.3f} {avg_f_sp - avg_z_sp:>+7.3f}")
    delta = avg_f_auc - avg_z_auc
    print(f"\n  Verdict: {'Few-Shot better ✅' if delta > 0 else 'Zero-Shot better 🔴'}")

# 1. Load BioMedClip

In [ ]:
# ================================================================
# Load BiomedCLIP (original model)
# Trained on 15M biomedical image-text pairs from PubMed.
# NOT trained on CheXpert ✅  — same open_clip interface as WhyXrayCLIP
# ================================================================
import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"

_BMC_MODEL_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
print("Loading BiomedCLIP...")

biomedclip_model, _, biomedclip_preprocess = open_clip.create_model_and_transforms(_BMC_MODEL_ID)
biomedclip_model     = biomedclip_model.to(device).eval()
biomedclip_tokenizer = open_clip.get_tokenizer(_BMC_MODEL_ID)

n_params = sum(p.numel() for p in biomedclip_model.parameters()) // 1_000_000
print(f"✅ BiomedCLIP loaded on {device} ({n_params}M params)")

# 2. BioMedCLIP Prompts

In [ ]:
BIOMEDCLIP_POSITIVE_PROMPTS = {
    "Edema":
        "chest radiograph showing pulmonary edema with symmetric bilateral "
        "perihilar bat-wing opacities, Kerley B lines at lung bases, "
        "peribronchial cuffing and upper lobe blood diversion — "
        "cardiogenic pattern consistent with elevated pulmonary venous pressure",

    "Atelectasis":
        "chest radiograph showing lobar atelectasis with ipsilateral "
        "mediastinal shift, hemidiaphragm elevation, bronchovascular "
        "crowding and fissural displacement indicating volume loss",
    
    "Pleural Effusion":
        "chest radiograph showing pleural effusion with blunting "
        "of the costophrenic angle, homogeneous basal opacity "
        "with meniscus sign along the lateral chest wall, "
        "hemidiaphragm obscuration, and layering fluid in the "
        "dependent portions of the pleural space",

    "Enlarged Cardiomediastinum":
        "chest radiograph showing enlarged cardiomediastinum with "
        "cardiothoracic ratio greater than 0.5 and water bottle heart "
        "appearance consistent with cardiomegaly or pericardial effusion",
    
    "Consolidation":
        "chest radiograph showing pulmonary consolidation with homogeneous "
        "airspace opacity obscuring vascular markings, presence of air bronchograms, "
        "segmental or lobar distribution with possible silhouette sign indicating "
        "alveolar filling process such as pneumonia",
}

BIOMEDCLIP_NEGATIVE_PROMPTS = {
    "Edema":
        "chest radiograph with clear lungs, normal pulmonary vascularity, "
        "no perihilar haze, no Kerley B lines, no pleural effusion, "
        "normal heart size — no evidence of cardiac or pulmonary venous hypertension",

    "Atelectasis":
        "chest radiograph showing fully expanded lungs bilaterally "
        "with normal fissure position, no volume loss, "
        "no mediastinal shift or hemidiaphragm elevation",

    "Pleural Effusion":
        "chest radiograph showing sharp and clear costophrenic angles "
        "bilaterally, fully visible hemidiaphragms, no basal opacity, "
        "no meniscus sign, no layering fluid, no blunting — "
        "pleural spaces are completely clear",

    "Enlarged Cardiomediastinum":
        "chest radiograph showing normal cardiothoracic ratio below 0.5 "
        "with normal mediastinal contours, "
        "no cardiomegaly or mediastinal widening",

    "Consolidation":
        "chest radiograph showing clear lung fields without any focal or diffuse "
        "airspace opacity, preserved vascular markings, no air bronchograms, "
        "no segmental or lobar consolidation or alveolar filling"
}

# 3. BioMedCLIP Feature-Context Strategy

In [ ]:
# ================================================================
# BioMedCLIP Feature-Context Strategy
#
# Mirrors WXRC_CONTEXT_STRATEGY / CZERO_CONTEXT_STRATEGY.
# All three format helpers (_build_old_format_context,
# _build_radiology_context, build_context) are already defined
# in Cell 25 (shared infrastructure) — we just add the
# BMC strategy map and a convenience wrapper here.
#
# UNCERTAIN (-1) HANDLING:
#   old_format       — uncertain values are SILENTLY SKIPPED
#                      (no noise from hedging in terse format)
#   radiology_format — uncertain values produce hedging sentences
#                      like "Mild edema cannot be excluded."
#                      via FEATURE_TO_RADIOLOGY_LANGUAGE[feat]["uncertain"]
#                      Only clinically relevant features (RELEVANT_NEGATIVES)
#                      get uncertain context — irrelevant ones are skipped.
#
# Values: "no_context" | "old_format" | "radiology_format"
# Seeded from BioViL-T ablation; the ablation cell below will
# auto-update this map before calibration runs.
# ================================================================

BMC_CONTEXT_STRATEGY = {
    "Edema":                      "radiology_format",
    "Atelectasis":                "old_format",
    "Pleural Effusion":           "old_format",
    "Enlarged Cardiomediastinum": "no_context",
    "Consolidation":              "no_context",
}


def build_bmc_context(label: str, feature_values: list) -> str:
    """
    Build feature context string for BioMedCLIP using BMC_CONTEXT_STRATEGY.
    Wraps the shared build_context() infrastructure.
    Uncertain (-1) values handled per strategy (see header comment).
    """
    return build_context(label, feature_values, BMC_CONTEXT_STRATEGY)


print("✅ BMC_CONTEXT_STRATEGY loaded.")
print(f"   Strategy map: {BMC_CONTEXT_STRATEGY}")

# 4. BioMedCLIP Context Format Ablation

In [ ]:
# ================================================================
# BioMedCLIP Context Format Ablation (Zero-Shot)
#
# PURPOSE: For each label, compare 3 context strategies:
#     no_context        — bare prompt only
#     old_format        — "X is present, Y is absent."
#                         (uncertain values silently skipped)
#     radiology_format  — "X is present with... A small Y cannot
#                         be excluded." (uncertain values produce
#                         hedging sentences for relevant features)
#
# Metric: AUC on GT-uncertain evaluation set (threshold-free).
# Winner auto-updates BMC_CONTEXT_STRATEGY before calibration.
#
# DESIGN NOTE: Context affects only the TEXT prompt, not any
# threshold or prototype — so there is no threshold leakage.
# AUC is threshold-free and is the primary metric.
# ================================================================
from sklearn.metrics import roc_auc_score

_BMC_ABL_RESULTS = {}


def _run_bmc_context_ablation():
    results = {label: {} for label in TARGET_LABELS}

    labeler_df = rule_labeler_df.copy()
    gt_df      = ground_truth_df.copy()
    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True)
                break

    for label in TARGET_LABELS:
        print(f"\n  [BMC] {label}")

        uncertain_idx    = labeler_df[labeler_df[label] == -1].index
        gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
        gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
        gt_for_uncertain = (gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]
                            .reset_index(drop=True))

        if len(gt_for_uncertain) < 5:
            print(f"    ⚠️  Too few samples — skipping")
            continue

        top_feats = top_features_per_label.get(label, [])

        # Collect image embeddings + feature values once
        sample_data = []
        for _, row in tqdm(gt_for_uncertain.iterrows(),
                           total=len(gt_for_uncertain),
                           desc=f"    Collecting [{label}]", leave=False):
            try:
                img_path = fix_test_path(row["Study"])
                img = biomedclip_preprocess(
                    Image.open(img_path).convert("RGB")
                ).unsqueeze(0).to(device)
                fv = get_sample_features(
                    row["Study"], label, top_feats, rule_labeler_df
                )
                with torch.no_grad():
                    img_f = biomedclip_model.encode_image(img)
                    img_f = img_f / img_f.norm(dim=-1, keepdim=True)
                sample_data.append({
                    "y_true": int(row["gt"]),
                    "img_f":  img_f,
                    "fv":     fv,
                })
            except Exception:
                continue

        if len(sample_data) < 5:
            continue

        for strategy in ["no_context", "old_format", "radiology_format"]:
            scores = []
            for sd in sample_data:
                try:
                    if strategy == "no_context":
                        ctx = ""
                    elif strategy == "old_format":
                        ctx = _build_old_format_context(sd["fv"], label)
                    else:
                        ctx = _build_radiology_context(sd["fv"], label)

                    pos_text = ctx + BIOMEDCLIP_POSITIVE_PROMPTS[label]
                    neg_text = ctx + BIOMEDCLIP_NEGATIVE_PROMPTS[label]

                    with torch.no_grad():
                        tok   = biomedclip_tokenizer([pos_text, neg_text]).to(device)
                        txt_f = biomedclip_model.encode_text(tok)
                        txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
                        sims  = (sd["img_f"] @ txt_f.T).squeeze(0)
                    scores.append(sims[0].item() - sims[1].item())
                except Exception:
                    scores.append(0.0)

            y_true = np.array([sd["y_true"] for sd in sample_data])
            scores = np.array(scores)
            try:
                auc = roc_auc_score(y_true, scores) if len(np.unique(y_true)) > 1 else 0.5
            except Exception:
                auc = 0.5
            results[label][strategy] = round(auc, 3)

        best_auc = max(results[label].values()) if results[label] else 0
        print(f"    {'Strategy':<22} {'AUC':>7}")
        print(f"    {'─'*32}")
        for strat, auc in results[label].items():
            marker = " ← best" if auc == best_auc else ""
            print(f"    {strat:<22} {auc:>7.3f}{marker}")

    return results


print("=" * 70)
print("  CONTEXT FORMAT ABLATION — BioMedCLIP (Zero-Shot)")
print("=" * 70)
_BMC_ABL_RESULTS = _run_bmc_context_ablation()

# Auto-update BMC_CONTEXT_STRATEGY
print("\n" + "=" * 70)
print("  UPDATING BMC_CONTEXT_STRATEGY FROM ABLATION RESULTS")
print("=" * 70)
for label in TARGET_LABELS:
    if _BMC_ABL_RESULTS.get(label):
        winner = max(_BMC_ABL_RESULTS[label], key=_BMC_ABL_RESULTS[label].get)
        BMC_CONTEXT_STRATEGY[label] = winner
        print(f"  BMC [{label}]: winner = {winner} "
              f"(AUC={_BMC_ABL_RESULTS[label][winner]:.3f})")

print(f"\n✅ BMC_CONTEXT_STRATEGY = {BMC_CONTEXT_STRATEGY}")
print("\n⚠️  Run threshold calibration next — strategy affects score distributions.")

# 5. BioMedCLIP Threshold Calibration

In [ ]:
# ================================================================
# BioMedCLIP Threshold Calibration — with feature context
#
# Uses build_bmc_context() per sample (same as inference).
# Uncertain (-1) values handled by the strategy:
#   radiology_format → hedging sentences for relevant features
#   old_format       → silently skipped
#   no_context       → no context at all
# ================================================================
from sklearn.metrics import roc_curve, balanced_accuracy_score


def calibrate_biomedclip_threshold(label, n_samples=1000,
                                    model=None, preprocess=None, tokenizer=None):
    """
    Calibrate BioMedCLIP zero-shot threshold on train_df certain (0/1) samples.
    Uses build_bmc_context() per sample for train/test consistency.

    model/preprocess/tokenizer: if None, uses the globals
    biomedclip_model / biomedclip_preprocess / biomedclip_tokenizer.
    Pass ft1_model etc. when calibrating the fine-tuned model.
    """
    _model      = model      if model      is not None else biomedclip_model
    _preprocess = preprocess if preprocess is not None else biomedclip_preprocess
    _tokenizer  = tokenizer  if tokenizer  is not None else biomedclip_tokenizer

    df = train_df[train_df[label].isin([0, 1])].copy()
    n_each = min(n_samples // 2, (df[label] == 1).sum(), (df[label] == 0).sum())

    if n_each < 10:
        print(f"  ⚠️  Not enough images for {label}, using 0.0")
        return 0.0

    pos    = df[df[label] == 1].sample(n_each, random_state=42).reset_index(drop=True)
    neg    = df[df[label] == 0].sample(n_each, random_state=42).reset_index(drop=True)
    df_cal = pd.concat([pos, neg], ignore_index=True).sample(
        frac=1, random_state=42).reset_index(drop=True)

    print(f"  {label}: calibrating on {len(df_cal)} samples (pos={n_each}, neg={n_each})")

    scores, true_labels = [], []
    top_feats = top_features_per_label.get(label, [])

    for _, row in tqdm(df_cal.iterrows(), total=len(df_cal),
                       desc=f"  Calibrating {label}", leave=False):
        try:
            clean    = (row["Path"]
                        .replace("CheXpert-v1.0-small/", "")
                        .replace("CheXpert-v1.0/", ""))
            img_path = os.path.join(BASE_PATH, clean)

            image = _preprocess(
                Image.open(img_path).convert("RGB")
            ).unsqueeze(0).to(device)

            # Per-sample feature context — train_df row uses "Path" as key
            fv  = get_sample_features(row["Path"], label, top_feats, train_df)
            ctx = build_bmc_context(label, fv)

            pos_text = ctx + BIOMEDCLIP_POSITIVE_PROMPTS[label]
            neg_text = ctx + BIOMEDCLIP_NEGATIVE_PROMPTS[label]

            texts = _tokenizer([pos_text, neg_text]).to(device)

            with torch.no_grad():
                img_f = _model.encode_image(image)
                txt_f = _model.encode_text(texts)
                img_f = img_f / img_f.norm(dim=-1, keepdim=True)
                txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
                sims  = (img_f @ txt_f.T).squeeze(0)

            scores.append(sims[0].item())
            true_labels.append(int(row[label]))

        except Exception as e:
            continue

    if len(scores) < 40:
        print(f"  ⚠️  Too few valid scores for {label}, using 0.0")
        return 0.0

    scores_arr = np.array(scores)
    labels_arr = np.array(true_labels)

    pos_mean = scores_arr[labels_arr == 1].mean()
    neg_mean = scores_arr[labels_arr == 0].mean()
    gap      = pos_mean - neg_mean

    print(f"  Score stats: min={scores_arr.min():.4f}, max={scores_arr.max():.4f}, "
          f"mean={scores_arr.mean():.4f}")
    print(f"  Score for positives : mean={pos_mean:.4f}")
    print(f"  Score for negatives : mean={neg_mean:.4f}")
    print(f"  Gap                 : {gap:.4f}  "
          f"{'✅ good' if gap > 0.03 else '⚠️ small'}")

    # Half-split cross-validated threshold selection (same as WXRC / CheXZero)
    idx        = np.random.RandomState(42).permutation(len(scores_arr))
    mid        = len(idx) // 2
    s_tr, l_tr = scores_arr[idx[:mid]], labels_arr[idx[:mid]]
    s_val, l_val = scores_arr[idx[mid:]], labels_arr[idx[mid:]]

    fpr, tpr, thresholds_roc = roc_curve(l_tr, s_tr)
    bal_acc    = (tpr + (1 - fpr)) / 2
    thresh_bal = float(thresholds_roc[np.argmax(bal_acc)])
    thresh_mid = float((s_tr[l_tr == 1].mean() + s_tr[l_tr == 0].mean()) / 2)
    thresh_med = float(np.median(s_tr[l_tr == 1]))

    candidates = {k: v for k, v in {
        "balanced_acc": thresh_bal,
        "midpoint":     thresh_mid,
        "median_pos":   thresh_med,
    }.items() if not np.isinf(v) and not np.isnan(v)}

    print(f"  Threshold candidates (val half):")
    best_name, best_thresh, best_score = None, None, -1
    for name, thresh in candidates.items():
        preds = (s_val > thresh).astype(int)
        score = balanced_accuracy_score(l_val, preds)
        print(f"    {name:15s}: {thresh:.4f} → val bal_acc={score:.3f}")
        if score > best_score:
            best_score, best_thresh, best_name = score, thresh, name

    print(f"  ✅ Chosen: {best_name} = {best_thresh:.4f} "
          f"(val balanced acc = {best_score:.3f})")
    return best_thresh


print("🔧 Calibrating BioMedCLIP thresholds from train_df...")
print("=" * 60)
BIOMEDCLIP_THRESHOLDS = {}
for label in TARGET_LABELS:
    BIOMEDCLIP_THRESHOLDS[label] = calibrate_biomedclip_threshold(
        label, n_samples=1000)

print("\n📊 BioMedCLIP Final Thresholds:")
for label, thresh in BIOMEDCLIP_THRESHOLDS.items():
    print(f"  {label}: {thresh:.4f}")

# 6. BioMedCLIP Inference Function

In [ ]:
# ================================================================
# BioMedCLIP Zero-Shot Inference — with feature context
#
# Uses build_bmc_context() which routes to old_format or
# radiology_format per BMC_CONTEXT_STRATEGY.
# Uncertain (-1) values:
#   old_format       → silently skipped (no noise)
#   radiology_format → hedging language for relevant features
# ================================================================

def infer_biomedclip(img_path: str, label: str,
                     feature_values: list = None,
                     model=None, preprocess=None, tokenizer=None,
                     threshold_dict=None) -> int:
    """
    Zero-shot BioMedCLIP inference with per-label feature context.

    model / preprocess / tokenizer: pass original_model or ft1_model etc.
        If None, falls back to globals biomedclip_model / _preprocess / _tokenizer.
    threshold_dict: dict of {label: thresh}. If None, uses BIOMEDCLIP_THRESHOLDS.
    """
    _model      = model          if model          is not None else biomedclip_model
    _preprocess = preprocess     if preprocess     is not None else biomedclip_preprocess
    _tokenizer  = tokenizer      if tokenizer      is not None else biomedclip_tokenizer
    _thresholds = threshold_dict if threshold_dict is not None else BIOMEDCLIP_THRESHOLDS

    try:
        ctx      = build_bmc_context(label, feature_values or [])
        pos_text = ctx + BIOMEDCLIP_POSITIVE_PROMPTS[label]
        neg_text = ctx + BIOMEDCLIP_NEGATIVE_PROMPTS[label]

        image = _preprocess(
            Image.open(img_path).convert("RGB")
        ).unsqueeze(0).to(device)
        texts = _tokenizer([pos_text, neg_text]).to(device)

        with torch.no_grad():
            img_f = _model.encode_image(image)
            txt_f = _model.encode_text(texts)
            img_f = img_f / img_f.norm(dim=-1, keepdim=True)
            txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
            sims  = (img_f @ txt_f.T).squeeze(0)

        threshold = _thresholds.get(label, 0.0)
        return 1 if sims[0].item() > threshold else 0

    except Exception as e:
        print(f"  [infer_biomedclip] Error — {label}: {e}")
        return 0

# 7. BioMedCLIP Benchmark Definition

In [ ]:
# ================================================================
# BioMedCLIP Benchmark — mirrors benchmark_uncertain_relabeling
# Calls infer_biomedclip which uses build_bmc_context per sample
# ================================================================

def benchmark_biomedclip(model=None, preprocess=None, tokenizer=None,
                          threshold_dict=None):
    """
    Run BioMedCLIP zero-shot benchmark on GT-uncertain samples.
    Pass model/preprocess/tokenizer to use a different model variant.
    """
    all_results = []
    labeler_df  = rule_labeler_df.copy()
    gt_df       = ground_truth_df.copy()

    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True)
                break

    for label in TARGET_LABELS:
        print(f"\n{'='*60}")
        print(f"  Label: {label}")
        print(f"{'='*60}")

        if label not in labeler_df.columns or label not in gt_df.columns:
            print(f"  ⚠️  '{label}' column missing — skipping")
            continue

        uncertain_idx = labeler_df[labeler_df[label] == -1].index
        if len(uncertain_idx) == 0:
            print(f"  ℹ️  No uncertain samples — skipping")
            continue

        gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
        gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
        gt_for_uncertain = gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]

        print(f"  Uncertain samples : {len(uncertain_idx)} out of {len(labeler_df)}")
        print(f"  With valid GT     : {len(gt_for_uncertain)} "
              f"(pos={int(gt_for_uncertain['gt'].sum())}, "
              f"neg={int((gt_for_uncertain['gt']==0).sum())})")

        top_features = top_features_per_label.get(label, [])

        for _, row in tqdm(gt_for_uncertain.iterrows(),
                           total=len(gt_for_uncertain),
                           desc=f"BioMedCLIP · {label}"):
            try:
                img_path       = fix_test_path(row["Study"])
                feature_values = get_sample_features(
                    row["Study"], label, top_features, rule_labeler_df
                )
                pred = infer_biomedclip(
                    img_path, label, feature_values,
                    model=model, preprocess=preprocess,
                    tokenizer=tokenizer, threshold_dict=threshold_dict
                )
                all_results.append({
                    "label":           label,
                    "study":           row["Study"],
                    "y_true":          int(row["gt"]),
                    "y_pred":          pred,
                    "feature_context": " | ".join(feature_values),
                })
            except Exception as e:
                all_results.append({
                    "label":           label,
                    "study":           row["Study"],
                    "y_true":          int(row["gt"]) if pd.notna(row.get("gt")) else -1,
                    "y_pred":          0,
                    "feature_context": f"ERROR: {e}",
                })

    return pd.DataFrame(all_results)

# 8. Run Original BioMedCLIP Benchmark

In [ ]:
print("🚀 Running ORIGINAL BioMedCLIP benchmark (zero-shot baseline)...")
print("=" * 80)
print("\n📊 BioMedCLIP Zero-Shot Thresholds:")
for label, thresh in BIOMEDCLIP_THRESHOLDS.items():
    print(f"  {label:<35} {thresh:.4f}  [strategy={BMC_CONTEXT_STRATEGY[label]}]")

biomedclip_results = benchmark_biomedclip()
biomedclip_results.to_csv("biomedclip_original_results.csv", index=False)
print("✅ Saved: biomedclip_original_results.csv")

biomedclip_metrics = evaluate_results_with_details(biomedclip_results)
print_detailed_results(biomedclip_metrics)

print("\n" + "=" * 80)
print("📈 ORIGINAL BioMedCLIP SUMMARY")
print("=" * 80)
if len(biomedclip_metrics) > 0:
    summary = biomedclip_metrics[[
        "label", "n_samples", "accuracy", "f1_score", "auc",
        "true_negative", "false_positive", "false_negative", "true_positive"
    ]]
    print(summary.to_string(index=False))

    print("\nBalanced Accuracy per label:")
    for _, row in biomedclip_metrics.iterrows():
        sens = row["true_positive"] / (row["true_positive"] + row["false_negative"]) \
               if (row["true_positive"] + row["false_negative"]) > 0 else 0
        spec = row["true_negative"] / (row["true_negative"] + row["false_positive"]) \
               if (row["true_negative"] + row["false_positive"]) > 0 else 0
        print(f"  {row['label']}: Sens={sens:.3f}, Spec={spec:.3f}, "
              f"Balanced Acc={(sens+spec)/2:.3f}")

print("\n" + "=" * 80)
print("✅ Baseline stored in 'biomedclip_metrics'")
print("=" * 80)

# FT-1 : LoRA layer + helper to inject into BioMedCLIP

In [ ]:
# ================================================================
# CELL FT-1 : LoRA layer + helper to inject into BioMedCLIP
# ================================================================
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from tqdm import tqdm
import open_clip
 
device = "cuda" if torch.cuda.is_available() else "cpu"
 
# ── LoRA Linear wrapper ──────────────────────────────────────────
class LoRALinear(nn.Module):
    def __init__(self, linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.linear = linear
        self.scaling = alpha / r
        din, dout = linear.in_features, linear.out_features
        self.loraA = nn.Parameter(torch.randn(r, din) * 0.01)
        self.loraB = nn.Parameter(torch.zeros(dout, r))
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Move LoRA params to input device
        loraA = self.loraA.to(x.device)
        loraB = self.loraB.to(x.device)
        return (self.linear(x) + 
                (self.dropout(x) @ loraA.T @ loraB.T) * self.scaling)

def injectlora(model, r=16, alpha=32, dropout=0.05):
    device = next(model.parameters()).device  # Get model device
    # Freeze everything
    for p in model.parameters(): 
        p.requires_grad = False
    
    loraparams = []
    replacemap = {}
    
    # Target visual encoder only
    for name, module in model.named_modules():
        if not name.startswith('visual'): continue
        if not isinstance(module, nn.Linear): continue
        if module.in_features <= 64 or module.out_features <= 64: continue
            
        loralayer = LoRALinear(module, r, alpha, dropout)
        loralayer.to(device)  # Move entire layer to GPU
        replacemap[name] = loralayer
        loraparams.extend([p for p in loralayer.parameters() 
                          if p.requires_grad])
    
    # Apply replacements
    for name, loralayer in replacemap.items():
        parts = name.split('.')
        parent = model
        for p in parts[:-1]: 
            parent = getattr(parent, p)
        setattr(parent, parts[-1], loralayer)
    
    ntrainable = sum(p.numel() for p in loraparams)
    ntotal = sum(p.numel() for p in model.parameters())
    print(f"LoRA injected: {len(replacemap)} layers replaced")
    print(f"Trainable params: {ntrainable:,} / {ntotal:,} ({100*ntrainable/ntotal:.2f}%)")
    return model, loraparams
 
 
# ── InfoNCE / CLIP contrastive loss ─────────────────────────────
class CLIPLoss(nn.Module):
    def __init__(self, init_temp=0.07):
        super().__init__()
        # log-temperature (learnable)
        self.log_temp = nn.Parameter(torch.tensor(init_temp).log())
 
    def forward(self, img_f, txt_f):
        img_f = F.normalize(img_f, dim=-1)
        txt_f = F.normalize(txt_f, dim=-1)
        temp  = self.log_temp.exp().clamp(min=0.01, max=0.5)
        logits = (img_f @ txt_f.T) / temp
        n      = logits.shape[0]
        labels = torch.arange(n, device=logits.device)
        loss   = (F.cross_entropy(logits, labels) +
                  F.cross_entropy(logits.T, labels)) / 2
        return loss
 
 
print("✅ LoRA utilities defined")

# FT-2 : Kaggle dataset paths

In [ ]:
# ================================================================
# NEW CELL FT-2 : Paths + CSV parser for MIMIC aug format
# (REPLACES Cell 49)
# ================================================================
 
import os, re, ast, random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import open_clip
 
device = "cuda" if torch.cuda.is_available() else "cpu"
 
# ── Root paths ───────────────────────────────────────────────────
MIMIC_ROOT      = "/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset"
MIMIC_IMG_ROOT  = f"{MIMIC_ROOT}/official_data_iccv_final"
MIMIC_TRAIN_CSV = f"{MIMIC_ROOT}/mimic_cxr_aug_train.csv"
MIMIC_VAL_CSV   = f"{MIMIC_ROOT}/mimic_cxr_aug_validate.csv"
 
# Target labels
FINETUNE_LABELS = [
    "Edema",
    "Atelectasis",
    "Pleural Effusion",
    "Enlarged Cardiomediastinum",
    "Consolidation",
]
 
# ── Parse stringified Python lists from CSV cells ────────────────
def parse_list_cell(val):
    """
    The CSV stores lists as strings like:
      "['files/p10/.../img.jpg', 'files/p11/.../img2.jpg']"
    This safely parses them back to Python lists.
    Returns [] on failure.
    """
    if pd.isna(val):
        return []
    try:
        result = ast.literal_eval(str(val))
        return result if isinstance(result, list) else [result]
    except Exception:
        # Fallback: strip brackets and split on comma+quote
        s = str(val).strip("[]").replace("'", "").replace('"', "")
        items = [x.strip() for x in s.split(",") if x.strip()]
        return items
 
 
def get_frontal_paths(row):
    """
    Return list of absolute paths for frontal (PA then AP) images
    in this row. PA is preferred over AP for CheXpert alignment.
    """
    paths = []
    for col in ["PA", "AP"]:
        for rel in parse_list_cell(row.get(col, [])):
            rel = str(rel).strip()
            if not rel:
                continue
            # rel is like "files/p10/p10000032/s50414267/02aa804e-....jpg"
            abs_path = os.path.join(MIMIC_IMG_ROOT, rel)
            if os.path.exists(abs_path):
                paths.append(abs_path)
    return paths
 
 
def get_report_texts(row):
    """
    Return list of report strings from the 'text' column.
    We use the original text (not augmented) for accurate
    report-based label extraction.
    """
    return parse_list_cell(row.get("text", []))
 
 
def get_augmented_texts(row):
    """
    Return augmented report texts — useful for Stage 1 to get
    more diverse text supervision.
    """
    return parse_list_cell(row.get("text_augment", []))
 
 
# ── Quick sanity check ───────────────────────────────────────────
_df   = pd.read_csv(MIMIC_TRAIN_CSV, nrows=3)
_row  = _df.iloc[0]
_imgs = get_frontal_paths(_row)
_txts = get_report_texts(_row)
 
print(f"Frontal paths found for row 0: {len(_imgs)}")
for p in _imgs[:2]:
    print(f"  {p}  exists={os.path.exists(p)}")
 
print(f"\nReport texts found for row 0: {len(_txts)}")
for t in _txts[:1]:
    print(f"  {str(t)[:200]}")
 
print("\n✅ MIMIC aug CSV paths configured")
 

# FT-3 : MIMIC image path resolver + report parser

In [ ]:
# ================================================================
# CELL FT-3 : MIMIC image path resolver + report parser
# ================================================================
 
def mimic_img_path(row):
    """
    Build absolute image path from a MIMIC-CXR-JPG split CSV row.
    Layout: files/p{sub[:2]}/p{subject_id}/s{study_id}/{dicom_id}.jpg
    """
    sub   = str(int(row["subject_id"]))
    study = str(int(row["study_id"]))
    dicom = str(row["dicom_id"])
    return os.path.join(
        MIMIC_IMG_ROOT,
        f"p{sub[:2]}", f"p{sub}", f"s{study}",
        f"{dicom}.jpg"
    )
 
 
def parse_report_from_jpg_tree(img_path):
    """
    MIMIC-CXR-JPG stores reports as s{study_id}.txt in the same
    folder as the images. Try to read it and extract the
    IMPRESSION or FINDINGS section.
    """
    report_path = os.path.splitext(img_path)[0]
    # The report is one level up from the image: s{study_id}.txt
    study_dir   = os.path.dirname(img_path)
    study_id    = os.path.basename(study_dir)          # e.g. s12345678
    report_txt  = os.path.join(study_dir, f"{study_id}.txt")
 
    if not os.path.exists(report_txt):
        return None
 
    try:
        with open(report_txt) as f:
            text = f.read()
    except Exception:
        return None
 
    for section in ["IMPRESSION:", "FINDINGS:", "CONCLUSION:"]:
        idx = text.upper().find(section)
        if idx != -1:
            snippet = text[idx + len(section):].strip()
            snippet = re.split(r'\n[A-Z ]{3,}:', snippet)[0].strip()
            snippet = re.sub(r'\s+', ' ', snippet)
            if len(snippet) > 15:
                return snippet[:350]
    return None
 
 
print("✅ MIMIC path/report utilities defined")

# FT-4 : Stage-1 Dataset — image + raw report (general CXR)

In [ ]:
# ================================================================
# NEW CELL FT-4 : Stage-1 Dataset — frontal image + report text
# (REPLACES Cell 53)
#
# General CXR domain adaptation.
# Each sample = (image_tensor, report_text_string).
# Uses ALL pathologies — broad exposure to CXR language.
# For each row we use the PA/AP image + original report text.
# We also add augmented texts as separate samples to double
# the effective dataset size cheaply.
# ================================================================
 
class Stage1Dataset(Dataset):
    """
    Stage 1: General CXR domain adaptation.
    Image: frontal (PA preferred, AP fallback) from MIMIC aug CSV.
    Text:  original report + augmented report (both used).
    Goal:  teach the image encoder CXR-specific visual features
           through broad image-text contrastive alignment.
    """
    def __init__(self, preprocess, split="train", max_samples=60000):
        self.preprocess = preprocess
        self.samples    = []   # (img_path, text_str)
 
        csv_path = MIMIC_TRAIN_CSV if split == "train" else MIMIC_VAL_CSV
        df = pd.read_csv(csv_path)
        print(f"  {split}: {len(df)} rows in CSV")
 
        # Shuffle for diversity up to max_samples
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)
 
        for _, row in tqdm(df.iterrows(), total=len(df),
                           desc=f"Stage-1 {split}", leave=False):
            img_paths = get_frontal_paths(row)
            if not img_paths:
                continue
 
            # Use first available frontal image
            img_path  = img_paths[0]
 
            orig_texts = get_report_texts(row)
            aug_texts  = get_augmented_texts(row)
 
            # Add (image, original_report) pair
            for txt in orig_texts[:1]:   # one original per image
                if txt and len(str(txt)) > 15:
                    self.samples.append((img_path, str(txt)[:400]))
 
            # Add (image, augmented_report) as extra supervision
            for txt in aug_texts[:1]:
                if txt and len(str(txt)) > 15:
                    self.samples.append((img_path, str(txt)[:400]))
 
            if len(self.samples) >= max_samples:
                break
 
        random.shuffle(self.samples)
        print(f"  ✅ Stage-1 {split}: {len(self.samples):,} samples")
 
    def __len__(self):  return len(self.samples)
 
    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        image = self.preprocess(Image.open(img_path).convert("RGB"))
        return image, text
 
 
print("✅ Stage-1 Dataset (MIMIC aug) defined")
 

# FT-5 : Stage-2 Dataset — confirmed labels + YOUR prompts

In [ ]:
# ================================================================
# NEW CELL FT-5 : Stage-2 Dataset — prompt-aligned confirmed labels
# (REPLACES Cell 55)
#
# For each of the 5 target labels, find MIMIC rows whose report
# text CLEARLY confirms presence (positive) or absence (negative).
# Pair the frontal image with YOUR BIOMEDCLIP_POSITIVE/NEGATIVE_PROMPTS.
# This closes the train/inference gap: the model trains with the
# exact prompts it will see at inference time on CheXpert.
# ================================================================
 
# Report patterns for CONFIRMED (unambiguous) findings
# These are stricter than Stage-3 patterns — we want clean labels here
CONFIRMED_POSITIVE = {
    "Edema": [
        r"(moderate|severe|mild)\s+pulmonary\s+edema",
        r"pulmonary\s+edema\s+(is\s+)?(present|seen|noted|increased|worsened)",
        r"(bilateral|interstitial)\s+(pulmonary\s+)?edema",
        r"(consistent\s+with|evidence\s+of)\s+(pulmonary\s+)?edema",
    ],
    "Atelectasis": [
        # Original patterns
        r"(bibasilar|basilar|lower\s+lobe|subsegmental|lobar)\s+atelectasis",
        r"atelectasis\s+(is\s+)?(present|seen|noted|identified|increased)",
        r"(new|worsened|persistent)\s+atelectasis",
        r"(evidence\s+of|consistent\s+with)\s+atelectasis",
        # ADD — these are extremely common in MIMIC reports
        r"(mild|moderate|severe)\s+atelectasis",
        r"(left|right)\s+(lower|upper|middle)\s+lobe\s+atelectasis",
        r"(plate[\-\s]like|discoid|linear)\s+atelectasis",
        r"atelectatic\s+(changes|collapse|process)",
        r"(bibasilar|basilar)\s+(atelectasis|opacities\s+likely\s+representing\s+atelectasis)",
        r"(volume\s+loss|collapse).{0,30}atelectasis",
        r"atelectasis.{0,30}(volume\s+loss|collapse)",
        r"(patchy|streaky)\s+atelectasis",
        r"(right|left|bilateral)\s+atelectasis",
        r"atelectasis\s+(at|of|in|involving)\s+the",
        r"(lower\s+lobe|lung\s+base).{0,20}atelectasis",
        r"atelectasis.{0,20}(lower\s+lobe|lung\s+base)",
    ],
    "Pleural Effusion": [
        r"(moderate|large|bilateral|left|right)\s+(pleural\s+)?effusion",
        r"pleural\s+effusion\s+(is\s+)?(present|seen|noted|increased)",
        r"(layering|free)\s+(pleural\s+)?fluid",
        r"(evidence\s+of|consistent\s+with)\s+(pleural\s+)?effusion",
    ],
    "Enlarged Cardiomediastinum": [
        r"(enlarged|increased|prominent)\s+(cardiac|cardiomediastinal)",
        r"cardiomegaly\s+(is\s+)?(present|seen|noted|increased)",
        r"(widened|enlarged)\s+mediastinum",
        r"cardiothoracic\s+ratio\s+(is\s+)?(increased|enlarged|greater)",
    ],
    "Consolidation": [
        r"(lobar|segmental|airspace)\s+consolidation",
        r"consolidation\s+(is\s+)?(present|seen|noted|identified)",
        r"(dense|new|worsened)\s+consolidation",
        r"(evidence\s+of|consistent\s+with)\s+consolidation",
    ],
}
 
CONFIRMED_NEGATIVE = {
    "Edema": [
        r"no\s+(pulmonary\s+)?edema",
        r"(pulmonary\s+)?edema\s+(is\s+)?(absent|resolved|cleared|improved)",
        r"no\s+evidence\s+of\s+(pulmonary\s+)?edema",
        r"clear\s+lung\s+fields?\s+without\s+edema",
    ],
    "Atelectasis": [
        r"no\s+(acute\s+)?atelectasis",
        r"atelectasis\s+(is\s+)?(absent|resolved|cleared|improved)",
        r"no\s+evidence\s+of\s+atelectasis",
        r"lungs?\s+(are\s+)?clear\s+(bilaterally|without\s+atelectasis)",
        # ADD
        r"no\s+(bibasilar|basilar|subsegmental|lobar)\s+atelectasis",
        r"(lungs?\s+are\s+)?(fully|well|adequately)\s+expanded",
        r"no\s+(new|acute|significant)\s+atelectasis",
        r"(previously\s+seen\s+)?atelectasis\s+(has\s+)?(resolved|cleared|improved|decreased)",
        r"(clear|clear\s+of)\s+(atelectasis|atelectatic\s+changes)",
    ],
    "Pleural Effusion": [
        r"no\s+(pleural\s+)?effusion",
        r"(pleural\s+)?effusion\s+(is\s+)?(absent|resolved|drained)",
        r"no\s+evidence\s+of\s+(pleural\s+)?effusion",
    ],
    "Enlarged Cardiomediastinum": [
        r"normal\s+(cardiac|cardiomediastinal)\s+(size|silhouette|contour)",
        r"heart\s+(size\s+is\s+)?(normal|stable|unchanged)",
        r"no\s+(cardiomegaly|mediastinal\s+widening)",
        r"cardiothoracic\s+ratio\s+(is\s+)?(normal|within\s+normal)",
    ],
    "Consolidation": [
        r"no\s+(focal\s+)?consolidation",
        r"consolidation\s+(is\s+)?(absent|resolved|cleared)",
        r"no\s+evidence\s+of\s+consolidation",
        r"(clear|no\s+airspace)\s+(opacity|consolidation|opacification)",
    ],
}
 
 
def text_confirms_positive(text, label):
    """Returns True if text clearly confirms presence of label."""
    t = text.lower()
    for pattern in CONFIRMED_POSITIVE.get(label, []):
        if re.search(pattern, t):
            return True
    return False
 
 
def text_confirms_negative(text, label):
    """Returns True if text clearly confirms absence of label."""
    t = text.lower()
    for pattern in CONFIRMED_NEGATIVE.get(label, []):
        if re.search(pattern, t):
            return True
    return False
 
 
class Stage2Dataset(Dataset):
    """
    Stage 2: Prompt-aligned fine-tuning.
    Scans MIMIC report texts to find confirmed positives/negatives
    for each of the 5 target labels, then pairs the frontal image
    with YOUR exact inference prompts.
    """
    def __init__(self, preprocess, pos_prompts, neg_prompts,
                 max_per_label=3000):
        self.preprocess = preprocess
        self.samples    = []   # (img_path, prompt_text, binary_label)
 
        df = pd.read_csv(MIMIC_TRAIN_CSV)
        print(f"  Scanning {len(df)} MIMIC rows for confirmed labels...")
 
        for label in FINETUNE_LABELS:
            pos_samples, neg_samples = [], []
 
            for _, row in df.iterrows():
                img_paths = get_frontal_paths(row)
                if not img_paths:
                    continue
 
                img_path   = img_paths[0]
                all_texts  = get_report_texts(row)
                combined   = " ".join(str(t) for t in all_texts).lower()
 
                if (len(pos_samples) < max_per_label and
                        text_confirms_positive(combined, label)):
                    pos_samples.append(img_path)
 
                elif (len(neg_samples) < max_per_label and
                          text_confirms_negative(combined, label)):
                    neg_samples.append(img_path)
 
                if (len(pos_samples) >= max_per_label and
                        len(neg_samples) >= max_per_label):
                    break
 
            # Balance pos/neg
            n = min(len(pos_samples), len(neg_samples))
            if n < 50:
                print(f"  ⚠️  {label}: only {n} balanced pairs — skipping")
                continue
 
            for p in pos_samples[:n]:
                self.samples.append((p, pos_prompts[label], 1))
            for p in neg_samples[:n]:
                self.samples.append((p, neg_prompts[label], 0))
 
            print(f"  Stage-2 {label}: {n} pos + {n} neg = {n*2} samples")
 
        random.shuffle(self.samples)
        print(f"  ✅ Stage-2 total: {len(self.samples):,} samples")
 
    def __len__(self): return len(self.samples)
 
    def __getitem__(self, idx):
        img_path, text, _ = self.samples[idx]
        image = self.preprocess(Image.open(img_path).convert("RGB"))
        return image, text
 
 
print("✅ Stage-2 Dataset (MIMIC aug, prompt-aligned) defined")
 
 


# FT-6 : Stage-3 Dataset — uncertainty-focused fine-tuning

In [ ]:
# ================================================================
# NEW CELL FT-6 : Stage-3 Dataset — uncertainty-focused
# (REPLACES Cell 57 entirely)
#
# Find MIMIC rows whose report language is AMBIGUOUS/UNCERTAIN —
# the same kind of borderline language that causes CheXpert's
# NLP labeler to assign -1 (uncertain).
# We resolve these using slightly softer regex patterns,
# then pair with YOUR prompts.
# These are the most valuable training samples for your task.
# ================================================================
 
# Uncertainty markers — language that a radiologist uses when unsure
# (same patterns that cause CheXpert NLP labeler to assign -1)
UNCERTAINTY_MARKERS = [
    r"cannot\s+(be\s+)?(excluded?|ruled\s+out)",
    r"(may|might|could)\s+(represent|reflect|be|suggest)",
    r"(possible|possible\s+mild|probable|likely)\s+",
    r"(question(able)?|suspect(ed)?)\s+",
    r"(not\s+excluded?|not\s+entirely\s+excluded?)",
    r"(if\s+clinical(ly)?\s+concerned|clinical\s+correlation)",
    r"(difficult\s+to|hard\s+to)\s+(exclude?|assess|evaluate)",
    r"(subtle|early|minimal|trace)\s+",
]
 
# Softer resolution patterns for uncertain text
# (less strict than Stage-2 confirmed patterns)
UNCERTAIN_POSITIVE_SOFT = {
    "Edema": [
        r"(possible|probable|likely|mild)\s+(pulmonary\s+)?edema",
        r"(cannot\s+exclude|not\s+excluded)\s+(pulmonary\s+)?edema",
        r"(pulmonary\s+)?edema\s+(cannot\s+be\s+excluded|is\s+possible)",
        r"(vascular\s+congestion|interstitial\s+(prominence|edema))",
    ],
    "Atelectasis": [
        r"(possible|probable|subtle|minimal)\s+atelectasis",
        r"(cannot\s+exclude|not\s+excluded)\s+atelectasis",
        r"(subsegmental|linear)\s+(atelectasis|opacity)",
        r"(plate[\-\s]like|discoid)\s+atelectasis",
    ],
    "Pleural Effusion": [
        r"(possible|probable|small|trace)\s+(pleural\s+)?effusion",
        r"(cannot\s+exclude|not\s+excluded)\s+(pleural\s+)?effusion",
        r"(blunting|blunted)\s+(of\s+the\s+)?(costophrenic|cp)\s+angle",
        r"(possible|probable)\s+(pleural\s+)?fluid",
    ],
    "Enlarged Cardiomediastinum": [
        r"(borderline|mildly|possibly)\s+(enlarged|prominent)\s+(cardiac|heart)",
        r"(cannot\s+exclude|not\s+excluded)\s+cardiomegaly",
        r"heart\s+(may\s+be|appears?)\s+(mildly\s+)?(enlarged|prominent)",
        r"(upper\s+limits?\s+of\s+normal|borderline)\s+cardiac",
    ],
    "Consolidation": [
        r"(possible|probable|cannot\s+exclude)\s+consolidation",
        r"(airspace|alveolar)\s+(opacity|opacification|disease)",
        r"(patchy|focal)\s+(opacity|opacification)",
        r"(infiltrate|infiltration)\s+(that\s+may|could|possibly)",
    ],
}
 
UNCERTAIN_NEGATIVE_SOFT = {
    "Edema": [
        r"(no\s+definite|no\s+convincing|no\s+obvious)\s+(pulmonary\s+)?edema",
        r"(lungs?\s+are\s+)?(relatively\s+)?clear",
        r"no\s+(significant|substantial)\s+(pulmonary\s+)?edema",
    ],
    "Atelectasis": [
        r"(no\s+definite|no\s+convincing)\s+atelectasis",
        r"(lungs?\s+are\s+)?(relatively\s+)?clear",
        r"(no\s+significant|without\s+significant)\s+atelectasis",
    ],
    "Pleural Effusion": [
        r"(no\s+definite|no\s+convincing|no\s+significant)\s+(pleural\s+)?effusion",
        r"(costophrenic\s+angles?\s+are\s+)?(relatively\s+)?sharp",
        r"(no\s+large|no\s+significant)\s+(pleural\s+)?fluid",
    ],
    "Enlarged Cardiomediastinum": [
        r"(heart\s+size\s+is\s+)?(upper\s+limits?\s+of\s+normal|borderline\s+normal)",
        r"(no\s+definite|no\s+significant)\s+cardiomegaly",
        r"(cardiac\s+silhouette\s+is\s+)?(relatively\s+)?normal",
    ],
    "Consolidation": [
        r"(no\s+definite|no\s+convincing|no\s+focal)\s+consolidation",
        r"(no\s+significant|without\s+significant)\s+(airspace|alveolar)\s+(disease|opacity)",
        r"(lungs?\s+are\s+)?(relatively\s+)?clear\s+without\s+(focal\s+)?consolidation",
    ],
}
 
 
def text_is_uncertain(text):
    """Returns True if the report contains uncertainty language."""
    t = text.lower()
    return any(re.search(p, t) for p in UNCERTAINTY_MARKERS)
 
 
def resolve_uncertain_soft(text, label):
    """
    Returns 1 (positive), 0 (negative), or None (unresolvable).
    Uses softer patterns appropriate for uncertain report language.
    """
    t = text.lower()
 
    # Positive resolution
    for pattern in UNCERTAIN_POSITIVE_SOFT.get(label, []):
        if re.search(pattern, t):
            return 1
 
    # Negative resolution
    for pattern in UNCERTAIN_NEGATIVE_SOFT.get(label, []):
        if re.search(pattern, t):
            return 0
 
    return None
 
 
class Stage3Dataset(Dataset):
    """
    Stage 3: Uncertainty-focused fine-tuning.
    Finds MIMIC rows whose report contains AMBIGUOUS language
    (same uncertainty markers that cause CheXpert NLP to assign -1),
    resolves the finding using soft patterns,
    and pairs with YOUR inference prompts.
    These samples are structurally identical to the CheXpert
    uncertain cases you will relabel.
    """
    def __init__(self, preprocess, pos_prompts, neg_prompts,
                 max_per_label=800):
        self.preprocess = preprocess
        self.samples    = []
 
        df = pd.read_csv(MIMIC_TRAIN_CSV)
        print(f"  Scanning {len(df)} MIMIC rows for uncertain cases...")
 
        for label in FINETUNE_LABELS:
            resolved_pos, resolved_neg = [], []
 
            for _, row in tqdm(df.iterrows(), total=len(df),
                               desc=f"  Stage-3 {label}", leave=False):
                if (len(resolved_pos) >= max_per_label and
                        len(resolved_neg) >= max_per_label):
                    break
 
                img_paths = get_frontal_paths(row)
                if not img_paths:
                    continue
 
                all_texts = get_report_texts(row)
                combined  = " ".join(str(t) for t in all_texts)
 
                # Only process rows with uncertain language
                if not text_is_uncertain(combined):
                    continue
 
                pseudo = resolve_uncertain_soft(combined, label)
                if pseudo is None:
                    continue
 
                img_path = img_paths[0]
                if pseudo == 1 and len(resolved_pos) < max_per_label:
                    resolved_pos.append(img_path)
                elif pseudo == 0 and len(resolved_neg) < max_per_label:
                    resolved_neg.append(img_path)
 
            # Balance
            n = min(len(resolved_pos), len(resolved_neg))
            if n < 20:
                print(f"  ⚠️  {label}: only {n} uncertain pairs resolved — skipping")
                continue
 
            for p in resolved_pos[:n]:
                self.samples.append((p, pos_prompts[label], 1))
            for p in resolved_neg[:n]:
                self.samples.append((p, neg_prompts[label], 0))
 
            print(f"  Stage-3 {label}: {n} pos + {n} neg = {n*2} resolved uncertain samples")
 
        random.shuffle(self.samples)
        print(f"  ✅ Stage-3 total: {len(self.samples):,} samples")
 
    def __len__(self): return len(self.samples)
 
    def __getitem__(self, idx):
        img_path, text, _ = self.samples[idx]
        image = self.preprocess(Image.open(img_path).convert("RGB"))
        return image, text
 
 
print("✅ Stage-3 Dataset (MIMIC aug, uncertainty-focused) defined")
 
 

# FT-7 : Generic training loop (shared across all 3 stages)

In [ ]:
# ================================================================
# CELL FT-7 : Generic training loop (shared across all 3 stages)
# ================================================================
 
def run_training_stage(
    model,
    tokenizer,
    train_dataset,
    lora_params,
    clip_loss,
    stage_name,
    n_epochs,
    lr,
    batch_size  = 32,
    save_path   = None,
    val_dataset = None,
):
    """
    Generic contrastive training loop.
    Returns (model, clip_loss) — same objects, updated in place.
    """
    if len(train_dataset) == 0:
        print(f"  ⚠️  {stage_name}: empty dataset — skipping")
        return model, clip_loss
 
    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True,  num_workers=2,
                              pin_memory=True, drop_last=True)
    val_loader   = None
    if val_dataset and len(val_dataset) > 0:
        val_loader = DataLoader(val_dataset, batch_size=batch_size,
                                shuffle=False, num_workers=2,
                                pin_memory=True)
 
    # Optimise LoRA params + learnable temperature
    optimizer = torch.optim.AdamW(
        lora_params + list(clip_loss.parameters()),
        lr=lr, weight_decay=0.01
    )
    total_steps = n_epochs * len(train_loader)
    warmup      = min(100, total_steps // 10)
    scheduler   = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=total_steps,
        pct_start=warmup / max(total_steps, 1)
    )
 
    best_val   = float("inf")
    print(f"\n{'='*60}")
    print(f"  {stage_name}  |  {len(train_dataset)} samples  "
          f"|  {n_epochs} epochs  |  lr={lr}")
    print(f"{'='*60}")
 
    for epoch in range(1, n_epochs + 1):
        # ── Train ────────────────────────────────────────────────
        model.train()
        train_losses = []
        for images, texts in tqdm(train_loader,
                                  desc=f"  [{stage_name}] Epoch {epoch}/{n_epochs}"):
            images = images.to(device)
            tokens = tokenizer(list(texts)).to(device)
 
            img_f  = model.encode_image(images)
            txt_f  = model.encode_text(tokens)
            loss   = clip_loss(img_f, txt_f)
 
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(lora_params, 1.0)
            optimizer.step()
            scheduler.step()
            train_losses.append(loss.item())
 
        avg_train = np.mean(train_losses)
 
        # ── Validate ─────────────────────────────────────────────
        if val_loader:
            model.eval()
            val_losses = []
            with torch.no_grad():
                for images, texts in val_loader:
                    images = images.to(device)
                    tokens = tokenizer(list(texts)).to(device)
                    img_f  = model.encode_image(images)
                    txt_f  = model.encode_text(tokens)
                    val_losses.append(clip_loss(img_f, txt_f).item())
            avg_val = np.mean(val_losses)
            print(f"    Epoch {epoch}: train={avg_train:.4f}  val={avg_val:.4f}  "
                  f"temp={clip_loss.log_temp.exp().item():.4f}")
            if avg_val < best_val and save_path:
                best_val = avg_val
                _save_lora(model, clip_loss, save_path)
        else:
            print(f"    Epoch {epoch}: train={avg_train:.4f}  "
                  f"temp={clip_loss.log_temp.exp().item():.4f}")
            if save_path:
                _save_lora(model, clip_loss, save_path)
 
    return model, clip_loss
 
 
def _save_lora(model, clip_loss, path):
    """Save only the LoRA (trainable) weights — small file."""
    state = {n: p.detach().cpu()
             for n, p in model.named_parameters()
             if p.requires_grad}
    torch.save({"lora_state":      state,
                "clip_loss_state": clip_loss.state_dict()}, path)
    size_mb = os.path.getsize(path) / 1e6
    print(f"    ✅ Checkpoint saved → {path}  ({size_mb:.1f} MB)")
 
 
print("✅ Training loop defined")

# FT-9 : Load Fine-Tuned Model + Per-Label Routing

In [ ]:
# ================================================================
# Load fine-tuned BioMedCLIP + per-label routing
# Update LABEL_MODEL_MAP after comparing biomedclip_metrics (original
# ZS benchmark above) vs fine-tuned ZS benchmark (run below).
# ================================================================
_BMC_MODEL_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"


def load_finetuned_biomedclip(checkpoint_path):
    model, _, preprocess = open_clip.create_model_and_transforms(_BMC_MODEL_ID)
    model, _             = injectlora(model, r=16, alpha=32, dropout=0.05)
    ckpt                 = torch.load(checkpoint_path, map_location=device)
    missing, unexpected  = model.load_state_dict(ckpt["lora_state"], strict=False)
    print(f"  LoRA weights loaded. Missing: {len(missing)}  "
          f"Unexpected: {len(unexpected)}")
    model     = model.to(device).eval()
    tokenizer = open_clip.get_tokenizer(_BMC_MODEL_ID)
    size_mb   = os.path.getsize(checkpoint_path) / 1e6
    print(f"✅ Fine-tuned BioMedCLIP loaded ({size_mb:.1f} MB)")
    return model, preprocess, tokenizer


# Load both models
original_model, _, original_preprocess = open_clip.create_model_and_transforms(_BMC_MODEL_ID)
original_model     = original_model.to(device).eval()
original_tokenizer = open_clip.get_tokenizer(_BMC_MODEL_ID)
print("✅ Original BioMedCLIP loaded")

ft1_model, ft1_preprocess, ft1_tokenizer = load_finetuned_biomedclip(
    "/kaggle/input/finetuned-biomedclip-model/biomedclip_stage3_final.pt"
)

# Per-label routing — update after comparing ZS benchmarks
LABEL_MODEL_MAP = {
    "Edema":                      "original",
    "Atelectasis":                "ft1",
    "Pleural Effusion":           "ft1",
    "Enlarged Cardiomediastinum": "ft1",
    "Consolidation":              "original",
}
print("\nLabel routing:")
for label, m in LABEL_MODEL_MAP.items():
    print(f"  {label:<35} → {m}")


def _bmc_parts(label):
    """Return (model, preprocess, tokenizer) for this label."""
    if LABEL_MODEL_MAP[label] == "original":
        return original_model, original_preprocess, original_tokenizer
    return ft1_model, ft1_preprocess, ft1_tokenizer

# FT-10 : Calibrate Thresholds (Original + Fine-Tuned, with Context)

In [ ]:
# ================================================================
# Calibrate thresholds separately for each model.
# Uses calibrate_biomedclip_threshold() which already calls
# build_bmc_context() per sample — so context is consistent.
# ================================================================

print("Calibrating ORIGINAL model thresholds...")
ORIGINAL_BMC_THRESHOLDS = {}
for label in [l for l, m in LABEL_MODEL_MAP.items() if m == "original"]:
    ORIGINAL_BMC_THRESHOLDS[label] = calibrate_biomedclip_threshold(
        label, n_samples=1000,
        model=original_model,
        preprocess=original_preprocess,
        tokenizer=original_tokenizer,
    )

print("\nCalibrating FINE-TUNED model thresholds...")
FT1_BMC_THRESHOLDS = {}
for label in [l for l, m in LABEL_MODEL_MAP.items() if m == "ft1"]:
    FT1_BMC_THRESHOLDS[label] = calibrate_biomedclip_threshold(
        label, n_samples=1000,
        model=ft1_model,
        preprocess=ft1_preprocess,
        tokenizer=ft1_tokenizer,
    )

BIOMEDCLIP_THRESHOLDS_SMART = {**ORIGINAL_BMC_THRESHOLDS, **FT1_BMC_THRESHOLDS}
print("\n📊 Final per-label thresholds:")
for label, thresh in BIOMEDCLIP_THRESHOLDS_SMART.items():
    print(f"  {label:<35} {thresh:.4f}  "
          f"({LABEL_MODEL_MAP[label]}, strategy={BMC_CONTEXT_STRATEGY[label]})")

# FT-11 : Run Zero-Shot Smart Benchmark (Original + FT per Label)

In [ ]:
# ================================================================
# Zero-Shot Smart Benchmark — routes each label to correct model.
# Uses build_bmc_context() inside infer_biomedclip() automatically
# via the model/preprocess/tokenizer/threshold_dict args.
# ================================================================

print("🚀 Running BioMedCLIP ZERO-SHOT smart benchmark...")
print("=" * 80)

def _run_bmc_smart_benchmark():
    all_results = []
    labeler_df  = rule_labeler_df.copy()
    gt_df       = ground_truth_df.copy()
    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True)
                break

    for label in TARGET_LABELS:
        print(f"\n{'='*60}\n  Label: {label}\n{'='*60}")
        m, p, t = _bmc_parts(label)
        thresh  = BIOMEDCLIP_THRESHOLDS_SMART[label]
        print(f"  Model: {LABEL_MODEL_MAP[label]}  "
              f"threshold={thresh:.4f}  "
              f"strategy={BMC_CONTEXT_STRATEGY[label]}")

        uncertain_idx    = labeler_df[labeler_df[label] == -1].index
        gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
        gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
        gt_for_uncertain = gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]

        print(f"  Samples: {len(gt_for_uncertain)} | "
              f"pos={int(gt_for_uncertain['gt'].sum())} | "
              f"neg={int((gt_for_uncertain['gt']==0).sum())}")

        top_features = top_features_per_label.get(label, [])

        for _, row in tqdm(gt_for_uncertain.iterrows(),
                           total=len(gt_for_uncertain),
                           desc=f"BMC-smart · {label}"):
            try:
                img_path = fix_test_path(row["Study"])
                fv       = get_sample_features(
                    row["Study"], label, top_features, rule_labeler_df
                )
                pred = infer_biomedclip(
                    img_path, label, fv,
                    model=m, preprocess=p, tokenizer=t,
                    threshold_dict=BIOMEDCLIP_THRESHOLDS_SMART,
                )
                all_results.append({
                    "label":           label,
                    "study":           row["Study"],
                    "y_true":          int(row["gt"]),
                    "y_pred":          pred,
                    "feature_context": " | ".join(fv),
                })
            except Exception as e:
                all_results.append({
                    "label": label, "study": row["Study"],
                    "y_true": int(row["gt"]) if pd.notna(row.get("gt")) else -1,
                    "y_pred": 0, "feature_context": f"ERROR: {e}",
                })

    return pd.DataFrame(all_results)


biomedclip_zs_results = _run_bmc_smart_benchmark()
biomedclip_zs_results.to_csv("biomedclip_zeroshot_smart_results.csv", index=False)
print("✅ Saved: biomedclip_zeroshot_smart_results.csv")

biomedclip_zs_metrics = evaluate_results_with_details(biomedclip_zs_results)
print_detailed_results(biomedclip_zs_metrics)

print("\n" + "=" * 80)
print("📈 ZERO-SHOT SMART SUMMARY")
print("=" * 80)
if len(biomedclip_zs_metrics) > 0:
    summary = biomedclip_zs_metrics[[
        "label", "n_samples", "accuracy", "f1_score", "auc",
        "true_negative", "false_positive", "false_negative", "true_positive"
    ]]
    print(summary.to_string(index=False))

# BioMedCLIP Few-Shot: N-Shot Sweep (with Feature Context)

In [ ]:
# ================================================================
# BioMedCLIP Few-Shot — Core Helpers
#
# build_bmc_prototype   — mean-pooled pos/neg image prototypes from
#                         train_df certain (0/1) samples.
#                         Uses _bmc_parts(label) for correct model.
#
# collect_bmc_scores    — for each GT-uncertain sample computes:
#   text_score  = sim(img, pos_txt) - sim(img, neg_txt)
#                 where pos_txt / neg_txt are built PER SAMPLE with
#                 build_bmc_context() — same as zero-shot inference.
#                 Uncertain (-1) feature values handled by strategy.
#   proto_score = sim(img, pos_proto) - sim(img, neg_proto)
#                 (image-only, no text context)
#
# No GT labels used for prototypes or scoring.
# ================================================================

BMC_N_SHOTS_LIST = [4, 8, 16, 32, 64]
BMC_ALPHA_SWEEP  = [0.0, 0.25, 0.5, 0.75, 1.0]
BMC_RANDOM_SEED  = 42
BMC_FS_THRESHOLDS = {}   # filled by sweep


def build_bmc_prototype(label: str, n_shots: int, seed: int = BMC_RANDOM_SEED):
    model, preprocess, _ = _bmc_parts(label)
    df      = train_df[train_df[label].isin([0, 1])].copy()
    n_pos_a = int((df[label] == 1).sum())
    n_neg_a = int((df[label] == 0).sum())
    n_pos   = min(n_shots, n_pos_a)
    n_neg   = min(n_shots, n_neg_a)

    if n_pos < 2 or n_neg < 2:
        print(f"    ⚠️  {label}: not enough train samples")
        return None, None, {}

    buf      = 2
    pos_pool = df[df[label] == 1].sample(min(n_pos * buf, n_pos_a), random_state=seed)
    neg_pool = df[df[label] == 0].sample(min(n_neg * buf, n_neg_a), random_state=seed)

    def load_embs(pool, target_n):
        embs, skipped = [], 0
        for _, row in pool.iterrows():
            if len(embs) >= target_n:
                break
            try:
                clean = (row["Path"]
                         .replace("CheXpert-v1.0-small/", "")
                         .replace("CheXpert-v1.0/", ""))
                img = preprocess(
                    Image.open(os.path.join(BASE_PATH, clean)).convert("RGB")
                ).unsqueeze(0).to(device)
                with torch.no_grad():
                    e = model.encode_image(img)
                    embs.append(e / e.norm(dim=-1, keepdim=True))
            except Exception:
                skipped += 1
        return embs, skipped

    pos_embs, sk_pos = load_embs(pos_pool, n_pos)
    neg_embs, sk_neg = load_embs(neg_pool, n_neg)

    if len(pos_embs) < 2 or len(neg_embs) < 2:
        print(f"    ⚠️  {label}: too few valid embeddings")
        return None, None, {}

    pos_proto = torch.cat(pos_embs).mean(0, keepdim=True)
    neg_proto = torch.cat(neg_embs).mean(0, keepdim=True)
    pos_proto = pos_proto / pos_proto.norm(dim=-1, keepdim=True)
    neg_proto = neg_proto / neg_proto.norm(dim=-1, keepdim=True)

    return pos_proto, neg_proto, {
        "n_pos": len(pos_embs), "n_neg": len(neg_embs),
        "skipped": sk_pos + sk_neg,
    }


def collect_bmc_scores(label: str, pos_proto, neg_proto) -> pd.DataFrame:
    """
    For every GT-uncertain sample compute text_score and proto_score.
    text_score is computed PER SAMPLE with build_bmc_context() so that
    uncertain (-1) feature values are handled exactly as in zero-shot
    inference (hedging language for radiology_format, silently skipped
    for old_format).
    GT labels are stored for evaluation only — never used in scoring.
    """
    model, preprocess, tokenizer = _bmc_parts(label)

    labeler_df = rule_labeler_df.copy()
    gt_df      = ground_truth_df.copy()
    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True)
                break

    uncertain_idx    = labeler_df[labeler_df[label] == -1].index
    gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
    gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
    gt_for_uncertain = (gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]
                        .reset_index(drop=True))

    if len(gt_for_uncertain) == 0:
        return pd.DataFrame()

    top_feats = top_features_per_label.get(label, [])
    rows = []

    for _, row in tqdm(gt_for_uncertain.iterrows(),
                       total=len(gt_for_uncertain),
                       desc=f"    [{label}]", leave=False):
        try:
            img_path = fix_test_path(row["Study"])
            image    = preprocess(
                Image.open(img_path).convert("RGB")
            ).unsqueeze(0).to(device)

            # Per-sample feature context — handles uncertain (-1) values
            # via the strategy map (radiology_format → hedging sentences,
            # old_format → skip, no_context → empty string)
            fv  = get_sample_features(row["Study"], label, top_feats, rule_labeler_df)
            ctx = build_bmc_context(label, fv)

            pos_text = ctx + BIOMEDCLIP_POSITIVE_PROMPTS[label]
            neg_text = ctx + BIOMEDCLIP_NEGATIVE_PROMPTS[label]

            with torch.no_grad():
                img_f = model.encode_image(image)
                img_f = img_f / img_f.norm(dim=-1, keepdim=True)

                pos_tok = tokenizer([pos_text]).to(device)
                neg_tok = tokenizer([neg_text]).to(device)
                pos_txt = model.encode_text(pos_tok)
                neg_txt = model.encode_text(neg_tok)
                pos_txt = pos_txt / pos_txt.norm(dim=-1, keepdim=True)
                neg_txt = neg_txt / neg_txt.norm(dim=-1, keepdim=True)

            text_score  = ((img_f @ pos_txt.T) - (img_f @ neg_txt.T)).item()
            proto_score = ((img_f @ pos_proto.T) - (img_f @ neg_proto.T)).item()

            rows.append({
                "study":       row["Study"],
                "y_true":      int(row["gt"]),
                "text_score":  text_score,
                "proto_score": proto_score,
            })
        except Exception:
            continue

    return pd.DataFrame(rows)


print("✅ BioMedCLIP few-shot core helpers defined")

# BioMedCLIP Few-Shot — Threshold Calibration + Evaluation Helpers

In [ ]:
# ================================================================
# BioMedCLIP Few-Shot — Threshold Calibration + Evaluation
#
# calibrate_bmc_combined_threshold:
#   Calibrates threshold for combined score on train_df.
#   Uses build_bmc_context() per sample — same as collect_bmc_scores.
#   Prototype samples excluded (no leakage).
#   Uncertain (-1) values handled identically to inference.
# ================================================================
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              balanced_accuracy_score, confusion_matrix,
                              f1_score, roc_curve)


def calibrate_bmc_combined_threshold(label: str, n_shots: int, alpha: float,
                                      n_cal: int = 300,
                                      cal_seed: int = 99) -> float:
    model, preprocess, tokenizer = _bmc_parts(label)
    df    = train_df[train_df[label].isin([0, 1])].copy()
    img_w = 1.0 - alpha

    # Prototype pool indices (seed=42 — same as build_bmc_prototype)
    n_pos_a = int((df[label] == 1).sum())
    n_neg_a = int((df[label] == 0).sum())
    buf     = 2
    proto_pool_pos = df[df[label] == 1].sample(
        min(n_shots * buf, n_pos_a), random_state=42)
    proto_pool_neg = df[df[label] == 0].sample(
        min(n_shots * buf, n_neg_a), random_state=42)
    proto_idx = set(proto_pool_pos.index) | set(proto_pool_neg.index)

    cal_df = df[~df.index.isin(proto_idx)]
    n_each = min(n_cal // 2, (cal_df[label] == 1).sum(),
                              (cal_df[label] == 0).sum())
    if n_each < 10:
        return 0.0

    cal_df = pd.concat([
        cal_df[cal_df[label] == 1].sample(n_each, random_state=cal_seed),
        cal_df[cal_df[label] == 0].sample(n_each, random_state=cal_seed),
    ], ignore_index=True)

    # Rebuild prototypes (same seed=42)
    pos_proto, neg_proto, _ = build_bmc_prototype(label, n_shots, seed=42)
    if pos_proto is None:
        return 0.0

    top_feats = top_features_per_label.get(label, [])
    scores, labels_list = [], []

    for _, row in tqdm(cal_df.iterrows(), total=len(cal_df),
                       desc=f"  Cal [{label}] n={n_shots} α={alpha:.2f}",
                       leave=False):
        try:
            clean    = (row["Path"]
                        .replace("CheXpert-v1.0-small/", "")
                        .replace("CheXpert-v1.0/", ""))
            image    = preprocess(
                Image.open(os.path.join(BASE_PATH, clean)).convert("RGB")
            ).unsqueeze(0).to(device)

            # Per-sample context — train_df row uses "Path" as key
            fv  = get_sample_features(row["Path"], label, top_feats, train_df)
            ctx = build_bmc_context(label, fv)

            pos_text = ctx + BIOMEDCLIP_POSITIVE_PROMPTS[label]
            neg_text = ctx + BIOMEDCLIP_NEGATIVE_PROMPTS[label]

            with torch.no_grad():
                img_f   = model.encode_image(image)
                img_f   = img_f / img_f.norm(dim=-1, keepdim=True)
                pos_tok = tokenizer([pos_text]).to(device)
                neg_tok = tokenizer([neg_text]).to(device)
                pos_txt = model.encode_text(pos_tok)
                neg_txt = model.encode_text(neg_tok)
                pos_txt = pos_txt / pos_txt.norm(dim=-1, keepdim=True)
                neg_txt = neg_txt / neg_txt.norm(dim=-1, keepdim=True)

            t_score = ((img_f @ pos_txt.T) - (img_f @ neg_txt.T)).item()
            p_score = ((img_f @ pos_proto.T) - (img_f @ neg_proto.T)).item()
            scores.append(alpha * t_score + img_w * p_score)
            labels_list.append(int(row[label]))
        except Exception:
            continue

    if len(scores) < 20:
        return 0.0

    scores_arr = np.array(scores)
    labels_arr = np.array(labels_list)

    # Half-split CV threshold selection
    idx          = np.random.RandomState(42).permutation(len(scores_arr))
    mid          = len(idx) // 2
    s_tr, l_tr   = scores_arr[idx[:mid]], labels_arr[idx[:mid]]
    s_val, l_val = scores_arr[idx[mid:]], labels_arr[idx[mid:]]

    try:
        fpr, tpr, thrs = roc_curve(l_tr, s_tr)
        thresh_bal     = float(thrs[np.argmax((tpr + (1 - fpr)) / 2)])
    except Exception:
        thresh_bal = 0.0

    thresh_mid = float((s_tr[l_tr == 1].mean() + s_tr[l_tr == 0].mean()) / 2)

    best_t, best_s = thresh_mid, -1
    for t in [thresh_bal, thresh_mid]:
        if np.isinf(t) or np.isnan(t):
            continue
        preds = (s_val > t).astype(int)
        sc    = balanced_accuracy_score(l_val, preds)
        if sc > best_s:
            best_s, best_t = sc, t

    return float(best_t)


def evaluate_bmc_scores(scores_arr, y_true_arr, label, method_name,
                         threshold=None) -> dict:
    if len(np.unique(y_true_arr)) < 2 or len(y_true_arr) < 4:
        return {}

    auc = roc_auc_score(y_true_arr, scores_arr)
    try:
        auprc = average_precision_score(y_true_arr, scores_arr)
    except Exception:
        auprc = float(y_true_arr.mean())

    if threshold is not None:
        thresh = threshold
    elif label in BIOMEDCLIP_THRESHOLDS_SMART:
        thresh = BIOMEDCLIP_THRESHOLDS_SMART[label]
    else:
        thresh = (scores_arr[y_true_arr == 1].mean() +
                  scores_arr[y_true_arr == 0].mean()) / 2.0

    preds          = (scores_arr > thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_arr, preds, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "method":    method_name,
        "auc":       round(float(auc),   3),
        "auprc":     round(float(auprc), 3),
        "bal_acc":   round((sens + spec) / 2, 3),
        "f1":        round(float(f1_score(y_true_arr, preds, zero_division=0)), 3),
        "sens":      round(float(sens),  3),
        "spec":      round(float(spec),  3),
        "threshold": round(float(thresh), 4),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
        "n":  len(y_true_arr),
    }


print("✅ BioMedCLIP few-shot calibration + evaluation helpers defined")

# BioMedCLIP Few-Shot — N-Shot × Alpha Sweep

In [ ]:
# ================================================================
# BioMedCLIP Few-Shot N-Shot × Alpha Sweep
#
# PROCEDURE (identical to WhyXrayCLIP and CheXZero):
#   For each label × n_shots × alpha:
#     1. Build prototypes from train_df (seed=42)
#     2. collect_bmc_scores() — text_score uses build_bmc_context()
#        per sample; uncertain (-1) values handled by strategy
#     3. Calibrate FS threshold on train_df combined scores
#        (proto samples excluded, seed=99)
#     4. AUC (primary, threshold-free) + binary metrics
#
# Best config per label = highest (AUC + bal_acc) / 2
# ================================================================

def run_bmc_fewshot_sweep():
    print("\n" + "=" * 80)
    print("🔬 BioMedCLIP FEW-SHOT N-SHOT SWEEP")
    print(f"   Shot counts : {BMC_N_SHOTS_LIST}")
    print(f"   Alpha sweep : {BMC_ALPHA_SWEEP}")
    print(f"   Labels      : {TARGET_LABELS}")
    print(f"   Context     : {BMC_CONTEXT_STRATEGY}")
    print("=" * 80)

    all_rows       = []
    best_per_label = {}

    for label in TARGET_LABELS:
        print(f"\n{'━' * 70}")
        print(f"  Label: {label}  [{LABEL_MODEL_MAP[label]} model | "
              f"strategy={BMC_CONTEXT_STRATEGY[label]}]")
        print(f"{'━' * 70}")
        BMC_FS_THRESHOLDS[label] = {}

        for n_shots in BMC_N_SHOTS_LIST:
            print(f"\n  ── {n_shots}-shot ────────────────────────────")

            pos_proto, neg_proto, meta = build_bmc_prototype(label, n_shots)
            if pos_proto is None:
                print(f"    ⚠️  Prototype build failed — skipping {n_shots}-shot")
                continue
            print(f"    ✅ Prototypes: pos={meta['n_pos']} | "
                  f"neg={meta['n_neg']} | skipped={meta['skipped']}")

            # collect_bmc_scores encodes text per-sample with context
            scores_df = collect_bmc_scores(label, pos_proto, neg_proto)
            if len(scores_df) < 4:
                print(f"    ⚠️  Too few samples ({len(scores_df)}) — skipping")
                continue

            y_true       = scores_df["y_true"].values
            text_scores  = scores_df["text_score"].values
            proto_scores = scores_df["proto_score"].values

            t_gap = text_scores[y_true==1].mean()  - text_scores[y_true==0].mean()
            p_gap = proto_scores[y_true==1].mean() - proto_scores[y_true==0].mean()
            print(f"    n={len(scores_df)} | pos={int(y_true.sum())} | "
                  f"neg={int((y_true==0).sum())}")
            print(f"    text_score  gap : {t_gap:>+.4f}  "
                  f"{'✅' if t_gap > 0.02 else '⚠️  weak'}")
            print(f"    proto_score gap : {p_gap:>+.4f}  "
                  f"{'✅' if p_gap > 0.02 else '⚠️  weak'}")

            print(f"\n    Calibrating FS thresholds...")
            BMC_FS_THRESHOLDS[label][n_shots] = {}

            for alpha in BMC_ALPHA_SWEEP:
                combined = alpha * text_scores + (1 - alpha) * proto_scores

                if alpha == 1.0:
                    fs_thresh  = BIOMEDCLIP_THRESHOLDS_SMART.get(label, 0.0)
                    thresh_src = "zs_calibrated"
                else:
                    fs_thresh  = calibrate_bmc_combined_threshold(
                        label, n_shots, alpha, n_cal=300)
                    thresh_src = "fs_calibrated"

                BMC_FS_THRESHOLDS[label][n_shots][alpha] = fs_thresh

                result = evaluate_bmc_scores(
                    combined, y_true,
                    label=label,
                    method_name=f"bmc_{n_shots}shot_a{alpha:.2f}",
                    threshold=fs_thresh,
                )
                if not result:
                    continue

                score = (result["auc"] + result["bal_acc"]) / 2
                result.update({
                    "label": label, "n_shots": n_shots,
                    "alpha": alpha, "score": score,
                    "model": LABEL_MODEL_MAP[label],
                    "strategy": BMC_CONTEXT_STRATEGY[label],
                })
                all_rows.append(result)

                if (label not in best_per_label or
                        score > best_per_label[label]["score"]):
                    best_per_label[label] = result

                print(f"      α={alpha:.2f} | AUC={result['auc']:.3f} | "
                      f"BA={result['bal_acc']:.3f} | F1={result['f1']:.3f} | "
                      f"score={score:.3f}  [{thresh_src}]")

    results_df = pd.DataFrame(all_rows)
    results_df.to_csv("bmc_fewshot_sweep_results.csv", index=False)
    print("\n✅ Sweep results saved: bmc_fewshot_sweep_results.csv")

    print("\n" + "=" * 80)
    print("📊 BEST CONFIG PER LABEL  (metric = (AUC + BalAcc) / 2)")
    print("=" * 80)
    for label, best in best_per_label.items():
        print(f"  {label:<35}  n={best['n_shots']:>2}, α={best['alpha']:.2f}  "
              f"AUC={best['auc']:.3f}  BA={best['bal_acc']:.3f}  "
              f"score={best['score']:.3f}  "
              f"[{LABEL_MODEL_MAP[label]} | {BMC_CONTEXT_STRATEGY[label]}]")

    return results_df, best_per_label


bmc_sweep_results, BMC_BEST_PER_LABEL = run_bmc_fewshot_sweep()

In [ ]:
# Apply best sweep configs
print("📋 Applying best sweep configs to inference pipeline...")
print("=" * 60)

BMC_BEST_CONFIGS = {}

for label, best in BMC_BEST_PER_LABEL .items():
    n     = best["n_shots"]
    alpha = best["alpha"]
    thresh = BMC_FS_THRESHOLDS[label][n][alpha]
    pos_proto, neg_proto, meta = build_bmc_prototype(label, n)

    BMC_BEST_CONFIGS[label] = {
        "n_shots":   n,
        "alpha":     alpha,
        "threshold": thresh,
        "pos_proto": pos_proto,
        "neg_proto": neg_proto,
        "model":     LABEL_MODEL_MAP[label],
    }
    BIOMEDCLIP_THRESHOLDS[label] = thresh

    print(f"  {label:<35} n={n:>2}  α={alpha:.2f}  "
          f"thresh={thresh:.4f}  model={LABEL_MODEL_MAP[label]}")

print("\n✅ BIOMEDCLIP_THRESHOLDS updated with best few-shot thresholds")

In [ ]:
# ================================================================
# HARDCODED BioMedCLIP BEST CONFIGS (from sweep output)
# Run this BEFORE infer_biomedclip
# ================================================================

BMC_BEST_CONFIGS = {
    "Edema": {
        "n_shots": 4,
        "alpha": 0.50,
        "threshold": -0.0023,
        "pos_proto": None,   # will be filled below
        "neg_proto": None,
        "model": "original",
    },
    "Atelectasis": {
        "n_shots": 4,
        "alpha": 0.50,
        "threshold": -0.0024,
        "pos_proto": None,
        "neg_proto": None,
        "model": "ft1",
    },
    "Pleural Effusion": {
        "n_shots": 4,
        "alpha": 0.50,
        "threshold": 0.0167,
        "pos_proto": None,
        "neg_proto": None,
        "model": "ft1",
    },
    "Enlarged Cardiomediastinum": {
        "n_shots": 64,
        "alpha": 0.00,
        "threshold": -0.0111,
        "pos_proto": None,
        "neg_proto": None,
        "model": "ft1",
    },
    "Consolidation": {
        "n_shots": 32,
        "alpha": 0.25,
        "threshold": 0.0095,
        "pos_proto": None,
        "neg_proto": None,
        "model": "original",
    },
}

# ------------------------------------------------
# Build prototypes once (IMPORTANT)
# ------------------------------------------------
print("🔧 Building hardcoded BioMedCLIP prototypes...")

for label, cfg in BMC_BEST_CONFIGS.items():
    n = cfg["n_shots"]

    if cfg["alpha"] < 1.0:
        pos_p, neg_p, meta = build_bmc_prototype(label, n, seed=42)
        cfg["pos_proto"] = pos_p
        cfg["neg_proto"] = neg_p
        print(f"  {label:<35} ✅ pos={meta.get('n_pos','?')} neg={meta.get('n_neg','?')}")
    else:
        print(f"  {label:<35} (text-only)")

# ------------------------------------------------
# Build threshold dict used by infer_biomedclip
# ------------------------------------------------
BIOMEDCLIP_THRESHOLDS = {
    label: cfg["threshold"] for label, cfg in BMC_BEST_CONFIGS.items()
}

print("\n✅ BioMedCLIP configs ready (hardcoded)")


In [ ]:
# ================================================================
# REDEFINE infer_biomedclip — few-shot aware version
# Overwrites the zero-shot version from Cell 40.
# Must run AFTER BMC_BEST_CONFIGS is built above.
# ================================================================

def infer_biomedclip(img_path: str, label: str,
                     feature_values: list = None) -> int:
    try:
        # Select correct model (original vs fine-tuned)
        if label in FT_LABELS and 'biomedclip_ft_model' in globals():
            model      = biomedclip_ft_model
            preprocess = biomedclip_ft_preprocess
            tokenizer  = biomedclip_ft_tokenizer
        else:
            model      = biomedclip_model
            preprocess = biomedclip_preprocess
            tokenizer  = biomedclip_tokenizer

        image = preprocess(
            Image.open(img_path).convert("RGB")
        ).unsqueeze(0).to(device)

        pos_text = BIOMEDCLIP_POSITIVE_PROMPTS[label]
        neg_text = BIOMEDCLIP_NEGATIVE_PROMPTS[label]
        texts    = tokenizer([pos_text, neg_text]).to(device)

        with torch.no_grad():
            img_f = model.encode_image(image)
            txt_f = model.encode_text(texts)
            img_f = img_f / img_f.norm(dim=-1, keepdim=True)
            txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
            sims  = (img_f @ txt_f.T).squeeze(0)

        text_score = sims[0].item()

        # Use few-shot combined score if sweep found a good config
        cfg   = BMC_BEST_CONFIGS.get(label)
        alpha = cfg["alpha"] if cfg else 1.0

        if cfg and alpha < 1.0 and cfg["pos_proto"] is not None:
            pos_proto   = cfg["pos_proto"].to(device)
            neg_proto   = cfg["neg_proto"].to(device)
            pos_sim     = (img_f @ pos_proto.T).mean().item()
            neg_sim     = (img_f @ neg_proto.T).mean().item()
            proto_score = pos_sim - neg_sim
            combined    = alpha * text_score + (1 - alpha) * proto_score
        else:
            # alpha=1.0 means text-only (zero-shot)
            combined = text_score

        threshold = BIOMEDCLIP_THRESHOLDS.get(label, 0.0)
        return 1 if combined > threshold else 0

    except Exception as e:
        print(f"  [infer_biomedclip] Error — {label}: {e}")
        return 0

print("✅ infer_biomedclip redefined — now uses few-shot prototypes + "
      "best alpha per label")
print("   Configs active:")
for label, cfg in BMC_BEST_CONFIGS.items():
    print(f"   {label:<35} α={cfg['alpha']:.2f}  "
          f"n={cfg['n_shots']}  model={cfg['model']}")

# BioMedCLIP Few-Shot — Final Benchmark with Best Config

In [ ]:
# ================================================================
# BioMedCLIP Final Few-Shot Benchmark
#
# Uses BMC_BEST_PER_LABEL for (n_shots, alpha) per label.
# Text re-encoded PER SAMPLE with build_bmc_context() so that
# uncertain (-1) feature values are handled correctly throughout.
# Prints full ZS vs FS comparison table.
# ================================================================

print("🚀 Running BioMedCLIP FINAL FEW-SHOT benchmark...")
print("=" * 80)

# Build final prototypes
BMC_FINAL_PROTO = {}
for label in TARGET_LABELS:
    best    = BMC_BEST_PER_LABEL.get(label, {})
    n_shots = best.get("n_shots", 16)
    pos_p, neg_p, meta = build_bmc_prototype(label, n_shots, seed=BMC_RANDOM_SEED)
    BMC_FINAL_PROTO[label] = (pos_p, neg_p, n_shots)
    print(f"  {label}: {n_shots}-shot  pos={meta.get('n_pos','?')} | "
          f"neg={meta.get('n_neg','?')}")

# Run final benchmark
final_rows = []
labeler_df = rule_labeler_df.copy()
gt_df      = ground_truth_df.copy()
for df_ in [gt_df, labeler_df]:
    for c in ["Path", "path", "study"]:
        if c in df_.columns and "Study" not in df_.columns:
            df_.rename(columns={c: "Study"}, inplace=True)
            break

for label in TARGET_LABELS:
    print(f"\n{'='*60}\n  Label: {label}\n{'='*60}")
    model, preprocess, tokenizer = _bmc_parts(label)
    best       = BMC_BEST_PER_LABEL.get(label, {})
    alpha      = best.get("alpha", 1.0)
    n_shots    = best.get("n_shots", 16)
    fs_thresh  = BMC_FS_THRESHOLDS.get(label, {}).get(n_shots, {}).get(
        alpha, BIOMEDCLIP_THRESHOLDS_SMART.get(label, 0.0))

    print(f"  Config: n={n_shots}, α={alpha:.2f}, "
          f"threshold={fs_thresh:.4f}, "
          f"model={LABEL_MODEL_MAP[label]}, "
          f"strategy={BMC_CONTEXT_STRATEGY[label]}")

    pos_proto, neg_proto, _ = BMC_FINAL_PROTO[label]

    uncertain_idx    = labeler_df[labeler_df[label] == -1].index
    gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
    gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
    gt_for_uncertain = gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]
    top_features     = top_features_per_label.get(label, [])

    print(f"  Samples: {len(gt_for_uncertain)} | "
          f"pos={int(gt_for_uncertain['gt'].sum())} | "
          f"neg={int((gt_for_uncertain['gt']==0).sum())}")

    for _, row in tqdm(gt_for_uncertain.iterrows(),
                       total=len(gt_for_uncertain),
                       desc=f"BMC-final · {label}"):
        try:
            img_path = fix_test_path(row["Study"])
            image    = preprocess(
                Image.open(img_path).convert("RGB")
            ).unsqueeze(0).to(device)

            fv  = get_sample_features(
                row["Study"], label, top_features, rule_labeler_df
            )
            # Per-sample context — uncertain (-1) handled by strategy
            ctx = build_bmc_context(label, fv)

            pos_text = ctx + BIOMEDCLIP_POSITIVE_PROMPTS[label]
            neg_text = ctx + BIOMEDCLIP_NEGATIVE_PROMPTS[label]

            with torch.no_grad():
                img_f   = model.encode_image(image)
                img_f   = img_f / img_f.norm(dim=-1, keepdim=True)

                if pos_proto is not None and alpha < 1.0:
                    pos_tok = tokenizer([pos_text]).to(device)
                    neg_tok = tokenizer([neg_text]).to(device)
                    pos_txt = model.encode_text(pos_tok)
                    neg_txt = model.encode_text(neg_tok)
                    pos_txt = pos_txt / pos_txt.norm(dim=-1, keepdim=True)
                    neg_txt = neg_txt / neg_txt.norm(dim=-1, keepdim=True)

                    t_score = ((img_f @ pos_txt.T) - (img_f @ neg_txt.T)).item()
                    p_score = ((img_f @ pos_proto.T) - (img_f @ neg_proto.T)).item()
                    combined = alpha * t_score + (1 - alpha) * p_score
                else:
                    # alpha == 1.0 or no prototype — text only
                    tok   = tokenizer([pos_text, neg_text]).to(device)
                    txt_f = model.encode_text(tok)
                    txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
                    sims  = (img_f @ txt_f.T).squeeze(0)
                    combined = sims[0].item() - sims[1].item()

            pred = 1 if combined > fs_thresh else 0
            final_rows.append({
                "label": label, "study": row["Study"],
                "y_true": int(row["gt"]), "y_pred": pred,
                "combined_score": combined, "threshold_used": fs_thresh,
                "feature_context": " | ".join(fv),
            })
        except Exception as e:
            final_rows.append({
                "label": label, "study": row["Study"],
                "y_true": int(row["gt"]) if pd.notna(row.get("gt")) else -1,
                "y_pred": 0, "combined_score": None,
                "threshold_used": fs_thresh, "feature_context": f"ERROR: {e}",
            })

bmc_fs_results = pd.DataFrame(final_rows)
bmc_fs_results.to_csv("bmc_fewshot_final_results.csv", index=False)
print("✅ Saved: bmc_fewshot_final_results.csv")

bmc_fs_metrics = evaluate_results_with_details(bmc_fs_results)
print_detailed_results(bmc_fs_metrics)

# ── ZS vs FS comparison ───────────────────────────────────────────
print("\n" + "=" * 80)
print("📊 BioMedCLIP: Zero-Shot vs Few-Shot (best config per label)")
print("   ZS threshold: train_df ZS-calibrated (with context)")
print("   FS threshold: train_df FS-calibrated combined score (with context)")
print("=" * 80)
print(f"  {'Label':<35} "
      f"{'ZS AUC':>7} {'FS AUC':>7} {'ΔAUC':>7} "
      f"{'ZS BA':>7} {'FS BA':>7} {'ΔBA':>7} "
      f"{'ZS F1':>6} {'FS F1':>6} {'ΔF1':>6}  Config")
print("  " + "-" * 110)

for label in TARGET_LABELS:
    z_row = biomedclip_zs_metrics[biomedclip_zs_metrics["label"] == label]
    f_row = bmc_fs_metrics[bmc_fs_metrics["label"] == label]
    if len(z_row) == 0 or len(f_row) == 0:
        continue
    z = z_row.iloc[0]
    f = f_row.iloc[0]

    best    = BMC_BEST_PER_LABEL.get(label, {})
    n_shot  = best.get("n_shots", "?")
    alpha   = best.get("alpha",   "?")

    def _g(row, *keys):
        for k in keys:
            if k in row and pd.notna(row[k]):
                return float(row[k])
        return 0.0

    z_auc, f_auc = _g(z, "auc"), _g(f, "auc")
    z_ba,  f_ba  = _g(z, "bal_acc"), _g(f, "bal_acc")
    z_f1,  f_f1  = _g(z, "f1", "f1_score"), _g(f, "f1", "f1_score")

    print(f"  {label:<35} "
          f"{z_auc:>7.3f} {f_auc:>7.3f} {f_auc-z_auc:>+7.3f} "
          f"{z_ba:>7.3f} {f_ba:>7.3f} {f_ba-z_ba:>+7.3f} "
          f"{z_f1:>6.3f} {f_f1:>6.3f} {f_f1-z_f1:>+6.3f}  "
          f"n={n_shot} α={alpha} [{LABEL_MODEL_MAP[label]}]")

print("=" * 80)
mean_zs = biomedclip_zs_metrics["auc"].mean()
mean_fs = bmc_fs_metrics["auc"].mean()
print(f"  Mean ZS AUC: {mean_zs:.3f}   Mean FS AUC: {mean_fs:.3f}   "
      f"Δ={mean_fs-mean_zs:>+.3f}")

# 1. Load CheXZero

In [ ]:
# ================================================================
# CELL 49: Install OpenAI CLIP then load CheXZero
# ================================================================
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "git+https://github.com/openai/CLIP.git"
], check=True)

import clip
import torch
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

CKPT_PATH = "/kaggle/input/chexpert-weights/best_64_0.0001_original_35000_0.864.pt"

print("Loading CheXZero via OpenAI CLIP (ViT-B/32)...")
chexzero_model, chexzero_preprocess = clip.load("ViT-B/32", device=device)

# Load CheXZero finetuned weights
checkpoint = torch.load(CKPT_PATH, map_location=device)
if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
missing, unexpected = chexzero_model.load_state_dict(state_dict, strict=False)
print(f"  Missing keys   : {len(missing)}")
print(f"  Unexpected keys: {len(unexpected)}")

chexzero_model.eval()
chexzero_tokenizer = clip.tokenize
print(f"✅ CheXZero loaded on {next(chexzero_model.parameters()).device}")

# Sanity check
print("\nSanity check...")
test_path = fix_test_path(rule_labeler_df["Study"].iloc[0])
test_img  = chexzero_preprocess(
    Image.open(test_path).convert("RGB")
).unsqueeze(0).to(device)

with torch.no_grad():
    test_txt = chexzero_tokenizer(
        ["pleural effusion", "no pleural effusion"], truncate=True
    ).to(device)
    img_f = chexzero_model.encode_image(test_img)
    txt_f = chexzero_model.encode_text(test_txt)
    img_f = img_f / img_f.norm(dim=-1, keepdim=True)
    txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
    sims  = (img_f @ txt_f.T).squeeze(0)

print(f"  pos={sims[0].item():.4f}, neg={sims[1].item():.4f}")
print("✅ Ready")

#  2. CheXZero Prompts

In [ ]:
CHEXZERO_POSITIVE_PROMPTS = {
    "Edema":
        "chest x-ray showing pulmonary edema with bilateral interstitial "
        "opacities perihilar haziness and vascular congestion",
    "Enlarged Cardiomediastinum":
        "chest x-ray showing enlarged cardiomediastinum with cardiomegaly "
        "increased cardiothoracic ratio and widened mediastinum",
    "Atelectasis":
        "collapsed lung with increased opacity and crowded vessels",
    "Pleural Effusion":
        "fluid in chest cavity with opaque hemithorax",
    "Consolidation":
        "dense lobar airspace consolidation with air bronchograms and opacified lung segment",
}

CHEXZERO_NEGATIVE_PROMPTS = {
    "Edema":
        "chest x-ray showing no pulmonary edema clear lung fields "
        "normal vascular markings without interstitial opacities",
    "Enlarged Cardiomediastinum":
        "chest x-ray showing normal cardiac size and mediastinal contours "
        "normal cardiothoracic ratio without enlargement",
    "Atelectasis":
        "normal aerated lung no collapse no opacity",
    "Pleural Effusion":
        "clear chest no fluid dry costophrenic angles",
    "Consolidation":
        "clear lung fields no consolidation no opacity no air bronchograms",
}

In [ ]:
# ================================================================
# CheXZero Context Strategy — Initial Defaults + build_czero_context
#
# CZERO_CONTEXT_STRATEGY is CheXZero's own strategy map — separate
# from WXRC_CONTEXT_STRATEGY (WhyXrayCLIP) and LABEL_CONTEXT_STRATEGY
# (BioViL-T). Each model has its own because ablation results differ.
#
# These defaults are overwritten by the CheXZero ablation cell
# (runs right after this, before calibration). DO NOT edit manually.
# ================================================================

CZERO_CONTEXT_STRATEGY = {
    "Edema":                      "radiology_format",
    "Atelectasis":                "old_format",
    "Pleural Effusion":           "old_format",
    "Enlarged Cardiomediastinum": "no_context",
    "Consolidation":              "no_context",
}


def build_czero_context(label: str, feature_values: list) -> str:
    """CheXZero context builder — uses CZERO_CONTEXT_STRATEGY."""
    return build_context(label, feature_values, CZERO_CONTEXT_STRATEGY)


print("✅ CheXZero context strategy defaults loaded.")
print(f"   CZERO_CONTEXT_STRATEGY: {CZERO_CONTEXT_STRATEGY}")
print("   Run CheXZero ablation next to determine winning strategy per label.")


# 3. CheXZero Inference Function

In [ ]:
# ================================================================
# CELL 51: CheXZero Inference Function
# Identical to infer_clip (WhyXrayCLIP) — same open_clip API
# Only uses chexzero_* variables and CHEXZERO_* prompts/thresholds
# ================================================================

def infer_chexzero(img_path: str, label: str, feature_values: list = None) -> int:
    try:
        feature_context = build_czero_context(label, feature_values or [])
        pos_text = feature_context + CHEXZERO_POSITIVE_PROMPTS[label]
        neg_text = feature_context + CHEXZERO_NEGATIVE_PROMPTS[label]

        image = chexzero_preprocess(
            Image.open(img_path).convert("RGB")
        ).unsqueeze(0).to(device)

        texts = chexzero_tokenizer([pos_text, neg_text], truncate=True).to(device)

        with torch.no_grad():
            img_f = chexzero_model.encode_image(image)
            txt_f = chexzero_model.encode_text(texts)
            img_f = img_f / img_f.norm(dim=-1, keepdim=True)
            txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
            sims  = (img_f @ txt_f.T).squeeze(0)

        # delta: same score space as calibration and ablation
        delta     = sims[0].item() - sims[1].item()
        threshold = CHEXZERO_THRESHOLDS.get(label, 0.0)
        return 1 if delta > threshold else 0

    except Exception as e:
        print(f"  [infer_chexzero] Error — {label}: {e}")
        return 0

In [ ]:
# ================================================================
# CheXZero Threshold Calibration
# Same cross-validated strategy as WhyXrayCLIP calibration
# Uses train_df clear (0/1) labels only — no leakage
# ================================================================
from sklearn.metrics import roc_curve, balanced_accuracy_score

def calibrate_chexzero_threshold(label, n_samples=1000):
    df = train_df[train_df[label].isin([0, 1])].copy()

    # def path_exists(p):
    #     clean = p.replace("CheXpert-v1.0-small/", "").replace("CheXpert-v1.0/", "")
    #     return os.path.exists(os.path.join(BASE_PATH, clean))

    # print(f"  {label}: scanning available images...")
    # df     = df[df["Path"].apply(path_exists)].copy()
    n_each = min(n_samples // 2, (df[label]==1).sum(), (df[label]==0).sum())

    if n_each < 10:
        print(f"  ⚠️  Not enough images for {label}, using 0.0")
        return 0.0

    pos    = df[df[label]==1].sample(n_each, random_state=42).reset_index(drop=True)
    neg    = df[df[label]==0].sample(n_each, random_state=42).reset_index(drop=True)
    df_cal = pd.concat([pos, neg], ignore_index=True).sample(
        frac=1, random_state=42
    ).reset_index(drop=True)

    print(f"  {label}: calibrating on {len(df_cal)} samples (pos={n_each}, neg={n_each})")

    scores, true_labels = [], []

    for _, row in tqdm(df_cal.iterrows(), total=len(df_cal),
                       desc=f"  Calibrating {label}", leave=False):
        try:
            clean    = row["Path"].replace("CheXpert-v1.0-small/", "").replace("CheXpert-v1.0/", "")
            img_path = os.path.join(BASE_PATH, clean)

            image = chexzero_preprocess(
                Image.open(img_path).convert("RGB")
            ).unsqueeze(0).to(device)

            fv  = get_sample_features(
                row["Path"], label, top_features_per_label.get(label, []), train_df
            ) if "top_features_per_label" in globals() else []
            ctx = build_czero_context(label, fv)
            pos_text = ctx + CHEXZERO_POSITIVE_PROMPTS[label]
            neg_text = ctx + CHEXZERO_NEGATIVE_PROMPTS[label]

            texts = chexzero_tokenizer([pos_text, neg_text], truncate=True).to(device)

            with torch.no_grad():
                img_f = chexzero_model.encode_image(image)
                txt_f = chexzero_model.encode_text(texts)
                img_f = img_f / img_f.norm(dim=-1, keepdim=True)
                txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
                sims  = (img_f @ txt_f.T).squeeze(0)

            # delta: same score space as ablation and infer_chexzero
            scores.append(sims[0].item() - sims[1].item())
            true_labels.append(int(row[label]))

        except Exception as e:
            print(f"  Error: {e}")
            continue

    if len(scores) < 40:
        print(f"  ⚠️  Too few valid scores for {label}, using 0.0")
        return 0.0

    scores_arr = np.array(scores)
    labels_arr = np.array(true_labels)

    pos_mean = scores_arr[labels_arr==1].mean()
    neg_mean = scores_arr[labels_arr==0].mean()
    gap      = pos_mean - neg_mean

    print(f"  Score stats: min={scores_arr.min():.4f}, max={scores_arr.max():.4f}, "
          f"mean={scores_arr.mean():.4f}")
    print(f"  Score for positives : mean={pos_mean:.4f}")
    print(f"  Score for negatives : mean={neg_mean:.4f}")
    print(f"  Gap                 : {gap:.4f}  "
          f"{'✅ good' if gap > 0.03 else '⚠️ small'}")

    # Cross-validated threshold selection
    idx          = np.random.RandomState(42).permutation(len(scores_arr))
    mid          = len(idx) // 2
    s_tr, l_tr   = scores_arr[idx[:mid]], labels_arr[idx[:mid]]
    s_val, l_val = scores_arr[idx[mid:]], labels_arr[idx[mid:]]

    fpr, tpr, thresholds_roc = roc_curve(l_tr, s_tr)
    bal_acc    = (tpr + (1 - fpr)) / 2
    thresh_bal = float(thresholds_roc[np.argmax(bal_acc)])
    thresh_mid = float((s_tr[l_tr==1].mean() + s_tr[l_tr==0].mean()) / 2)
    thresh_med = float(np.median(s_tr[l_tr==1]))

    candidates = {k: v for k, v in {
        "balanced_acc": thresh_bal,
        "midpoint":     thresh_mid,
        "median_pos":   thresh_med,
    }.items() if not np.isinf(v) and not np.isnan(v)}

    print(f"  Threshold candidates (val half):")
    best_name, best_thresh, best_score = None, None, -1
    for name, thresh in candidates.items():
        preds = (s_val > thresh).astype(int)
        score = balanced_accuracy_score(l_val, preds)
        print(f"    {name:15s}: {thresh:.4f} → val bal_acc={score:.3f}")
        if score > best_score:
            best_score, best_thresh, best_name = score, thresh, name

    print(f"  ✅ Chosen: {best_name} = {best_thresh:.4f} "
          f"(val balanced acc = {best_score:.3f})")
    return best_thresh


print("🔧 Calibrating CheXZero thresholds from train_df...")
print(f"   Strategy: {CZERO_CONTEXT_STRATEGY}")
print("="*60)
CHEXZERO_THRESHOLDS = {}
for label in TARGET_LABELS:
    CHEXZERO_THRESHOLDS[label] = calibrate_chexzero_threshold(label, n_samples=1000)

print("\n📊 CheXZero Final Thresholds:")
for label, thresh in CHEXZERO_THRESHOLDS.items():
    print(f"  {label}: {thresh:.4f}")
print("\n✅ Calibration done. Run the ablation cell next.")
print("   After ablation updates CZERO_CONTEXT_STRATEGY, run the")
print("   Post-Ablation Recalibration cell to re-align thresholds")
print("   with the winning strategy before benchmarking.")

# Context Format Ablation — CheXZero

> **Run after threshold calibration.** `CHEXZERO_THRESHOLDS` must exist.
> After this cell updates `CZERO_CONTEXT_STRATEGY`, run the
> **Post-Ablation Recalibration** cell.

In [ ]:
# ================================================================
# Context Format Ablation — CheXZero (Zero-Shot)
#
# PURPOSE:
#   For each target label, compare 3 context strategies:
#     no_context        : bare prompt only
#     old_format        : "X is present, Y is absent."
#     radiology_format  : "X is present with... No Y is identified."
#
# SELECTION METRIC:
#   Composite score = 0.5 × AUC + 0.5 × Balanced Accuracy.
#   AUC alone is unreliable for severely imbalanced labels (e.g.
#   Consolidation: 3 pos / 60 neg). Balanced accuracy penalises
#   strategies that win AUC by predicting everything as negative.
#   The composite gives a robust signal across both balanced and
#   imbalanced label distributions in our GT-uncertain eval set.
#
# PREREQUISITE: Run the CheXZero Threshold Calibration cell
#   (immediately before this one) to populate CHEXZERO_THRESHOLDS.
#   Score space: delta = sims[0] - sims[1], same as calibration
#   and infer_chexzero — no mismatch.
#
# DESIGN:
#   Image embedding computed once per sample and reused across
#   strategies. Balanced accuracy uses CHEXZERO_THRESHOLDS[label]
#   (train_df calibrated, no leakage). AUC is threshold-free.
#   Winner per label updates CZERO_CONTEXT_STRATEGY.
# ================================================================
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

_CZERO_ABL_RESULTS = {}   # label → {strategy: {"auc", "bal_acc", "composite"}}

def _run_czero_context_ablation():
    """
    Runs context format ablation for CheXZero.
    Returns dict: {label: {strategy: {auc, bal_acc, composite}}}
    """
    encode_img  = lambda img: chexzero_model.encode_image(img)
    tokenize    = lambda texts: chexzero_tokenizer(texts, truncate=True).to(device)
    encode_txt  = lambda tok:  chexzero_model.encode_text(tok)
    preprocess  = chexzero_preprocess
    pos_prompts = CHEXZERO_POSITIVE_PROMPTS
    neg_prompts = CHEXZERO_NEGATIVE_PROMPTS

    results = {label: {} for label in TARGET_LABELS}

    labeler_df = rule_labeler_df.copy()
    gt_df      = ground_truth_df.copy()
    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True)
                break

    for label in TARGET_LABELS:
        uncertain_idx    = labeler_df[labeler_df[label] == -1].index
        gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
        gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
        gt_for_uncertain = (gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]
                            .reset_index(drop=True))

        n_pos    = int(gt_for_uncertain["gt"].sum())
        n_neg    = int((gt_for_uncertain["gt"] == 0).sum())
        balance  = n_pos / (n_pos + n_neg) if (n_pos + n_neg) > 0 else 0
        print(f"\n  [CheXZero] {label}  "
              f"(n={len(gt_for_uncertain)}, pos={n_pos}, neg={n_neg}, "
              f"balance={balance:.2f})")

        if len(gt_for_uncertain) < 5:
            print(f"    ⚠️  Too few samples — skipping")
            continue

        # Use CheXZero zero-shot threshold (populated by calibration cells 61/62)
        thresh = CHEXZERO_THRESHOLDS.get(label, 0.0)
        print(f"    Calibrated threshold: {thresh:.4f}")
        top_feats = top_features_per_label.get(label, [])

        sample_data = []
        for _, row in tqdm(gt_for_uncertain.iterrows(),
                           total=len(gt_for_uncertain),
                           desc=f"    Embedding [{label}]", leave=False):
            try:
                img_path = fix_test_path(row["Study"])
                img = preprocess(
                    Image.open(img_path).convert("RGB")
                ).unsqueeze(0).to(device)
                fv = get_sample_features(row["Study"], label, top_feats, rule_labeler_df)
                with torch.no_grad():
                    img_f = encode_img(img)
                    img_f = img_f / img_f.norm(dim=-1, keepdim=True)
                sample_data.append({"y_true": int(row["gt"]), "img_f": img_f, "fv": fv})
            except Exception:
                continue

        if len(sample_data) < 5:
            continue

        y_true = np.array([sd["y_true"] for sd in sample_data])

        print(f"    {'Strategy':<22} {'AUC':>7} {'Bal-Acc':>9} {'Composite':>10}")
        print(f"    {'─'*52}")

        for strategy in STRATEGIES_TO_TEST:
            scores = []
            for sd in sample_data:
                try:
                    if strategy == "no_context":
                        ctx = ""
                    elif strategy == "old_format":
                        ctx = _build_old_format_context(sd["fv"], label)
                    else:
                        ctx = _build_radiology_context(sd["fv"], label)

                    pos_text = ctx + pos_prompts[label]
                    neg_text = ctx + neg_prompts[label]

                    with torch.no_grad():
                        tok   = tokenize([pos_text, neg_text])
                        txt_f = encode_txt(tok)
                        txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
                        sims  = (sd["img_f"] @ txt_f.T).squeeze(0)
                    scores.append(sims[0].item() - sims[1].item())
                except Exception:
                    scores.append(0.0)

            scores_arr = np.array(scores)
            try:
                auc = roc_auc_score(y_true, scores_arr) if len(np.unique(y_true)) > 1 else 0.5
            except Exception:
                auc = 0.5

            preds    = (scores_arr > thresh).astype(int)
            bal_acc  = balanced_accuracy_score(y_true, preds)
            composite = 0.5 * auc + 0.5 * bal_acc

            results[label][strategy] = {
                "auc":       round(auc, 3),
                "bal_acc":   round(bal_acc, 3),
                "composite": round(composite, 3),
            }
            print(f"    {strategy:<22} {auc:>7.3f} {bal_acc:>9.3f} {composite:>10.3f}")

        if results[label]:
            winner    = max(results[label], key=lambda s: results[label][s]["composite"])
            best      = results[label][winner]
            baseline  = results[label].get("no_context", {}).get("composite", 0.0)
            delta     = best["composite"] - baseline
            sign      = "✅" if delta > 0.01 else ("🔁" if delta > -0.01 else "❌")
            print(f"\n    Winner: {winner}  "
                  f"(composite={best['composite']:.3f}, "
                  f"AUC={best['auc']:.3f}, "
                  f"Bal-Acc={best['bal_acc']:.3f}, "
                  f"Δ composite vs no_context={delta:+.3f}) {sign}")

    return results


print("="*70)
print("  CONTEXT FORMAT ABLATION — CheXZero (Zero-Shot)")
print("="*70)
_CZERO_ABL_RESULTS = _run_czero_context_ablation()

# ── Auto-update CZERO_CONTEXT_STRATEGY ───────────────────────────
print("\n" + "="*70)
print("  UPDATING CZERO_CONTEXT_STRATEGY FROM ABLATION RESULTS")
print("="*70)
for label in TARGET_LABELS:
    if _CZERO_ABL_RESULTS.get(label):
        winner = max(_CZERO_ABL_RESULTS[label],
                     key=lambda s: _CZERO_ABL_RESULTS[label][s]["composite"])
        old    = CZERO_CONTEXT_STRATEGY.get(label, "not set")
        CZERO_CONTEXT_STRATEGY[label] = winner
        marker = "✅ (unchanged)" if old == winner else f"🔄 updated: {old} → {winner}"
        best   = _CZERO_ABL_RESULTS[label][winner]
        print(f"  [{label}]: {winner}  "
              f"(composite={best['composite']:.3f}, "
              f"AUC={best['auc']:.3f}, "
              f"Bal-Acc={best['bal_acc']:.3f})  {marker}")

print(f"\n✅ CZERO_CONTEXT_STRATEGY = {CZERO_CONTEXT_STRATEGY}")
print("\n✅ CZERO_CONTEXT_STRATEGY updated. Run the")
print("   Post-Ablation Recalibration cell to re-align thresholds")
print("   with the winning strategy before benchmarking.")


# CheXZero Post-Ablation Recalibration

> Runs **after** ablation. `CZERO_CONTEXT_STRATEGY` now holds the winning
> strategy. Re-runs `calibrate_chexzero_threshold()` so `CHEXZERO_THRESHOLDS`
> is calibrated with the exact same context used at inference.
> Identical pattern to WhyXrayCLIP and BioViLT.

In [ ]:
# # ================================================================
# # CELL 52: CheXZero Threshold Calibration
# # Same cross-validated strategy as WhyXrayCLIP calibration
# # Uses train_df clear (0/1) labels only — no leakage
# # ================================================================
# from sklearn.metrics import roc_curve, balanced_accuracy_score

# def calibrate_chexzero_threshold(label, n_samples=1000):
#     df = train_df[train_df[label].isin([0, 1])].copy()

#     # def path_exists(p):
#     #     clean = p.replace("CheXpert-v1.0-small/", "").replace("CheXpert-v1.0/", "")
#     #     return os.path.exists(os.path.join(BASE_PATH, clean))

#     # print(f"  {label}: scanning available images...")
#     # df     = df[df["Path"].apply(path_exists)].copy()
#     n_each = min(n_samples // 2, (df[label]==1).sum(), (df[label]==0).sum())

#     if n_each < 10:
#         print(f"  ⚠️  Not enough images for {label}, using 0.0")
#         return 0.0

#     pos    = df[df[label]==1].sample(n_each, random_state=42).reset_index(drop=True)
#     neg    = df[df[label]==0].sample(n_each, random_state=42).reset_index(drop=True)
#     df_cal = pd.concat([pos, neg], ignore_index=True).sample(
#         frac=1, random_state=42
#     ).reset_index(drop=True)

#     print(f"  {label}: calibrating on {len(df_cal)} samples (pos={n_each}, neg={n_each})")

#     scores, true_labels = [], []

#     for _, row in tqdm(df_cal.iterrows(), total=len(df_cal),
#                        desc=f"  Calibrating {label}", leave=False):
#         try:
#             clean    = row["Path"].replace("CheXpert-v1.0-small/", "").replace("CheXpert-v1.0/", "")
#             img_path = os.path.join(BASE_PATH, clean)

#             image = chexzero_preprocess(
#                 Image.open(img_path).convert("RGB")
#             ).unsqueeze(0).to(device)

#             fv  = get_sample_features(
#                 row["Path"], label, top_features_per_label.get(label, []), train_df
#             ) if "top_features_per_label" in globals() else []
#             ctx = build_czero_context(label, fv)
#             pos_text = ctx + CHEXZERO_POSITIVE_PROMPTS[label]
#             neg_text = ctx + CHEXZERO_NEGATIVE_PROMPTS[label]

#             texts = chexzero_tokenizer([pos_text, neg_text], truncate=True).to(device)

#             with torch.no_grad():
#                 img_f = chexzero_model.encode_image(image)
#                 txt_f = chexzero_model.encode_text(texts)
#                 img_f = img_f / img_f.norm(dim=-1, keepdim=True)
#                 txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
#                 sims  = (img_f @ txt_f.T).squeeze(0)

#             # delta: same score space as ablation and infer_chexzero
#             scores.append(sims[0].item() - sims[1].item())
#             true_labels.append(int(row[label]))

#         except Exception as e:
#             print(f"  Error: {e}")
#             continue

#     if len(scores) < 40:
#         print(f"  ⚠️  Too few valid scores for {label}, using 0.0")
#         return 0.0

#     scores_arr = np.array(scores)
#     labels_arr = np.array(true_labels)

#     pos_mean = scores_arr[labels_arr==1].mean()
#     neg_mean = scores_arr[labels_arr==0].mean()
#     gap      = pos_mean - neg_mean

#     print(f"  Score stats: min={scores_arr.min():.4f}, max={scores_arr.max():.4f}, "
#           f"mean={scores_arr.mean():.4f}")
#     print(f"  Score for positives : mean={pos_mean:.4f}")
#     print(f"  Score for negatives : mean={neg_mean:.4f}")
#     print(f"  Gap                 : {gap:.4f}  "
#           f"{'✅ good' if gap > 0.03 else '⚠️ small'}")

#     # Cross-validated threshold selection
#     idx          = np.random.RandomState(42).permutation(len(scores_arr))
#     mid          = len(idx) // 2
#     s_tr, l_tr   = scores_arr[idx[:mid]], labels_arr[idx[:mid]]
#     s_val, l_val = scores_arr[idx[mid:]], labels_arr[idx[mid:]]

#     fpr, tpr, thresholds_roc = roc_curve(l_tr, s_tr)
#     bal_acc    = (tpr + (1 - fpr)) / 2
#     thresh_bal = float(thresholds_roc[np.argmax(bal_acc)])
#     thresh_mid = float((s_tr[l_tr==1].mean() + s_tr[l_tr==0].mean()) / 2)
#     thresh_med = float(np.median(s_tr[l_tr==1]))

#     candidates = {k: v for k, v in {
#         "balanced_acc": thresh_bal,
#         "midpoint":     thresh_mid,
#         "median_pos":   thresh_med,
#     }.items() if not np.isinf(v) and not np.isnan(v)}

#     print(f"  Threshold candidates (val half):")
#     best_name, best_thresh, best_score = None, None, -1
#     for name, thresh in candidates.items():
#         preds = (s_val > thresh).astype(int)
#         score = balanced_accuracy_score(l_val, preds)
#         print(f"    {name:15s}: {thresh:.4f} → val bal_acc={score:.3f}")
#         if score > best_score:
#             best_score, best_thresh, best_name = score, thresh, name

#     print(f"  ✅ Chosen: {best_name} = {best_thresh:.4f} "
#           f"(val balanced acc = {best_score:.3f})")
#     return best_thresh


# # print("🔧 Calibrating CheXZero thresholds from train_df...")
# # print("="*60)
# # CHEXZERO_THRESHOLDS = {}
# # for label in TARGET_LABELS:
# #     CHEXZERO_THRESHOLDS[label] = calibrate_chexzero_threshold(label, n_samples=1000)

# # print("\n📊 CheXZero Final Thresholds:")
# # for label, thresh in CHEXZERO_THRESHOLDS.items():
# #     print(f"  {label}: {thresh:.4f}")

In [ ]:
# Runs AFTER ablation — CZERO_CONTEXT_STRATEGY holds the winning strategy.
# calibrate_chexzero_threshold() → build_czero_context() → CZERO_CONTEXT_STRATEGY.
# Threshold calibrated with the same context used at inference.
print("🔧 Calibrating CheXZero thresholds (post-ablation, winning strategy)...")
print("="*60)
CHEXZERO_THRESHOLDS = {}
for label in TARGET_LABELS:
    CHEXZERO_THRESHOLDS[label] = calibrate_chexzero_threshold(label, n_samples=1000)
print("\n📊 CheXZero Final Thresholds (calibrated with winning strategy):")
for label, thresh in CHEXZERO_THRESHOLDS.items():
    print(f"  {label}: {thresh:.4f}")
print(f"\n  Strategy used: {CZERO_CONTEXT_STRATEGY}")


# 5. CheXZero Zero Shot Benchmark

In [ ]:
# ================================================================
# CELL 53: CheXZero Benchmark
# ================================================================

def benchmark_chexzero():
    all_results = []
    labeler_df  = rule_labeler_df.copy()
    gt_df       = ground_truth_df.copy()

    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True)
                break

    for label in TARGET_LABELS:
        print(f"\n{'='*60}")
        print(f"  Label: {label}")
        print(f"{'='*60}")

        if label not in labeler_df.columns or label not in gt_df.columns:
            print(f"  ⚠️  '{label}' column missing — skipping")
            continue

        uncertain_idx = labeler_df[labeler_df[label] == -1].index
        if len(uncertain_idx) == 0:
            print(f"  ℹ️  No uncertain samples — skipping")
            continue

        gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
        gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
        gt_for_uncertain = gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]

        print(f"  Uncertain samples : {len(uncertain_idx)} out of {len(labeler_df)}")
        print(f"  With valid GT     : {len(gt_for_uncertain)} "
              f"(pos={int(gt_for_uncertain['gt'].sum())}, "
              f"neg={int((gt_for_uncertain['gt']==0).sum())})")

        top_features = top_features_per_label.get(label, [])

        for idx, row in tqdm(gt_for_uncertain.iterrows(),
                             total=len(gt_for_uncertain),
                             desc=f"CheXZero · {label}"):
            try:
                img_path       = fix_test_path(row["Study"])
                feature_values = get_sample_features(
                    row["Study"], label, top_features, rule_labeler_df
                )
                pred = infer_chexzero(img_path, label, feature_values)

                all_results.append({
                    "label":           label,
                    "study":           row["Study"],
                    "y_true":          int(row["gt"]),
                    "y_pred":          pred,
                    "feature_context": " | ".join(feature_values),
                })

            except Exception as e:
                all_results.append({
                    "label":           label,
                    "study":           row["Study"],
                    "y_true":          int(row["gt"]) if pd.notna(row.get("gt")) else -1,
                    "y_pred":          0,
                    "feature_context": f"ERROR: {e}",
                })

    return pd.DataFrame(all_results)

# 6. Run 

In [ ]:
# ================================================================
# CELL 54: Run CheXZero 
# ================================================================

print("🚀 Running uncertain label relabeling with CheXZero...")
print("="*80)

chexzero_results = benchmark_chexzero()
chexzero_results.to_csv("chexzero_uncertain_relabeling_results.csv", index=False)
print("✅ Saved: chexzero_uncertain_relabeling_results.csv")

chexzero_metrics = evaluate_results_with_details(chexzero_results)
print_detailed_results(chexzero_metrics)

# CheXZero Summary
print("\n" + "="*80)
print("📈 CheXZero SUMMARY")
print("="*80)
if len(chexzero_metrics) > 0:
    summary = chexzero_metrics[["label", "n_samples", "accuracy", "f1_score", "auc",
                                 "true_negative", "false_positive",
                                 "false_negative", "true_positive"]]
    print(summary.to_string(index=False))

    print("\nBalanced Accuracy per label:")
    for _, row in chexzero_metrics.iterrows():
        sens = row["true_positive"] / (row["true_positive"] + row["false_negative"]) \
               if (row["true_positive"] + row["false_negative"]) > 0 else 0
        spec = row["true_negative"] / (row["true_negative"] + row["false_positive"]) \
               if (row["true_negative"] + row["false_positive"]) > 0 else 0
        bal  = (sens + spec) / 2
        print(f"  {row['label']}: Sens={sens:.3f}, Spec={spec:.3f}, Balanced Acc={bal:.3f}")


# Compute Combined Prototypes

In [ ]:
CHEXZERO_N_SHOTS_LIST = [4, 8, 16, 32, 64]

def find_best_weights(label, n_shot=5, n_cal=200):
    """Find optimal txt/img weight on train_df — no test data"""
    
    df = train_df[train_df[label].isin([0, 1])].copy()
    # def path_exists(p):
    #     clean = p.replace("CheXpert-v1.0-small/", "").replace("CheXpert-v1.0/", "")
    #     return os.path.exists(os.path.join(BASE_PATH, clean))
    # df = df[df["Path"].apply(path_exists)].copy()

    # Calibration samples — separate from prototype samples
    proto_pos = df[df[label]==1].sample(n_shot, random_state=42)
    proto_neg = df[df[label]==0].sample(n_shot, random_state=42)
    proto_idx = set(proto_pos.index) | set(proto_neg.index)

    # Cal samples — exclude prototype rows
    cal_df    = df[~df.index.isin(proto_idx)]
    n_each    = min(n_cal//2, (cal_df[label]==1).sum(), (cal_df[label]==0).sum())
    cal_pos   = cal_df[cal_df[label]==1].sample(n_each, random_state=99)
    cal_neg   = cal_df[cal_df[label]==0].sample(n_each, random_state=99)
    cal_df    = pd.concat([cal_pos, cal_neg], ignore_index=True)

    # Get text embeddings
    def get_txt_emb(text):
        tokens = chexzero_tokenizer([text], truncate=True).to(device)
        with torch.no_grad():
            e = chexzero_model.encode_text(tokens)
            return e / e.norm(dim=-1, keepdim=True)

    # Get image embeddings for prototypes
    def get_img_embs(samples):
        embs = []
        for _, row in samples.iterrows():
            try:
                clean = row["Path"].replace(
                    "CheXpert-v1.0-small/", ""
                ).replace("CheXpert-v1.0/", "")
                img = chexzero_preprocess(
                    Image.open(os.path.join(BASE_PATH, clean)).convert("RGB")
                ).unsqueeze(0).to(device)
                with torch.no_grad():
                    e = chexzero_model.encode_image(img)
                    embs.append(e / e.norm(dim=-1, keepdim=True))
            except:
                continue
        return torch.cat(embs).mean(0, keepdim=True) if embs else None

    pos_img = get_img_embs(proto_pos)
    neg_img = get_img_embs(proto_neg)
    pos_txt = get_txt_emb(CHEXZERO_POSITIVE_PROMPTS[label])
    neg_txt = get_txt_emb(CHEXZERO_NEGATIVE_PROMPTS[label])

    # Try different weights — purely on train_df cal set
    best_w, best_score = 0.5, -1
    print(f"  {label}: searching best weight on train_df cal set...")

    for txt_w in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
        img_w      = 1.0 - txt_w
        pos_anchor = txt_w * pos_txt + img_w * pos_img
        neg_anchor = txt_w * neg_txt + img_w * neg_img
        pos_anchor = pos_anchor / pos_anchor.norm(dim=-1, keepdim=True)
        neg_anchor = neg_anchor / neg_anchor.norm(dim=-1, keepdim=True)

        preds, trues = [], []
        for _, row in cal_df.iterrows():
            try:
                clean    = row["Path"].replace(
                    "CheXpert-v1.0-small/", ""
                ).replace("CheXpert-v1.0/", "")
                img      = chexzero_preprocess(
                    Image.open(os.path.join(BASE_PATH, clean)).convert("RGB")
                ).unsqueeze(0).to(device)
                with torch.no_grad():
                    f = chexzero_model.encode_image(img)
                    f = f / f.norm(dim=-1, keepdim=True)
                pred = 1 if (f @ pos_anchor.T).item() > (f @ neg_anchor.T).item() else 0
                preds.append(pred)
                trues.append(int(row[label]))
            except:
                continue

        score = balanced_accuracy_score(trues, preds)
        print(f"    txt_w={txt_w:.1f}: bal_acc={score:.3f}")
        if score > best_score:
            best_score, best_w = score, txt_w

    print(f"  ✅ Best: txt_w={best_w:.1f} (bal_acc={best_score:.3f})")
    return best_w


def compute_label_weights(n_shot, n_cal=200):
    print("🔍 Finding optimal weights per label on train_df...")
    print(f"   n_shot = {n_shot}")
    print("="*60)
    label_weights = {}
    for label in TARGET_LABELS:
        best_txt_w = find_best_weights(label, n_shot=n_shot, n_cal=n_cal)
        label_weights[label] = (best_txt_w, 1.0 - best_txt_w)

    print("\n📊 Optimal weights per label:")
    for label, (tw, iw) in label_weights.items():
        print(f"  {label}: txt={tw:.1f}, img={iw:.1f}")
    return label_weights

In [ ]:
def get_combined_prototypes(label, n_shot=5, label_weights=None):
    df = train_df[train_df[label].isin([0, 1])].copy()

    # def path_exists(p):
    #     clean = p.replace("CheXpert-v1.0-small/", "").replace("CheXpert-v1.0/", "")
    #     return os.path.exists(os.path.join(BASE_PATH, clean))

    # df = df[df["Path"].apply(path_exists)].copy()

    pos_samples = df[df[label] == 1].sample(n_shot, random_state=42)
    neg_samples = df[df[label] == 0].sample(n_shot, random_state=42)

    print(f"  {label}: {len(pos_samples)} pos + {len(neg_samples)} neg prototypes")

    def get_image_embeddings(samples):
        embeddings = []
        for _, row in samples.iterrows():
            try:
                clean    = row["Path"].replace(
                    "CheXpert-v1.0-small/", ""
                ).replace("CheXpert-v1.0/", "")
                img_path = os.path.join(BASE_PATH, clean)
                img      = chexzero_preprocess(
                    Image.open(img_path).convert("RGB")
                ).unsqueeze(0).to(device)
                with torch.no_grad():
                    emb = chexzero_model.encode_image(img)
                    emb = emb / emb.norm(dim=-1, keepdim=True)
                embeddings.append(emb)
            except Exception as e:
                print(f"    Image error: {e}")
                continue
        if not embeddings:
            return None
        return torch.cat(embeddings, dim=0).mean(dim=0, keepdim=True)

    def get_text_embedding(text):
        tokens = chexzero_tokenizer([text], truncate=True).to(device)
        with torch.no_grad():
            emb = chexzero_model.encode_text(tokens)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        return emb

    pos_img_emb = get_image_embeddings(pos_samples)
    neg_img_emb = get_image_embeddings(neg_samples)
    # Build mean-pooled contextual text embeddings from the prototype samples.
    # At inference time, each test image gets its own context; here we
    # approximate by averaging contexts of prototype train samples.
    def _mean_ctx_txt_emb(samples, polarity):
        top_f = top_features_per_label.get(label, [])
        embs  = []
        for _, r in samples.iterrows():
            try:
                fv  = get_sample_features(r["Path"], label, top_f, train_df)
                ctx = build_czero_context(label, fv)
                prompt = ctx + (CHEXZERO_POSITIVE_PROMPTS[label]
                                if polarity == "pos"
                                else CHEXZERO_NEGATIVE_PROMPTS[label])
                embs.append(get_text_embedding(prompt))
            except Exception:
                continue
        if not embs:
            return get_text_embedding(CHEXZERO_POSITIVE_PROMPTS[label]
                                      if polarity == "pos"
                                      else CHEXZERO_NEGATIVE_PROMPTS[label])
        stacked = torch.cat(embs, dim=0).mean(dim=0, keepdim=True)
        return stacked / stacked.norm(dim=-1, keepdim=True)

    pos_txt_emb = _mean_ctx_txt_emb(pos_samples, "pos")
    neg_txt_emb = _mean_ctx_txt_emb(neg_samples, "neg")

    if pos_img_emb is None or neg_img_emb is None:
        print(f"  ⚠️  Could not load prototype images for {label}")
        return None, None, set()

    if label_weights is None:
        if "LABEL_WEIGHTS" not in globals():
            raise ValueError("LABEL_WEIGHTS not set; pass label_weights.")
        label_weights = LABEL_WEIGHTS

    # ── Use train_df optimized weights — no hardcoding ──
    txt_weight, img_weight = label_weights[label]
    print(f"  {label}: using txt_w={txt_weight:.1f}, img_w={img_weight:.1f}")

    pos_combined = (txt_weight * pos_txt_emb) + (img_weight * pos_img_emb)
    neg_combined = (txt_weight * neg_txt_emb) + (img_weight * neg_img_emb)
    pos_combined = pos_combined / pos_combined.norm(dim=-1, keepdim=True)
    neg_combined = neg_combined / neg_combined.norm(dim=-1, keepdim=True)

    return pos_combined, neg_combined, set()


def build_chexzero_prototypes(n_shot, label_weights):
    print("Computing prototypes with optimized weights...")
    print(f"   n_shot = {n_shot}")
    print("="*60)
    combined_proto = {}
    proto_excl = {}

    for label in TARGET_LABELS:
        pos_proto, neg_proto, used = get_combined_prototypes(
            label, n_shot=n_shot, label_weights=label_weights
        )
        combined_proto[label] = (pos_proto, neg_proto)
        proto_excl[label] = used
        print(f"  ✅ {label}: done")

    return combined_proto, proto_excl

# CheXZero Few-Shot Threshold Calibration Helper

In [ ]:
# ================================================================
# CheXZero Few-Shot — Combined Score Threshold Calibration
#
# THRESHOLD POLICY:
#   For each (label, n_shots, txt_w) configuration, calibrate a
#   threshold on the combined score from train_df certain samples.
#   Prototype samples (seed=42) are excluded from the calibration set.
#   Calibration uses seed=99 to pick different cal samples.
#
# This gives paper-valid binary metrics (Acc/F1/Sens/Spec) for the
# few-shot setting. AUC remains threshold-free (primary metric).
# ================================================================

def calibrate_chexzero_combined_threshold(label: str,
                                           n_shot: int,
                                           txt_w: float,
                                           label_weights: dict,
                                           n_cal: int = 300,
                                           cal_seed: int = 99) -> float:
    """
    Calibrate threshold for CheXZero combined (text+proto) score on train_df.
    Excludes prototype samples to prevent leakage.

    Returns optimal threshold (float).
    """
    from sklearn.metrics import roc_curve, balanced_accuracy_score

    df = train_df[train_df[label].isin([0, 1])].copy()
    img_w = 1.0 - txt_w

    # Identify prototype indices (seed=42 — same as prototype builder)
    n_pos_avail = int((df[label] == 1).sum())
    n_neg_avail = int((df[label] == 0).sum())
    proto_pos = df[df[label]==1].sample(n_shot, random_state=42)
    proto_neg = df[df[label]==0].sample(n_shot, random_state=42)
    proto_idx = set(proto_pos.index) | set(proto_neg.index)

    cal_df = df[~df.index.isin(proto_idx)]
    n_each = min(n_cal // 2, (cal_df[label]==1).sum(), (cal_df[label]==0).sum())
    if n_each < 10:
        return 0.0

    cal_pos = cal_df[cal_df[label]==1].sample(n_each, random_state=cal_seed)
    cal_neg = cal_df[cal_df[label]==0].sample(n_each, random_state=cal_seed)
    cal_df  = pd.concat([cal_pos, cal_neg], ignore_index=True)

    # Build prototypes (same as sweep)
    pos_txt_emb = None; neg_txt_emb = None
    pos_img_proto = None; neg_img_proto = None
    try:
        def get_txt_emb(text):
            tokens = chexzero_tokenizer([text], truncate=True).to(device)
            with torch.no_grad():
                e = chexzero_model.encode_text(tokens)
                return e / e.norm(dim=-1, keepdim=True)

        def get_img_embs(samples):
            embs = []
            for _, row in samples.iterrows():
                try:
                    clean = row["Path"].replace("CheXpert-v1.0-small/","").replace("CheXpert-v1.0/","")
                    img = chexzero_preprocess(
                        Image.open(os.path.join(BASE_PATH, clean)).convert("RGB")
                    ).unsqueeze(0).to(device)
                    with torch.no_grad():
                        e = chexzero_model.encode_image(img)
                        embs.append(e / e.norm(dim=-1, keepdim=True))
                except: continue
            return torch.cat(embs).mean(0, keepdim=True) if embs else None

        # We use mean-pooled contextual text embeddings from cal samples.
        # This matches what happens at inference (context prepended per sample).
        def get_mean_contextual_txt_emb(df_samples, polarity):
            embs = []
            top_f = top_features_per_label.get(label, [])
            for _, r in df_samples.iterrows():
                try:
                    fv  = get_sample_features(r["Path"], label, top_f, train_df)
                    ctx = build_czero_context(label, fv)
                    prompt = ctx + (CHEXZERO_POSITIVE_PROMPTS[label]
                                    if polarity == "pos"
                                    else CHEXZERO_NEGATIVE_PROMPTS[label])
                    embs.append(get_txt_emb(prompt))
                except Exception:
                    continue
            if not embs:
                return get_txt_emb(CHEXZERO_POSITIVE_PROMPTS[label]
                                   if polarity == "pos"
                                   else CHEXZERO_NEGATIVE_PROMPTS[label])
            stacked = torch.cat(embs, dim=0).mean(dim=0, keepdim=True)
            return stacked / stacked.norm(dim=-1, keepdim=True)

        all_cal = pd.concat([cal_pos, cal_neg], ignore_index=True)
        pos_txt_emb = get_mean_contextual_txt_emb(all_cal[all_cal[label]==1], "pos")
        neg_txt_emb = get_mean_contextual_txt_emb(all_cal[all_cal[label]==0], "neg")
        pos_img_proto = get_img_embs(proto_pos)
        neg_img_proto = get_img_embs(proto_neg)
    except Exception as e:
        print(f"    ⚠️  Prototype build failed for calibration: {e}")
        return 0.0

    if pos_img_proto is None or neg_img_proto is None:
        return 0.0

    pos_anchor = txt_w * pos_txt_emb + img_w * pos_img_proto
    neg_anchor = txt_w * neg_txt_emb + img_w * neg_img_proto
    pos_anchor = pos_anchor / pos_anchor.norm(dim=-1, keepdim=True)
    neg_anchor = neg_anchor / neg_anchor.norm(dim=-1, keepdim=True)

    scores, labels_list = [], []
    for _, row in cal_df.iterrows():
        try:
            clean = row["Path"].replace("CheXpert-v1.0-small/","").replace("CheXpert-v1.0/","")
            img = chexzero_preprocess(
                Image.open(os.path.join(BASE_PATH, clean)).convert("RGB")
            ).unsqueeze(0).to(device)
            with torch.no_grad():
                f = chexzero_model.encode_image(img)
                f = f / f.norm(dim=-1, keepdim=True)
            # combined score = sim(img, pos_anchor) - sim(img, neg_anchor)
            combined_score = (f @ pos_anchor.T).item() - (f @ neg_anchor.T).item()
            scores.append(combined_score)
            labels_list.append(int(row[label]))
        except: continue

    if len(scores) < 20:
        return 0.0

    scores_arr = np.array(scores)
    labels_arr = np.array(labels_list)

    idx = np.random.RandomState(42).permutation(len(scores_arr))
    mid = len(idx) // 2
    s_tr, l_tr   = scores_arr[idx[:mid]], labels_arr[idx[:mid]]
    s_val, l_val = scores_arr[idx[mid:]], labels_arr[idx[mid:]]

    try:
        fpr, tpr, thrs = roc_curve(l_tr, s_tr)
        thresh_bal = float(thrs[np.argmax((tpr + (1-fpr)) / 2)])
    except:
        thresh_bal = 0.0

    thresh_mid = float((s_tr[l_tr==1].mean() + s_tr[l_tr==0].mean()) / 2)

    best_t, best_s = thresh_mid, -1
    for t in [thresh_bal, thresh_mid]:
        if np.isinf(t) or np.isnan(t): continue
        preds = (s_val > t).astype(int)
        sc = balanced_accuracy_score(l_val, preds)
        if sc > best_s:
            best_s, best_t = sc, t

    return float(best_t)

# Storage: CZFS_THRESHOLDS[label][(n_shot, txt_w)] = threshold
CZFS_THRESHOLDS = {}
print("✅ CheXZero few-shot threshold calibration helper ready.")
print("   Will calibrate per (label, n_shot, txt_w) during the sweep.")


# Few-Shot Inference Function

In [ ]:
# ================================================================
# CheXZero Few-Shot Inference
#
# Returns combined SCORE (float), not binary prediction.
# Thresholding is done in the benchmark loop using CZFS_THRESHOLDS.
# This separation allows:
#   (a) AUC computation (threshold-free)
#   (b) Binary metrics using the train_df-calibrated FS threshold
# ================================================================

def score_chexzero_combined(img_path: str, label: str) -> float:
    """
    Returns the combined score: sim(img, pos_anchor) - sim(img, neg_anchor).
    Positive → predict positive, but thresholding is done externally.
    """
    try:
        image = chexzero_preprocess(
            Image.open(img_path).convert("RGB")
        ).unsqueeze(0).to(device)

        with torch.no_grad():
            img_f = chexzero_model.encode_image(image)
            img_f = img_f / img_f.norm(dim=-1, keepdim=True)

        pos_proto, neg_proto = CHEXZERO_COMBINED_PROTO[label]
        pos_score = (img_f @ pos_proto.T).item()
        neg_score = (img_f @ neg_proto.T).item()

        return pos_score - neg_score   # combined score

    except Exception as e:
        print(f"  [score_chexzero_combined] Error: {e}")
        return 0.0


def infer_chexzero_combined(img_path: str, label: str,
                             threshold: float = None) -> int:
    """
    Binary prediction using combined score.
    threshold: if None, uses midpoint 0.0 (relative comparison).
               For paper reporting, pass CZFS_THRESHOLDS[label][(n_shot, txt_w)].
    """
    score = score_chexzero_combined(img_path, label)
    t = threshold if threshold is not None else 0.0
    return 1 if score > t else 0


# Few-Shot Benchmark

In [ ]:
# ================================================================
# CheXZero Few-Shot Benchmark — Score-Based with FS Threshold
#
# THRESHOLD POLICY:
#   1. Collects combined scores on GT-uncertain samples (no threshold)
#   2. Applies CZFS_THRESHOLDS[label][(n_shot, txt_w)] for binary metrics
#   3. Also computes AUC (threshold-free)
#   4. Excludes prototype samples from test set
# ================================================================

def benchmark_chexzero_combined(n_shot: int, label_weights: dict):
    """
    Run few-shot benchmark for a given n_shot configuration.

    Args:
        n_shot        : number of prototypes per class
        label_weights : {label: (txt_w, img_w)} from find_best_weights

    Returns DataFrame with scores + binary predictions.
    """
    from sklearn.metrics import roc_auc_score

    all_results = []
    labeler_df  = rule_labeler_df.copy()
    gt_df       = ground_truth_df.copy()

    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True)
                break

    for label in TARGET_LABELS:
        print(f"\n{'='*60}")
        print(f"  Label: {label}")
        print(f"{'='*60}")

        if label not in labeler_df.columns or label not in gt_df.columns:
            print(f"  ⚠️  '{label}' column missing — skipping")
            continue

        uncertain_idx    = labeler_df[labeler_df[label] == -1].index
        gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
        gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
        gt_for_uncertain = gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]

        # Exclude prototype samples
        excluded         = CHEXZERO_PROTO_EXCL.get(label, set())
        before           = len(gt_for_uncertain)
        gt_for_uncertain = gt_for_uncertain[~gt_for_uncertain["Study"].isin(excluded)]
        after = len(gt_for_uncertain)
        print(f"  Samples: {before} → {after} (excluded {before-after} prototypes)")
        print(f"  pos={int(gt_for_uncertain['gt'].sum())}, "
              f"neg={int((gt_for_uncertain['gt']==0).sum())}")

        # Get calibrated FS threshold for this (label, n_shot, txt_w)
        txt_w = label_weights.get(label, (0.5, 0.5))[0]
        fs_key = (n_shot, txt_w)
        if label in CZFS_THRESHOLDS and fs_key in CZFS_THRESHOLDS[label]:
            fs_thresh = CZFS_THRESHOLDS[label][fs_key]
        else:
            print(f"  ⚠️  FS threshold not calibrated for {label} "
                  f"(n={n_shot}, txt_w={txt_w:.1f}) — calibrating now...")
            fs_thresh = calibrate_chexzero_combined_threshold(
                label, n_shot, txt_w, label_weights, n_cal=1000)
            if label not in CZFS_THRESHOLDS:
                CZFS_THRESHOLDS[label] = {}
            CZFS_THRESHOLDS[label][fs_key] = fs_thresh

        print(f"  FS threshold (train_df calibrated): {fs_thresh:.4f}")

        scores, y_trues = [], []
        for idx, row in tqdm(gt_for_uncertain.iterrows(),
                             total=len(gt_for_uncertain),
                             desc=f"CheXZero+FS · {label}"):
            try:
                img_path = fix_test_path(row["Study"])
                score    = score_chexzero_combined(img_path, label)
                pred     = 1 if score > fs_thresh else 0
                scores.append(score)
                y_trues.append(int(row["gt"]))
                all_results.append({
                    "label":  label, "study":  row["Study"],
                    "y_true": int(row["gt"]), "y_pred": pred,
                    "score":  score, "threshold": fs_thresh,
                })
            except Exception as e:
                all_results.append({
                    "label":  label, "study":  row["Study"],
                    "y_true": int(row["gt"]) if pd.notna(row.get("gt")) else -1,
                    "y_pred": 0, "score": 0.0, "threshold": fs_thresh,
                })

        if len(scores) >= 4 and len(np.unique(y_trues)) > 1:
            auc = roc_auc_score(y_trues, scores)
            print(f"  AUC (threshold-free) : {auc:.3f}")
            print(f"  Threshold source     : train_df combined-score calibration")

    return pd.DataFrame(all_results)


#  Run + Compare Zero-Shot vs Few-Shot

In [ ]:
# ================================================================
# CheXZero Few-Shot: N-Shot Sweep with Calibrated FS Thresholds
#
# THRESHOLD POLICY:
#   For each (label, n_shot, txt_w):
#     → calibrate_chexzero_combined_threshold() on train_df
#     → store in CZFS_THRESHOLDS[label][(n_shot, txt_w)]
#   AUC: threshold-free (primary)
#   Acc/F1/Sens/Spec: use CZFS_THRESHOLDS
# ================================================================

print("🚀 Running CheXZero Text+Image Few-Shot sweep...")
print("="*80)

chexzero_fewshot_summary = []
chexzero_fewshot_metrics_by_shot = {}

for n_shot in CHEXZERO_N_SHOTS_LIST:
    print("\n" + "="*80)
    print(f"CheXZero few-shot: {n_shot}-shot")
    print("="*80)

    LABEL_WEIGHTS = compute_label_weights(n_shot, n_cal=200)

    # Calibrate FS thresholds for this n_shot
    print(f"\n🔧 Calibrating few-shot combined-score thresholds "
          f"for {n_shot}-shot...")
    for label in TARGET_LABELS:
        txt_w = LABEL_WEIGHTS.get(label, (0.5, 0.5))[0]
        print(f"  [{label}] txt_w={txt_w:.1f} ...", end="", flush=True)
        thresh = calibrate_chexzero_combined_threshold(
            label, n_shot, txt_w, LABEL_WEIGHTS, n_cal=1000)
        if label not in CZFS_THRESHOLDS:
            CZFS_THRESHOLDS[label] = {}
        CZFS_THRESHOLDS[label][(n_shot, txt_w)] = thresh
        print(f" threshold={thresh:.4f}")

    CHEXZERO_COMBINED_PROTO, CHEXZERO_PROTO_EXCL = build_chexzero_prototypes(
        n_shot, LABEL_WEIGHTS
    )

    chexzero_fs_results = benchmark_chexzero_combined(n_shot, LABEL_WEIGHTS)
    out_csv = f"chexzero_fewshot_results_{n_shot}shot.csv"
    chexzero_fs_results.to_csv(out_csv, index=False)
    print(f"✅ Saved: {out_csv}")

    chexzero_fs_metrics = evaluate_results_with_details(chexzero_fs_results)
    chexzero_fewshot_metrics_by_shot[n_shot] = chexzero_fs_metrics

    if len(chexzero_fs_metrics) == 0:
        chexzero_fewshot_summary.append({
            "n_shots": n_shot, "mean_auc": 0.0, "mean_bal_acc": 0.0, "mean_score": 0.0})
        continue

    print_detailed_results(chexzero_fs_metrics)

    summary = chexzero_fs_metrics[[
        "label", "n_samples", "accuracy", "f1_score", "auc",
        "true_negative", "false_positive", "false_negative", "true_positive"
    ]]
    print("\n" + "-"*80)
    print(f"CheXZero {n_shot}-shot summary")
    print("-"*80)
    print(summary.to_string(index=False))

    mean_auc = chexzero_fs_metrics["auc"].mean()
    mean_bal = chexzero_fs_metrics["bal_acc"].mean()
    mean_score = (mean_auc + mean_bal) / 2
    chexzero_fewshot_summary.append({
        "n_shots": n_shot, "mean_auc": mean_auc, "mean_bal_acc": mean_bal, "mean_score": mean_score})
    print(f"\n  Mean AUC      : {mean_auc:.3f}")
    print(f"  Mean Bal Acc  : {mean_bal:.3f}")
    print(f"  Mean Score    : {mean_score:.3f}")

summary_df = (pd.DataFrame(chexzero_fewshot_summary)
              .sort_values("n_shots").reset_index(drop=True))

print("\n" + "="*80)
print("📊 CheXZero Few-Shot Summary by Shot")
print("  Note: Score = (AUC + BA) / 2")
print("  Acc/F1/Sens/Spec = train_df-calibrated FS threshold")
print("="*80)
print(summary_df.to_string(index=False))

if len(summary_df) > 0:
    best_idx  = summary_df["mean_score"].idxmax()
    best_row  = summary_df.loc[best_idx]
    best_shot = int(best_row["n_shots"])
    print(f"\nBest by mean score: {best_shot}-shot "
          f"(mean score = {best_row['mean_score']:.3f})")

In [ ]:
# ================================================================
# CheXZero Best Few-Shot vs Zero-Shot
# ================================================================

if "summary_df" not in globals() or len(summary_df) == 0:
    raise ValueError("summary_df is missing. Run the few-shot sweep cell first.")

best_idx  = summary_df["mean_score"].idxmax()
best_shot = int(summary_df.loc[best_idx, "n_shots"])

print("="*80)
print(f"CheXZero best config: {best_shot}-shot")
print("="*80)

LABEL_WEIGHTS = compute_label_weights(best_shot, n_cal=200)
CHEXZERO_COMBINED_PROTO, CHEXZERO_PROTO_EXCL = build_chexzero_prototypes(
    best_shot, LABEL_WEIGHTS
)

chexzero_best_results = benchmark_chexzero_combined(best_shot, LABEL_WEIGHTS)
best_csv = f"chexzero_fewshot_best_{best_shot}shot.csv"
chexzero_best_results.to_csv(best_csv, index=False)
print(f"✅ Saved: {best_csv}")

chexzero_best_metrics = evaluate_results_with_details(chexzero_best_results)
print_detailed_results(chexzero_best_metrics)

if "chexzero_metrics" not in globals() or len(chexzero_metrics) == 0:
    print("⚠️  chexzero_metrics missing. Run zero-shot CheXZero benchmark first.")
else:
    print("\n" + "="*80)
    print(f"📊 CheXZero: Zero-Shot vs Few-Shot ({best_shot}-shot)")
    print(f"   ZS: train_df ZS-calibrated threshold (text score)")
    print(f"   FS: train_df FS-calibrated threshold (combined score)")
    print("="*80)
    print(f"  {'Label':<30} "
          f"{'ZS AUC':>7} {'FS AUC':>7} {'ΔAUC':>7} "
          f"{'ZS BA':>7} {'FS BA':>7} {'ΔBA':>7}")
    print(f"  {'':<30} "
          f"{'ZS AUPRC':>8} {'FS AUPRC':>8} {'ΔAUPRC':>7} "
          f"{'ZS F1':>7} {'FS F1':>7} {'ΔF1':>7} "
          f"{'ZS Se':>7} {'FS Se':>7} {'ΔSe':>7} "
          f"{'ZS Sp':>7} {'FS Sp':>7} {'ΔSp':>7}")
    print("  " + "-" * 125)

    def _metric(row, key, fallback_key=None):
        if key in row:
            return float(row[key])
        if fallback_key and fallback_key in row:
            return float(row[fallback_key])
        return 0.0

    zs_auc, fs_auc = [], []
    zs_ba, fs_ba = [], []
    zs_auprc, fs_auprc = [], []
    zs_f1, fs_f1 = [], []
    zs_sens, fs_sens = [], []
    zs_spec, fs_spec = [], []
    for label in TARGET_LABELS:
        z_row = chexzero_metrics[chexzero_metrics["label"] == label]
        f_row = chexzero_best_metrics[chexzero_best_metrics["label"] == label]
        if len(z_row) == 0 or len(f_row) == 0:
            continue
        z = z_row.iloc[0]
        f = f_row.iloc[0]

        z_auc   = _metric(z, "auc");                    f_auc   = _metric(f, "auc")
        z_ba    = _metric(z, "bal_acc", "balanced_acc"); f_ba    = _metric(f, "bal_acc", "balanced_acc")
        z_auprc = _metric(z, "auprc");                  f_auprc = _metric(f, "auprc")
        z_f1    = _metric(z, "f1", "f1_score");          f_f1    = _metric(f, "f1", "f1_score")
        z_se    = _metric(z, "sens", "sensitivity");     f_se    = _metric(f, "sens", "sensitivity")
        z_sp    = _metric(z, "spec", "specificity");     f_sp    = _metric(f, "spec", "specificity")

        zs_auc.append(z_auc);     fs_auc.append(f_auc)
        zs_ba.append(z_ba);       fs_ba.append(f_ba)
        zs_auprc.append(z_auprc); fs_auprc.append(f_auprc)
        zs_f1.append(z_f1);       fs_f1.append(f_f1)
        zs_sens.append(z_se);     fs_sens.append(f_se)
        zs_spec.append(z_sp);     fs_spec.append(f_sp)

        print(f"  {label:<30} "
              f"{z_auc:>7.3f} {f_auc:>7.3f} {f_auc - z_auc:>+7.3f} "
              f"{z_ba:>7.3f} {f_ba:>7.3f} {f_ba - z_ba:>+7.3f}")
        print(f"  {'':<30} "
              f"{z_auprc:>8.3f} {f_auprc:>8.3f} {f_auprc - z_auprc:>+7.3f} "
              f"{z_f1:>7.3f} {f_f1:>7.3f} {f_f1 - z_f1:>+7.3f} "
              f"{z_se:>7.3f} {f_se:>7.3f} {f_se - z_se:>+7.3f} "
              f"{z_sp:>7.3f} {f_sp:>7.3f} {f_sp - z_sp:>+7.3f}")
    print("  " + "-" * 125)
    if zs_auc:
        avg_z_auc   = float(np.mean(zs_auc));   avg_f_auc   = float(np.mean(fs_auc))
        avg_z_ba    = float(np.mean(zs_ba));    avg_f_ba    = float(np.mean(fs_ba))
        avg_z_auprc = float(np.mean(zs_auprc)); avg_f_auprc = float(np.mean(fs_auprc))
        avg_z_f1    = float(np.mean(zs_f1));    avg_f_f1    = float(np.mean(fs_f1))
        avg_z_se    = float(np.mean(zs_sens));  avg_f_se    = float(np.mean(fs_sens))
        avg_z_sp    = float(np.mean(zs_spec));  avg_f_sp    = float(np.mean(fs_spec))
        print(f"  {'Average':<30} "
              f"{avg_z_auc:>7.3f} {avg_f_auc:>7.3f} {avg_f_auc - avg_z_auc:>+7.3f} "
              f"{avg_z_ba:>7.3f} {avg_f_ba:>7.3f} {avg_f_ba - avg_z_ba:>+7.3f}")
        print(f"  {'':<30} "
              f"{avg_z_auprc:>8.3f} {avg_f_auprc:>8.3f} {avg_f_auprc - avg_z_auprc:>+7.3f} "
              f"{avg_z_f1:>7.3f} {avg_f_f1:>7.3f} {avg_f_f1 - avg_z_f1:>+7.3f} "
              f"{avg_z_se:>7.3f} {avg_f_se:>7.3f} {avg_f_se - avg_z_se:>+7.3f} "
              f"{avg_z_sp:>7.3f} {avg_f_sp:>7.3f} {avg_f_sp - avg_z_sp:>+7.3f}")
        delta = avg_f_auc - avg_z_auc
        print(f"\n  Verdict: {'Few-Shot better ✅' if delta > 0 else 'Zero-Shot better ❌'}")

# Load BioViLT

In [ ]:
# ================================================================
# CELL 60: Load BioViLT (Biomedical Vision-Language Transformer)
# - BioViL-T: Trained on biomedical images (MIMIC-CXR) and radiology reports  
# - Uses health_multimodal package from Microsoft Research
# - Proper multimodal image-text inference engine
# ================================================================


from health_multimodal.text import get_bert_inference
from health_multimodal.text.utils import BertEncoderType
from health_multimodal.image import get_image_inference
from health_multimodal.image.utils import ImageModelType
from health_multimodal.vlp import ImageTextInferenceEngine

print("Loading BioViL-T models...")
biovilt_text_inference = get_bert_inference(BertEncoderType.BIOVIL_T_BERT)
biovilt_image_inference = get_image_inference(ImageModelType.BIOVIL_T)

biovilt_engine = ImageTextInferenceEngine(
    image_inference_engine=biovilt_image_inference,
    text_inference_engine=biovilt_text_inference
)

biovilt_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
biovilt_engine.to(biovilt_device)
print(f"✅ BioViL-T loaded on {biovilt_device}")

# BioViLT Prompts

In [ ]:
BIOVILT_POSITIVE_PROMPTS = {
    
    # Edema: current prompts are directionally correct, minor strengthening only
    "Edema": [
        "pulmonary edema with bilateral perihilar hazy opacities",
        "interstitial pulmonary edema showing Kerley B lines and peribronchial cuffing",
        "alveolar pulmonary edema with diffuse bilateral airspace consolidation",
        "vascular congestion with upper lobe blood diversion and interstitial thickening",
        "bilateral ground glass opacities consistent with pulmonary edema",
    ],

    # Atelectasis: make MORE SPECIFIC to reduce false positives
    # Current prompts match too many images — add location/mechanism specificity
    "Atelectasis": [
        "linear plate-like atelectasis at the lung base with horizontal opacity",
        "subsegmental atelectasis showing discoid opacity parallel to diaphragm",
        "lobar atelectasis with ipsilateral mediastinal shift and fissure displacement",
        "passive atelectasis with volume loss and ipsilateral hemidiaphragm elevation",
        "compressive atelectasis adjacent to pleural effusion at lung base",
    ],

    # Pleural Effusion: current prompts have embedding bias — rewrite with
    # stronger, more unambiguous radiological language
    "Pleural Effusion": [
        "moderate to large pleural effusion with meniscus sign and blunted costophrenic angle",
        "layering pleural fluid seen as homogeneous basal opacity with upward concave border",
        "bilateral pleural effusions with bilateral costophrenic angle blunting",
        "pleural fluid collection causing passive atelectasis of the adjacent lung",
        "subpulmonic pleural effusion with elevated hemidiaphragm appearance",
    ],

    # Enlarged Cardiomediastinum: add specific anatomical measurements and comparisons
    # Current prompts are too vague — model needs stronger anchors
    "Enlarged Cardiomediastinum": [
        "cardiomegaly with cardiothoracic ratio greater than 0.5 on PA chest radiograph",
        "enlarged cardiac silhouette with bilateral extension beyond normal cardiac borders",
        "mediastinal widening with right paratracheal stripe enlargement",
        "enlarged cardiac shadow with prominent left ventricular contour",
        "dilated cardiomyopathy appearance with globular cardiac enlargement",
    ],

    # Add to BIOVILT_POSITIVE_PROMPTS:
    "Consolidation": [
        "dense homogeneous lobar airspace consolidation with complete opacification of lung segment",
        "airspace consolidation with air bronchograms and silhouette sign at cardiac border",
        "segmental consolidation with volume preservation and surrounding ground glass haze",
        "lobar consolidation with ipsilateral fissure bowing and air bronchogram pattern",
        "sublobar consolidation with wedge-shaped opacity and pleural-based density",
    ],
}

BIOVILT_NEGATIVE_PROMPTS = {

    # Edema: make negative prompts as descriptively specific as positives
    "Edema": [
        "clear lung fields with sharp pulmonary vascular markings and no interstitial thickening",
        "no perihilar haze, no Kerley B lines, and no peribronchial cuffing",
        "lung fields are well aerated with normal vascular redistribution pattern",
        "no alveolar or interstitial opacities, lung parenchyma appears clear",
        "normal pulmonary vascularity without upper lobe diversion or congestion",
    ],

    # Atelectasis: mirror the specificity of new positive prompts
    "Atelectasis": [
        "lung bases are clear with no horizontal linear opacities or plate-like densities",
        "lungs are fully expanded bilaterally with no subsegmental or discoid opacities",
        "no volume loss, hemidiaphragms in normal position, no fissure displacement",
        "no lobar collapse, mediastinum is midline without shift",
        "no compressive or passive atelectasis identified at the lung bases",
    ],

    # Pleural Effusion: match the stronger positive prompt language
    "Pleural Effusion": [
        "sharp bilateral costophrenic angles with no blunting or meniscus sign",
        "lung bases are clear with no layering fluid or basal opacity",
        "hemidiaphragm contours are normal with no subpulmonic fluid collection",
        "no pleural fluid identified, costophrenic angles are acute bilaterally",
        "no pleural effusion, lung bases show normal aeration to the diaphragm",
    ],

    # Enlarged Cardiomediastinum: match the anatomical specificity
    "Enlarged Cardiomediastinum": [
        "normal cardiothoracic ratio less than 0.5 with normal cardiac borders",
        "cardiac silhouette is normal in size with no left ventricular enlargement",
        "mediastinum is normal in width with no paratracheal stripe widening",
        "heart size is within normal limits on this frontal chest radiograph",
        "normal cardiac contours without globular enlargement or cardiomegaly",
    ],
    # Add to BIOVILT_NEGATIVE_PROMPTS:
    "Consolidation": [
        "lungs are clear with no airspace consolidation or lobar opacity identified",
        "no consolidation, air bronchograms or silhouette sign, lung fields fully aerated",
        "no focal airspace disease, pulmonary vascularity normal throughout both lungs",
        "no segmental or lobar opacity, hemidiaphragms and costophrenic angles are sharp",
        "clear lung fields bilaterally with no consolidative process identified",
    ],
}

# BioViLT Inference Function

In [ ]:
# ================================================================
# Feature → Radiology Report Language Converter
# Maps CheXpert feature names + values to natural clinical language
# that matches BioViL-T's pretraining distribution (MIMIC-CXR reports)
# ================================================================

FEATURE_TO_RADIOLOGY_LANGUAGE = {
    "Enlarged Cardiomediastinum": {
        "positive":  "The cardiomediastinal silhouette is enlarged.",
        "negative":  "The cardiomediastinal silhouette is within normal limits.",
        "uncertain": "The cardiomediastinal silhouette may be mildly prominent.",
    },
    "Cardiomegaly": {
        "positive":  "Cardiomegaly is present with an increased cardiothoracic ratio.",
        "negative":  "The cardiac size is normal.",
        "uncertain": "The cardiac size is borderline.",
    },
    "Lung Opacity": {
        "positive":  "There is lung opacity present.",
        "negative":  "The lung fields are clear without opacity.",
        "uncertain": "There is possible subtle lung opacity.",
    },
    "Lung Lesion": {
        "positive":  "A lung lesion is identified.",
        "negative":  "No lung lesion is identified.",
        "uncertain": "A possible lung lesion cannot be excluded.",
    },
    "Edema": {
        "positive":  "Pulmonary edema is present with perihilar hazy infiltrates.",
        "negative":  "No pulmonary edema is identified.",
        "uncertain": "Mild pulmonary edema cannot be excluded.",
    },
    "Consolidation": {
        "positive":  "There is airspace consolidation present.",
        "negative":  "No consolidation is identified.",
        "uncertain": "Possible early consolidation cannot be excluded.",
    },
    "Atelectasis": {
        "positive":  "Atelectasis is present at the lung base.",
        "negative":  "No atelectasis is identified.",
        "uncertain": "Subsegmental atelectasis cannot be excluded.",
    },
    "Pneumothorax": {
        "positive":  "Pneumothorax is present.",
        "negative":  "No pneumothorax is identified.",
        "uncertain": "A small pneumothorax cannot be excluded.",
    },
    "Pleural Effusion": {
        "positive":  "There is pleural effusion with blunting of the costophrenic angle.",
        "negative":  "No pleural effusion is identified. The costophrenic angles are sharp.",
        "uncertain": "A small pleural effusion cannot be excluded.",
    },
    "Pleural Other": {
        "positive":  "There is pleural thickening or other pleural abnormality.",
        "negative":  "No pleural abnormality is identified.",
        "uncertain": "Pleural abnormality cannot be excluded.",
    },
    "Fracture": {
        "positive":  "A rib or bony fracture is identified.",
        "negative":  "No acute fracture is identified.",
        "uncertain": "A fracture cannot be excluded.",
    },
    "Support Devices": {
        "positive":  "Support devices are present including lines and tubes.",
        "negative":  "No support devices are identified.",
        "uncertain": "Support device positioning is difficult to assess.",
    },
    "No Finding": {
        "positive":  "No acute cardiopulmonary finding is identified.",
        "negative":  "Findings are present on this examination.",
        "uncertain": "Findings are equivocal on this examination.",
    },
}


def features_to_radiology_context(feature_values: list, target_label: str) -> str:
    """
    Convert feature_values list into natural radiology report language.

    Design decisions:
    1. Skip the target label itself (avoid circular reasoning).
    2. For uncertain (-1) features: include hedging language ("cannot be
       excluded") only for clinically relevant features (same gate as
       negatives). This is in-distribution for MIMIC-CXR trained models.
       Fractional values (0 < val < 1) are always skipped.
    3. For negative findings, only include ones that are clinically
       contrastive for the target label (see relevant_negatives below).
    4. Cap at 2 context sentences to avoid diluting the target prompt.
    Priority order within the cap: positives → negatives → uncertain.
    """
    positive_findings = []
    negative_findings = []
    uncertain_findings = []

    # Per-label clinically relevant negative contrasts.
    # Rationale:
    #   Pleural Effusion   — knowing Consolidation/Lung Opacity is absent
    #                        helps distinguish fluid from airspace disease.
    #   Atelectasis        — knowing Pleural Effusion/Consolidation is absent
    #                        isolates volume-loss pattern.
    #   Edema              — knowing Consolidation/Pleural Effusion is absent
    #                        isolates interstitial pattern.
    #   Enlarged CardMed   — knowing Cardiomegaly/Pleural Effusion is absent
    #                        focuses on mediastinal contour alone.
    #   Consolidation      — knowing Atelectasis/Pleural Effusion is absent
    #                        isolates true airspace filling from collapse/fluid.
    #                        Previously missing — now added for completeness.
    relevant_negatives = {
        "Pleural Effusion":           ["Consolidation", "Lung Opacity"],
        "Atelectasis":                ["Pleural Effusion", "Consolidation"],
        "Edema":                      ["Consolidation", "Pleural Effusion"],
        "Enlarged Cardiomediastinum": ["Cardiomegaly",   "Pleural Effusion"],
        "Consolidation":              ["Atelectasis",    "Pleural Effusion"],
    }

    for fv in feature_values:
        try:
            feat, val = fv.split("=")
            val  = float(val)
            feat = feat.strip()

            if feat.lower() == target_label.lower():
                continue  # skip circular reference

            if val > 0 and val < 1:
                continue  # skip fractional / semi-certain values

            if feat not in FEATURE_TO_RADIOLOGY_LANGUAGE:
                continue

            lang     = FEATURE_TO_RADIOLOGY_LANGUAGE[feat]
            relevant = relevant_negatives.get(target_label, [])

            if val >= 1.0:
                positive_findings.append(lang["positive"])
            elif val == 0.0:
                if feat in relevant:
                    negative_findings.append(lang["negative"])
            elif val == -1:
                # Include uncertain context for clinically relevant features
                # using hedging radiology language ("cannot be excluded").
                # Only gated by relevant_negatives to avoid noise.
                if feat in relevant:
                    uncertain_findings.append(lang["uncertain"])

        except Exception:
            continue

    if not positive_findings and not negative_findings and not uncertain_findings:
        return ""

    parts   = positive_findings + negative_findings + uncertain_findings
    # Cap at 2 to avoid overwhelming the main target prompt
    context = " ".join(parts[:2]) + " "
    return context


# ================================================================
# Per-Label Optimal Context Strategy
#
# Determined by ablation study (see Ablation Study cell below).
# Each label uses the context format that maximised AUC on the
# GT-uncertain evaluation set:
#
#   Atelectasis              → old_format        (+0.049 AUC)
#   Edema                    → radiology_format  (+0.032 AUC)
#   Pleural Effusion         → old_format        (+0.100 AUC)
#   Enlarged Cardiomediastinum → no_context      (context hurts)
#   Consolidation            → no_context (TBD — run ablation to confirm)
#
# Justification:
#   Context helps when co-occurring findings are informative
#   (Atelectasis, Pleural Effusion), is neutral when findings are
#   ambiguous (Edema), and hurts when it dilutes a strong visual
#   signal (Enlarged Cardiomediastinum). Consolidation is set to
#   no_context pending ablation; update after running ablation cell.
# ================================================================

LABEL_CONTEXT_STRATEGY = {
    "Edema":                      "radiology_format",
    "Atelectasis":                "old_format",
    "Pleural Effusion":           "old_format",
    "Enlarged Cardiomediastinum": "no_context",
    "Consolidation":              "no_context",   # update from ablation results
}


def build_prompts_with_context(label: str, feature_values: list) -> tuple:
    """
    Build positive and negative prompt lists using the per-label
    optimal context strategy from LABEL_CONTEXT_STRATEGY.

    Returns: (pos_prompts: list[str], neg_prompts: list[str])
    """
    strategy = LABEL_CONTEXT_STRATEGY.get(label, "no_context")

    if strategy == "no_context" or not feature_values:
        context = ""

    elif strategy == "old_format":
        parts = []
        for fv in feature_values:
            try:
                feat, val = fv.split("=")
                val  = float(val)
                feat = feat.strip()
                if feat.lower() == label.lower():
                    continue
                if val == -1 or (val > 0 and val < 1):
                    continue  # explicit guard: skip uncertain / fractional
                if val >= 1.0:
                    parts.append(f"{feat} is present")
                elif val == 0.0:
                    parts.append(f"{feat} is absent")
            except Exception:
                continue
        context = ", ".join(parts) + ". " if parts else ""

    elif strategy == "radiology_format":
        context = features_to_radiology_context(feature_values, label)

    else:
        context = ""

    pos_prompts = [context + p for p in BIOVILT_POSITIVE_PROMPTS[label]]
    neg_prompts = [context + p for p in BIOVILT_NEGATIVE_PROMPTS[label]]

    return pos_prompts, neg_prompts


# ================================================================
# infer_biovilt — uses per-label context strategy
# ================================================================

def infer_biovilt(img_path: str, label: str,
                  feature_values: list = None) -> int:
    """
    BioViL-T zero-shot inference with per-label feature context.
    Returns hard prediction (0/1) using calibrated BIOVILT_THRESHOLDS.
    """
    try:
        pos_prompts, neg_prompts = build_prompts_with_context(
            label, feature_values or []
        )

        pos_scores = [
            biovilt_engine.get_similarity_score_from_raw_data(
                image_path=Path(img_path), query_text=p
            )
            for p in pos_prompts
        ]
        neg_scores = [
            biovilt_engine.get_similarity_score_from_raw_data(
                image_path=Path(img_path), query_text=p
            )
            for p in neg_prompts
        ]

        delta     = np.mean(pos_scores) - np.mean(neg_scores)
        threshold = (BIOVILT_THRESHOLDS.get(label, 0.0)
                     if 'BIOVILT_THRESHOLDS' in globals() else 0.0)
        return 1 if delta > threshold else 0

    except Exception as e:
        print(f"  [infer_biovilt] Error — {label}: {e}")
        return 0


# BioViLT Threshold Calibration

In [ ]:
# ================================================================
# BioViL-T Threshold Calibration
#
# Strategy: identical half-split cross-validated approach used by
# WhyXrayCLIP (calibrate_threshold) and CheXZero
# (calibrate_chexzero_threshold) for full consistency.
#
# Steps:
#   1. Sample balanced pos/neg from train_df certain (0/1) labels
#   2. Compute BioViL-T delta scores (mean_pos_sim - mean_neg_sim)
#   3. Shuffle and split into two halves:
#        • First half  → find threshold candidates (ROC curve + midpoint + median_pos)
#        • Second half → evaluate each candidate on held-out data
#   4. Select the candidate with highest balanced accuracy on the
#      held-out half (no GT from the evaluation benchmark is touched)
#
# This replaces the previous 5-fold KFold CV. The half-split approach
# is simpler, faster, and consistent with the other two models.
# ================================================================
from sklearn.metrics import roc_curve, balanced_accuracy_score

RUN_BIOVILT_CALIBRATION = True

def calibrate_biovilt_threshold(label, n_samples=1000):
    """
    Calibrate BioViL-T decision threshold on train_df certain samples.

    Identical half-split CV strategy as WhyXrayCLIP / CheXZero:
      - Find candidates on first half (ROC balanced_acc, midpoint, median_pos)
      - Evaluate each candidate on second held-out half
      - Return the candidate with highest balanced accuracy on held-out half

    Args:
        label     : target label name
        n_samples : total calibration samples (pos + neg, balanced)

    Returns:
        Optimal threshold float
    """
    df = train_df[train_df[label].isin([0, 1])].copy()

    n_each = min(n_samples // 2, (df[label]==1).sum(), (df[label]==0).sum())
    if n_each < 10:
        print(f"  ⚠️  Not enough images for {label}, using 0.0")
        return 0.0

    pos    = df[df[label]==1].sample(n_each, random_state=42).reset_index(drop=True)
    neg    = df[df[label]==0].sample(n_each, random_state=42).reset_index(drop=True)
    df_cal = pd.concat([pos, neg], ignore_index=True).sample(
        frac=1, random_state=42
    ).reset_index(drop=True)

    print(f"  {label}: calibrating on {len(df_cal)} samples "
          f"(pos={n_each}, neg={n_each})")

    scores, true_labels = [], []
    pos_prompts = BIOVILT_POSITIVE_PROMPTS[label]
    neg_prompts = BIOVILT_NEGATIVE_PROMPTS[label]

    for _, row in tqdm(df_cal.iterrows(), total=len(df_cal),
                       desc=f"  Calibrating {label}", leave=False):
        try:
            clean    = (row["Path"]
                        .replace("CheXpert-v1.0-small/", "")
                        .replace("CheXpert-v1.0/", ""))
            img_path = os.path.join(BASE_PATH, clean)

            pos_scores_row = [
                biovilt_engine.get_similarity_score_from_raw_data(
                    image_path=Path(img_path), query_text=p)
                for p in pos_prompts
            ]
            neg_scores_row = [
                biovilt_engine.get_similarity_score_from_raw_data(
                    image_path=Path(img_path), query_text=p)
                for p in neg_prompts
            ]
            delta = np.mean(pos_scores_row) - np.mean(neg_scores_row)
            scores.append(delta)
            true_labels.append(int(row[label]))
        except Exception as e:
            print(f"  Error: {e}")
            continue

    if len(scores) < 40:
        print(f"  ⚠️  Too few valid scores for {label}, using 0.0")
        return 0.0

    scores_arr = np.array(scores)
    labels_arr = np.array(true_labels)

    pos_mean = scores_arr[labels_arr==1].mean()
    neg_mean = scores_arr[labels_arr==0].mean()
    gap      = pos_mean - neg_mean

    print(f"  Score stats: min={scores_arr.min():.4f}, "
          f"max={scores_arr.max():.4f}, mean={scores_arr.mean():.4f}")
    print(f"  Score for positives : mean={pos_mean:.4f}")
    print(f"  Score for negatives : mean={neg_mean:.4f}")
    print(f"  Gap                 : {gap:.4f}  "
          f"{'✅ good' if gap > 0.03 else '⚠️ small'}")

    # ── Half-split cross-validation (same as WhyXrayCLIP / CheXZero) ──
    # Shuffle → split 50/50 → find on first half → evaluate on second half
    idx          = np.random.RandomState(42).permutation(len(scores_arr))
    mid          = len(idx) // 2
    s_tr, l_tr   = scores_arr[idx[:mid]], labels_arr[idx[:mid]]
    s_val, l_val = scores_arr[idx[mid:]], labels_arr[idx[mid:]]

    # Compute threshold candidates on the FIRST half only
    fpr, tpr, thresholds_roc = roc_curve(l_tr, s_tr)
    bal_acc    = (tpr + (1 - fpr)) / 2
    thresh_bal = float(thresholds_roc[np.argmax(bal_acc)])
    thresh_mid = float((s_tr[l_tr==1].mean() + s_tr[l_tr==0].mean()) / 2)
    thresh_med = float(np.median(s_tr[l_tr==1]))

    candidates = {k: v for k, v in {
        "balanced_acc": thresh_bal,
        "midpoint":     thresh_mid,
        "median_pos":   thresh_med,
    }.items() if not np.isinf(v) and not np.isnan(v)}

    # Evaluate each candidate on the SECOND (held-out) half
    print(f"  Threshold candidates (evaluated on held-out half):")
    best_name, best_thresh, best_score = None, None, -1

    for name, thresh in candidates.items():
        preds = (s_val > thresh).astype(int)
        score = balanced_accuracy_score(l_val, preds)
        print(f"    {name:15s}: {thresh:.4f} → val bal_acc={score:.3f}")
        if score > best_score:
            best_score, best_thresh, best_name = score, thresh, name

    print(f"  ✅ Chosen: {best_name} = {best_thresh:.4f} "
          f"(val balanced acc = {best_score:.3f})")
    return best_thresh


if RUN_BIOVILT_CALIBRATION:
    print("🔧 Calibrating BioViL-T thresholds from train_df...")
    print("=" * 60)
    BIOVILT_THRESHOLDS = {}
    for label in TARGET_LABELS:
        BIOVILT_THRESHOLDS[label] = calibrate_biovilt_threshold(
            label, n_samples=1000
        )

    print("\n📊 BioViL-T Final Thresholds:")
    for label, thresh in BIOVILT_THRESHOLDS.items():
        print(f"  {label}: {thresh:.4f}")
else:
    BIOVILT_THRESHOLDS = {}
    print("⚙️  BioViL-T calibration skipped; using threshold=0.0 for all labels")


In [ ]:
# Display calibrated thresholds with interpretation
if BIOVILT_THRESHOLDS:
    print("\n📊 Calibrated BioViL-T Thresholds:")
    print("="*60)
    for label, thresh in BIOVILT_THRESHOLDS.items():
        if thresh < -0.05:
            interpretation = "Strong negative bias - positive prompts much weaker"
        elif thresh < 0:
            interpretation = "Slight negative bias - calibration adjusted downward"
        elif thresh < 0.05:
            interpretation = "Well-balanced - close to default"
        else:
            interpretation = "Positive bias - calibration adjusted upward"
        
        print(f"{label}:")
        print(f"  Threshold: {thresh:.4f}")
        print(f"  Interpretation: {interpretation}\n")
else:
    print("⚠️ No thresholds calibrated (RUN_BIOVILT_CALIBRATION was False)")

In [ ]:
# ================================================================
# BioViL-T Context Strategy Ablation Study
#
# PURPOSE:
#   Determine the optimal feature-context format for each target label
#   by comparing three strategies on the GT-uncertain evaluation set:
#
#     no_context       : no feature context prepended to prompts
#     old_format       : "Feature is present/absent" comma list
#     radiology_format : natural radiology report language sentences
#
# SELECTION METRIC:
#   Composite score = 0.5 × AUC + 0.5 × Balanced Accuracy.
#   AUC alone is unreliable for severely imbalanced labels (e.g.
#   Consolidation: 3 pos / 60 neg). Balanced accuracy penalises
#   strategies that win AUC by predicting everything as negative.
#   The composite gives a robust signal across both balanced and
#   imbalanced label distributions in our GT-uncertain eval set.
#
# STRUCTURE:
#   For each label × strategy:
#     1. Collect raw delta scores on GT uncertain samples
#        (delta = mean_pos_sim - mean_neg_sim)
#     2. Evaluate with calibrated BIOVILT_THRESHOLDS[label]
#        (train_df calibrated, zero-shot threshold — no leakage)
#     3. Report AUC, Balanced Accuracy, F1, Sens, Spec
#     4. Select winner by: composite = 0.5 × AUC + 0.5 × Bal-Acc
#
# PREREQUISITE: Run BioViL-T threshold calibration cells (91, 92)
#   first — balanced accuracy requires BIOVILT_THRESHOLDS[label].
#
# DESIGN NOTES:
#   - GT labels are used ONLY for evaluation, never for selection
#   - Thresholds come from train_df calibration — no leakage
#   - Winners are auto-applied to LABEL_CONTEXT_STRATEGY
# ================================================================

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import (roc_auc_score, confusion_matrix,
                              f1_score, balanced_accuracy_score)

# ── Strategies to compare ────────────────────────────────────────
ABLATION_STRATEGIES = ["no_context", "old_format", "radiology_format"]

# ── Temp override: run ablation with each strategy for every label ─
def collect_biovilt_scores_with_strategy(label: str,
                                          strategy: str) -> pd.DataFrame:
    """
    Collect raw delta scores on GT-uncertain samples for a given
    label and context strategy.
    Returns DataFrame with columns: study, y_true, delta_score.
    """
    labeler_df = rule_labeler_df.copy()
    gt_df      = ground_truth_df.copy()
    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True)
                break

    if label not in labeler_df.columns or label not in gt_df.columns:
        return pd.DataFrame()

    uncertain_idx    = labeler_df[labeler_df[label] == -1].index
    gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
    gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
    gt_for_uncertain = (gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]
                        .reset_index(drop=True))

    if len(gt_for_uncertain) == 0:
        return pd.DataFrame()

    top_features = top_features_per_label.get(label, [])
    rows = []

    for _, row in tqdm(gt_for_uncertain.iterrows(),
                       total=len(gt_for_uncertain),
                       desc=f"  [{label}][{strategy}]",
                       leave=False):
        try:
            img_path       = fix_test_path(row["Study"])
            feature_values = get_sample_features(
                row["Study"], label, top_features, rule_labeler_df
            )

            # Build context according to the ablation strategy
            if strategy == "no_context" or not feature_values:
                context = ""
            elif strategy == "old_format":
                parts = []
                for fv in feature_values:
                    try:
                        feat, val = fv.split("=")
                        val  = float(val)
                        feat = feat.strip()
                        if feat.lower() == label.lower():
                            continue
                        if val >= 1.0:
                            parts.append(f"{feat} is present")
                        elif val == 0.0:
                            parts.append(f"{feat} is absent")
                    except Exception:
                        continue
                context = ", ".join(parts) + ". " if parts else ""
            elif strategy == "radiology_format":
                context = features_to_radiology_context(feature_values, label)
            else:
                context = ""

            pos_prompts = [context + p for p in BIOVILT_POSITIVE_PROMPTS[label]]
            neg_prompts = [context + p for p in BIOVILT_NEGATIVE_PROMPTS[label]]

            pos_scores = [
                float(biovilt_engine.get_similarity_score_from_raw_data(
                    image_path=Path(img_path), query_text=p))
                for p in pos_prompts
            ]
            neg_scores = [
                float(biovilt_engine.get_similarity_score_from_raw_data(
                    image_path=Path(img_path), query_text=p))
                for p in neg_prompts
            ]
            delta = float(np.mean(pos_scores) - np.mean(neg_scores))
            rows.append({"study": row["Study"], "y_true": int(row["gt"]),
                         "delta_score": delta})

        except Exception as e:
            continue

    return pd.DataFrame(rows)


def evaluate_ablation(scores_arr: np.ndarray,
                      y_true_arr: np.ndarray,
                      label: str) -> dict:
    """
    Evaluate using calibrated threshold (train_df, no leakage).
    Primary metric: AUC (threshold-free).
    """
    if len(np.unique(y_true_arr)) < 2 or len(y_true_arr) < 4:
        return {}

    auc    = roc_auc_score(y_true_arr, scores_arr)
    thresh = (BIOVILT_THRESHOLDS.get(label, 0.0)
              if "BIOVILT_THRESHOLDS" in globals() else 0.0)
    preds  = (scores_arr > thresh).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true_arr, preds, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "auc":     round(float(auc), 3),
        "bal_acc": round((sens + spec) / 2, 3),
        "f1":      round(float(f1_score(y_true_arr, preds, zero_division=0)), 3),
        "sens":    round(float(sens), 3),
        "spec":    round(float(spec), 3),
        "tp": int(tp), "tn": int(tn),
        "fp": int(fp), "fn": int(fn),
        "n":  int(len(y_true_arr)),
    }


# ── Main ablation loop ───────────────────────────────────────────
print("=" * 80)
print("🔬 BioViL-T CONTEXT STRATEGY ABLATION STUDY")
print(f"   Labels     : {TARGET_LABELS}")
print(f"   Strategies : {ABLATION_STRATEGIES}")
print("=" * 80)

ablation_rows = []
winner_per_label = {}

for label in TARGET_LABELS:
    print(f"\n{'━' * 70}")
    print(f"  Label: {label}")
    cal_thresh = (BIOVILT_THRESHOLDS.get(label, 0.0)
                  if "BIOVILT_THRESHOLDS" in globals() else 0.0)
    print(f"  Calibrated threshold (train_df): {cal_thresh:.4f}")
    print(f"  {'━' * 68}")

    print(f"  {'Strategy':<22} {'n':>5} {'AUC':>7} {'BA':>7} "
          f"{'F1':>7} {'Sens':>7} {'Spec':>7}")
    print(f"  {'-' * 65}")

    label_results = {}

    for strategy in ABLATION_STRATEGIES:
        df = collect_biovilt_scores_with_strategy(label, strategy)

        if len(df) < 4:
            print(f"  {strategy:<22} {'—':>5}  (too few samples)")
            continue

        scores_arr = df["delta_score"].values
        y_true_arr = df["y_true"].values
        res = evaluate_ablation(scores_arr, y_true_arr, label)

        if not res:
            continue

        label_results[strategy] = res
        ablation_rows.append({
            "label": label, "strategy": strategy, **res
        })

        print(f"  {strategy:<22} {res['n']:>5} {res['auc']:>7.3f} "
              f"{res['bal_acc']:>7.3f} {res['f1']:>7.3f} "
              f"{res['sens']:>7.3f} {res['spec']:>7.3f}")

    # Pick winner by composite score = 0.5 × AUC + 0.5 × Balanced Accuracy
    if label_results:
        for s, r in label_results.items():
            r["composite"] = round(0.5 * r["auc"] + 0.5 * r["bal_acc"], 3)
        winner         = max(label_results, key=lambda s: label_results[s]["composite"])
        best           = label_results[winner]
        baseline_comp  = label_results.get("no_context", {}).get("composite", 0.0)
        delta          = best["composite"] - baseline_comp
        winner_per_label[label] = winner
        sign = "✅" if delta > 0.01 else ("🔁" if delta > -0.01 else "❌")
        print(f"\n  Winner: {winner}  "
              f"(composite={best['composite']:.3f}, "
              f"AUC={best['auc']:.3f}, "
              f"Bal-Acc={best['bal_acc']:.3f}, "
              f"Δ composite vs no_context={delta:+.3f}) {sign}")

# ── Summary table ────────────────────────────────────────────────
print("\n\n" + "=" * 80)
print("📊 ABLATION SUMMARY — Winner per Label")
print("=" * 80)
print(f"  {'Label':<35} {'Winner Strategy':<22} {'Composite':>10} {'AUC':>7} {'Bal-Acc':>9} {'Δ composite':>12}")
print("  " + "─" * 95)

ablation_df = pd.DataFrame(ablation_rows)

for label in TARGET_LABELS:
    winner = winner_per_label.get(label, "—")
    sub    = ablation_df[ablation_df["label"] == label]
    if len(sub) == 0:
        print(f"  {label:<35} no results")
        continue
    w_row   = sub[sub["strategy"] == winner]
    nc_row  = sub[sub["strategy"] == "no_context"]
    w_auc   = float(w_row["auc"].values[0])   if len(w_row)  > 0 else 0.0
    w_ba    = float(w_row["bal_acc"].values[0]) if len(w_row) > 0 else 0.0
    w_comp  = round(0.5 * w_auc + 0.5 * w_ba, 3)
    nc_auc  = float(nc_row["auc"].values[0])  if len(nc_row) > 0 else 0.0
    nc_ba   = float(nc_row["bal_acc"].values[0]) if len(nc_row) > 0 else 0.0
    nc_comp = round(0.5 * nc_auc + 0.5 * nc_ba, 3)
    delta   = w_comp - nc_comp
    print(f"  {label:<35} {winner:<22} {w_comp:>10.3f} {w_auc:>7.3f} {w_ba:>9.3f} {delta:>+12.3f}")

# ── Auto-apply winning strategy to LABEL_CONTEXT_STRATEGY ───────
# Mirror the same pattern used by WhyXrayCLIP/CheXZero ablation (cell 30),
# which does WXRC_CONTEXT_STRATEGY[label] = winner automatically.
# This ensures the strategy is applied consistently in inference AND
# calibration without requiring manual edits to cell 87.
print("\n\n📌 Auto-applying winners to LABEL_CONTEXT_STRATEGY...")
for lbl, winner in winner_per_label.items():
    old = LABEL_CONTEXT_STRATEGY.get(lbl, "not set")
    LABEL_CONTEXT_STRATEGY[lbl] = winner
    marker = "✅ (unchanged)" if old == winner else f"🔄 updated: {old} → {winner}"
    print(f"   {lbl:<35}: {winner:<22} {marker}")

print(f"\n✅ LABEL_CONTEXT_STRATEGY = {LABEL_CONTEXT_STRATEGY}")
print("\n✅ LABEL_CONTEXT_STRATEGY updated. Recalibration runs next with winning strategy.")

ablation_df.to_csv("biovilt_ablation_results.csv", index=False)
print("\n💾 Saved: biovilt_ablation_results.csv")


# BioViL-T Threshold Recalibration (Post-Ablation)

> Runs **after** ablation. `LABEL_CONTEXT_STRATEGY` holds the winning strategy — thresholds match inference exactly.

In [ ]:
# ================================================================
# BioViL-T Threshold Recalibration — Post-Ablation
#
# Runs AFTER ablation: LABEL_CONTEXT_STRATEGY holds the winning strategy.
# calibrate_biovilt_threshold() uses LABEL_CONTEXT_STRATEGY via
# build_prompts_with_context() — thresholds are calibrated with the
# same context used at inference (no train/test mismatch).
# ================================================================
print("🔧 Recalibrating BioViL-T thresholds (post-ablation, winning strategy)...")
print("="*60)
BIOVILT_THRESHOLDS = {}
for label in TARGET_LABELS:
    print(f"\n  [{label}]")
    BIOVILT_THRESHOLDS[label] = calibrate_biovilt_threshold(label, n_samples=1000)

print("\n📊 BioViL-T Final Thresholds (calibrated with winning strategy):")
for label, thresh in BIOVILT_THRESHOLDS.items():
    print(f"  {label}: {thresh:.4f}")
print(f"\n  Strategy used: {LABEL_CONTEXT_STRATEGY}")


# BioViLT Benchmark

In [ ]:
# ================================================================
# BioViL-T Benchmark
# Mirrors benchmark_uncertain_relabeling with BioViL-T
# ================================================================

def benchmark_biovilt():
    all_results = []
    labeler_df  = rule_labeler_df.copy()
    gt_df       = ground_truth_df.copy()

    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True)
                break

    for label in TARGET_LABELS:
        print(f"\n{'='*60}")
        print(f"  Label: {label}")
        print(f"{'='*60}")

        if label not in labeler_df.columns or label not in gt_df.columns:
            print(f"  ⚠️  '{label}' column missing — skipping")
            continue

        uncertain_idx = labeler_df[labeler_df[label] == -1].index
        if len(uncertain_idx) == 0:
            print(f"  ℹ️  No uncertain samples — skipping")
            continue

        gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
        gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
        gt_for_uncertain = gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]

        print(f"  Uncertain samples : {len(uncertain_idx)} out of {len(labeler_df)}")
        print(f"  With valid GT     : {len(gt_for_uncertain)} "
              f"(pos={int(gt_for_uncertain['gt'].sum())}, "
              f"neg={int((gt_for_uncertain['gt']==0).sum())})")

        top_features = top_features_per_label.get(label, [])

        for idx, row in tqdm(gt_for_uncertain.iterrows(),
                             total=len(gt_for_uncertain),
                             desc=f"BioViL-T · {label}"):
            try:
                img_path       = fix_test_path(row["Study"])
                feature_values = get_sample_features(
                    row["Study"], label, top_features, rule_labeler_df
                )
                pred = infer_biovilt(img_path, label, feature_values)

                all_results.append({
                    "label":           label,
                    "study":           row["Study"],
                    "y_true":          int(row["gt"]),
                    "y_pred":          pred,
                    "feature_context": " | ".join(feature_values),
                })

            except Exception as e:
                all_results.append({
                    "label":           label,
                    "study":           row["Study"],
                    "y_true":          int(row["gt"]) if pd.notna(row.get("gt")) else -1,
                    "y_pred":          0,
                    "feature_context": f"ERROR: {e}",
                })

    return pd.DataFrame(all_results)

# Run BioViLT

In [ ]:
print("🚀 Running uncertain label relabeling with BioViL-T...")
print("="*80)

biovilt_results = benchmark_biovilt()
biovilt_results.to_csv("biovilt_uncertain_relabeling_results.csv", index=False)
print("✅ Saved: biovilt_uncertain_relabeling_results.csv")

biovilt_metrics = evaluate_results_with_details(biovilt_results)
print_detailed_results(biovilt_metrics)

print("\n" + "="*80)
print("📈 BioViL-T SUMMARY")
print("="*80)
if len(biovilt_metrics) > 0:
    summary = biovilt_metrics[["label", "n_samples", "accuracy", "f1_score", "auc",
                                "true_negative", "false_positive",
                                "false_negative", "true_positive"]]
    print(summary.to_string(index=False))

    print("\nBalanced Accuracy per label:")
    for _, row in biovilt_metrics.iterrows():
        sens = row["true_positive"] / (row["true_positive"] + row["false_negative"]) \
               if (row["true_positive"] + row["false_negative"]) > 0 else 0
        spec = row["true_negative"] / (row["true_negative"] + row["false_positive"]) \
               if (row["true_negative"] + row["false_positive"]) > 0 else 0
        bal  = (sens + spec) / 2
        print(f"  {row['label']}: Sens={sens:.3f}, Spec={spec:.3f}, Balanced Acc={bal:.3f}")

# BioViL-T Few-Shot Threshold Calibration Helper

In [ ]:
# ================================================================
# BioViL-T Few-Shot — Combined Score Threshold Calibration
#
# THRESHOLD POLICY:
#   - Zero-shot (text_only):  BIOVILT_THRESHOLDS[label]
#                             calibrated on text_score from train_df
#   - Few-shot (hybrid):      BIOVILT_FS_THRESHOLDS[label][(n_shots, alpha)]
#                             calibrated on combined_score from train_df
#                             (prototype samples excluded — no leakage)
#
# Why separate thresholds?
#   The hybrid combined score = alpha*text_score + (1-alpha)*proto_score
#   has a different distribution than the pure text_score.
#   Using the text-score threshold for the hybrid score leads to
#   miscalibrated binary metrics (too many FP or FN).
#   AUC is unaffected (threshold-free) and remains the primary metric.
# ================================================================

def calibrate_biovilt_combined_threshold(label: str,
                                          n_shots: int,
                                          alpha: float,
                                          n_cal: int = 300,
                                          cal_seed: int = 99) -> float:
    """
    Calibrate threshold for BioViL-T combined score on train_df.

    Returns optimal threshold (float).
    """
    from sklearn.metrics import roc_curve, balanced_accuracy_score

    df = train_df[train_df[label].isin([0, 1])].copy()

    # Build prototypes (same seed as main run — seed=42)
    pos_proto, neg_proto, _ = build_prototype(label, n_shots, seed=42)
    if pos_proto is None:
        print(f"  ⚠️  {label}: prototype failed, using midpoint fallback")
        return 0.0

    # Prototype sample indices to exclude from calibration
    n_avail_pos = int((df[label] == 1).sum())
    n_avail_neg = int((df[label] == 0).sum())
    buf = 2
    proto_pool_pos = df[df[label]==1].sample(
        min(n_shots*buf, n_avail_pos), random_state=42)
    proto_pool_neg = df[df[label]==0].sample(
        min(n_shots*buf, n_avail_neg), random_state=42)
    proto_idx = set(proto_pool_pos.index) | set(proto_pool_neg.index)

    cal_df = df[~df.index.isin(proto_idx)]
    n_each = min(n_cal // 2, (cal_df[label]==1).sum(), (cal_df[label]==0).sum())
    if n_each < 10:
        print(f"  ⚠️  {label}: too few calibration samples")
        return 0.0

    cal_pos = cal_df[cal_df[label]==1].sample(n_each, random_state=cal_seed)
    cal_neg = cal_df[cal_df[label]==0].sample(n_each, random_state=cal_seed)
    cal_df  = pd.concat([cal_pos, cal_neg], ignore_index=True)

    # Use context-aware prompts — must match inference (cell 98).
    # Cell 98 calls build_prompts_with_context(label, feature_values) which
    # applies LABEL_CONTEXT_STRATEGY per label. Using bare prompts here
    # would create a train/test mismatch: threshold calibrated on bare-prompt
    # score distribution, but inference scores include context → miscalibrated.
    # For train_df rows we look up correlated features from train_df itself.
    scores, labels_list = [], []

    for _, row in tqdm(cal_df.iterrows(), total=len(cal_df),
                       desc=f"  Cal [{label}] n={n_shots} α={alpha:.2f}",
                       leave=False):
        try:
            clean    = (str(row["Path"])
                        .replace("CheXpert-v1.0-small/", "")
                        .replace("CheXpert-v1.0/", ""))
            img_path = os.path.join(BASE_PATH, clean)

            # Per-sample context (same as cell 98 inference)
            top_feats = top_features_per_label.get(label, [])
            fv        = get_sample_features(row["Path"], label, top_feats, train_df)
            pos_prompts, neg_prompts = build_prompts_with_context(label, fv)

            pos_text_scores = [
                float(biovilt_engine.get_similarity_score_from_raw_data(
                    image_path=Path(img_path), query_text=p))
                for p in pos_prompts
            ]
            neg_text_scores = [
                float(biovilt_engine.get_similarity_score_from_raw_data(
                    image_path=Path(img_path), query_text=p))
                for p in neg_prompts
            ]
            text_score = float(np.mean(pos_text_scores) - np.mean(neg_text_scores))

            proto_score = 0.0
            if alpha < 1.0:
                img_emb = safe_image_embedding(img_path)
                if img_emb is not None:
                    proto_score = compute_proto_score(img_emb, pos_proto, neg_proto)

            combined = alpha * text_score + (1 - alpha) * proto_score
            scores.append(combined)
            labels_list.append(int(row[label]))
        except Exception:
            continue

    if len(scores) < 20:
        return 0.0

    scores_arr = np.array(scores)
    labels_arr = np.array(labels_list)

    idx = np.random.RandomState(42).permutation(len(scores_arr))
    mid = len(idx) // 2
    s_tr, l_tr   = scores_arr[idx[:mid]], labels_arr[idx[:mid]]
    s_val, l_val = scores_arr[idx[mid:]], labels_arr[idx[mid:]]

    try:
        fpr, tpr, thrs = roc_curve(l_tr, s_tr)
        thresh_bal = float(thrs[np.argmax((tpr + (1-fpr)) / 2)])
    except:
        thresh_bal = 0.0

    thresh_mid = float((s_tr[l_tr==1].mean() + s_tr[l_tr==0].mean()) / 2)

    best_t, best_s = thresh_mid, -1
    for t in [thresh_bal, thresh_mid]:
        if np.isinf(t) or np.isnan(t): continue
        preds = (s_val > t).astype(int)
        sc = balanced_accuracy_score(l_val, preds)
        if sc > best_s:
            best_s, best_t = sc, t

    return float(best_t)


# Storage: BIOVILT_FS_THRESHOLDS[label][(n_shots, alpha)] = threshold
BIOVILT_FS_THRESHOLDS = {}
print("✅ BioViL-T few-shot threshold calibration helper ready.")
print("   Will run calibration before each strategy is evaluated.")


In [ ]:
BIOVILT_RANDOM_SEED  = 42
# ----------------------------------------------------------------
# PATH HELPER
# ----------------------------------------------------------------

def get_train_img_path(row) -> str:
    clean = (str(row["Path"])
             .replace("CheXpert-v1.0-small/", "")
             .replace("CheXpert-v1.0/", ""))
    return os.path.join(BASE_PATH, clean)

# ----------------------------------------------------------------
# IMAGE EMBEDDING
# ----------------------------------------------------------------

def safe_image_embedding(img_path: str):
    """Returns normalised [1, D] tensor or None on failure."""
    if not os.path.exists(img_path):
        return None
    try:
        raw = biovilt_image_inference.get_projected_patch_embeddings(
            Path(img_path))
        emb = raw[0] if isinstance(raw, tuple) else raw

        if emb.ndim == 4:
            b, h, w, d = emb.shape
            emb = emb.reshape(b, h * w, d).mean(dim=1)
        elif emb.ndim == 3:
            emb = emb.mean(dim=1)
        elif emb.ndim == 2 and emb.shape[0] != 1:
            emb = emb.mean(dim=0, keepdim=True)

        emb = emb.reshape(1, -1).to(biovilt_device)
        return F.normalize(emb, dim=-1)
    except Exception:
        return None


def compute_proto_score(img_emb, pos_proto, neg_proto) -> float:
    pos = float((img_emb @ pos_proto.T).reshape(1, 1).squeeze().item())
    neg = float((img_emb @ neg_proto.T).reshape(1, 1).squeeze().item())
    return pos - neg

# ----------------------------------------------------------------
# PROTOTYPE BUILDER
# ----------------------------------------------------------------

def build_prototype(label: str, n_shots: int, seed: int = BIOVILT_RANDOM_SEED):
    df    = train_df[train_df[label].isin([0, 1])].copy()
    n_pos = min(n_shots, int((df[label] == 1).sum()))
    n_neg = min(n_shots, int((df[label] == 0).sum()))

    if n_pos < 2 or n_neg < 2:
        print(f"    ⚠️  {label}: insufficient train samples")
        return None, None, {}

    pos_pool = df[df[label]==1].sample(
        min(n_pos*2, int((df[label]==1).sum())), random_state=seed)
    neg_pool = df[df[label]==0].sample(
        min(n_neg*2, int((df[label]==0).sum())), random_state=seed)

    pos_embs, neg_embs, skipped = [], [], 0

    for _, row in pos_pool.iterrows():
        if len(pos_embs) >= n_pos: break
        emb = safe_image_embedding(get_train_img_path(row))
        if emb is not None: pos_embs.append(emb)
        else: skipped += 1

    for _, row in neg_pool.iterrows():
        if len(neg_embs) >= n_neg: break
        emb = safe_image_embedding(get_train_img_path(row))
        if emb is not None: neg_embs.append(emb)
        else: skipped += 1

    if len(pos_embs) < 2 or len(neg_embs) < 2:
        print(f"    ⚠️  {label}: too few valid embeddings")
        return None, None, {}

    pos_proto = F.normalize(
        torch.cat(pos_embs).mean(dim=0, keepdim=True), dim=-1)
    neg_proto = F.normalize(
        torch.cat(neg_embs).mean(dim=0, keepdim=True), dim=-1)

    return pos_proto, neg_proto, {
        "n_pos": len(pos_embs), "n_neg": len(neg_embs),
        "skipped": skipped
    }

# ----------------------------------------------------------------
# SCORE CACHE BUILDER  (core efficiency step)
# ----------------------------------------------------------------

def build_score_cache(label: str,
                      proto_bank: dict) -> pd.DataFrame:
    """
    Computes text_score ONCE and proto_score for each n_shots ONCE.
    Returns a DataFrame with columns:
        study, y_true, text_score,
        proto_score_{n_shots} for each n_shots in BIOVILT_N_SHOTS_LIST

    proto_bank : {n_shots → (pos_proto, neg_proto)}
                 pass {} for text-only mode (no proto columns)
    """
    labeler_df = rule_labeler_df.copy()
    gt_df      = ground_truth_df.copy()

    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True)
                break

    uncertain_idx    = labeler_df[labeler_df[label] == -1].index
    gt_for_uncertain = (gt_df.loc[uncertain_idx, ["Study", label]]
                        .rename(columns={label: "gt"})
                        .pipe(lambda d: d[d["gt"].isin([0, 1])])
                        .reset_index(drop=True))

    if len(gt_for_uncertain) == 0:
        print(f"  ⚠️  No GT uncertain samples for {label}")
        return pd.DataFrame()

    top_features = top_features_per_label.get(label, [])
    rows = []
    proto_errors = {n: 0 for n in proto_bank}

    for _, row in tqdm(gt_for_uncertain.iterrows(),
                       total=len(gt_for_uncertain),
                       desc=f"  Caching [{label}]",
                       leave=False):
        try:
            img_path = fix_test_path(row["Study"])
            feature_values = get_sample_features(
                row["Study"], label, top_features, rule_labeler_df)
            pos_prompts, neg_prompts = build_prompts_with_context(
                label, feature_values)

            # ── Text score (computed once) ───────────────────────
            pos_text = [
                float(biovilt_engine.get_similarity_score_from_raw_data(
                    image_path=Path(img_path), query_text=p))
                for p in pos_prompts
            ]
            neg_text = [
                float(biovilt_engine.get_similarity_score_from_raw_data(
                    image_path=Path(img_path), query_text=p))
                for p in neg_prompts
            ]
            text_score = float(np.mean(pos_text) - np.mean(neg_text))

            # ── Proto scores (one per n_shots) ───────────────────
            proto_scores = {}
            if proto_bank:
                img_emb = safe_image_embedding(img_path)
                for n_shots, (pos_proto, neg_proto) in proto_bank.items():
                    if pos_proto is None:
                        proto_scores[n_shots] = 0.0
                    elif img_emb is not None:
                        proto_scores[n_shots] = compute_proto_score(
                            img_emb, pos_proto, neg_proto)
                    else:
                        proto_scores[n_shots] = 0.0
                        proto_errors[n_shots] += 1

            entry = {
                "study":      row["Study"],
                "y_true":     int(row["gt"]),
                "text_score": text_score,
            }
            for n_shots in proto_bank:
                entry[f"proto_score_{n_shots}"] = proto_scores.get(
                    n_shots, 0.0)
            rows.append(entry)

        except Exception as e:
            print(f"    ⚠️  {row['Study']}: {e}")
            continue

    for n, errs in proto_errors.items():
        if errs > 0:
            print(f"  ℹ️  {n}-shot: {errs} embedding failures "
                  f"(proto_score=0.0 used)")

    return pd.DataFrame(rows)

# ----------------------------------------------------------------
# METRICS
# ----------------------------------------------------------------

def compute_metrics(scores_arr: np.ndarray,
                    y_true_arr: np.ndarray,
                    label: str,
                    n_shots=None, alpha=1.0) -> dict:
    if len(np.unique(y_true_arr)) < 2:
        return {}

    auc = roc_auc_score(y_true_arr, scores_arr)

    try:
        auprc = average_precision_score(y_true_arr, scores_arr)
    except Exception:
        auprc = float(y_true_arr.mean())

    # Threshold depends on mode:
    #   text_only (alpha=1.0) → ZS text-score calibrated threshold
    #   hybrid (alpha<1.0)    → FS combined-score calibrated threshold
    if alpha == 1.0:
        if label in BIOVILT_THRESHOLDS:
            thresh     = BIOVILT_THRESHOLDS[label]
            thresh_src = "zs_calibrated"
        else:
            thresh     = ((scores_arr[y_true_arr==1].mean() +
                           scores_arr[y_true_arr==0].mean()) / 2.0)
            thresh_src = "midpoint_fallback"
    else:
        fs_key = (n_shots, alpha)
        if label in BIOVILT_FS_THRESHOLDS and fs_key in BIOVILT_FS_THRESHOLDS[label]:
            thresh     = BIOVILT_FS_THRESHOLDS[label][fs_key]
            thresh_src = "fs_calibrated"
        else:
            thresh     = ((scores_arr[y_true_arr==1].mean() +
                           scores_arr[y_true_arr==0].mean()) / 2.0)
            thresh_src = "midpoint_fallback (calibrate first)"

    preds = (scores_arr > thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true_arr, preds, labels=[0,1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "auc":       round(auc, 3),
        "auprc":     round(float(auprc), 3),
        "accuracy":  round(float(accuracy_score(y_true_arr, preds)), 3),
        "bal_acc":   round((sens + spec) / 2, 3),
        "f1":        round(float(f1_score(y_true_arr, preds,
                                          zero_division=0)), 3),
        "sens":      round(float(sens), 3),
        "spec":      round(float(spec), 3),
        "tp": int(tp), "tn": int(tn),
        "fp": int(fp), "fn": int(fn),
        "n":         len(y_true_arr),
        "n_pos":     int(y_true_arr.sum()),
        "n_neg":     int((y_true_arr == 0).sum()),
        "threshold": round(float(thresh), 4),
        "thresh_src": thresh_src,
    }

In [ ]:
# ================================================================
# BioViL-T FINAL EVALUATION — Full Strategy Comparison (Option A)
#
# Runs ALL strategy variations in one pass using score caching:
#
#   Strategies evaluated (generated from sweep lists):
#     - text_only (α=1.0 baseline)
#     - hybrid_{n_shots}_a{alpha} for each n_shots in BIOVILT_N_SHOTS_LIST and
#       each alpha in BIOVILT_ALPHA_SWEEP where alpha < 1.0
#
#   Compute efficiency via caching:
#     - text_score      : computed ONCE per sample, reused for all strategies
#     - proto_score     : computed ONCE per (label, n_shots), reused for
#                         different alpha values of the same shot count
#     → 1 inference pass + len(BIOVILT_N_SHOTS_LIST) prototype builds instead of
#       len(STRATEGIES) full passes
#
#   Strategy selection:
#     Winner = highest mean AUC across all 4 labels (macro average).
#     Decision made from this table — no further test set access after.
#
#   Threshold:
#     BIOVILT_THRESHOLDS (calibrated on train_df) for binary metrics.
#     AUC is always threshold-free (primary metric).
#
#   Saves:
#     biovilt_strategy_comparison.csv   — per-label metrics for all strategies
#     biovilt_scores_cache.csv          — text + proto scores per sample
#     biovilt_final_predictions.csv     — predictions from winning strategy
# ================================================================

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             confusion_matrix, f1_score, accuracy_score)

# ----------------------------------------------------------------
# STRATEGIES TO EVALUATE
# ----------------------------------------------------------------
# Each entry: (strategy_name, alpha, n_shots)
# n_shots=None → text_only (no prototype computation)
BIOVILT_N_SHOTS_LIST = [4, 8, 16, 32, 64]
BIOVILT_ALPHA_SWEEP  = [0.0, 0.25, 0.5, 0.75, 1.0]
BIOVILT_RANDOM_SEED  = 42

STRATEGIES = []
if 1.0 in BIOVILT_ALPHA_SWEEP:
    STRATEGIES.append(("text_only", 1.00, None))

for n_shots in BIOVILT_N_SHOTS_LIST:
    for alpha in BIOVILT_ALPHA_SWEEP:
        if alpha >= 1.0:
            continue
        STRATEGIES.append((f"hybrid_{n_shots}_a{alpha:.2f}", alpha, n_shots))

UNIQUE_N_SHOTS = sorted(BIOVILT_N_SHOTS_LIST)
# → prototype built once per shot count

# ----------------------------------------------------------------
# PATH HELPER
# ----------------------------------------------------------------

def get_train_img_path(row) -> str:
    clean = (str(row["Path"])
             .replace("CheXpert-v1.0-small/", "")
             .replace("CheXpert-v1.0/", ""))
    return os.path.join(BASE_PATH, clean)

# ----------------------------------------------------------------
# IMAGE EMBEDDING
# ----------------------------------------------------------------

def safe_image_embedding(img_path: str):
    """Returns normalised [1, D] tensor or None on failure."""
    if not os.path.exists(img_path):
        return None
    try:
        raw = biovilt_image_inference.get_projected_patch_embeddings(
            Path(img_path))
        emb = raw[0] if isinstance(raw, tuple) else raw

        if emb.ndim == 4:
            b, h, w, d = emb.shape
            emb = emb.reshape(b, h * w, d).mean(dim=1)
        elif emb.ndim == 3:
            emb = emb.mean(dim=1)
        elif emb.ndim == 2 and emb.shape[0] != 1:
            emb = emb.mean(dim=0, keepdim=True)

        emb = emb.reshape(1, -1).to(biovilt_device)
        return F.normalize(emb, dim=-1)
    except Exception:
        return None


def compute_proto_score(img_emb, pos_proto, neg_proto) -> float:
    pos = float((img_emb @ pos_proto.T).reshape(1, 1).squeeze().item())
    neg = float((img_emb @ neg_proto.T).reshape(1, 1).squeeze().item())
    return pos - neg

# ----------------------------------------------------------------
# PROTOTYPE BUILDER
# ----------------------------------------------------------------

def build_prototype(label: str, n_shots: int, seed: int = BIOVILT_RANDOM_SEED):
    df    = train_df[train_df[label].isin([0, 1])].copy()
    n_pos = min(n_shots, int((df[label] == 1).sum()))
    n_neg = min(n_shots, int((df[label] == 0).sum()))

    if n_pos < 2 or n_neg < 2:
        print(f"    ⚠️  {label}: insufficient train samples")
        return None, None, {}

    pos_pool = df[df[label]==1].sample(
        min(n_pos*2, int((df[label]==1).sum())), random_state=seed)
    neg_pool = df[df[label]==0].sample(
        min(n_neg*2, int((df[label]==0).sum())), random_state=seed)

    pos_embs, neg_embs, skipped = [], [], 0

    for _, row in pos_pool.iterrows():
        if len(pos_embs) >= n_pos: break
        emb = safe_image_embedding(get_train_img_path(row))
        if emb is not None: pos_embs.append(emb)
        else: skipped += 1

    for _, row in neg_pool.iterrows():
        if len(neg_embs) >= n_neg: break
        emb = safe_image_embedding(get_train_img_path(row))
        if emb is not None: neg_embs.append(emb)
        else: skipped += 1

    if len(pos_embs) < 2 or len(neg_embs) < 2:
        print(f"    ⚠️  {label}: too few valid embeddings")
        return None, None, {}

    pos_proto = F.normalize(
        torch.cat(pos_embs).mean(dim=0, keepdim=True), dim=-1)
    neg_proto = F.normalize(
        torch.cat(neg_embs).mean(dim=0, keepdim=True), dim=-1)

    return pos_proto, neg_proto, {
        "n_pos": len(pos_embs), "n_neg": len(neg_embs),
        "skipped": skipped
    }

# ----------------------------------------------------------------
# SCORE CACHE BUILDER  (core efficiency step)
# ----------------------------------------------------------------

def build_score_cache(label: str,
                      proto_bank: dict) -> pd.DataFrame:
    """
    Computes text_score ONCE and proto_score for each n_shots ONCE.
    Returns a DataFrame with columns:
        study, y_true, text_score,
        proto_score_{n_shots} for each n_shots in BIOVILT_N_SHOTS_LIST

    proto_bank : {n_shots → (pos_proto, neg_proto)}
                 pass {} for text-only mode (no proto columns)
    """
    labeler_df = rule_labeler_df.copy()
    gt_df      = ground_truth_df.copy()

    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True)
                break

    uncertain_idx    = labeler_df[labeler_df[label] == -1].index
    gt_for_uncertain = (gt_df.loc[uncertain_idx, ["Study", label]]
                        .rename(columns={label: "gt"})
                        .pipe(lambda d: d[d["gt"].isin([0, 1])])
                        .reset_index(drop=True))

    if len(gt_for_uncertain) == 0:
        print(f"  ⚠️  No GT uncertain samples for {label}")
        return pd.DataFrame()

    top_features = top_features_per_label.get(label, [])
    rows = []
    proto_errors = {n: 0 for n in proto_bank}

    for _, row in tqdm(gt_for_uncertain.iterrows(),
                       total=len(gt_for_uncertain),
                       desc=f"  Caching [{label}]",
                       leave=False):
        try:
            img_path = fix_test_path(row["Study"])
            feature_values = get_sample_features(
                row["Study"], label, top_features, rule_labeler_df)
            pos_prompts, neg_prompts = build_prompts_with_context(
                label, feature_values)

            # ── Text score (computed once) ───────────────────────
            pos_text = [
                float(biovilt_engine.get_similarity_score_from_raw_data(
                    image_path=Path(img_path), query_text=p))
                for p in pos_prompts
            ]
            neg_text = [
                float(biovilt_engine.get_similarity_score_from_raw_data(
                    image_path=Path(img_path), query_text=p))
                for p in neg_prompts
            ]
            text_score = float(np.mean(pos_text) - np.mean(neg_text))

            # ── Proto scores (one per n_shots) ───────────────────
            proto_scores = {}
            if proto_bank:
                img_emb = safe_image_embedding(img_path)
                for n_shots, (pos_proto, neg_proto) in proto_bank.items():
                    if pos_proto is None:
                        proto_scores[n_shots] = 0.0
                    elif img_emb is not None:
                        proto_scores[n_shots] = compute_proto_score(
                            img_emb, pos_proto, neg_proto)
                    else:
                        proto_scores[n_shots] = 0.0
                        proto_errors[n_shots] += 1

            entry = {
                "study":      row["Study"],
                "y_true":     int(row["gt"]),
                "text_score": text_score,
            }
            for n_shots in proto_bank:
                entry[f"proto_score_{n_shots}"] = proto_scores.get(
                    n_shots, 0.0)
            rows.append(entry)

        except Exception as e:
            print(f"    ⚠️  {row['Study']}: {e}")
            continue

    for n, errs in proto_errors.items():
        if errs > 0:
            print(f"  ℹ️  {n}-shot: {errs} embedding failures "
                  f"(proto_score=0.0 used)")

    return pd.DataFrame(rows)

# ----------------------------------------------------------------
# METRICS
# ----------------------------------------------------------------

def compute_metrics(scores_arr: np.ndarray,
                    y_true_arr: np.ndarray,
                    label: str,
                    n_shots=None, alpha=1.0) -> dict:
    if len(np.unique(y_true_arr)) < 2:
        return {}

    auc = roc_auc_score(y_true_arr, scores_arr)

    try:
        auprc = average_precision_score(y_true_arr, scores_arr)
    except Exception:
        auprc = float(y_true_arr.mean())

    # Threshold depends on mode:
    #   text_only (alpha=1.0) → ZS text-score calibrated threshold
    #   hybrid (alpha<1.0)    → FS combined-score calibrated threshold
    if alpha == 1.0:
        if label in BIOVILT_THRESHOLDS:
            thresh     = BIOVILT_THRESHOLDS[label]
            thresh_src = "zs_calibrated"
        else:
            thresh     = ((scores_arr[y_true_arr==1].mean() +
                           scores_arr[y_true_arr==0].mean()) / 2.0)
            thresh_src = "midpoint_fallback"
    else:
        fs_key = (n_shots, alpha)
        if label in BIOVILT_FS_THRESHOLDS and fs_key in BIOVILT_FS_THRESHOLDS[label]:
            thresh     = BIOVILT_FS_THRESHOLDS[label][fs_key]
            thresh_src = "fs_calibrated"
        else:
            thresh     = ((scores_arr[y_true_arr==1].mean() +
                           scores_arr[y_true_arr==0].mean()) / 2.0)
            thresh_src = "midpoint_fallback (calibrate first)"

    preds = (scores_arr > thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true_arr, preds, labels=[0,1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "auc":       round(auc, 3),
        "auprc":     round(float(auprc), 3),
        "accuracy":  round(float(accuracy_score(y_true_arr, preds)), 3),
        "bal_acc":   round((sens + spec) / 2, 3),
        "f1":        round(float(f1_score(y_true_arr, preds,
                                          zero_division=0)), 3),
        "sens":      round(float(sens), 3),
        "spec":      round(float(spec), 3),
        "tp": int(tp), "tn": int(tn),
        "fp": int(fp), "fn": int(fn),
        "n":         len(y_true_arr),
        "n_pos":     int(y_true_arr.sum()),
        "n_neg":     int((y_true_arr == 0).sum()),
        "threshold": round(float(thresh), 4),
        "thresh_src": thresh_src,
    }

# ----------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------

def run_full_strategy_comparison():

    # ── Threshold check ───────────────────────────────────────────
    print("=" * 70)
    print("🔧 BIOVILT_THRESHOLDS")
    print("=" * 70)
    if "BIOVILT_THRESHOLDS" not in globals():
        print("⚠️  Not found — binary metrics will use midpoint fallback.")
    else:
        for lbl, thr in BIOVILT_THRESHOLDS.items():
            print(f"  {lbl:<35}: {thr:.4f}")

    # ── Step 1: Build all prototypes upfront ─────────────────────
    # proto_bank[label][n_shots] = (pos_proto, neg_proto)
    print(f"\n{'=' * 70}")
    print(f"  STEP 1: Building prototypes "
          f"for n_shots = {UNIQUE_N_SHOTS}")
    print(f"{'=' * 70}")

    proto_bank = {label: {} for label in TARGET_LABELS}

    for n_shots in UNIQUE_N_SHOTS:
        print(f"\n  ── {n_shots}-shot ──────────────────────────────")
        for label in TARGET_LABELS:
            pos_proto, neg_proto, meta = build_prototype(label, n_shots)
            proto_bank[label][n_shots] = (pos_proto, neg_proto)
            if pos_proto is not None:
                print(f"    ✅ {label}: "
                      f"pos={meta['n_pos']} neg={meta['n_neg']} "
                      f"skipped={meta['skipped']}")
            else:
                print(f"    ❌ {label}: prototype build failed")

    # ── Step 2: Build score cache per label ───────────────────────
    print(f"\n{'=' * 70}")
    print(f"  STEP 2: Caching text + proto scores "
          f"(1 inference pass per label)")
    print(f"{'=' * 70}")

    # score_cache[label] = DataFrame with text_score + proto_score_N cols
    score_cache = {}

    for label in TARGET_LABELS:
        print(f"\n  {label}")
        score_cache[label] = build_score_cache(
            label, proto_bank[label])
        df = score_cache[label]
        if len(df) > 0:
            print(f"  ✅ {len(df)} samples cached "
                  f"(pos={int(df['y_true'].sum())}, "
                  f"neg={int((df['y_true']==0).sum())})")


    # ── Calibrate FS thresholds BEFORE evaluating strategies ─────
    print(f"\n{'=' * 70}")
    print(f"  STEP 2b: Calibrating FS thresholds for hybrid strategies")
    print(f"{'=' * 70}")
    for sname, salpha, sshots in STRATEGIES:
        if salpha == 1.0:
            continue  # text-only uses ZS threshold already in BIOVILT_THRESHOLDS
        for lbl in TARGET_LABELS:
            fs_key = (sshots, salpha)
            if lbl not in BIOVILT_FS_THRESHOLDS:
                BIOVILT_FS_THRESHOLDS[lbl] = {}
            if fs_key not in BIOVILT_FS_THRESHOLDS[lbl]:
                print(f"  Calibrating [{lbl}] n={sshots} α={salpha:.2f}...", end="", flush=True)
                t = calibrate_biovilt_combined_threshold(lbl, sshots, salpha, n_cal=1000)
                BIOVILT_FS_THRESHOLDS[lbl][fs_key] = t
                print(f" thresh={t:.4f}")
            else:
                print(f"  [{lbl}] n={sshots} α={salpha:.2f}: thresh="
                      f"{BIOVILT_FS_THRESHOLDS[lbl][fs_key]:.4f} (cached)")

    # ── Step 3: Evaluate all strategies from cache ────────────────
    print(f"\n{'=' * 70}")
    print(f"  STEP 3: Evaluating {len(STRATEGIES)} strategies from cache")
    print(f"{'=' * 70}")

    # all_results[strategy_name][label] = metrics dict
    all_results = {}

    for strategy_name, alpha, n_shots in STRATEGIES:
        all_results[strategy_name] = {}
        is_hybrid = n_shots is not None

        for label in TARGET_LABELS:
            df = score_cache[label]
            if len(df) < 5:
                continue

            y_true     = df["y_true"].values
            text_scores = df["text_score"].values

            if is_hybrid:
                proto_col = f"proto_score_{n_shots}"
                if proto_col not in df.columns:
                    # prototype build failed for this n_shots — skip
                    continue
                proto_scores = df[proto_col].values
            else:
                proto_scores = np.zeros(len(df))

            combined = alpha * text_scores + (1 - alpha) * proto_scores

            m = compute_metrics(combined, y_true, label, n_shots=n_shots, alpha=alpha)
            if m:
                all_results[strategy_name][label] = {
                    **m,
                    "label":    label,
                    "strategy": strategy_name,
                    "alpha":    alpha,
                    "n_shots":  n_shots,
                }

    # ── Step 4: Print per-strategy summaries ──────────────────────
    print(f"\n{'=' * 70}")
    print(f"  STEP 4: Per-Strategy Summaries")
    print(f"{'=' * 70}")

    strategy_mean_aucs = {}

    for strategy_name, alpha, n_shots in STRATEGIES:
        results = all_results[strategy_name]
        if not results:
            continue

        rows    = list(results.values())
        m_aucs   = [r["auc"]      for r in rows]
        m_auprcs = [r.get("auprc", 0.0) for r in rows]
        m_accs   = [r["accuracy"] for r in rows]
        m_bas    = [r["bal_acc"]  for r in rows]
        m_f1s    = [r["f1"]       for r in rows]
        m_senss  = [r["sens"]     for r in rows]
        m_specs  = [r["spec"]     for r in rows]

        mean_auc   = np.mean(m_aucs)
        mean_ba    = np.mean(m_bas)
        mean_score = (mean_auc + mean_ba) / 2  # primary selection criterion
        strategy_mean_aucs[strategy_name] = mean_score  # stored as score, not raw AUC

        shots_str = f"{n_shots}-shot" if n_shots else "no-proto"
        print(f"\n  ── {strategy_name} "
              f"(α={alpha}, {shots_str}) "
              f"─────────────────────")
        print(f"  {'Label':<35} {'AUC':>7} {'AUPRC':>7} {'Acc':>7} "
              f"{'BA':>7} {'F1':>7} {'Sens':>7} {'Spec':>7}")
        print("  " + "─" * 82)
        for r in rows:
            print(f"  {r['label']:<35} "
                  f"{r['auc']:>7.3f} {r.get('auprc',0.0):>7.3f} {r['accuracy']:>7.3f} "
                  f"{r['bal_acc']:>7.3f} {r['f1']:>7.3f} "
                  f"{r['sens']:>7.3f} {r['spec']:>7.3f}")
        print("  " + "─" * 82)
        print(f"  {'MEAN':<35} "
              f"{np.mean(m_aucs):>7.3f} {np.mean(m_auprcs):>7.3f} {np.mean(m_accs):>7.3f} "
              f"{np.mean(m_bas):>7.3f} {np.mean(m_f1s):>7.3f} "
              f"{np.mean(m_senss):>7.3f} {np.mean(m_specs):>7.3f}")
        print(f"  Score=(AUC+BA)/2: {mean_score:.3f}")

    # ── Step 5: Head-to-head comparison table ─────────────────────
    print(f"\n\n{'=' * 100}")
    print(f"  STEP 5: HEAD-TO-HEAD COMPARISON (all strategies × all labels)")
    print(f"{'=' * 100}")

    # Header
    strat_names = [s[0] for s in STRATEGIES]
    header_strats = "  " + f"{'Label':<35}"
    for sn in strat_names:
        header_strats += f" {sn[:12]:>12}"
    print(header_strats + "  ← Score=(AUC+BA)/2")
    print("  " + "─" * (35 + 13 * len(strat_names)))

    for label in TARGET_LABELS:
        line = f"  {label:<35}"
        best_auc = max(
            all_results[sn].get(label, {}).get("auc", 0)
            for sn in strat_names
        )
        for sn in strat_names:
            auc = all_results[sn].get(label, {}).get("auc", None)
            if auc is None:
                line += f" {'—':>12}"
            elif auc == best_auc:
                line += f" {f'{auc:.3f}★':>12}"   # mark best per label
            else:
                line += f" {auc:>12.3f}"
        print(line)

    # Mean row
    print("  " + "─" * (35 + 13 * len(strat_names)))
    mean_line = f"  {'MEAN SCORE (AUC+BA)/2':<35}"
    for sn in strat_names:
        mean_line += f" {strategy_mean_aucs.get(sn, 0):>12.3f}"
    print(mean_line)

    # ── Step 6: Winner ────────────────────────────────────────────
    print(f"\n\n{'=' * 70}")
    print(f"  STEP 6: WINNER SELECTION")
    print(f"{'=' * 70}")

    sorted_strategies = sorted(
        strategy_mean_aucs.items(), key=lambda x: x[1], reverse=True)

    print(f"\n  Ranking by mean Score = (AUC + BA) / 2 (macro across 4 labels):")
    print(f"  {'Rank':<6} {'Strategy':<25} {'Mean Score':>11} {'Δ vs text_only':>15}")
    print("  " + "─" * 60)

    text_only_score = strategy_mean_aucs.get("text_only", 0)
    for rank, (sn, score) in enumerate(sorted_strategies, 1):
        delta     = score - text_only_score
        marker    = " ← WINNER" if rank == 1 else ""
        delta_str = f"{delta:+.3f}" if sn != "text_only" else "baseline"
        print(f"  {rank:<6} {sn:<25} {score:>11.3f} "
              f"{delta_str:>15}{marker}")

    winner_name = sorted_strategies[0][0]
    winner_score = sorted_strategies[0][1]
    winner_cfg  = next(s for s in STRATEGIES if s[0] == winner_name)

    print(f"\n  ✅ WINNER : {winner_name}")
    print(f"     α      : {winner_cfg[1]}")
    print(f"     n_shots: {winner_cfg[2] if winner_cfg[2] else '— (text-only)'}")
    print(f"     Mean Score (AUC+BA)/2: {winner_score:.3f}  "
          f"(Δ vs text-only: "
          f"{winner_score - text_only_score:+.3f})")

    print(f"\n  ⚠️  This is the FINAL strategy decision.")
    print(f"     Do not re-run with different hyperparameters.")
    print(f"     Use '{winner_name}' as your reported method.")

    # ── Save outputs ─────────────────────────────────────────────
    # Flat comparison CSV
    comp_rows = []
    for sn, alpha, n_shots in STRATEGIES:
        for label in TARGET_LABELS:
            m = all_results[sn].get(label)
            if m:
                comp_rows.append(m)
    comparison_df = pd.DataFrame(comp_rows)
    comparison_df.to_csv("biovilt_strategy_comparison.csv", index=False)

    # Score cache CSV (useful for downstream analysis)
    cache_rows = []
    for label, df in score_cache.items():
        df_copy = df.copy()
        df_copy["label"] = label
        cache_rows.append(df_copy)
    cache_df = pd.concat(cache_rows, ignore_index=True) if cache_rows else pd.DataFrame()
    cache_df.to_csv("biovilt_scores_cache.csv", index=False)

    # Final predictions from winner
    if winner_name in all_results:
        winner_alpha  = winner_cfg[1]
        winner_shots  = winner_cfg[2]
        pred_rows = []
        for label in TARGET_LABELS:
            df = score_cache[label]
            if len(df) == 0:
                continue
            text_scores  = df["text_score"].values
            proto_col    = f"proto_score_{winner_shots}" if winner_shots else None
            proto_scores = (df[proto_col].values
                            if proto_col and proto_col in df.columns
                            else np.zeros(len(df)))
            combined     = winner_alpha * text_scores + (1 - winner_alpha) * proto_scores
            thresh       = BIOVILT_THRESHOLDS.get(
                label,
                (combined[df["y_true"].values==1].mean() +
                 combined[df["y_true"].values==0].mean()) / 2.0
            )
            for i, row in df.iterrows():
                pred_rows.append({
                    "label":          label,
                    "strategy":       winner_name,
                    "study":          row["study"],
                    "y_true":         int(row["y_true"]),
                    "combined_score": float(combined[i] if hasattr(combined, '__len__') else combined),
                    "y_pred":         int(combined[i] > thresh),
                    "threshold":      round(float(thresh), 4),
                })
        predictions_df = pd.DataFrame(pred_rows)
        predictions_df.to_csv("biovilt_final_predictions.csv", index=False)
    else:
        predictions_df = pd.DataFrame()

    print(f"\n\n{'=' * 70}")
    print(f"  💾 Saved files")
    print(f"{'=' * 70}")
    print(f"  biovilt_strategy_comparison.csv — all strategies × all labels")
    print(f"  biovilt_scores_cache.csv        — raw text + proto scores")
    print(f"  biovilt_final_predictions.csv   — predictions from winner")

    return comparison_df, cache_df, predictions_df, all_results


# ── RUN ───────────────────────────────────────────────────────────
comparison_df, cache_df, predictions_df, all_results = \
    run_full_strategy_comparison()

# Inferring with the Best Config

In [ ]:
# ================================================================
# BioViL-T FINAL INFERENCE CELL
#
# Winning strategy : hybrid_16_a0.25
#   α      = 0.25  → combined = 0.25×text_score + 0.75×proto_score
#   n_shots= 16    → 16 pos + 16 neg prototypes per label from train_df
#
# Selected from full strategy comparison (Step 6):
#   Mean AUC 0.791  (Δ +0.002 vs text_only baseline 0.789)
#   Per-label wins: Edema (+0.038), Atelectasis (+0.041)
#
# Threshold note:
#   AUC     → threshold-free, primary metric
#   Acc/F1  → BIOVILT_THRESHOLDS calibrated on train_df text scores.
#             Since hybrid scores have different scale, AUC is the
#             most reliable metric to report.
#
# Output:
#   biovilt_final_results.csv   — per-label metrics summary
#   biovilt_final_resolved.csv  — per-sample resolved labels
#                                 (used as input for Phase 3 downstream)
# ================================================================

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             confusion_matrix, f1_score, accuracy_score)

# ── Winning config ────────────────────────────────────────────────
FINAL_ALPHA   = 1.0
FINAL_N_SHOTS = None
RANDOM_SEED   = 42

# ----------------------------------------------------------------
# HELPERS  (same as comparison cell — no changes)
# ----------------------------------------------------------------

def get_train_img_path(row) -> str:
    clean = (str(row["Path"])
             .replace("CheXpert-v1.0-small/", "")
             .replace("CheXpert-v1.0/", ""))
    return os.path.join(BASE_PATH, clean)


def safe_image_embedding(img_path: str):
    if not os.path.exists(img_path):
        return None
    try:
        raw = biovilt_image_inference.get_projected_patch_embeddings(
            Path(img_path))
        emb = raw[0] if isinstance(raw, tuple) else raw
        if emb.ndim == 4:
            b, h, w, d = emb.shape
            emb = emb.reshape(b, h * w, d).mean(dim=1)
        elif emb.ndim == 3:
            emb = emb.mean(dim=1)
        elif emb.ndim == 2 and emb.shape[0] != 1:
            emb = emb.mean(dim=0, keepdim=True)
        emb = emb.reshape(1, -1).to(biovilt_device)
        return F.normalize(emb, dim=-1)
    except Exception:
        return None


def compute_proto_score(img_emb, pos_proto, neg_proto) -> float:
    pos = float((img_emb @ pos_proto.T).reshape(1, 1).squeeze().item())
    neg = float((img_emb @ neg_proto.T).reshape(1, 1).squeeze().item())
    return pos - neg


def build_prototype(label: str, n_shots: int, seed: int = RANDOM_SEED):
    df    = train_df[train_df[label].isin([0, 1])].copy()
    n_pos = min(n_shots, int((df[label] == 1).sum()))
    n_neg = min(n_shots, int((df[label] == 0).sum()))
    if n_pos < 2 or n_neg < 2:
        return None, None

    pos_pool = df[df[label]==1].sample(
        min(n_pos*2, int((df[label]==1).sum())), random_state=seed)
    neg_pool = df[df[label]==0].sample(
        min(n_neg*2, int((df[label]==0).sum())), random_state=seed)

    pos_embs, neg_embs = [], []
    for _, row in pos_pool.iterrows():
        if len(pos_embs) >= n_pos: break
        emb = safe_image_embedding(get_train_img_path(row))
        if emb is not None: pos_embs.append(emb)
    for _, row in neg_pool.iterrows():
        if len(neg_embs) >= n_neg: break
        emb = safe_image_embedding(get_train_img_path(row))
        if emb is not None: neg_embs.append(emb)

    if len(pos_embs) < 2 or len(neg_embs) < 2:
        return None, None

    pos_proto = F.normalize(
        torch.cat(pos_embs).mean(dim=0, keepdim=True), dim=-1)
    neg_proto = F.normalize(
        torch.cat(neg_embs).mean(dim=0, keepdim=True), dim=-1)
    return pos_proto, neg_proto


def compute_metrics(scores_arr, y_true_arr, label, n_shots=None, alpha=None) -> dict:
    if len(np.unique(y_true_arr)) < 2:
        return {}
    auc = roc_auc_score(y_true_arr, scores_arr)
    try:
        auprc = average_precision_score(y_true_arr, scores_arr)
    except Exception:
        auprc = float(y_true_arr.mean())
    # FINAL_ALPHA < 1.0 → hybrid → use FS combined-score threshold
    fs_key = (n_shots if n_shots is not None else FINAL_N_SHOTS, alpha if alpha is not None else FINAL_ALPHA)
    if label in BIOVILT_FS_THRESHOLDS and fs_key in BIOVILT_FS_THRESHOLDS[label]:
        thresh     = BIOVILT_FS_THRESHOLDS[label][fs_key]
        thresh_src = "fs_calibrated"
    elif FINAL_ALPHA == 1.0 and label in BIOVILT_THRESHOLDS:
        thresh     = BIOVILT_THRESHOLDS[label]
        thresh_src = "zs_calibrated"
    else:
        thresh     = ((scores_arr[y_true_arr==1].mean() +
                       scores_arr[y_true_arr==0].mean()) / 2.0)
        thresh_src = "midpoint_fallback (run calibration first)"
    preds = (scores_arr > thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true_arr, preds, labels=[0,1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        "auc":       round(auc, 3),
        "auprc":     round(float(auprc), 3),
        "accuracy":  round(float(accuracy_score(y_true_arr, preds)), 3),
        "bal_acc":   round((sens + spec) / 2, 3),
        "f1":        round(float(f1_score(y_true_arr, preds,
                                          zero_division=0)), 3),
        "sens":      round(float(sens), 3),
        "spec":      round(float(spec), 3),
        "tp": int(tp), "tn": int(tn),
        "fp": int(fp), "fn": int(fn),
        "n":         len(y_true_arr),
        "n_pos":     int(y_true_arr.sum()),
        "n_neg":     int((y_true_arr == 0).sum()),
        "threshold": round(float(thresh), 4),
        "thresh_src": thresh_src,
    }

# ----------------------------------------------------------------
# MAIN FINAL INFERENCE
# ----------------------------------------------------------------

def run_biovilt_final_inference():

    print("=" * 70)
    print("🏁 BioViL-T FINAL INFERENCE")
    print(f"   Strategy : hybrid_16_a0.25  (winner from comparison)")
    print(f"   α        : {FINAL_ALPHA}   "
          f"(combined = {FINAL_ALPHA}×text + {1-FINAL_ALPHA}×proto)")
    print(f"   n_shots  : {FINAL_N_SHOTS}  per class, from train_df")
    print(f"   Labels   : {TARGET_LABELS}")
    print("=" * 70)

    # ── Step 0: Calibrate FS threshold for the winning config ────
    print(f"\n  Calibrating FS threshold for hybrid_{FINAL_N_SHOTS}_a{FINAL_ALPHA}...")
    if "BIOVILT_FS_THRESHOLDS" not in globals():
        print("  ⚠️  BIOVILT_FS_THRESHOLDS not found — run calibration helper cell first")
    else:
        for lbl in TARGET_LABELS:
            fs_key = (FINAL_N_SHOTS, FINAL_ALPHA)
            if lbl not in BIOVILT_FS_THRESHOLDS:
                BIOVILT_FS_THRESHOLDS[lbl] = {}
            if fs_key not in BIOVILT_FS_THRESHOLDS[lbl]:
                print(f"    [{lbl}] calibrating...", end="", flush=True)
                t = calibrate_biovilt_combined_threshold(
                    lbl, FINAL_N_SHOTS, FINAL_ALPHA, n_cal=1000)
                BIOVILT_FS_THRESHOLDS[lbl][fs_key] = t
                print(f" thresh={t:.4f}")
            else:
                print(f"    [{lbl}]: thresh={BIOVILT_FS_THRESHOLDS[lbl][fs_key]:.4f} (cached)")

    # ── Step 1: Build prototypes ──────────────────────────────────
    print(f"\n  Building {FINAL_N_SHOTS}-shot prototypes from train_df...")
    prototypes = {}
    for label in TARGET_LABELS:
        pos_proto, neg_proto = build_prototype(label, FINAL_N_SHOTS)
        if pos_proto is None:
            print(f"  ❌ {label}: prototype failed — falling back to text_only")
        else:
            print(f"  ✅ {label}")
        prototypes[label] = (pos_proto, neg_proto)

    # ── Step 2: Inference per label ───────────────────────────────
    print(f"\n  Running inference on GT uncertain samples...")

    label_metrics  = {}   # label → metrics dict
    all_sample_rows = []  # for per-sample CSV

    for label in TARGET_LABELS:
        pos_proto, neg_proto = prototypes[label]
        eff_alpha = FINAL_ALPHA if pos_proto is not None else 1.0

        if eff_alpha != FINAL_ALPHA:
            print(f"\n  ━━ {label}  [text_only fallback] ━━")
        else:
            print(f"\n  ━━ {label} ━━")

        # ── Get uncertain GT samples ──────────────────────────────
        labeler_df = rule_labeler_df.copy()
        gt_df      = ground_truth_df.copy()
        for df_ in [gt_df, labeler_df]:
            for c in ["Path", "path", "study"]:
                if c in df_.columns and "Study" not in df_.columns:
                    df_.rename(columns={c: "Study"}, inplace=True)
                    break

        uncertain_idx = labeler_df[labeler_df[label] == -1].index
        gt_samples    = (gt_df.loc[uncertain_idx, ["Study", label]]
                         .rename(columns={label: "gt"})
                         .pipe(lambda d: d[d["gt"].isin([0, 1])])
                         .reset_index(drop=True))

        if len(gt_samples) == 0:
            print(f"  ⚠️  No GT uncertain samples")
            continue

        top_features = top_features_per_label.get(label, [])
        rows = []

        for _, row in tqdm(gt_samples.iterrows(),
                           total=len(gt_samples),
                           desc=f"  {label}",
                           leave=False):
            try:
                img_path = fix_test_path(row["Study"])
                feature_values = get_sample_features(
                    row["Study"], label, top_features, rule_labeler_df)
                pos_prompts, neg_prompts = build_prompts_with_context(
                    label, feature_values)

                # text score
                pos_text = [
                    float(biovilt_engine.get_similarity_score_from_raw_data(
                        image_path=Path(img_path), query_text=p))
                    for p in pos_prompts
                ]
                neg_text = [
                    float(biovilt_engine.get_similarity_score_from_raw_data(
                        image_path=Path(img_path), query_text=p))
                    for p in neg_prompts
                ]
                text_score = float(np.mean(pos_text) - np.mean(neg_text))

                # proto score
                proto_score = 0.0
                if pos_proto is not None:
                    img_emb = safe_image_embedding(img_path)
                    if img_emb is not None:
                        proto_score = compute_proto_score(
                            img_emb, pos_proto, neg_proto)

                combined = eff_alpha * text_score + (1 - eff_alpha) * proto_score

                rows.append({
                    "study":          row["Study"],
                    "y_true":         int(row["gt"]),
                    "text_score":     text_score,
                    "proto_score":    proto_score,
                    "combined_score": combined,
                })

            except Exception as e:
                print(f"    ⚠️  {row['Study']}: {e}")
                continue

        if len(rows) < 5:
            print(f"  ❌ Too few samples ({len(rows)}) — skipping")
            continue

        scores_df = pd.DataFrame(rows)
        y_true    = scores_df["y_true"].values
        combined  = scores_df["combined_score"].values

        m = compute_metrics(combined, y_true, label)
        if not m:
            continue

        label_metrics[label] = {**m, "label": label}

        # Print per-label results
        print(f"  n={m['n']} (pos={m['n_pos']}, neg={m['n_neg']})")
        print(f"  ├─ AUC         : {m['auc']:.3f}  ← primary metric")
        print(f"  ├─ Accuracy    : {m['accuracy']:.3f}")
        print(f"  ├─ Bal. Acc.   : {m['bal_acc']:.3f}")
        print(f"  ├─ F1          : {m['f1']:.3f}")
        print(f"  ├─ Sensitivity : {m['sens']:.3f}")
        print(f"  ├─ Specificity : {m['spec']:.3f}")
        print(f"  ├─ TP={m['tp']} | TN={m['tn']} | "
              f"FP={m['fp']} | FN={m['fn']}")
        print(f"  └─ Threshold   : {m['threshold']:.4f} "
              f"[{m['thresh_src']}]")

        for _, r in scores_df.iterrows():
            all_sample_rows.append({
                "label":          label,
                "study":          r["study"],
                "y_true":         int(r["y_true"]),
                "text_score":     r["text_score"],
                "proto_score":    r["proto_score"],
                "combined_score": r["combined_score"],
                "y_pred":         int(r["combined_score"] > m["threshold"]),
                "threshold":      m["threshold"],
            })

    # ── Final summary ─────────────────────────────────────────────
    if not label_metrics:
        print("\n⚠️  No results.")
        return pd.DataFrame(), pd.DataFrame()

    rows_list = list(label_metrics.values())

    print(f"\n\n{'=' * 72}")
    print(f"  FINAL RESULTS — BioViL-T (hybrid_16_a0.25)")
    print(f"{'=' * 72}")
    print(f"  {'Label':<35} {'n':>4} {'AUC':>7} {'AUPRC':>7} {'Acc':>7} "
          f"{'BA':>7} {'F1':>7} {'Sens':>7} {'Spec':>7}")
    print("  " + "─" * 82)

    aucs, auprcs, accs, bas, f1s, senss, specs = [], [], [], [], [], [], []
    for r in rows_list:
        print(f"  {r['label']:<35} {r['n']:>4} "
              f"{r['auc']:>7.3f} {r.get('auprc',0.0):>7.3f} {r['accuracy']:>7.3f} "
              f"{r['bal_acc']:>7.3f} {r['f1']:>7.3f} "
              f"{r['sens']:>7.3f} {r['spec']:>7.3f}")
        aucs.append(r['auc']);    auprcs.append(r.get('auprc', 0.0))
        accs.append(r['accuracy']); bas.append(r['bal_acc'])
        f1s.append(r['f1']);      senss.append(r['sens']); specs.append(r['spec'])

    total_n = sum(r['n'] for r in rows_list)
    print("  " + "─" * 82)
    print(f"  {'MEAN (macro)':<35} {total_n:>4} "
          f"{np.mean(aucs):>7.3f} {np.mean(auprcs):>7.3f} {np.mean(accs):>7.3f} "
          f"{np.mean(bas):>7.3f} {np.mean(f1s):>7.3f} "
          f"{np.mean(senss):>7.3f} {np.mean(specs):>7.3f}")

    # Confusion totals
    tp_t = sum(r['tp'] for r in rows_list)
    tn_t = sum(r['tn'] for r in rows_list)
    fp_t = sum(r['fp'] for r in rows_list)
    fn_t = sum(r['fn'] for r in rows_list)
    print(f"\n  Confusion (aggregate): "
          f"TP={tp_t} TN={tn_t} FP={fp_t} FN={fn_t} "
          f"Total={tp_t+tn_t+fp_t+fn_t}")

    # Note on AUC vs binary metrics
    print(f"\n  📌 Note: AUC is threshold-free and the primary metric.")
    print(f"     Binary metrics (Acc/F1/Sens/Spec) use BIOVILT_THRESHOLDS")
    print(f"     calibrated on text scores. Hybrid score scale may differ,")
    print(f"     causing slight degradation in binary metrics vs ablation.")
    print(f"     Use AUC as the reported comparison metric for the paper.")

    # ── Save ─────────────────────────────────────────────────────
    results_df  = pd.DataFrame(rows_list)
    resolved_df = pd.DataFrame(all_sample_rows)

    # resolved_df is the key output for Phase 3 downstream training:
    # y_pred column = resolved label (0/1) for each uncertain sample
    results_df.to_csv("biovilt_final_results.csv",   index=False)
    resolved_df.to_csv("biovilt_final_resolved.csv", index=False)

    print(f"\n  💾 biovilt_final_results.csv   — per-label metrics")
    print(f"  💾 biovilt_final_resolved.csv  — per-sample resolved labels")
    print(f"     (biovilt_final_resolved.csv → input for Phase 3)")

    return results_df, resolved_df


# ── RUN ───────────────────────────────────────────────────────────
final_results_df, final_resolved_df = run_biovilt_final_inference()

# Ablation Study

In [ ]:
# # ================================================================
# # BioViL-T Few-Shot Prototype Evaluation — FULL FIXED VERSION
# #
# # Fix: safe_image_embedding now correctly handles [1, N, D] patch
# #      embeddings (N=14) by mean-pooling to [1, D] befor,e any
# #      dot product. compute_proto_score extracted as separate
# #      function to guarantee scalar output.
# #
# # Prototype source : train_df certain (0/1) samples — NO GT leakage
# # Evaluation set   : GT uncertain samples — eval only, never used
# #                    for prototype construction or threshold selection
# # ================================================================

# import torch
# import torch.nn.functional as F
# import numpy as np
# import pandas as pd
# from pathlib import Path
# from tqdm import tqdm
# from sklearn.metrics import (roc_auc_score, average_precision_score,
#                              balanced_accuracy_score,
#                              confusion_matrix, f1_score, accuracy_score)

# # ----------------------------------------------------------------
# # CONFIG
# # ----------------------------------------------------------------
# N_SHOTS_LIST = [8, 16, 32]
# ALPHA_SWEEP  = [0.0, 0.25, 0.5, 0.75, 1.0]
# RANDOM_SEED  = 42

# # ----------------------------------------------------------------
# # PATH HELPER
# # ----------------------------------------------------------------

# def get_train_img_path(row) -> str:
#     """
#     Construct absolute path for a train_df row.
#     train_df["Path"] = "CheXpert-v1.0/train/patient.../study.../view.jpg"
#     Strip the CheXpert-v1.0 prefix and join with BASE_PATH.
#     Same logic used in calibrate_biovilt_threshold.
#     """
#     clean = (str(row["Path"])
#              .replace("CheXpert-v1.0-small/", "")
#              .replace("CheXpert-v1.0/", ""))
#     return os.path.join(BASE_PATH, clean)


# # ----------------------------------------------------------------
# # FIXED IMAGE EMBEDDING
# # ----------------------------------------------------------------

# def safe_image_embedding(img_path: str):
#     """
#     Get BioViL-T image embedding as guaranteed [1, D] tensor.

#     Root cause of previous error:
#         get_projected_patch_embeddings returns [1, N, D] where N=14
#         patches. The dot product img_emb @ proto.T then gives [1, 14]
#         which cannot be converted to scalar.

#     Fix: aggressively pool/reshape to [1, D] before returning.

#     Returns: normalized [1, D] Tensor, or None on any failure.
#     """
#     if not os.path.exists(img_path):
#         return None

#     try:
#         raw = biovilt_image_inference.get_projected_patch_embeddings(
#             Path(img_path)
#         )
#         emb = raw[0] if isinstance(raw, tuple) else raw

#         # ── Flatten to [1, D] regardless of returned shape ────────
#         # Observed shapes from BioViL-T:
#         #   [1, 14, D]  ← most common (14 patch tokens, pooled below)
#         #   [1,  D]     ← already correct
#         #   [1, H, W, D]← spatial grid (flatten then pool)
#         #   [N, D]      ← no batch dim

#         if emb.ndim == 4:
#             # [B, H, W, D] → [B, H*W, D] → [B, D]
#             b, h, w, d = emb.shape
#             emb = emb.reshape(b, h * w, d).mean(dim=1)

#         elif emb.ndim == 3:
#             # [B, N, D] → [B, D]  (handles the N=14 patch case)
#             emb = emb.mean(dim=1)

#         elif emb.ndim == 2 and emb.shape[0] != 1:
#             # [N, D] → [1, D]
#             emb = emb.mean(dim=0, keepdim=True)

#         # Final safety: force to [1, D]
#         emb = emb.reshape(1, -1).to(biovilt_device)
#         return F.normalize(emb, dim=-1)  # [1, D]

#     except Exception:
#         return None


# def compute_proto_score(img_emb, pos_proto, neg_proto) -> float:
#     """
#     Compute prototype score as a guaranteed Python float scalar.

#     img_emb  : [1, D]
#     pos_proto: [1, D]
#     neg_proto: [1, D]

#     [1, D] @ [D, 1] = [1, 1] → reshape(1,1).squeeze() → scalar
#     """
#     pos_sim = float((img_emb @ pos_proto.T).reshape(1, 1).squeeze().item())
#     neg_sim = float((img_emb @ neg_proto.T).reshape(1, 1).squeeze().item())
#     return pos_sim - neg_sim


# # ----------------------------------------------------------------
# # PART 1: Build Prototypes from train_df  (NO GT data touched)
# # ----------------------------------------------------------------

# def build_prototype(label: str, n_shots: int, seed: int = RANDOM_SEED):
#     """
#     Build positive and negative prototype embeddings from train_df
#     certain (0/1) samples only. GT set is never touched here.

#     Prototype = mean of n_shots image embeddings, re-normalized.
#     Samples 2x buffer to handle missing/corrupt files gracefully.

#     Returns:
#         pos_proto : Tensor [1, D] or None
#         neg_proto : Tensor [1, D] or None
#         meta      : build statistics dict
#     """
#     df = train_df[train_df[label].isin([0, 1])].copy()

#     n_pos_avail = int((df[label] == 1).sum())
#     n_neg_avail = int((df[label] == 0).sum())
#     n_pos = min(n_shots, n_pos_avail)
#     n_neg = min(n_shots, n_neg_avail)

#     if n_pos < 4 or n_neg < 4:
#         print(f"    ⚠️  {label}: not enough train samples "
#               f"(pos={n_pos_avail}, neg={n_neg_avail})")
#         return None, None, {}

#     # Sample with 2x buffer to survive missing files
#     buffer    = 2
#     pos_pool  = df[df[label] == 1].sample(
#         min(n_pos * buffer, n_pos_avail), random_state=seed
#     )
#     neg_pool  = df[df[label] == 0].sample(
#         min(n_neg * buffer, n_neg_avail), random_state=seed
#     )

#     pos_embs, neg_embs = [], []
#     skipped = 0

#     for _, row in pos_pool.iterrows():
#         if len(pos_embs) >= n_pos:
#             break
#         emb = safe_image_embedding(get_train_img_path(row))
#         if emb is not None:
#             pos_embs.append(emb)
#         else:
#             skipped += 1

#     for _, row in neg_pool.iterrows():
#         if len(neg_embs) >= n_neg:
#             break
#         emb = safe_image_embedding(get_train_img_path(row))
#         if emb is not None:
#             neg_embs.append(emb)
#         else:
#             skipped += 1

#     if len(pos_embs) < 2 or len(neg_embs) < 2:
#         print(f"    ⚠️  {label}: too few valid embeddings "
#               f"(pos={len(pos_embs)}, neg={len(neg_embs)}, "
#               f"skipped={skipped})")
#         sample_path = get_train_img_path(pos_pool.iloc[0])
#         print(f"    ℹ️  Sample path: {sample_path}")
#         print(f"    ℹ️  Exists: {os.path.exists(sample_path)}")
#         return None, None, {}

#     # Mean pool across n_shots embeddings, then re-normalize
#     pos_proto = F.normalize(
#         torch.cat(pos_embs, dim=0).mean(dim=0, keepdim=True), dim=-1
#     )   # [1, D]
#     neg_proto = F.normalize(
#         torch.cat(neg_embs, dim=0).mean(dim=0, keepdim=True), dim=-1
#     )   # [1, D]

#     return pos_proto, neg_proto, {
#         "n_pos": len(pos_embs),
#         "n_neg": len(neg_embs),
#         "skipped": skipped,
#     }


# # ----------------------------------------------------------------
# # PART 2: Collect text + prototype scores on GT uncertain samples
# # ----------------------------------------------------------------

# def collect_fewshot_scores(label: str,
#                             pos_proto,
#                             neg_proto) -> pd.DataFrame:
#     """
#     For every uncertain GT sample of `label`, compute:
#         text_score  : mean(pos_text_sims) - mean(neg_text_sims)
#                       using per-label optimal context (ablation result)
#         proto_score : cosine(img, pos_proto) - cosine(img, neg_proto)
#                       using safe_image_embedding → compute_proto_score

#     Uncertain sample selection mirrors benchmark_biovilt exactly:
#         rule_labeler_df[label] == -1  → uncertain samples
#         ground_truth_df[label]        → GT label (used for eval ONLY)
#     Both dfs share the same index (500 test samples).
#     """
#     labeler_df = rule_labeler_df.copy()
#     gt_df      = ground_truth_df.copy()

#     # Normalise Study column — same as benchmark_biovilt
#     for df_ in [gt_df, labeler_df]:
#         for c in ["Path", "path", "study"]:
#             if c in df_.columns and "Study" not in df_.columns:
#                 df_.rename(columns={c: "Study"}, inplace=True)
#                 break

#     if label not in labeler_df.columns or label not in gt_df.columns:
#         print(f"    ⚠️  '{label}' column not found in labeler or GT df")
#         return pd.DataFrame()

#     uncertain_idx    = labeler_df[labeler_df[label] == -1].index
#     gt_for_uncertain = gt_df.loc[uncertain_idx, ["Study", label]].copy()
#     gt_for_uncertain = gt_for_uncertain.rename(columns={label: "gt"})
#     gt_for_uncertain = (gt_for_uncertain[gt_for_uncertain["gt"].isin([0, 1])]
#                         .reset_index(drop=True))

#     if len(gt_for_uncertain) == 0:
#         print(f"    ⚠️  No valid GT uncertain samples for {label}")
#         return pd.DataFrame()

#     top_features = top_features_per_label.get(label, [])
#     rows = []
#     proto_errors = 0

#     for _, row in tqdm(gt_for_uncertain.iterrows(),
#                        total=len(gt_for_uncertain),
#                        desc=f"    [{label}]",
#                        leave=False):
#         try:
#             img_path = fix_test_path(row["Study"])

#             # ── Text score (per-label optimal context) ─────────────
#             feature_values = get_sample_features(
#                 row["Study"], label, top_features, rule_labeler_df
#             )
#             pos_prompts, neg_prompts = build_prompts_with_context(
#                 label, feature_values
#             )
#             pos_text = [
#                 float(biovilt_engine.get_similarity_score_from_raw_data(
#                     image_path=Path(img_path), query_text=p))
#                 for p in pos_prompts
#             ]
#             neg_text = [
#                 float(biovilt_engine.get_similarity_score_from_raw_data(
#                     image_path=Path(img_path), query_text=p))
#                 for p in neg_prompts
#             ]
#             text_score = float(np.mean(pos_text) - np.mean(neg_text))

#             # ── Prototype score (fixed) ────────────────────────────
#             img_emb = safe_image_embedding(img_path)
#             if img_emb is None:
#                 proto_score = 0.0
#                 proto_errors += 1
#             else:
#                 # compute_proto_score guarantees a Python float scalar
#                 proto_score = compute_proto_score(img_emb, pos_proto, neg_proto)

#             rows.append({
#                 "study":       row["Study"],
#                 "y_true":      int(row["gt"]),
#                 "text_score":  text_score,
#                 "proto_score": proto_score,
#             })

#         except Exception as e:
#             print(f"      ⚠️  [{label}] {row['Study']}: {e}")
#             continue

#     if proto_errors > 0:
#         print(f"    ℹ️  {proto_errors}/{len(rows)} samples used "
#               f"proto_score=0.0 (embedding failed, text_score used)")

#     return pd.DataFrame(rows)


# # ----------------------------------------------------------------
# # PART 3: Metrics
# # ----------------------------------------------------------------

# def evaluate_scores(scores_arr: np.ndarray,
#                     y_true_arr: np.ndarray,
#                     method_name: str,
#                     label: str = None,
#                     n_shots: int = None,
#                     alpha: float = None) -> dict:
#     """
#     Primary metric : AUC — threshold-independent.
#     Threshold policy (paper-consistent):
#       - alpha == 1.0 (text_only): use BIOVILT_THRESHOLDS[label] (ZS calibrated)
#       - alpha < 1.0 (hybrid)    : use BIOVILT_FS_THRESHOLDS[label][(n_shots, alpha)]
#         (FS calibrated on train_df; calibrated via calibrate_biovilt_combined_threshold)
#       - Fallback: midpoint threshold if no calibrated threshold available
#     """
#     if len(np.unique(y_true_arr)) < 2 or len(y_true_arr) < 4:
#         return {}

#     auc = roc_auc_score(y_true_arr, scores_arr)
#     try:
#         auprc = average_precision_score(y_true_arr, scores_arr)
#     except Exception:
#         auprc = float(y_true_arr.mean())

#     # Threshold selection — mirrors compute_metrics in cell 96
#     thresh_src = "midpoint_fallback"
#     if alpha is not None and alpha == 1.0 and label and label in BIOVILT_THRESHOLDS:
#         thresh = BIOVILT_THRESHOLDS[label]
#         thresh_src = "zs_calibrated"
#     elif (alpha is not None and alpha < 1.0 and label and n_shots
#           and "BIOVILT_FS_THRESHOLDS" in globals()
#           and label in BIOVILT_FS_THRESHOLDS
#           and (n_shots, alpha) in BIOVILT_FS_THRESHOLDS[label]):
#         thresh = BIOVILT_FS_THRESHOLDS[label][(n_shots, alpha)]
#         thresh_src = "fs_calibrated"
#     else:
#         thresh = ((scores_arr[y_true_arr == 1].mean() +
#                    scores_arr[y_true_arr == 0].mean()) / 2.0)

#     preds  = (scores_arr > thresh).astype(int)

#     tn, fp, fn, tp = confusion_matrix(y_true_arr, preds, labels=[0, 1]).ravel()
#     sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
#     spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

#     return {
#         "method":     method_name,
#         "auc":        round(auc, 3),
#         "auprc":      round(float(auprc), 3),
#         "bal_acc":    round((sens + spec) / 2, 3),
#         "f1":         round(float(f1_score(y_true_arr, preds, zero_division=0)), 3),
#         "sens":       round(float(sens), 3),
#         "spec":       round(float(spec), 3),
#         "threshold":  round(float(thresh), 4),
#         "thresh_src": thresh_src,
#         "tp": int(tp), "tn": int(tn),
#         "fp": int(fp), "fn": int(fn),
#         "n": len(y_true_arr),
#     }


# # ----------------------------------------------------------------
# # PART 4: Main evaluation loop
# # ----------------------------------------------------------------

# def run_fewshot_evaluation():

#     print("\n" + "=" * 80)
#     print("🔬 BioViL-T FEW-SHOT PROTOTYPE EVALUATION")
#     print(f"   Prototype source : train_df ({BASE_PATH})")
#     print(f"   Evaluation set   : GT uncertain samples (eval only)")
#     print(f"   Shot counts      : {N_SHOTS_LIST}")
#     print(f"   Alpha sweep      : {ALPHA_SWEEP}")
#     print("=" * 80)

#     # ── Sanity check train path before starting ───────────────────
#     print("\n🔍 Path sanity check...")
#     _sample_row  = train_df[train_df[TARGET_LABELS[0]].isin([0, 1])].iloc[0]
#     _sample_path = get_train_img_path(_sample_row)
#     print(f"   Sample train path : {_sample_path}")
#     print(f"   Exists            : {os.path.exists(_sample_path)}")

#     if not os.path.exists(_sample_path):
#         print("\n   ❌ Train image not found. Listing BASE_PATH:")
#         try:
#             print("  ", os.listdir(BASE_PATH)[:10])
#         except Exception as e:
#             print(f"   Cannot list: {e}")
#         print("   Fix get_train_img_path() before proceeding.")
#         return pd.DataFrame()

#     # ── Quick embedding shape check ───────────────────────────────
#     print("\n🔍 Embedding shape check...")
#     _test_emb = safe_image_embedding(_sample_path)
#     if _test_emb is None:
#         print("   ❌ safe_image_embedding returned None on valid path.")
#         print("   Check biovilt_image_inference is loaded correctly.")
#         return pd.DataFrame()
#     print(f"   Embedding shape   : {_test_emb.shape}  "
#           f"(should be [1, D])")
#     print("   ✅ Ready\n")

#     all_rows       = []
#     best_per_label = {}

#     for label in TARGET_LABELS:
#         print(f"\n{'━' * 70}")
#         print(f"  Label : {label}  "
#               f"[context: {LABEL_CONTEXT_STRATEGY.get(label, 'no_context')}]")
#         print(f"{'━' * 70}")

#         for n_shots in N_SHOTS_LIST:
#             print(f"\n  ── {n_shots}-shot "
#                   f"────────────────────────────────────────")

#             # ── Build prototypes from train_df ────────────────────
#             print(f"    Building prototypes "
#                   f"({n_shots}/class from train_df)...")
#             pos_proto, neg_proto, meta = build_prototype(
#                 label, n_shots
#             )
#             if pos_proto is None:
#                 continue
#             print(f"    ✅ pos={meta['n_pos']} | "
#                   f"neg={meta['n_neg']} | "
#                   f"skipped={meta['skipped']}")

#             # ── Collect text + proto scores on GT set ─────────────
#             print(f"    Collecting scores on GT uncertain samples...")
#             scores_df = collect_fewshot_scores(
#                 label, pos_proto, neg_proto
#             )

#             if len(scores_df) < 5:
#                 print(f"    ⚠️  Too few samples "
#                       f"({len(scores_df)}) — skipping")
#                 continue

#             y_true       = scores_df["y_true"].values
#             text_scores  = scores_df["text_score"].values
#             proto_scores = scores_df["proto_score"].values

#             # Score gap diagnostic — tells you if prototypes help
#             t_gap = (text_scores[y_true == 1].mean() -
#                      text_scores[y_true == 0].mean())
#             p_gap = (proto_scores[y_true == 1].mean() -
#                      proto_scores[y_true == 0].mean())

#             print(f"    n={len(scores_df)} | "
#                   f"pos={int(y_true.sum())} | "
#                   f"neg={int((y_true == 0).sum())}")
#             print(f"    text_score  gap : {t_gap:>+.4f}  "
#                   f"{'✅' if t_gap > 0.05 else '⚠️'}")
#             print(f"    proto_score gap : {p_gap:>+.4f}  "
#                   f"{'✅' if p_gap > 0.02 else '⚠️ weak — prototypes may not help'}")

#             # ── Alpha sweep ───────────────────────────────────────
#             print(f"\n    {'Method':<26} "
#                   f"{'AUC':>7} {'AUPRC':>7} {'BA':>7} "
#                   f"{'Sens':>7} {'Spec':>7} {'F1':>7}  ← Score=(AUC+BA)/2")
#             print(f"    {'─' * 72}")

#             best_auc_shots = 0.0
#             best_res_shots = {}

#             for alpha in ALPHA_SWEEP:
#                 combined = (alpha * text_scores +
#                             (1 - alpha) * proto_scores)

#                 mname = ("text_only"        if alpha == 1.0 else
#                          "proto_only"       if alpha == 0.0 else
#                          f"hybrid_a{alpha:.2f}")

#                 res = evaluate_scores(combined, y_true, mname,
#                                       label=label, n_shots=n_shots, alpha=alpha)
#                 if not res:
#                     continue

#                 res.update({
#                     "label":   label,
#                     "n_shots": n_shots,
#                     "alpha":   alpha,
#                 })
#                 all_rows.append(res)

#                 res["score"] = (res["auc"] + res["bal_acc"]) / 2
#                 marker = ""
#                 if res["score"] > best_auc_shots:
#                     best_auc_shots = res["score"]
#                     best_res_shots = res
#                     marker = " ←"

#                 print(f"    {mname:<26} "
#                       f"{res['auc']:>7.3f} "
#                       f"{res.get('auprc',0.0):>7.3f} "
#                       f"{res['bal_acc']:>7.3f} "
#                       f"{res['sens']:>7.3f} "
#                       f"{res['spec']:>7.3f} "
#                       f"{res['f1']:>7.3f}{marker}")

#             if best_res_shots:
#                 print(f"\n    🏆 Best {n_shots}-shot: "
#                       f"[{best_res_shots['method']}] "
#                       f"Score={best_res_shots['score']:.3f} | "
#                       f"AUC={best_res_shots['auc']:.3f} | "
#                       f"BA={best_res_shots['bal_acc']:.3f}")
#                 if (label not in best_per_label or
#                         best_res_shots["score"] >
#                         best_per_label[label].get("score", 0.0)):
#                     best_per_label[label] = best_res_shots

#     # ----------------------------------------------------------------
#     # PART 5: Summary tables
#     # ----------------------------------------------------------------
#     if not all_rows:
#         print("\n⚠️  No results collected — check errors above.")
#         return pd.DataFrame()

#     results_df = pd.DataFrame(all_rows)

#     # ── Table 1: text-only baseline ───────────────────────────────
#     print("\n\n" + "=" * 80)
#     print("📊 TABLE 1: TEXT-ONLY BASELINE (alpha=1.0)")
#     print("=" * 80)
#     text_only = (results_df[results_df["alpha"] == 1.0]
#                  .sort_values("auc", ascending=False)
#                  .groupby("label").first()
#                  .reset_index())
#     print(text_only[["label", "n", "auc", "bal_acc",
#                       "sens", "spec", "f1"]].to_string(index=False))
#     mean_text_auc = text_only["auc"].mean()
#     mean_text_ba_val = text_only["bal_acc"].mean()
#     print(f"\n  Mean AUC   (text-only): {mean_text_auc:.3f}")
#     print(f"  Mean BA    (text-only): {mean_text_ba_val:.3f}")
#     print(f"  Mean Score (text-only): {(mean_text_auc + mean_text_ba_val) / 2:.3f}")

#     # ── Table 2: best few-shot per label ──────────────────────────
#     print("\n\n" + "=" * 80)
#     print("📊 TABLE 2: BEST FEW-SHOT RESULT PER LABEL")
#     print("=" * 80)
#     print(f"  {'Label':<35} {'Method':<20} "
#           f"{'Shots':>6} {'Score':>7} {'AUC':>7} {'BA':>7} {'Δ Score':>8}")
#     print("  " + "─" * 80)

#     best_scores_list = []
#     for label in TARGET_LABELS:
#         if label not in best_per_label:
#             print(f"  {label:<35} {'N/A (no results)'}")
#             continue
#         best      = best_per_label[label]
#         to_row    = text_only[text_only["label"] == label]
#         to_auc    = (float(to_row["auc"].values[0])  if len(to_row) > 0 else 0.0)
#         to_ba     = (float(to_row["bal_acc"].values[0]) if len(to_row) > 0 else 0.0)
#         to_score  = (to_auc + to_ba) / 2
#         best_score = best.get("score", (best["auc"] + best["bal_acc"]) / 2)
#         delta  = best_score - to_score
#         sign   = ("✅" if delta > 0.01 else
#                   "🔁" if delta > -0.01 else "❌")
#         best_scores_list.append(best_score)
#         print(f"  {label:<35} {best['method']:<20} "
#               f"{best['n_shots']:>6} "
#               f"{best_score:>7.3f} "
#               f"{best['auc']:>7.3f} "
#               f"{best['bal_acc']:>7.3f} "
#               f"{delta:>+8.3f}  {sign}")

#     mean_best_score = np.mean(best_scores_list) if best_scores_list else 0.0
#     mean_text_score_val = (mean_text_auc + text_only["bal_acc"].mean()) / 2
#     print("  " + "─" * 80)
#     print(f"  Mean Score (best few-shot) : {mean_best_score:.3f}")
#     print(f"  Δ vs text-only             : "
#           f"{mean_best_score - mean_text_score_val:+.3f}")

#     # ── Table 3: mean AUC by alpha ────────────────────────────────
#     print("\n\n" + "=" * 80)
#     print("📊 TABLE 3: MEAN SCORE BY ALPHA  (Score = (AUC+BA)/2)")
#     print("   (averaged across all labels and shot counts)")
#     print("=" * 80)
#     if "score" not in results_df.columns:
#         results_df["score"] = (results_df["auc"] + results_df["bal_acc"]) / 2
#     alpha_grp = (results_df
#                  .groupby("alpha")
#                  .agg(mean_score=("score", "mean"),
#                       mean_auc=("auc", "mean"),
#                       mean_ba=("bal_acc", "mean"))
#                  .reset_index())
#     best_alpha_score = alpha_grp["mean_score"].max()
#     print(f"  {'Alpha':<8} {'Method':<22} "
#           f"{'Mean Score':>11} {'Mean AUC':>10} {'Mean BA':>10}")
#     print("  " + "─" * 65)
#     for _, r in alpha_grp.sort_values("alpha").iterrows():
#         mname  = ("text_only"  if r["alpha"] == 1.0 else
#                   "proto_only" if r["alpha"] == 0.0 else
#                   f"hybrid_a{r['alpha']:.2f}")
#         marker = " ← best" if r["mean_score"] == best_alpha_score else ""
#         print(f"  {r['alpha']:<8.2f} {mname:<22} "
#               f"{r['mean_score']:>11.3f} {r['mean_auc']:>10.3f} "
#               f"{r['mean_ba']:>10.3f}{marker}")

#     # ── Table 4: mean AUC by n_shots ─────────────────────────────
#     print("\n\n" + "=" * 80)
#     print("📊 TABLE 4: MEAN AUC BY N_SHOTS")
#     print("=" * 80)
#     shots_grp = (results_df
#                  .groupby("n_shots")
#                  .agg(mean_auc=("auc", "mean"),
#                       mean_ba=("bal_acc", "mean"))
#                  .reset_index())
#     print(f"  {'N-Shots':<10} {'Mean AUC':>10} {'Mean BA':>10}")
#     print("  " + "─" * 34)
#     for _, r in shots_grp.iterrows():
#         print(f"  {int(r['n_shots']):<10} "
#               f"{r['mean_auc']:>10.3f} "
#               f"{r['mean_ba']:>10.3f}")

#     # ── Verdict ───────────────────────────────────────────────────
#     delta_overall = mean_best_score - mean_text_score_val
#     score_col_name = "mean_score" if "mean_score" in alpha_grp.columns else "mean_auc"
#     best_alpha    = alpha_grp.loc[
#         alpha_grp[score_col_name].idxmax(), "alpha"
#     ]
#     shots_score_col = "mean_score" if "mean_score" in shots_grp.columns else "mean_auc"
#     best_shots    = int(shots_grp.loc[
#         shots_grp[shots_score_col].idxmax(), "n_shots"
#     ])

#     print("\n\n" + "=" * 80)
#     print("🏁 VERDICT")
#     print("=" * 80)
#     if delta_overall > 0.01:
#         print(f"  ✅ Few-shot prototypes IMPROVE over text-only "
#               f"(Δ Score = {delta_overall:+.3f})")
#         print(f"     → Use hybrid inference in final benchmark.")
#         print(f"     → Best alpha  : {best_alpha}")
#         print(f"     → Best n_shots: {best_shots}")
#     elif delta_overall > -0.01:
#         print(f"  🔁 Few-shot prototypes COMPARABLE to text-only "
#               f"(Δ Score = {delta_overall:+.3f})")
#         print(f"     → Report both; text-only is simpler to reproduce.")
#     else:
#         print(f"  ❌ Few-shot prototypes HURT performance "
#               f"(Δ Score = {delta_overall:+.3f})")
#         print(f"     → Stick with text-only.")

#     # ── Save ──────────────────────────────────────────────────────
#     results_df.to_csv("biovilt_fewshot_results.csv", index=False)
#     print(f"\n  💾 Saved: biovilt_fewshot_results.csv")

#     return results_df


# # ── RUN ───────────────────────────────────────────────────────────
# fewshot_results_df = run_fewshot_evaluation()

In [ ]:
# ================================================================
# BioViL-T: Zero-Shot vs Best Few-Shot Comparison
#
# Mirrors the ZS vs FS comparison tables in WhyXrayCLIP (cell 40)
# and CheXZero (cell 78). Shows all 6 metrics:
#   AUC, AUPRC, Balanced Acc, F1, Sensitivity, Specificity
#
# Sources:
#   ZS  → biovilt_metrics       (from benchmark_biovilt, cell 93)
#   FS  → final_results_df      (from run_biovilt_final_inference, cell 98)
#         OR fewshot_results_df  (from run_fewshot_evaluation, cell 100)
# ================================================================

# ── Identify the best FS result source ──────────────────────────
# Prefer the final-inference output (cell 98) which uses the winning config.
if "final_results_df" in globals() and len(final_results_df) > 0:
    biovilt_fs_metrics = final_results_df
    fs_source_label   = "hybrid_16_a0.25 (final inference, cell 98)"
elif "fewshot_results_df" in globals() and len(fewshot_results_df) > 0:
    # Aggregate fewshot_results_df to best config per label
    if "score" not in fewshot_results_df.columns:
        fewshot_results_df["score"] = (
            fewshot_results_df["auc"] + fewshot_results_df["bal_acc"]) / 2
    biovilt_fs_metrics = (
        fewshot_results_df.sort_values("score", ascending=False)
        .groupby("label").first().reset_index()
    )
    fs_source_label = "best config from sweep (cell 100)"
else:
    print("⚠️  No BioViL-T few-shot results found. "
          "Run cell 98 (final inference) or cell 100 (sweep) first.")
    biovilt_fs_metrics = None

if "biovilt_metrics" not in globals() or len(biovilt_metrics) == 0:
    print("⚠️  biovilt_metrics (zero-shot) not found. "
          "Run benchmark_biovilt + evaluate_results_with_details (cell 93) first.")
    biovilt_zs_metrics = None
else:
    biovilt_zs_metrics = biovilt_metrics

if biovilt_zs_metrics is not None and biovilt_fs_metrics is not None:

    print("\n" + "="*80)
    print(f"📊 BioViL-T: Zero-Shot vs Few-Shot")
    print(f"   ZS source: benchmark_biovilt (cell 93) | "
          f"FS source: {fs_source_label}")
    print(f"   ZS threshold: BIOVILT_THRESHOLDS (train_df ZS-calibrated)")
    print(f"   FS threshold: BIOVILT_FS_THRESHOLDS (train_df FS-calibrated)")
    print("="*80)
    print(f"  {'Label':<30} "
          f"{'ZS AUC':>7} {'FS AUC':>7} {'ΔAUC':>7} "
          f"{'ZS BA':>7} {'FS BA':>7} {'ΔBA':>7}")
    print(f"  {'':<30} "
          f"{'ZS AUPRC':>8} {'FS AUPRC':>8} {'ΔAUPRC':>7} "
          f"{'ZS F1':>7} {'FS F1':>7} {'ΔF1':>7} "
          f"{'ZS Se':>7} {'FS Se':>7} {'ΔSe':>7} "
          f"{'ZS Sp':>7} {'FS Sp':>7} {'ΔSp':>7}")
    print("  " + "-"*125)

    def _bvt_metric(row, *keys):
        for k in keys:
            if k in row and pd.notna(row[k]):
                return float(row[k])
        return 0.0

    zs_auc, fs_auc = [], []
    zs_ba, fs_ba   = [], []
    zs_auprc, fs_auprc = [], []
    zs_f1, fs_f1   = [], []
    zs_sens, fs_sens = [], []
    zs_spec, fs_spec = [], []

    for label in TARGET_LABELS:
        z_row = biovilt_zs_metrics[biovilt_zs_metrics["label"] == label]
        f_row = biovilt_fs_metrics[biovilt_fs_metrics["label"] == label]
        if len(z_row) == 0 or len(f_row) == 0:
            print(f"  {label:<30}  — missing ZS or FS data")
            continue
        z = z_row.iloc[0]
        f = f_row.iloc[0]

        z_auc_v   = _bvt_metric(z, "auc")
        f_auc_v   = _bvt_metric(f, "auc")
        z_ba_v    = _bvt_metric(z, "bal_acc", "balanced_acc")
        f_ba_v    = _bvt_metric(f, "bal_acc", "balanced_acc")
        z_auprc_v = _bvt_metric(z, "auprc")
        f_auprc_v = _bvt_metric(f, "auprc")
        z_f1_v    = _bvt_metric(z, "f1", "f1_score")
        f_f1_v    = _bvt_metric(f, "f1", "f1_score")
        z_se_v    = _bvt_metric(z, "sens", "sensitivity")
        f_se_v    = _bvt_metric(f, "sens", "sensitivity")
        z_sp_v    = _bvt_metric(z, "spec", "specificity")
        f_sp_v    = _bvt_metric(f, "spec", "specificity")

        zs_auc.append(z_auc_v);     fs_auc.append(f_auc_v)
        zs_ba.append(z_ba_v);       fs_ba.append(f_ba_v)
        zs_auprc.append(z_auprc_v); fs_auprc.append(f_auprc_v)
        zs_f1.append(z_f1_v);       fs_f1.append(f_f1_v)
        zs_sens.append(z_se_v);     fs_sens.append(f_se_v)
        zs_spec.append(z_sp_v);     fs_spec.append(f_sp_v)

        print(f"  {label:<30} "
              f"{z_auc_v:>7.3f} {f_auc_v:>7.3f} {f_auc_v - z_auc_v:>+7.3f} "
              f"{z_ba_v:>7.3f} {f_ba_v:>7.3f} {f_ba_v - z_ba_v:>+7.3f}")
        print(f"  {'':<30} "
              f"{z_auprc_v:>8.3f} {f_auprc_v:>8.3f} {f_auprc_v - z_auprc_v:>+7.3f} "
              f"{z_f1_v:>7.3f} {f_f1_v:>7.3f} {f_f1_v - z_f1_v:>+7.3f} "
              f"{z_se_v:>7.3f} {f_se_v:>7.3f} {f_se_v - z_se_v:>+7.3f} "
              f"{z_sp_v:>7.3f} {f_sp_v:>7.3f} {f_sp_v - z_sp_v:>+7.3f}")

    print("  " + "-"*125)
    if zs_auc:
        avg_z_auc   = float(np.mean(zs_auc));   avg_f_auc   = float(np.mean(fs_auc))
        avg_z_ba    = float(np.mean(zs_ba));    avg_f_ba    = float(np.mean(fs_ba))
        avg_z_auprc = float(np.mean(zs_auprc)); avg_f_auprc = float(np.mean(fs_auprc))
        avg_z_f1    = float(np.mean(zs_f1));    avg_f_f1    = float(np.mean(fs_f1))
        avg_z_se    = float(np.mean(zs_sens));  avg_f_se    = float(np.mean(fs_sens))
        avg_z_sp    = float(np.mean(zs_spec));  avg_f_sp    = float(np.mean(fs_spec))

        print(f"  {'Average':<30} "
              f"{avg_z_auc:>7.3f} {avg_f_auc:>7.3f} {avg_f_auc - avg_z_auc:>+7.3f} "
              f"{avg_z_ba:>7.3f} {avg_f_ba:>7.3f} {avg_f_ba - avg_z_ba:>+7.3f}")
        print(f"  {'':<30} "
              f"{avg_z_auprc:>8.3f} {avg_f_auprc:>8.3f} {avg_f_auprc - avg_z_auprc:>+7.3f} "
              f"{avg_z_f1:>7.3f} {avg_f_f1:>7.3f} {avg_f_f1 - avg_z_f1:>+7.3f} "
              f"{avg_z_se:>7.3f} {avg_f_se:>7.3f} {avg_f_se - avg_z_se:>+7.3f} "
              f"{avg_z_sp:>7.3f} {avg_f_sp:>7.3f} {avg_f_sp - avg_z_sp:>+7.3f}")

        zs_score = (avg_z_auc + avg_z_ba) / 2
        fs_score = (avg_f_auc + avg_f_ba) / 2
        delta_auc = avg_f_auc - avg_z_auc
        delta_score = fs_score - zs_score
        print(f"\n  ZS mean Score (AUC+BA)/2: {zs_score:.3f} | "
              f"FS mean Score: {fs_score:.3f} | Δ Score: {delta_score:+.3f}")
        print(f"  Verdict: {'Few-Shot better ✅' if delta_score > 0 else 'Zero-Shot better 🔴'}")


# Ensemble new code

In [ ]:
# ================================================================
# CELL 1 of 4 -- ENSEMBLE WINNING CONFIGS (MANUAL)
#
# Fill in the best configs from each model's individual benchmark.
# No automatic detection -- set these based on your printed output.
#
# WEIGHTS: w = (AUC + BAC) / 2, taken from each model's best-config
#          benchmark results and hardcoded here so they are always
#          correct even in a fresh session (no globals dependency).
# ================================================================

# ----------------------------------------------------------------
# 1. WhyXrayCLIP -- uniform config, 64-shot alpha=0.75
# ----------------------------------------------------------------
WXRC_ENS_N_SHOTS = 64
WXRC_ENS_ALPHA   = 0.75

WXRC_ENS_CONTEXT_STRATEGY = {
    "Edema":                      "old_format",
    "Atelectasis":                "no_context",
    "Pleural Effusion":           "no_context",
    "Enlarged Cardiomediastinum": "radiology_format",
    "Consolidation":              "old_format",
}

# w = (AUC + BAC) / 2  from FS 64-shot alpha=0.75 benchmark
WXRC_ENS_WEIGHTS = {
    "Edema":                      0.8090,  # AUC=0.910, BAC=0.708
    "Atelectasis":                0.7025,  # AUC=0.736, BAC=0.669
    "Pleural Effusion":           0.8535,  # AUC=0.923, BAC=0.784
    "Enlarged Cardiomediastinum": 0.8280,  # AUC=0.862, BAC=0.794
    "Consolidation":              0.8525,  # AUC=0.972, BAC=0.733
}

# FS thresholds for the winning config (train_df calibrated)
WXRC_FS_THRESHOLDS = {
    "Edema": {
        (64, 0.75): -0.0019,
    },
    "Atelectasis": {
        (64, 0.75): -0.0163,
    },
    "Pleural Effusion": {
        (64, 0.75): -0.0280,
    },
    "Enlarged Cardiomediastinum": {
        (64, 0.75): -0.0200,
    },
    "Consolidation": {
        (64, 0.75): -0.0430,
    },
}

# ----------------------------------------------------------------
# 2. CheXZero -- 8-shot, per-label txt_w
# ----------------------------------------------------------------
CZ_ENS_N_SHOTS = 8

CZ_ENS_LABEL_WEIGHTS = {
    # label: (txt_w, img_w)
    "Edema":                      (0.2, 0.8),
    "Atelectasis":                (0.2, 0.8),
    "Pleural Effusion":           (0.5, 0.5),
    "Enlarged Cardiomediastinum": (0.4, 0.6),
    "Consolidation":              (0.3, 0.7),
}

CZERO_ENS_CONTEXT_STRATEGY = {
    "Edema":                      "radiology_format",
    "Atelectasis":                "radiology_format",
    "Pleural Effusion":           "radiology_format",
    "Enlarged Cardiomediastinum": "old_format",
    "Consolidation":              "radiology_format",
}

# w = (AUC + BAC) / 2  from FS 8-shot benchmark
CZ_ENS_WEIGHTS = {
    "Edema":                      0.7885,  # AUC=0.827, BAC=0.750
    "Atelectasis":                0.7085,  # AUC=0.749, BAC=0.668
    "Pleural Effusion":           0.8885,  # AUC=0.927, BAC=0.850
    "Enlarged Cardiomediastinum": 0.7320,  # AUC=0.758, BAC=0.706
    "Consolidation":              0.8390,  # AUC=0.978, BAC=0.700
}

# FS thresholds for the winning config (train_df calibrated)
CZFS_THRESHOLDS = {
    "Edema": {
        (8, 0.2): 0.0142,
    },
    "Atelectasis": {
        (8, 0.2): 0.0117,
    },
    "Pleural Effusion": {
        (8, 0.5): 0.0142,
    },
    "Enlarged Cardiomediastinum": {
        (8, 0.4): -0.0185,
    },
    "Consolidation": {
        (8, 0.3): 0.0405,
    },
}

# ----------------------------------------------------------------
# 3. BioViL-T -- text_only won (alpha=1.0, no prototypes)
# ----------------------------------------------------------------
BV_ENS_ALPHA   = 1.0   # alpha=1.0 means text_only
BV_ENS_N_SHOTS = None  # no prototypes

BIOVILT_ENS_CONTEXT_STRATEGY = {
    "Edema":                      "radiology_format",
    "Atelectasis":                "radiology_format",
    "Pleural Effusion":           "old_format",
    "Enlarged Cardiomediastinum": "no_context",
    "Consolidation":              "old_format",
}

# w = (AUC + BAC) / 2  from text_only benchmark
BV_ENS_WEIGHTS = {
    "Edema":                      0.7325,  # AUC=0.750, BAC=0.715
    "Atelectasis":                0.7120,  # AUC=0.773, BAC=0.651
    "Pleural Effusion":           0.8105,  # AUC=0.841, BAC=0.780
    "Enlarged Cardiomediastinum": 0.7475,  # AUC=0.789, BAC=0.706
    "Consolidation":              0.8335,  # AUC=0.917, BAC=0.750
}

# Thresholds for the winning strategy (train_df calibrated)
BIOVILT_THRESHOLDS = {
    "Edema": -0.0518,
    "Atelectasis": 0.0069,
    "Pleural Effusion": -0.1039,
    "Enlarged Cardiomediastinum": -0.0085,
    "Consolidation": 0.0297,
}

# ----------------------------------------------------------------
# 4. BioMedCLIP (finetuned) -- per-label best config
# ----------------------------------------------------------------
BMC_ENS_BEST_PER_LABEL = {
    "Edema": {
        "n_shots": 4, "alpha": 0.50,
        "model": "original", "context_strategy": "old_format",
    },
    "Atelectasis": {
        "n_shots": 4, "alpha": 0.50,
        "model": "ft1", "context_strategy": "no_context",
    },
    "Pleural Effusion": {
        "n_shots": 4, "alpha": 0.50,
        "model": "ft1", "context_strategy": "no_context",
    },
    "Enlarged Cardiomediastinum": {
        "n_shots": 64, "alpha": 0.00,  # pure prototype score
        "model": "ft1", "context_strategy": "no_context",
    },
    "Consolidation": {
        "n_shots": 32, "alpha": 0.25,
        "model": "original", "context_strategy": "no_context",
    },
}

# w = (AUC + BAC) / 2  from best-per-label FS benchmark
BMC_ENS_WEIGHTS = {
    "Edema":                      0.9165,  # AUC=0.955, BAC=0.878
    "Atelectasis":                0.6825,  # AUC=0.711, BAC=0.654
    "Pleural Effusion":           0.6690,  # AUC=0.695, BAC=0.643
    "Enlarged Cardiomediastinum": 0.7600,  # AUC=0.785, BAC=0.735
    "Consolidation":              0.8900,  # AUC=0.972, BAC=0.808
}

# FS thresholds — from BMC_FS_THRESHOLDS[label][n_shots][alpha]
BMC_FS_THRESHOLDS = {
    "Edema":                      -0.0023,
    "Atelectasis":                -0.0024,
    "Pleural Effusion":           0.0167,
    "Enlarged Cardiomediastinum": -0.0111,
    "Consolidation":              0.0095,
}


LABEL_MODEL_INCLUSION = {
    # Score = (AUC + BAC) / 2 from Ensemble B individual model results
    # Include model if score >= best_score_for_label * 0.93
    #
    # Edema scores:    wxrc=0.809 cz=0.788 bv=0.732 bmc=0.916
    #   best=0.916 → threshold=0.852 → wxrc✅ cz✅ bv❌ bmc✅
    "Edema": {
        "wx":  True,   # 0.809 ✅
        "cz":  True,   # 0.788 ✅
        "bv":  False,  # 0.732 ❌ (below 0.852)
        "bmc": True,   # 0.916 ✅ best
    },
    # Atelectasis scores: wxrc=0.702 cz=0.715 bv=0.712 bmc=0.682
    #   best=0.715 → threshold=0.665 → wxrc✅ cz✅ bv✅ bmc❌
    "Atelectasis": {
        "wx":  True,   # 0.702 ✅
        "cz":  True,   # 0.715 ✅ best
        "bv":  True,   # 0.712 ✅
        "bmc": False,   # 0.682 ❌
    },
    # Pleural Effusion: wxrc=0.853 cz=0.920 bv=0.810 bmc=0.669
    #   best=0.920 → threshold=0.856 → wxrc✅ cz✅ bv✅ bmc❌
    "Pleural Effusion": {
        "wx":  True,   # 0.853 ✅
        "cz":  True,   # 0.920 ✅ best
        "bv":  True,  # 0.810 ✅
        "bmc": False,  # 0.669 ❌
    },
    # Enlarged Cardiomediastinum: wxrc=0.828 cz=0.722 bv=0.747 bmc=0.760
    #   best=0.828 → threshold=0.770 → wxrc✅ cz❌ bv❌ bmc✅
    "Enlarged Cardiomediastinum": {
        "wx":  True,   # 0.828 ✅ best
        "cz":  False,  # 0.722 ❌
        "bv":  False,  # 0.747 ❌
        "bmc": True,  # 0.760 ✅
    },
    # Consolidation: wxrc=0.852 cz=0.839 bv=0.833 bmc=0.890
    #   best=0.890 → threshold=0.828 → all ✅
    "Consolidation": {
        "wx":  True,   # 0.852 ✅
        "cz":  True,   # 0.839 ✅
        "bv":  True,   # 0.833 ✅
        "bmc": True,   # 0.890 ✅ best
    },
}


# ----------------------------------------------------------------
# Sanity print
# ----------------------------------------------------------------
print("=" * 75)
print("  ENSEMBLE WINNING CONFIGS + WEIGHTS (manually set)")
print("=" * 75)
print(f"\n  WhyXrayCLIP  : {WXRC_ENS_N_SHOTS}-shot, alpha={WXRC_ENS_ALPHA}")
for lbl in TARGET_LABELS:
    ctx = WXRC_ENS_CONTEXT_STRATEGY[lbl]
    w   = WXRC_ENS_WEIGHTS[lbl]
    t   = WXRC_FS_THRESHOLDS.get(lbl, {}).get((WXRC_ENS_N_SHOTS, WXRC_ENS_ALPHA))
    t_s = f"{t:.4f}" if t is not None else "n/a"
    print(f"    {lbl:<35} ctx={ctx:<18}  w={w:.4f}  t={t_s}")

print(f"\n  CheXZero     : {CZ_ENS_N_SHOTS}-shot, per-label txt_w")
for lbl in TARGET_LABELS:
    tw, iw = CZ_ENS_LABEL_WEIGHTS[lbl]
    ctx    = CZERO_ENS_CONTEXT_STRATEGY[lbl]
    w      = CZ_ENS_WEIGHTS[lbl]
    t      = CZFS_THRESHOLDS.get(lbl, {}).get((CZ_ENS_N_SHOTS, tw))
    t_s    = f"{t:.4f}" if t is not None else "n/a"
    print(f"    {lbl:<35} txt_w={tw}, img_w={iw}  ctx={ctx:<18}  w={w:.4f}  t={t_s}")

print(f"\n  BioViL-T     : text_only (alpha={BV_ENS_ALPHA}, n_shots={BV_ENS_N_SHOTS})")
for lbl in TARGET_LABELS:
    ctx = BIOVILT_ENS_CONTEXT_STRATEGY[lbl]
    w   = BV_ENS_WEIGHTS[lbl]
    t   = BIOVILT_THRESHOLDS.get(lbl)
    t_s = f"{t:.4f}" if t is not None else "n/a"
    print(f"    {lbl:<35} ctx={ctx:<18}  w={w:.4f}  t={t_s}")

print(f"\n  BioMedCLIP(FT) : per-label best config")
for lbl, cfg in BMC_ENS_BEST_PER_LABEL.items():
    w = BMC_ENS_WEIGHTS[lbl]
    print(f"    {lbl:<35} n={cfg['n_shots']:<3}  alpha={cfg['alpha']:.2f}"
          f"  model={cfg['model']:<8}  ctx={cfg['context_strategy']:<18}  w={w:.4f}")

print(f"\n  {'Label':<35} {'WXRC':>7}  {'CZ':>7}  {'BV':>7}  {'BMC':>7}")
print("  " + "-" * 65)
for lbl in TARGET_LABELS:
    print(f"  {lbl:<35} {WXRC_ENS_WEIGHTS[lbl]:>7.4f}  "
          f"{CZ_ENS_WEIGHTS[lbl]:>7.4f}  "
          f"{BV_ENS_WEIGHTS[lbl]:>7.4f}  "
          f"{BMC_ENS_WEIGHTS[lbl]:>7.4f}")
print("\n  Config cell complete -- run Ensemble A or Ensemble B next.")


# Sanity check
for _lbl, _inc in LABEL_MODEL_INCLUSION.items():
    assert any(_inc.values()), f"⚠️ {_lbl}: all models disabled"
 
print("\n  PER-LABEL MODEL INCLUSION  (based on (AUC+BAC)/2 >= best*0.93):")
print(f"  {'Label':<35} {'WXRC':>6} {'CZ':>6} {'BV':>6} {'BMC':>6}")
print("  " + "-" * 60)
for _lbl, _inc in LABEL_MODEL_INCLUSION.items():
    print(f"  {_lbl:<35} "
          f"{'✅' if _inc.get('wx')  else '❌':>6} "
          f"{'✅' if _inc.get('cz')  else '❌':>6} "
          f"{'✅' if _inc.get('bv')  else '❌':>6} "
          f"{'✅' if _inc.get('bmc') else '❌':>6}")


In [ ]:
# ================================================================
# CELL 2 of 4 -- ENSEMBLE A: WhyXrayCLIP + CheXZero + BioViL-T
#
# Three-model ensemble using the best FS config for each model.
# BioViL-T contributes text_only scores (alpha=1.0, no prototypes).
# Fusion : performance-weighted rank fusion using hardcoded weights
#          from Cell 1  (w = (AUC+BAC)/2 per label per model).
# Threshold: calibrated on train_df certain samples (no GT leakage).
# ================================================================

import os, torch, torch.nn.functional as F, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from scipy.stats import rankdata
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              confusion_matrix, f1_score, accuracy_score,
                              balanced_accuracy_score)
from sklearn.model_selection import StratifiedShuffleSplit

RANDOM_SEED_ENS = 42

# ----------------------------------------------------------------
# SCORE FUNCTIONS
# ----------------------------------------------------------------

def _ens_a_score_wxrc(img_path, label, feature_values, pos_proto, neg_proto):
    """WhyXrayCLIP: combined = alpha*text + (1-alpha)*proto, winning context."""
    _saved = dict(WXRC_CONTEXT_STRATEGY)
    WXRC_CONTEXT_STRATEGY.update(WXRC_ENS_CONTEXT_STRATEGY)
    ctx      = build_wxrc_context(label, feature_values or [])
    pos_text = ctx + POSITIVE_PROMPTS[label]
    neg_text = ctx + NEGATIVE_PROMPTS[label]
    WXRC_CONTEXT_STRATEGY.update(_saved)

    image = clip_preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        img_f = clip_model.encode_image(image)
        img_f = img_f / img_f.norm(dim=-1, keepdim=True)
        tok   = clip_tokenizer([pos_text, neg_text]).to(device)
        txt_f = clip_model.encode_text(tok)
        txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)

    text_score = float((img_f @ txt_f[0:1].T).item() - (img_f @ txt_f[1:2].T).item())
    if pos_proto is None or neg_proto is None:
        return text_score
    proto_score = float((img_f @ pos_proto.T).item() - (img_f @ neg_proto.T).item())
    return WXRC_ENS_ALPHA * text_score + (1 - WXRC_ENS_ALPHA) * proto_score


def _ens_a_score_czero(img_path, label, cz_pos_anchor, cz_neg_anchor):
    """CheXZero: sim(img, pos_anchor) - sim(img, neg_anchor).
    Context is baked into the anchor via build_chexzero_prototypes."""
    image = chexzero_preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        img_f = chexzero_model.encode_image(image)
        img_f = img_f / img_f.norm(dim=-1, keepdim=True)
    return float((img_f @ cz_pos_anchor.T).item() - (img_f @ cz_neg_anchor.T).item())


def _ens_a_score_biovilt(img_path, label, feature_values):
    """BioViL-T text_only score using winning context strategy."""
    _saved = dict(LABEL_CONTEXT_STRATEGY)
    LABEL_CONTEXT_STRATEGY.update(BIOVILT_ENS_CONTEXT_STRATEGY)
    pos_prompts, neg_prompts = build_prompts_with_context(label, feature_values or [])
    LABEL_CONTEXT_STRATEGY.update(_saved)

    pos_sims = [float(biovilt_engine.get_similarity_score_from_raw_data(
                    image_path=Path(img_path), query_text=p)) for p in pos_prompts]
    neg_sims = [float(biovilt_engine.get_similarity_score_from_raw_data(
                    image_path=Path(img_path), query_text=p)) for p in neg_prompts]
    return float(np.mean(pos_sims) - np.mean(neg_sims))


# ----------------------------------------------------------------
# RANK FUSION (shared by Cells 2 & 3)
# ----------------------------------------------------------------

def _rank_fuse(scores_dict, weights_dict):
    """Performance-weighted rank fusion.
    scores_dict  : {model_key: np.array[N]}
    weights_dict : {model_key: float}  hardcoded (AUC+BAC)/2 from Cell 1
    norm_rank in (0,1]; fused = sum(w * norm_rank) / sum(w)
    """
    n       = len(next(iter(scores_dict.values())))
    fused   = np.zeros(n)
    total_w = sum(weights_dict.get(m, 1.0) for m in scores_dict)
    for model, scores in scores_dict.items():
        w          = weights_dict.get(model, 1.0)
        norm_ranks = rankdata(scores, method="average") / n
        fused     += w * norm_ranks
    return fused / total_w


# ----------------------------------------------------------------
# METRICS
# ----------------------------------------------------------------

def _ens_metrics(scores_arr, y_true_arr, threshold):
    if len(np.unique(y_true_arr)) < 2:
        return {}
    auc = roc_auc_score(y_true_arr, scores_arr)
    try:  auprc = average_precision_score(y_true_arr, scores_arr)
    except: auprc = float(y_true_arr.mean())
    preds = (scores_arr > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_arr, preds, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        "auc":      round(float(auc),   3),
        "auprc":    round(float(auprc), 3),
        "bal_acc":  round((sens + spec) / 2, 3),
        "f1":       round(float(f1_score(y_true_arr, preds, zero_division=0)), 3),
        "sens":     round(float(sens), 3),
        "spec":     round(float(spec), 3),
        "accuracy": round(float(accuracy_score(y_true_arr, preds)), 3),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
        "n":     len(y_true_arr),
        "n_pos": int(y_true_arr.sum()),
        "n_neg": int((y_true_arr == 0).sum()),
        "threshold": round(float(threshold), 4),
    }


# ----------------------------------------------------------------
# INDIVIDUAL MODEL METRICS (uses each model's own FS threshold)
# ----------------------------------------------------------------

def _ind_metrics(arr, y_true, model_name, label):
    if len(np.unique(y_true)) < 2:
        return {"auc": 0.5, "auprc": float(y_true.mean()),
                "bac": 0.5, "f1": 0.0, "sens": 0.0, "spec": 0.0}
    auc_  = float(roc_auc_score(y_true, arr))
    try:   auprc_ = float(average_precision_score(y_true, arr))
    except: auprc_ = float(y_true.mean())

    t = None
    if model_name == "wx":
        fs_key = (WXRC_ENS_N_SHOTS, WXRC_ENS_ALPHA)
        t = (WXRC_FS_THRESHOLDS.get(label, {}).get(fs_key)
             if "WXRC_FS_THRESHOLDS" in globals() else None)
    elif model_name == "cz":
        txt_w  = CZ_ENS_LABEL_WEIGHTS.get(label, (0.5, 0.5))[0]
        fs_key = (CZ_ENS_N_SHOTS, txt_w)
        t = (CZFS_THRESHOLDS.get(label, {}).get(fs_key)
             if "CZFS_THRESHOLDS" in globals() else None)
    elif model_name == "bv":
        t = (BIOVILT_THRESHOLDS.get(label)
             if "BIOVILT_THRESHOLDS" in globals() else None)

    if t is None:
        t = float((arr[y_true==1].mean() + arr[y_true==0].mean()) / 2) \
            if (y_true==1).sum() > 0 and (y_true==0).sum() > 0 else 0.0

    preds = (arr > t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    sens_ = tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else 0.0
    spec_ = tn_ / (tn_ + fp_) if (tn_ + fp_) > 0 else 0.0
    return {
        "auc":   round(auc_, 3),   "auprc": round(auprc_, 3),
        "bac":   round((sens_ + spec_) / 2, 3),
        "f1":    round(float(f1_score(y_true, preds, zero_division=0)), 3),
        "sens":  round(float(sens_), 3), "spec": round(float(spec_), 3),
    }


# ----------------------------------------------------------------
# THRESHOLD CALIBRATION (train_df, half-split BAC, no GT leakage)
# ----------------------------------------------------------------

def _calibrate_ens_threshold(label, score_fn_dict, label_weights_dict,
                               n_samples=600):
    """Half-split BAC calibration on train_df certain (0/1) samples.
    score_fn_dict      : {model_key: callable(img_path, label, fv)}
    label_weights_dict : {model_key: float}  hardcoded weights from Cell 1
    """
    df     = train_df[train_df[label].isin([0, 1])].copy()
    n_each = min(n_samples // 2,
                 int((df[label] == 1).sum()),
                 int((df[label] == 0).sum()))
    if n_each < 20:
        print(f"  !! {label}: too few cal samples -> using 0.5")
        return 0.5

    cal_df = pd.concat([
        df[df[label] == 1].sample(n_each, random_state=RANDOM_SEED_ENS),
        df[df[label] == 0].sample(n_each, random_state=RANDOM_SEED_ENS),
    ]).sample(frac=1, random_state=RANDOM_SEED_ENS).reset_index(drop=True)
    print(f"  {label}: calibrating on {len(cal_df)} samples (pos={n_each}, neg={n_each})...")

    top_feats    = top_features_per_label.get(label, [])
    model_scores = {k: [] for k in score_fn_dict}
    labs         = []

    for _, row in tqdm(cal_df.iterrows(), total=len(cal_df),
                       desc=f"    Cal [{label}]", leave=False):
        try:
            clean    = str(row["Path"]).replace("CheXpert-v1.0-small/","").replace("CheXpert-v1.0/","")
            img_path = os.path.join(BASE_PATH, clean)
            fv       = get_sample_features(row["Path"], label, top_feats, train_df)
            for k, fn in score_fn_dict.items():
                model_scores[k].append(fn(img_path, label, fv))
            labs.append(int(row[label]))
        except Exception:
            continue

    if len(labs) < 20:
        print(f"  !! {label}: too few valid cal samples -> using 0.5")
        return 0.5

    fused    = _rank_fuse({k: np.array(v) for k, v in model_scores.items()},
                           label_weights_dict)
    labs_arr = np.array(labs)

    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=RANDOM_SEED_ENS)
    find_idx, val_idx = next(sss.split(fused, labs_arr))
    s_f, l_f = fused[find_idx], labs_arr[find_idx]
    s_v, l_v = fused[val_idx],  labs_arr[val_idx]

    best_t_find, best_ba_find = 0.5, 0.0
    for t in np.linspace(0.2, 0.8, 61):
        preds = (s_f > t).astype(int)
        if preds.sum() in (0, len(preds)): continue
        ba = balanced_accuracy_score(l_f, preds)
        if ba > best_ba_find:
            best_ba_find, best_t_find = ba, t

    midpoint   = float((s_f[l_f==1].mean() + s_f[l_f==0].mean()) / 2) \
                 if (l_f==1).sum() > 0 and (l_f==0).sum() > 0 else 0.5
    median_pos = float(np.median(s_f[l_f==1])) if (l_f==1).sum() > 0 else 0.5

    best_final_t, best_val_ba = 0.5, 0.0
    for t in sorted({best_t_find, midpoint, median_pos}):
        preds = (s_v > t).astype(int)
        if preds.sum() in (0, len(preds)): continue
        ba = balanced_accuracy_score(l_v, preds)
        if ba > best_val_ba:
            best_val_ba, best_final_t = ba, t

    print(f"  {label}: threshold={best_final_t:.4f}  "
          f"(find_BAC={best_ba_find:.3f} | val_BAC={best_val_ba:.3f})")
    return float(best_final_t)


In [ ]:
# ----------------------------------------------------------------
# MAIN -- Ensemble A
# ----------------------------------------------------------------

def run_ensemble_a():
    print("=" * 75)
    print("  ENSEMBLE A: WhyXrayCLIP + CheXZero + BioViL-T")
    print(f"  WhyXrayCLIP : {WXRC_ENS_N_SHOTS}-shot, alpha={WXRC_ENS_ALPHA}")
    print(f"  CheXZero    : {CZ_ENS_N_SHOTS}-shot, per-label txt_w")
    print(f"  BioViL-T    : text_only (alpha={BV_ENS_ALPHA})")
    print(f"  Weights     : hardcoded (AUC+BAC)/2 from best-config benchmarks")
    print(f"  Fusion      : performance-weighted rank fusion")
    print("=" * 75)

    # Step 1: Build / reuse prototypes
    print("\n" + "-" * 75)
    print("  STEP 1: Building prototypes / anchors")
    print("-" * 75)

    print("\n  [WhyXrayCLIP]")
    if ("WXRC_FINAL_PROTO" in globals()
            and globals().get("WXRC_FINAL_N_SHOTS") == WXRC_ENS_N_SHOTS
            and globals().get("WXRC_FINAL_ALPHA")   == WXRC_ENS_ALPHA):
        wx_protos = WXRC_FINAL_PROTO
        print("  Reusing WXRC_FINAL_PROTO from memory.")
    else:
        wx_protos = {}
        for label in TARGET_LABELS:
            pos_p, neg_p, meta = build_wxrc_prototype(
                label, WXRC_ENS_N_SHOTS, seed=RANDOM_SEED_ENS)
            wx_protos[label] = (pos_p, neg_p)
            status = (f"pos={meta.get('n_pos','?')} neg={meta.get('n_neg','?')}"
                      if pos_p is not None else "FAILED")
            print(f"    {label}: {status}")

    print("\n  [CheXZero]")
    _saved_czero_ctx = dict(CZERO_CONTEXT_STRATEGY) if "CZERO_CONTEXT_STRATEGY" in globals() else {}
    if "CZERO_CONTEXT_STRATEGY" in globals():
        CZERO_CONTEXT_STRATEGY.update(CZERO_ENS_CONTEXT_STRATEGY)

    if ("CHEXZERO_COMBINED_PROTO" in globals() and CHEXZERO_COMBINED_PROTO
            and globals().get("best_shot", -1) == CZ_ENS_N_SHOTS):
        cz_anchors = CHEXZERO_COMBINED_PROTO
        print("  Reusing CHEXZERO_COMBINED_PROTO from memory.")
    else:
        cz_anchors, _ = build_chexzero_prototypes(CZ_ENS_N_SHOTS, CZ_ENS_LABEL_WEIGHTS)

    if "CZERO_CONTEXT_STRATEGY" in globals():
        CZERO_CONTEXT_STRATEGY.update(_saved_czero_ctx)

    print("\n  [BioViL-T] text_only -- no prototypes needed.")

    # Step 2: Print hardcoded weights
    print("\n" + "-" * 75)
    print("  STEP 2: Ensemble weights  w = (AUC+BAC)/2  [hardcoded in Cell 1]")
    print("-" * 75)
    print(f"\n  {'Label':<35} {'WXRC':>7}  {'CZ':>7}  {'BV':>7}")
    print("  " + "-" * 60)
    for label in TARGET_LABELS:
        print(f"  {label:<35} {WXRC_ENS_WEIGHTS[label]:>7.4f}  "
              f"{CZ_ENS_WEIGHTS[label]:>7.4f}  "
              f"{BV_ENS_WEIGHTS[label]:>7.4f}")

    # Step 3: Calibrate ensemble thresholds
    print("\n" + "-" * 75)
    print("  STEP 3: Calibrating ensemble thresholds on train_df")
    print("-" * 75)

    ENS_A_THRESHOLDS = {}
    for label in TARGET_LABELS:
        wx_pp, wx_np = wx_protos[label]
        cz_pa, cz_na = cz_anchors[label]
        inc = LABEL_MODEL_INCLUSION.get(
            label, {"wx": True, "cz": True, "bv": True})
 
        # Build score_fns and label_w for active models only
        score_fns = {}
        label_w   = {}
        if inc.get("wx", True):
            score_fns["wx"] = (lambda p, lbl, fv, _pp=wx_pp, _np=wx_np:
                               _ens_a_score_wxrc(p, lbl, fv, _pp, _np))
            label_w["wx"]   = WXRC_ENS_WEIGHTS[label]
        if inc.get("cz", True):
            score_fns["cz"] = (lambda p, lbl, fv, _pa=cz_pa, _na=cz_na:
                               _ens_a_score_czero(p, lbl, _pa, _na))
            label_w["cz"]   = CZ_ENS_WEIGHTS[label]
        if inc.get("bv", True):
            score_fns["bv"] = lambda p, lbl, fv: _ens_a_score_biovilt(p, lbl, fv)
            label_w["bv"]   = BV_ENS_WEIGHTS[label]
 
        # CALL _calibrate_ens_threshold (was missing before)
        ENS_A_THRESHOLDS[label] = _calibrate_ens_threshold(
            label, score_fns, label_w)
 
    print("\n  Calibrated thresholds:")
    for lbl, t in ENS_A_THRESHOLDS.items():
        print(f"    {lbl:<35}: {t:.4f}")
        
    for lbl, t in ENS_A_THRESHOLDS.items():
        print(f"    {lbl:<35}: {t:.4f}")

    # Step 4: Score GT uncertain samples
    print("\n" + "-" * 75)
    print("  STEP 4: Scoring GT uncertain samples")
    print("-" * 75)

    labeler_df = rule_labeler_df.copy()
    gt_df      = ground_truth_df.copy()
    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True); break

    label_metrics_a   = {}
    all_sample_rows_a = []

    for label in TARGET_LABELS:
        print(f"\n  -- {label} --")
        uncertain_idx = labeler_df[labeler_df[label] == -1].index
        gt_samples    = (gt_df.loc[uncertain_idx, ["Study", label]]
                         .rename(columns={label: "gt"})
                         .pipe(lambda d: d[d["gt"].isin([0, 1])])
                         .reset_index(drop=True))
        if len(gt_samples) == 0:
            print("  No GT uncertain samples -- skipping"); continue
        print(f"  n={len(gt_samples)}  pos={int(gt_samples['gt'].sum())}  "
              f"neg={int((gt_samples['gt']==0).sum())}")

        top_feats    = top_features_per_label.get(label, [])
        wx_pp, wx_np = wx_protos[label]
        cz_pa, cz_na = cz_anchors[label]

        wx_scores, cz_scores, bv_scores = [], [], []
        y_true_list, study_list         = [], []

        for _, row in tqdm(gt_samples.iterrows(), total=len(gt_samples),
                           desc=f"  Ens-A [{label}]", leave=False):
            try:
                img_path = fix_test_path(row["Study"])
                fv       = get_sample_features(row["Study"], label, top_feats, rule_labeler_df)
                wx_scores.append(_ens_a_score_wxrc(img_path, label, fv, wx_pp, wx_np))
                cz_scores.append(_ens_a_score_czero(img_path, label, cz_pa, cz_na))
                bv_scores.append(_ens_a_score_biovilt(img_path, label, fv))
                y_true_list.append(int(row["gt"]))
                study_list.append(row["Study"])
            except Exception as e:
                print(f"    SKIP {row['Study']}: {e}"); continue

        if len(y_true_list) < 5:
            print(f"  Too few valid samples ({len(y_true_list)}) -- skipping"); continue

        y_true = np.array(y_true_list)
        wx_arr = np.array(wx_scores)
        cz_arr = np.array(cz_scores)
        bv_arr = np.array(bv_scores)
 
        # Build active scores/weights and CALL _rank_fuse directly
        inc = LABEL_MODEL_INCLUSION.get(
            label, {"wx": True, "cz": True, "bv": True})
        scores_dict  = {}
        weights_dict = {}
        if inc.get("wx", True):
            scores_dict["wx"]  = wx_arr
            weights_dict["wx"] = WXRC_ENS_WEIGHTS[label]
        if inc.get("cz", True):
            scores_dict["cz"]  = cz_arr
            weights_dict["cz"] = CZ_ENS_WEIGHTS[label]
        if inc.get("bv", True):
            scores_dict["bv"]  = bv_arr
            weights_dict["bv"] = BV_ENS_WEIGHTS[label]
 
        # fused_scores now exists — _ens_metrics will work
        fused_scores = _rank_fuse(scores_dict, weights_dict)
        label_w      = weights_dict   # used for printing weights below
        threshold    = ENS_A_THRESHOLDS[label]

        wx_m = _ind_metrics(wx_arr, y_true, "wx", label)
        cz_m = _ind_metrics(cz_arr, y_true, "cz", label)
        bv_m = _ind_metrics(bv_arr, y_true, "bv", label)
        m    = _ens_metrics(fused_scores, y_true, threshold)
        if not m: continue

        best_ind_auc = max(wx_m["auc"], cz_m["auc"], bv_m["auc"])
        delta_auc    = round(m["auc"] - best_ind_auc, 3)
        marker       = " [IMPROVES]" if delta_auc > 0.01 else (" [~]" if delta_auc > -0.01 else " [WORSE]")

        label_metrics_a[label] = {
            **m, "label": label,
            "wx_auc": wx_m["auc"],   "cz_auc": cz_m["auc"],   "bv_auc": bv_m["auc"],
            "wx_auprc":wx_m["auprc"],"cz_auprc":cz_m["auprc"],"bv_auprc":bv_m["auprc"],
            "wx_bac": wx_m["bac"],   "cz_bac": cz_m["bac"],   "bv_bac": bv_m["bac"],
            "wx_f1":  wx_m["f1"],    "cz_f1":  cz_m["f1"],    "bv_f1":  bv_m["f1"],
            "wx_sens":wx_m["sens"],  "cz_sens":cz_m["sens"],  "bv_sens":bv_m["sens"],
            "wx_spec":wx_m["spec"],  "cz_spec":cz_m["spec"],  "bv_spec":bv_m["spec"],
            "ens_score": round((m["auc"]+m["bal_acc"])/2, 3),
            "delta_vs_best_ind_auc": delta_auc,
        }

        print(f"\n  {'Model':<15} {'AUC':>6} {'AUPRC':>6} {'BAC':>6} {'F1':>6} "
              f"{'Sens':>6} {'Spec':>6}  {'(wt)':>7}")
        print(f"  {'-'*67}")
        # Safe print — only show active models
        model_print = [
            ("WhyXrayCLIP", wx_m, "wx"),
            ("CheXZero",    cz_m, "cz"),
            ("BioViL-T",    bv_m, "bv"),
        ]
        for mname, mm, mkey in model_print:
            wt_str = f"({label_w[mkey]:.4f})" if mkey in label_w else "(disabled)"
            print(f"  {mname:<15} {mm['auc']:>6.3f} {mm['auprc']:>6.3f} "
                  f"{mm['bac']:>6.3f} {mm['f1']:>6.3f} {mm['sens']:>6.3f} "
                  f"{mm['spec']:>6.3f}  {wt_str}")
        print(f"  {'-'*67}")
        print(f"  {'ENSEMBLE A':<15} {m['auc']:>6.3f} {m['auprc']:>6.3f} "
              f"{m['bal_acc']:>6.3f} {m['f1']:>6.3f} {m['sens']:>6.3f} "
              f"{m['spec']:>6.3f}  dAUC={delta_auc:+.3f}{marker}")
        print(f"  TP={m['tp']}  TN={m['tn']}  FP={m['fp']}  FN={m['fn']}  "
              f"threshold={threshold:.4f}")

        for i in range(len(y_true_list)):
            all_sample_rows_a.append({
                "label": label, "study": study_list[i],
                "y_true": y_true_list[i],
                "wx_score":    round(float(wx_scores[i]), 6),
                "cz_score":    round(float(cz_scores[i]), 6),
                "bv_score":    round(float(bv_scores[i]), 6),
                "fused_score": round(float(fused_scores[i]), 6),
                "y_pred":      int(fused_scores[i] > threshold),
                "threshold":   threshold,
            })

    if not label_metrics_a:
        print("\n  No results collected.")
        return pd.DataFrame(), pd.DataFrame()

    rows = list(label_metrics_a.values())
    print("\n\n" + "=" * 110)
    print("  ENSEMBLE A FINAL SUMMARY -- WhyXrayCLIP + CheXZero + BioViL-T")
    print("  Weights: (AUC+BAC)/2 hardcoded from best-config benchmarks")
    print("=" * 110)
    print(f"  {'Label':<30} {'Model':<13} {'AUC':>6} {'AUPRC':>6} {'BAC':>6} "
          f"{'F1':>6} {'Sens':>6} {'Spec':>6}  {'n':>4}")
    print("  " + "-" * 95)

    aucs, auprcs, bacs, f1s, senss, specs = [], [], [], [], [], []
    wx_auc_all, cz_auc_all, bv_auc_all   = [], [], []
    for r in rows:
        dlt = r["delta_vs_best_ind_auc"]
        mrk = " [+]" if dlt > 0.01 else (" [~]" if dlt > -0.01 else " [-]")
        if "wx" in label_w:
            print(f"{'WhyXrayCLIP':<20} {wx_m['auc']:>6.3f} {wx_m['auprc']:>6.3f} "
                  f"{wx_m['bac']:>6.3f} {wx_m['f1']:>6.3f} {wx_m['sens']:>6.3f} "
                  f"{wx_m['spec']:>6.3f}  ({label_w['wx']:.4f})")
        
        if "cz" in label_w:
            print(f"{'CheXZero':<20} {cz_m['auc']:>6.3f} {cz_m['auprc']:>6.3f} "
                  f"{cz_m['bac']:>6.3f} {cz_m['f1']:>6.3f} {cz_m['sens']:>6.3f} "
                  f"{cz_m['spec']:>6.3f}  ({label_w['cz']:.4f})")
        
        if "bv" in label_w:
            print(f"{'BioViL-T':<20} {bv_m['auc']:>6.3f} {bv_m['auprc']:>6.3f} "
                  f"{bv_m['bac']:>6.3f} {bv_m['f1']:>6.3f} {bv_m['sens']:>6.3f} "
                  f"{bv_m['spec']:>6.3f}  ({label_w['bv']:.4f})")
        
        if "bmc" in label_w:
            print(f"{'BioMedCLIP(FT)':<20} {bmc_m['auc']:>6.3f} {bmc_m['auprc']:>6.3f} "
                  f"{bmc_m['bac']:>6.3f} {bmc_m['f1']:>6.3f} {bmc_m['sens']:>6.3f} "
                  f"{bmc_m['spec']:>6.3f}  ({label_w['bmc']:.4f})")
        print(f"  {'':<30} {'ENSEMBLE A':<13} "
              f"{r['auc']:>6.3f} {r['auprc']:>6.3f} {r['bal_acc']:>6.3f} "
              f"{r['f1']:>6.3f} {r['sens']:>6.3f} {r['spec']:>6.3f}"
              f"  dAUC={dlt:+.3f}{mrk}")
        print(f"  {'':<30}  TP={r['tp']}  TN={r['tn']}  FP={r['fp']}  FN={r['fn']}")
        print("  " + "-" * 95)
        aucs.append(r["auc"]);     auprcs.append(r["auprc"])
        bacs.append(r["bal_acc"]); f1s.append(r["f1"])
        senss.append(r["sens"]);   specs.append(r["spec"])
        wx_auc_all.append(r["wx_auc"])
        cz_auc_all.append(r["cz_auc"])
        bv_auc_all.append(r["bv_auc"])

    ens_mauc       = np.mean(aucs)
    best_ind_mauc  = max(np.mean(wx_auc_all), np.mean(cz_auc_all), np.mean(bv_auc_all))
    delta_auc_mean = round(ens_mauc - best_ind_mauc, 3)
    total_n        = sum(r["n"] for r in rows)
    print(f"  {'MACRO MEAN -- Ensemble A':<44} "
          f"{ens_mauc:>6.3f} {np.mean(auprcs):>6.3f} {np.mean(bacs):>6.3f} "
          f"{np.mean(f1s):>6.3f} {np.mean(senss):>6.3f} {np.mean(specs):>6.3f}  n={total_n}")
    print(f"  dAUC ensemble A vs best individual (macro): {delta_auc_mean:+.3f}")

    tp_t = sum(r["tp"] for r in rows); tn_t = sum(r["tn"] for r in rows)
    fp_t = sum(r["fp"] for r in rows); fn_t = sum(r["fn"] for r in rows)
    print(f"\n  Aggregate confusion: TP={tp_t}  TN={tn_t}  FP={fp_t}  FN={fn_t}")
    verdict = ("IMPROVES" if delta_auc_mean > 0.01
               else ("COMPARABLE" if delta_auc_mean > -0.01 else "UNDERPERFORMS"))
    print(f"\n  Verdict: Ensemble A {verdict} vs best individual (dAUC = {delta_auc_mean:+.3f})")

    results_a_df  = pd.DataFrame(rows)
    resolved_a_df = pd.DataFrame(all_sample_rows_a)
    results_a_df.to_csv("ensemble_a_results.csv",  index=False)
    resolved_a_df.to_csv("ensemble_a_resolved.csv", index=False)
    print(f"\n  Saved: ensemble_a_results.csv  |  ensemble_a_resolved.csv")
    globals()["ENS_A_THRESHOLDS"] = ENS_A_THRESHOLDS
    return results_a_df, resolved_a_df


ens_a_results_df, ens_a_resolved_df = run_ensemble_a()

In [ ]:
# ================================================================
# CELL 3 of 4 -- ENSEMBLE B: WhyXrayCLIP + CheXZero + BioViL-T + BioMedCLIP(FT)
#
# Four-model ensemble adding fine-tuned BioMedCLIP on top of Ensemble A.
# BioMedCLIP uses per-label (n_shots, alpha, model_variant, context).
# Enlarged Cardiomediastinum: alpha=0.00 -> pure prototype score (no text).
# Fusion : performance-weighted rank fusion using hardcoded weights
#          from Cell 1  (w = (AUC+BAC)/2 per label per model).
# Threshold: calibrated on train_df certain samples (no GT leakage).
#
# NOTE: Requires Cell 2 to have been run first so that _rank_fuse,
#       _ens_metrics, _ind_metrics, _calibrate_ens_threshold,
#       _ens_a_score_wxrc, _ens_a_score_czero, _ens_a_score_biovilt
#       are all available in memory.
# ================================================================

import os, torch, torch.nn.functional as F, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# Per-label prototype store for BioMedCLIP (populated below)
_bmc_ens_protos = {}

RANDOM_SEED_ENS = 42

# ----------------------------------------------------------------
# BioMedCLIP SCORE FUNCTION
# ----------------------------------------------------------------

def _ens_b_score_bmc(img_path, label, feature_values):
    """BioMedCLIP few-shot score using per-label best config.
    combined = alpha*text + (1-alpha)*proto.
    For alpha=0.0 (Enlarged Cardiomediastinum): pure prototype score,
    text encoding is skipped entirely.
    """
    cfg       = BMC_ENS_BEST_PER_LABEL[label]
    alpha     = cfg["alpha"]
    pos_proto, neg_proto = _bmc_ens_protos.get(label, (None, None))

    _saved_bmc = dict(BMC_CONTEXT_STRATEGY) if "BMC_CONTEXT_STRATEGY" in globals() else {}
    if "BMC_CONTEXT_STRATEGY" in globals():
        BMC_CONTEXT_STRATEGY[label] = cfg["context_strategy"]
    ctx      = build_bmc_context(label, feature_values or [])
    pos_text = ctx + BIOMEDCLIP_POSITIVE_PROMPTS[label]
    neg_text = ctx + BIOMEDCLIP_NEGATIVE_PROMPTS[label]
    if "BMC_CONTEXT_STRATEGY" in globals():
        BMC_CONTEXT_STRATEGY.update(_saved_bmc)

    model, preprocess, tokenizer = _bmc_parts(label)
    image = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)

    with torch.no_grad():
        img_f = model.encode_image(image)
        img_f = img_f / img_f.norm(dim=-1, keepdim=True)

    if alpha == 0.0 and pos_proto is not None:
        # Pure prototype score -- skip text encoding
        return float((img_f @ pos_proto.T).item() - (img_f @ neg_proto.T).item())

    with torch.no_grad():
        pos_tok = tokenizer([pos_text]).to(device)
        neg_tok = tokenizer([neg_text]).to(device)
        pos_txt = model.encode_text(pos_tok)
        neg_txt = model.encode_text(neg_tok)
        pos_txt = pos_txt / pos_txt.norm(dim=-1, keepdim=True)
        neg_txt = neg_txt / neg_txt.norm(dim=-1, keepdim=True)

    text_score = float((img_f @ pos_txt.T).item() - (img_f @ neg_txt.T).item())
    if pos_proto is None or alpha >= 1.0:
        return text_score
    proto_score = float((img_f @ pos_proto.T).item() - (img_f @ neg_proto.T).item())
    return alpha * text_score + (1 - alpha) * proto_score


def _ind_metrics_bmc(arr, y_true, label):
    """Metrics for BioMedCLIP using its FS calibrated threshold."""
    from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, f1_score
    if len(np.unique(y_true)) < 2:
        return {"auc": 0.5, "auprc": float(y_true.mean()),
                "bac": 0.5, "f1": 0.0, "sens": 0.0, "spec": 0.0}
    auc_  = float(roc_auc_score(y_true, arr))
    try:   auprc_ = float(average_precision_score(y_true, arr))
    except: auprc_ = float(y_true.mean())

    cfg    = BMC_ENS_BEST_PER_LABEL[label]
    n_shot = cfg["n_shots"]
    alpha  = cfg["alpha"]
    t      = None
    if "BMC_FS_THRESHOLDS" in globals():
        val = BMC_FS_THRESHOLDS.get(label)
        if isinstance(val, dict):
            # nested structure
            t = val.get(n_shot, {}).get(alpha)
        else:
            # flat structure (float)
            t = val
    if t is None and "BIOMEDCLIP_THRESHOLDS_SMART" in globals():
        t = BIOMEDCLIP_THRESHOLDS_SMART.get(label)
    if t is None:
        t = float((arr[y_true==1].mean() + arr[y_true==0].mean()) / 2) \
            if (y_true==1).sum() > 0 and (y_true==0).sum() > 0 else 0.0

    preds = (arr > t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    sens_ = tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else 0.0
    spec_ = tn_ / (tn_ + fp_) if (tn_ + fp_) > 0 else 0.0
    return {
        "auc":   round(auc_, 3),   "auprc": round(auprc_, 3),
        "bac":   round((sens_ + spec_) / 2, 3),
        "f1":    round(float(f1_score(y_true, preds, zero_division=0)), 3),
        "sens":  round(float(sens_), 3), "spec": round(float(spec_), 3),
    }


def _ens_b_active_score_fns(label, wx_pp, wx_np, cz_pa, cz_na):
    """Return score_fns and label_w filtered by LABEL_MODEL_INCLUSION (4-model)."""
    inc = LABEL_MODEL_INCLUSION.get(
        label, {"wx": True, "cz": True, "bv": True, "bmc": True})
    score_fns = {}
    label_w   = {}
    if inc.get("wx", True):
        score_fns["wx"] = (lambda p, lbl, fv, _pp=wx_pp, _np=wx_np:
                           _ens_a_score_wxrc(p, lbl, fv, _pp, _np))
        label_w["wx"]   = WXRC_ENS_WEIGHTS[label]
    if inc.get("cz", True):
        score_fns["cz"] = (lambda p, lbl, fv, _pa=cz_pa, _na=cz_na:
                           _ens_a_score_czero(p, lbl, _pa, _na))
        label_w["cz"]   = CZ_ENS_WEIGHTS[label]
    if inc.get("bv", True):
        score_fns["bv"] = lambda p, lbl, fv: _ens_a_score_biovilt(p, lbl, fv)
        label_w["bv"]   = BV_ENS_WEIGHTS[label]
    if inc.get("bmc", True) and "BMC_ENS_WEIGHTS" in globals():
        score_fns["bmc"] = lambda p, lbl, fv: _ens_b_score_bmc(p, lbl, fv)
        label_w["bmc"]   = BMC_ENS_WEIGHTS[label]
    return score_fns, label_w
 
 
def _ens_b_fuse_for_label(label, wx_arr, cz_arr, bv_arr, bmc_arr):
    """Rank-fuse only the active models for this label (Ensemble B)."""
    inc = LABEL_MODEL_INCLUSION.get(
        label, {"wx": True, "cz": True, "bv": True, "bmc": True})
    scores_dict  = {}
    weights_dict = {}
    if inc.get("wx",  True):
        scores_dict["wx"]  = wx_arr;  weights_dict["wx"]  = WXRC_ENS_WEIGHTS[label]
    if inc.get("cz",  True):
        scores_dict["cz"]  = cz_arr;  weights_dict["cz"]  = CZ_ENS_WEIGHTS[label]
    if inc.get("bv",  True):
        scores_dict["bv"]  = bv_arr;  weights_dict["bv"]  = BV_ENS_WEIGHTS[label]
    if inc.get("bmc", True) and "BMC_ENS_WEIGHTS" in globals():
        scores_dict["bmc"] = bmc_arr; weights_dict["bmc"] = BMC_ENS_WEIGHTS[label]
    return _rank_fuse(scores_dict, weights_dict)



# ----------------------------------------------------------------
# THRESHOLD CALIBRATION (train_df, half-split BAC, no GT leakage)
# ----------------------------------------------------------------

def _calibrate_ens_threshold(label, score_fn_dict, label_weights_dict,
                               n_samples=600):
    """Half-split BAC calibration on train_df certain (0/1) samples.
    score_fn_dict      : {model_key: callable(img_path, label, fv)}
    label_weights_dict : {model_key: float}  hardcoded weights from Cell 1
    """
    df     = train_df[train_df[label].isin([0, 1])].copy()
    n_each = min(n_samples // 2,
                 int((df[label] == 1).sum()),
                 int((df[label] == 0).sum()))
    if n_each < 20:
        print(f"  !! {label}: too few cal samples -> using 0.5")
        return 0.5

    cal_df = pd.concat([
        df[df[label] == 1].sample(n_each, random_state=RANDOM_SEED_ENS),
        df[df[label] == 0].sample(n_each, random_state=RANDOM_SEED_ENS),
    ]).sample(frac=1, random_state=RANDOM_SEED_ENS).reset_index(drop=True)
    print(f"  {label}: calibrating on {len(cal_df)} samples (pos={n_each}, neg={n_each})...")

    top_feats    = top_features_per_label.get(label, [])
    model_scores = {k: [] for k in score_fn_dict}
    labs         = []

    for _, row in tqdm(cal_df.iterrows(), total=len(cal_df),
                       desc=f"    Cal [{label}]", leave=False):
        try:
            clean    = str(row["Path"]).replace("CheXpert-v1.0-small/","").replace("CheXpert-v1.0/","")
            img_path = os.path.join(BASE_PATH, clean)
            fv       = get_sample_features(row["Path"], label, top_feats, train_df)
            for k, fn in score_fn_dict.items():
                model_scores[k].append(fn(img_path, label, fv))
            labs.append(int(row[label]))
        except Exception:
            continue

    if len(labs) < 20:
        print(f"  !! {label}: too few valid cal samples -> using 0.5")
        return 0.5

    fused    = _rank_fuse({k: np.array(v) for k, v in model_scores.items()},
                           label_weights_dict)
    labs_arr = np.array(labs)

    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=RANDOM_SEED_ENS)
    find_idx, val_idx = next(sss.split(fused, labs_arr))
    s_f, l_f = fused[find_idx], labs_arr[find_idx]
    s_v, l_v = fused[val_idx],  labs_arr[val_idx]

    best_t_find, best_ba_find = 0.5, 0.0
    for t in np.linspace(0.2, 0.8, 61):
        preds = (s_f > t).astype(int)
        if preds.sum() in (0, len(preds)): continue
        ba = balanced_accuracy_score(l_f, preds)
        if ba > best_ba_find:
            best_ba_find, best_t_find = ba, t

    midpoint   = float((s_f[l_f==1].mean() + s_f[l_f==0].mean()) / 2) \
                 if (l_f==1).sum() > 0 and (l_f==0).sum() > 0 else 0.5
    median_pos = float(np.median(s_f[l_f==1])) if (l_f==1).sum() > 0 else 0.5

    best_final_t, best_val_ba = 0.5, 0.0
    for t in sorted({best_t_find, midpoint, median_pos}):
        preds = (s_v > t).astype(int)
        if preds.sum() in (0, len(preds)): continue
        ba = balanced_accuracy_score(l_v, preds)
        if ba > best_val_ba:
            best_val_ba, best_final_t = ba, t

    print(f"  {label}: threshold={best_final_t:.4f}  "
          f"(find_BAC={best_ba_find:.3f} | val_BAC={best_val_ba:.3f})")
    return float(best_final_t)

print("✅ Per-label fusion helpers defined.")

In [ ]:
# ----------------------------------------------------------------
# MAIN -- Ensemble B
# ----------------------------------------------------------------

ENS_B_THRESHOLDS = {}

def run_ensemble_b():
    print("=" * 75)
    print("  ENSEMBLE B: WhyXrayCLIP + CheXZero + BioViL-T + BioMedCLIP(FT)")
    print(f"  WhyXrayCLIP    : {WXRC_ENS_N_SHOTS}-shot, alpha={WXRC_ENS_ALPHA}")
    print(f"  CheXZero       : {CZ_ENS_N_SHOTS}-shot, per-label txt_w")
    print(f"  BioViL-T       : text_only (alpha={BV_ENS_ALPHA})")
    print(f"  BioMedCLIP(FT) : per-label (n_shots, alpha, model, context)")
    print(f"  Weights        : hardcoded (AUC+BAC)/2 from best-config benchmarks")
    print(f"  Fusion         : performance-weighted rank fusion")
    print("=" * 75)

    # Step 1: Reuse / build prototypes
    print("\n" + "-" * 75)
    print("  STEP 1: Building prototypes")
    print("-" * 75)

    # WhyXrayCLIP
    if ("WXRC_FINAL_PROTO" in globals()
            and globals().get("WXRC_FINAL_N_SHOTS") == WXRC_ENS_N_SHOTS
            and globals().get("WXRC_FINAL_ALPHA")   == WXRC_ENS_ALPHA):
        wx_protos = WXRC_FINAL_PROTO
        print("  WhyXrayCLIP: reusing WXRC_FINAL_PROTO")
    else:
        wx_protos = {}
        for label in TARGET_LABELS:
            pos_p, neg_p, meta = build_wxrc_prototype(
                label, WXRC_ENS_N_SHOTS, seed=RANDOM_SEED_ENS)
            wx_protos[label] = (pos_p, neg_p)
            print(f"    {label}: pos={meta.get('n_pos','?')} neg={meta.get('n_neg','?')}")

    # CheXZero
    if ("CHEXZERO_COMBINED_PROTO" in globals() and CHEXZERO_COMBINED_PROTO
            and globals().get("best_shot", -1) == CZ_ENS_N_SHOTS):
        cz_anchors = CHEXZERO_COMBINED_PROTO
        print("  CheXZero: reusing CHEXZERO_COMBINED_PROTO")
    else:
        cz_anchors, _ = build_chexzero_prototypes(CZ_ENS_N_SHOTS, CZ_ENS_LABEL_WEIGHTS)

    # BioMedCLIP -- build per-label prototypes
    print("\n  [BioMedCLIP(FT)] building per-label prototypes...")
    for label in TARGET_LABELS:
        cfg     = BMC_ENS_BEST_PER_LABEL[label]
        n_shots = cfg["n_shots"]
        alpha   = cfg["alpha"]
        print(f"    {label}  [n={n_shots}, alpha={alpha:.2f}, "
              f"model={cfg['model']}, ctx={cfg['context_strategy']}]", end="")
        if alpha >= 1.0:
            _bmc_ens_protos[label] = (None, None)
            print("  -> text_only, no prototype")
        else:
            pos_p, neg_p, meta = build_bmc_prototype(label, n_shots, seed=RANDOM_SEED_ENS)
            _bmc_ens_protos[label] = (pos_p, neg_p)
            if pos_p is not None:
                print(f"  ok  pos={meta.get('n_pos','?')} neg={meta.get('n_neg','?')}")
            else:
                print("  FAILED -- text_only fallback")

    # Step 2: Print hardcoded weights
    print("\n" + "-" * 75)
    print("  STEP 2: Ensemble weights  w = (AUC+BAC)/2  [hardcoded in Cell 1]")
    print("-" * 75)
    print(f"\n  {'Label':<35} {'WXRC':>7}  {'CZ':>7}  {'BV':>7}  {'BMC':>7}")
    print("  " + "-" * 65)
    for label in TARGET_LABELS:
        print(f"  {label:<35} {WXRC_ENS_WEIGHTS[label]:>7.4f}  "
              f"{CZ_ENS_WEIGHTS[label]:>7.4f}  "
              f"{BV_ENS_WEIGHTS[label]:>7.4f}  "
              f"{BMC_ENS_WEIGHTS[label]:>7.4f}")

    # Step 3: Calibrate ensemble B thresholds
    print("\n" + "-" * 75)
    print("  STEP 3: Calibrating ensemble B thresholds on train_df")
    print("-" * 75)

    for label in TARGET_LABELS:
        wx_pp, wx_np = wx_protos[label]
        cz_pa, cz_na = cz_anchors[label]
        inc = LABEL_MODEL_INCLUSION.get(
            label, {"wx": True, "cz": True, "bv": True, "bmc": True})
 
        score_fns = {}
        label_w   = {}
        if inc.get("wx", True):
            score_fns["wx"] = (lambda p, lbl, fv, _pp=wx_pp, _np=wx_np:
                               _ens_a_score_wxrc(p, lbl, fv, _pp, _np))
            label_w["wx"]   = WXRC_ENS_WEIGHTS[label]
        if inc.get("cz", True):
            score_fns["cz"] = (lambda p, lbl, fv, _pa=cz_pa, _na=cz_na:
                               _ens_a_score_czero(p, lbl, _pa, _na))
            label_w["cz"]   = CZ_ENS_WEIGHTS[label]
        if inc.get("bv", True):
            score_fns["bv"] = lambda p, lbl, fv: _ens_a_score_biovilt(p, lbl, fv)
            label_w["bv"]   = BV_ENS_WEIGHTS[label]
        if inc.get("bmc", True) and "BMC_ENS_WEIGHTS" in globals():
            score_fns["bmc"] = lambda p, lbl, fv: _ens_b_score_bmc(p, lbl, fv)
            label_w["bmc"]   = BMC_ENS_WEIGHTS[label]
 
        ENS_B_THRESHOLDS[label] = _calibrate_ens_threshold(
            label, score_fns, label_w)
 
    print("\n  Calibrated thresholds:")
    for lbl, t in ENS_B_THRESHOLDS.items():
        print(f"    {lbl:<35}: {t:.4f}")

    # Step 4: Score GT uncertain samples
    print("\n" + "-" * 75)
    print("  STEP 4: Scoring GT uncertain samples")
    print("-" * 75)

    labeler_df = rule_labeler_df.copy()
    gt_df      = ground_truth_df.copy()
    for df_ in [gt_df, labeler_df]:
        for c in ["Path", "path", "study"]:
            if c in df_.columns and "Study" not in df_.columns:
                df_.rename(columns={c: "Study"}, inplace=True); break

    label_metrics_b   = {}
    all_sample_rows_b = []

    for label in TARGET_LABELS:
        print(f"\n  -- {label} --")
        uncertain_idx = labeler_df[labeler_df[label] == -1].index
        gt_samples    = (gt_df.loc[uncertain_idx, ["Study", label]]
                         .rename(columns={label: "gt"})
                         .pipe(lambda d: d[d["gt"].isin([0, 1])])
                         .reset_index(drop=True))
        if len(gt_samples) == 0:
            print("  No GT uncertain samples -- skipping"); continue
        print(f"  n={len(gt_samples)}  pos={int(gt_samples['gt'].sum())}  "
              f"neg={int((gt_samples['gt']==0).sum())}")

        top_feats    = top_features_per_label.get(label, [])
        wx_pp, wx_np = wx_protos[label]
        cz_pa, cz_na = cz_anchors[label]

        wx_scores, cz_scores, bv_scores, bmc_scores = [], [], [], []
        y_true_list, study_list = [], []

        for _, row in tqdm(gt_samples.iterrows(), total=len(gt_samples),
                           desc=f"  Ens-B [{label}]", leave=False):
            try:
                img_path = fix_test_path(row["Study"])
                fv       = get_sample_features(row["Study"], label, top_feats, rule_labeler_df)
                wx_scores.append(_ens_a_score_wxrc(img_path, label, fv, wx_pp, wx_np))
                cz_scores.append(_ens_a_score_czero(img_path, label, cz_pa, cz_na))
                bv_scores.append(_ens_a_score_biovilt(img_path, label, fv))
                bmc_scores.append(_ens_b_score_bmc(img_path, label, fv))
                y_true_list.append(int(row["gt"]))
                study_list.append(row["Study"])
            except Exception as e:
                print(f"    SKIP {row['Study']}: {e}"); continue

        if len(y_true_list) < 5:
            print(f"  Too few valid samples ({len(y_true_list)}) -- skipping"); continue

        y_true  = np.array(y_true_list)
        wx_arr  = np.array(wx_scores)
        cz_arr  = np.array(cz_scores)
        bv_arr  = np.array(bv_scores)
        bmc_arr = np.array(bmc_scores)

        label_w = {
            "wx":  WXRC_ENS_WEIGHTS[label],
            "cz":  CZ_ENS_WEIGHTS[label],
            "bv":  BV_ENS_WEIGHTS[label],
            "bmc": BMC_ENS_WEIGHTS[label],
        }


        inc = LABEL_MODEL_INCLUSION.get(
            label, {"wx": True, "cz": True, "bv": True, "bmc": True})
        scores_dict  = {}
        weights_dict = {}
        if inc.get("wx", True):
            scores_dict["wx"]  = wx_arr
            weights_dict["wx"] = WXRC_ENS_WEIGHTS[label]
        if inc.get("cz", True):
            scores_dict["cz"]  = cz_arr
            weights_dict["cz"] = CZ_ENS_WEIGHTS[label]
        if inc.get("bv", True):
            scores_dict["bv"]  = bv_arr
            weights_dict["bv"] = BV_ENS_WEIGHTS[label]
        if inc.get("bmc", True) and "BMC_ENS_WEIGHTS" in globals():
            scores_dict["bmc"]  = bmc_arr
            weights_dict["bmc"] = BMC_ENS_WEIGHTS[label]
 
        fused_scores = _rank_fuse(scores_dict, weights_dict)
        label_w      = weights_dict
        
        threshold    = ENS_B_THRESHOLDS[label]

        wx_m  = _ind_metrics(wx_arr,  y_true, "wx",  label)
        cz_m  = _ind_metrics(cz_arr,  y_true, "cz",  label)
        bv_m  = _ind_metrics(bv_arr,  y_true, "bv",  label)
        bmc_m = _ind_metrics_bmc(bmc_arr, y_true, label)
        m     = _ens_metrics(fused_scores, y_true, threshold)
        if not m: continue

        ind_aucs = []
        if "wx" in label_w:  ind_aucs.append(wx_m["auc"])
        if "cz" in label_w:  ind_aucs.append(cz_m["auc"])
        if "bv" in label_w:  ind_aucs.append(bv_m["auc"])
        if "bmc" in label_w: ind_aucs.append(bmc_m["auc"])
        
        best_ind_auc = max(ind_aucs)
        delta_auc    = round(m["auc"] - best_ind_auc, 3)
        marker       = " [IMPROVES]" if delta_auc > 0.01 else (" [~]" if delta_auc > -0.01 else " [WORSE]")

        label_metrics_b[label] = {
            **m, "label": label,
            "wx_auc":  wx_m["auc"],  "cz_auc":  cz_m["auc"],
            "bv_auc":  bv_m["auc"],  "bmc_auc": bmc_m["auc"],
            "wx_auprc":wx_m["auprc"],"cz_auprc":cz_m["auprc"],
            "bv_auprc":bv_m["auprc"],"bmc_auprc":bmc_m["auprc"],
            "wx_bac":  wx_m["bac"],  "cz_bac":  cz_m["bac"],
            "bv_bac":  bv_m["bac"],  "bmc_bac": bmc_m["bac"],
            "wx_f1":   wx_m["f1"],   "cz_f1":   cz_m["f1"],
            "bv_f1":   bv_m["f1"],   "bmc_f1":  bmc_m["f1"],
            "wx_sens": wx_m["sens"], "cz_sens": cz_m["sens"],
            "bv_sens": bv_m["sens"], "bmc_sens":bmc_m["sens"],
            "wx_spec": wx_m["spec"], "cz_spec": cz_m["spec"],
            "bv_spec": bv_m["spec"], "bmc_spec":bmc_m["spec"],
            "ens_score": round((m["auc"]+m["bal_acc"])/2, 3),
            "delta_vs_best_ind_auc": delta_auc,
        }

        print(f"\n  {'Model':<20} {'AUC':>6} {'AUPRC':>6} {'BAC':>6} {'F1':>6} "
              f"{'Sens':>6} {'Spec':>6}  {'(wt)':>7}")
        print(f"  {'-'*75}")
        if "wx" in label_w:
            print(f"{'WhyXrayCLIP':<20} {wx_m['auc']:>6.3f} {wx_m['auprc']:>6.3f} "
                  f"{wx_m['bac']:>6.3f} {wx_m['f1']:>6.3f} {wx_m['sens']:>6.3f} "
                  f"{wx_m['spec']:>6.3f}  ({label_w['wx']:.4f})")
        
        if "cz" in label_w:
            print(f"{'CheXZero':<20} {cz_m['auc']:>6.3f} {cz_m['auprc']:>6.3f} "
                  f"{cz_m['bac']:>6.3f} {cz_m['f1']:>6.3f} {cz_m['sens']:>6.3f} "
                  f"{cz_m['spec']:>6.3f}  ({label_w['cz']:.4f})")
        
        if "bv" in label_w:
            print(f"{'BioViL-T':<20} {bv_m['auc']:>6.3f} {bv_m['auprc']:>6.3f} "
                  f"{bv_m['bac']:>6.3f} {bv_m['f1']:>6.3f} {bv_m['sens']:>6.3f} "
                  f"{bv_m['spec']:>6.3f}  ({label_w['bv']:.4f})")
        
        if "bmc" in label_w:
            print(f"{'BioMedCLIP(FT)':<20} {bmc_m['auc']:>6.3f} {bmc_m['auprc']:>6.3f} "
                  f"{bmc_m['bac']:>6.3f} {bmc_m['f1']:>6.3f} {bmc_m['sens']:>6.3f} "
                  f"{bmc_m['spec']:>6.3f}  ({label_w['bmc']:.4f})")
        print(f"  {'-'*75}")
        print(f"  {'ENSEMBLE B':<20} {m['auc']:>6.3f} {m['auprc']:>6.3f} "
              f"{m['bal_acc']:>6.3f} {m['f1']:>6.3f} {m['sens']:>6.3f} "
              f"{m['spec']:>6.3f}  dAUC={delta_auc:+.3f}{marker}")
        print(f"  TP={m['tp']}  TN={m['tn']}  FP={m['fp']}  FN={m['fn']}  "
              f"threshold={threshold:.4f}")

        for i in range(len(y_true_list)):
            all_sample_rows_b.append({
                "label": label, "study": study_list[i],
                "y_true": y_true_list[i],
                "wx_score":    round(float(wx_scores[i]),  6),
                "cz_score":    round(float(cz_scores[i]),  6),
                "bv_score":    round(float(bv_scores[i]),  6),
                "bmc_score":   round(float(bmc_scores[i]), 6),
                "fused_score": round(float(fused_scores[i]), 6),
                "y_pred":      int(fused_scores[i] > threshold),
                "threshold":   threshold,
            })

    if not label_metrics_b:
        print("\n  No results collected.")
        return pd.DataFrame(), pd.DataFrame()

    rows = list(label_metrics_b.values())
    print("\n\n" + "=" * 115)
    print("  ENSEMBLE B FINAL SUMMARY -- WhyXrayCLIP + CheXZero + BioViL-T + BioMedCLIP(FT)")
    print("  Weights: (AUC+BAC)/2 hardcoded from best-config benchmarks")
    print("=" * 115)
    print(f"  {'Label':<30} {'Model':<15} {'AUC':>6} {'AUPRC':>6} {'BAC':>6} "
          f"{'F1':>6} {'Sens':>6} {'Spec':>6}  {'n':>4}")
    print("  " + "-" * 100)

    aucs, auprcs, bacs, f1s, senss, specs = [], [], [], [], [], []
    wx_auc_all, cz_auc_all, bv_auc_all, bmc_auc_all = [], [], [], []
    for r in rows:
        dlt = r["delta_vs_best_ind_auc"]
        mrk = " [+]" if dlt > 0.01 else (" [~]" if dlt > -0.01 else " [-]")
        if "wx_auc" in r:
            print(f"  {r['label']:<30} {'WhyXrayCLIP':<15} "
                  f"{r['wx_auc']:>6.3f} {r['wx_auprc']:>6.3f} {r['wx_bac']:>6.3f} "
                  f"{r['wx_f1']:>6.3f} {r['wx_sens']:>6.3f} {r['wx_spec']:>6.3f}  {r['n']:>4}")
        if "cz_auc" in r:
            print(f"  {'':<30} {'CheXZero':<15} "
                  f"{r['cz_auc']:>6.3f} {r['cz_auprc']:>6.3f} {r['cz_bac']:>6.3f} "
                  f"{r['cz_f1']:>6.3f} {r['cz_sens']:>6.3f} {r['cz_spec']:>6.3f}")
        if "bv_auc" in r:
            print(f"  {'':<30} {'BioViL-T':<15} "
                  f"{r['bv_auc']:>6.3f} {r['bv_auprc']:>6.3f} {r['bv_bac']:>6.3f} "
                  f"{r['bv_f1']:>6.3f} {r['bv_sens']:>6.3f} {r['bv_spec']:>6.3f}")
        if "bmc_auc" in r:
            print(f"  {'':<30} {'BioMedCLIP(FT)':<15} "
                  f"{r['bmc_auc']:>6.3f} {r['bmc_auprc']:>6.3f} {r['bmc_bac']:>6.3f} "
                  f"{r['bmc_f1']:>6.3f} {r['bmc_sens']:>6.3f} {r['bmc_spec']:>6.3f}")
        print(f"  {'':<30} {'ENSEMBLE B':<15} "
              f"{r['auc']:>6.3f} {r['auprc']:>6.3f} {r['bal_acc']:>6.3f} "
              f"{r['f1']:>6.3f} {r['sens']:>6.3f} {r['spec']:>6.3f}"
              f"  dAUC={dlt:+.3f}{mrk}")
        print(f"  {'':<30}  TP={r['tp']}  TN={r['tn']}  FP={r['fp']}  FN={r['fn']}")
        print("  " + "-" * 100)
        aucs.append(r["auc"]);     auprcs.append(r["auprc"])
        bacs.append(r["bal_acc"]); f1s.append(r["f1"])
        senss.append(r["sens"]);   specs.append(r["spec"])
        if "wx_auc" in r:
            wx_auc_all.append(r["wx_auc"]);  cz_auc_all.append(r["cz_auc"])
        if "bv_auc" in r:
            bv_auc_all.append(r["bv_auc"]);  bmc_auc_all.append(r["bmc_auc"])

    ens_mauc       = np.mean(aucs)
    means = []
    if wx_auc_all:  means.append(np.mean(wx_auc_all))
    if cz_auc_all:  means.append(np.mean(cz_auc_all))
    if bv_auc_all:  means.append(np.mean(bv_auc_all))
    if bmc_auc_all: means.append(np.mean(bmc_auc_all))
    
    best_ind_mauc = max(means)
    delta_auc_mean = round(ens_mauc - best_ind_mauc, 3)
    total_n        = sum(r["n"] for r in rows)
    print(f"  {'MACRO MEAN -- Ensemble B':<46} "
          f"{ens_mauc:>6.3f} {np.mean(auprcs):>6.3f} {np.mean(bacs):>6.3f} "
          f"{np.mean(f1s):>6.3f} {np.mean(senss):>6.3f} {np.mean(specs):>6.3f}  n={total_n}")
    print(f"  dAUC ensemble B vs best individual (macro): {delta_auc_mean:+.3f}")

    tp_t = sum(r["tp"] for r in rows); tn_t = sum(r["tn"] for r in rows)
    fp_t = sum(r["fp"] for r in rows); fn_t = sum(r["fn"] for r in rows)
    print(f"\n  Aggregate confusion: TP={tp_t}  TN={tn_t}  FP={fp_t}  FN={fn_t}")
    verdict = ("IMPROVES" if delta_auc_mean > 0.01
               else ("COMPARABLE" if delta_auc_mean > -0.01 else "UNDERPERFORMS"))
    print(f"\n  Verdict: Ensemble B {verdict} vs best individual (dAUC = {delta_auc_mean:+.3f})")

    results_b_df  = pd.DataFrame(rows)
    resolved_b_df = pd.DataFrame(all_sample_rows_b)
    results_b_df.to_csv("ensemble_b_results.csv",  index=False)
    resolved_b_df.to_csv("ensemble_b_resolved.csv", index=False)
    print(f"\n  Saved: ensemble_b_results.csv  |  ensemble_b_resolved.csv")
    globals()["ENS_B_THRESHOLDS"] = ENS_B_THRESHOLDS
    return results_b_df, resolved_b_df


ens_b_results_df, ens_b_resolved_df = run_ensemble_b()

In [ ]:
# ================================================================
# CELL 4 of 4 -- ENSEMBLE A vs ENSEMBLE B COMPARISON
#
# Compares the two ensemble strategies head-to-head:
#   Ensemble A: WhyXrayCLIP + CheXZero + BioViL-T         (3 models)
#   Ensemble B: WhyXrayCLIP + CheXZero + BioViL-T + BMC   (4 models)
#
# Loads from memory if available, otherwise from saved CSVs.
# ================================================================

import numpy as np, pandas as pd

if "ens_a_results_df" not in globals() or len(ens_a_results_df) == 0:
    print("ens_a_results_df not in memory -- loading from CSV...")
    ens_a_results_df = pd.read_csv("ensemble_a_results.csv")

if "ens_b_results_df" not in globals() or len(ens_b_results_df) == 0:
    print("ens_b_results_df not in memory -- loading from CSV...")
    ens_b_results_df = pd.read_csv("ensemble_b_results.csv")

METRICS       = ["auc", "auprc", "bal_acc", "f1", "sens", "spec"]
METRIC_LABELS = {"auc": "AUC", "auprc": "AUPRC", "bal_acc": "BAC",
                 "f1": "F1", "sens": "Sens", "spec": "Spec"}

print("\n" + "=" * 110)
print("  ENSEMBLE A vs ENSEMBLE B -- HEAD-TO-HEAD COMPARISON")
print("  Ensemble A : WhyXrayCLIP + CheXZero + BioViL-T")
print("  Ensemble B : WhyXrayCLIP + CheXZero + BioViL-T + BioMedCLIP(FT)")
print("  Weights    : (AUC+BAC)/2 hardcoded from each model best-config benchmark")
print("=" * 110)
print(f"\n  {'Label':<35} {'Metric':<8}  {'Ens A':>7}  {'Ens B':>7}  {'D(B-A)':>8}  {'Winner':>10}")
print("  " + "-" * 85)

summary_rows = []
for label in TARGET_LABELS:
    a_row = ens_a_results_df[ens_a_results_df["label"] == label]
    b_row = ens_b_results_df[ens_b_results_df["label"] == label]
    if len(a_row) == 0 or len(b_row) == 0:
        print(f"  !! {label}: missing from one ensemble -- skipping")
        continue
    a, b  = a_row.iloc[0], b_row.iloc[0]
    first = True
    for metric in METRICS:
        a_val  = float(a[metric]) if metric in a and pd.notna(a[metric]) else float("nan")
        b_val  = float(b[metric]) if metric in b and pd.notna(b[metric]) else float("nan")
        delta  = b_val - a_val
        winner = "Ens B" if delta > 0.005 else ("Ens A" if delta < -0.005 else "Tie")
        lbl_col = label if first else ""
        print(f"  {lbl_col:<35} {METRIC_LABELS[metric]:<8}  {a_val:>7.3f}  {b_val:>7.3f}  "
              f"{delta:>+8.3f}  {winner:>10}")
        first = False
        summary_rows.append({"label": label, "metric": metric,
                              "ens_a": round(a_val, 3), "ens_b": round(b_val, 3),
                              "delta": round(delta, 3)})
    print("  " + "-" * 85)

# Macro-average
print(f"\n  {'MACRO AVERAGE':<35} {'Metric':<8}  {'Ens A':>7}  {'Ens B':>7}  {'D(B-A)':>8}  {'Winner':>10}")
print("  " + "-" * 85)
for metric in METRICS:
    a_vals = [r["ens_a"] for r in summary_rows if r["metric"] == metric and not np.isnan(r["ens_a"])]
    b_vals = [r["ens_b"] for r in summary_rows if r["metric"] == metric and not np.isnan(r["ens_b"])]
    if not a_vals: continue
    a_mean, b_mean = float(np.mean(a_vals)), float(np.mean(b_vals))
    delta  = b_mean - a_mean
    winner = "Ens B" if delta > 0.005 else ("Ens A" if delta < -0.005 else "Tie")
    print(f"  {'':<35} {METRIC_LABELS[metric]:<8}  {a_mean:>7.3f}  {b_mean:>7.3f}  "
          f"{delta:>+8.3f}  {winner:>10}")
print("  " + "-" * 85)

# Verdict
a_auc_mean = float(np.mean([r["ens_a"] for r in summary_rows if r["metric"] == "auc"]))
b_auc_mean = float(np.mean([r["ens_b"] for r in summary_rows if r["metric"] == "auc"]))
a_bac_mean = float(np.mean([r["ens_a"] for r in summary_rows if r["metric"] == "bal_acc"]))
b_bac_mean = float(np.mean([r["ens_b"] for r in summary_rows if r["metric"] == "bal_acc"]))
delta_auc  = b_auc_mean - a_auc_mean
delta_bac  = b_bac_mean - a_bac_mean

print(f"\n  Mean AUC -- Ens A: {a_auc_mean:.3f}  |  Ens B: {b_auc_mean:.3f}  D={delta_auc:+.3f}")
print(f"  Mean BAC -- Ens A: {a_bac_mean:.3f}  |  Ens B: {b_bac_mean:.3f}  D={delta_bac:+.3f}")
print()
if delta_auc > 0.01:
    print("  VERDICT: Ensemble B is BETTER -- BioMedCLIP(FT) improves the fusion.")
    print("  -> Use Ensemble B resolved labels for Phase 3.")
elif delta_auc > -0.01:
    print("  VERDICT: Ensemble A and B are COMPARABLE in AUC.")
    if delta_bac > 0.0:
        print("  Ensemble B has higher BAC -> prefer B for balanced class performance.")
        print("  -> Use Ensemble B resolved labels for Phase 3.")
    else:
        print("  Ensemble A has equivalent or better BAC -> prefer A (simpler, faster).")
        print("  -> Use Ensemble A resolved labels for Phase 3.")
else:
    print("  VERDICT: Ensemble A is BETTER -- BioMedCLIP(FT) hurts the fusion.")
    print("  -> Use Ensemble A resolved labels for Phase 3.")

comparison_df = pd.DataFrame(summary_rows)
comparison_df.to_csv("ensemble_ab_comparison.csv", index=False)
print(f"\n  Saved: ensemble_ab_comparison.csv")


# ENSEMBLE THRESHOLD CALIBRATION ON TRAIN DATASET

In [ ]:
# ================================================================
# ENSEMBLE THRESHOLD CALIBRATION — TRAIN_DF ONLY
#
# Exactly the same approach used by the individual model calibration
# cells (WhyXrayCLIP, CheXZero, BioViL-T):
#
#   1. Sample a balanced set of certain (0/1) rows from train_df
#      — n_each per class (default 300 per class = 600 total)
#   2. Score every sample with the full active ensemble
#      (per-label model inclusion from LABEL_MODEL_INCLUSION)
#   3. Rank-fuse the scores using the same weights as relabeling
#   4. Half-split the calibration set (stratified):
#        Find-half  → scan 61 threshold candidates, pick best BAC
#        Val-half   → validate find-half winner + midpoint + median-pos
#   5. Return the threshold with best BAC on the held-out val-half
#
# This is identical to _calibrate_ens_threshold() already used in
# run_ensemble_a() / run_ensemble_b(), but run here explicitly so
# ENSEMBLE_THRESHOLDS is set before relabeling without needing to
# run the full benchmark cells.
#
# Result stored in ENSEMBLE_THRESHOLDS, picked up by Cell 159/160.
# ================================================================

import os, torch, torch.nn.functional as F
import numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from scipy.stats import rankdata
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit

USE_ENSEMBLE_B  = True   # True = include BioMedCLIP (Ensemble B)
                          # False = 3-model Ensemble A only
CAL_N_EACH      = 300    # samples per class from train_df (300 pos + 300 neg = 600 total)
                          # same default as _calibrate_ens_threshold in Cell 148
RANDOM_SEED_CAL = 42

print("=" * 70)
print("  ENSEMBLE THRESHOLD CALIBRATION — TRAIN_DF ONLY")
print(f"  Ensemble  : {'B (WXRC+CZ+BV+BMC)' if USE_ENSEMBLE_B else 'A (WXRC+CZ+BV)'}")
print(f"  Cal set   : {CAL_N_EACH} pos + {CAL_N_EACH} neg per label (balanced)")
print(f"  Method    : half-split BAC (find → validate)")
print(f"  Seed      : {RANDOM_SEED_CAL}")
print("=" * 70)


# ----------------------------------------------------------------
# STEP 0: Build / reuse prototypes
# (same reuse logic as the GT calibration cell — nothing changes here)
# ----------------------------------------------------------------
print("\n  Building / reusing prototypes...")

# WhyXrayCLIP
if (  "WXRC_FINAL_PROTO" in globals()
    and globals().get("WXRC_FINAL_N_SHOTS") == WXRC_ENS_N_SHOTS
    and globals().get("WXRC_FINAL_ALPHA")   == WXRC_ENS_ALPHA):
    _cal_wx = WXRC_FINAL_PROTO
    print("  WhyXrayCLIP : reusing WXRC_FINAL_PROTO")
else:
    _cal_wx = {}
    for _lbl in TARGET_LABELS:
        _pp, _np, _meta = build_wxrc_prototype(
            _lbl, WXRC_ENS_N_SHOTS, seed=RANDOM_SEED_CAL)
        _cal_wx[_lbl] = (_pp, _np)
        print(f"  WhyXrayCLIP [{_lbl}]: "
              f"pos={_meta.get('n_pos','?')} neg={_meta.get('n_neg','?')}")

# CheXZero
if (  "CHEXZERO_COMBINED_PROTO" in globals()
    and CHEXZERO_COMBINED_PROTO
    and globals().get("best_shot", -1) == CZ_ENS_N_SHOTS):
    _cal_cz = CHEXZERO_COMBINED_PROTO
    print("  CheXZero    : reusing CHEXZERO_COMBINED_PROTO")
else:
    _saved_ctx = dict(CZERO_CONTEXT_STRATEGY) \
                 if "CZERO_CONTEXT_STRATEGY" in globals() else {}
    if "CZERO_CONTEXT_STRATEGY" in globals():
        CZERO_CONTEXT_STRATEGY.update(CZERO_ENS_CONTEXT_STRATEGY)
    _cal_cz, _ = build_chexzero_prototypes(
        CZ_ENS_N_SHOTS, CZ_ENS_LABEL_WEIGHTS)
    if "CZERO_CONTEXT_STRATEGY" in globals():
        CZERO_CONTEXT_STRATEGY.update(_saved_ctx)
    print("  CheXZero    : built fresh")

# BioViL-T — text_only, no prototypes
print("  BioViL-T    : text_only (alpha=1.0) — no prototypes needed")

# BioMedCLIP
_cal_bmc = {}
if USE_ENSEMBLE_B and "BMC_ENS_BEST_PER_LABEL" in globals():
    if "_bmc_ens_protos" in globals() and _bmc_ens_protos:
        _cal_bmc = _bmc_ens_protos
        print("  BioMedCLIP  : reusing _bmc_ens_protos")
    else:
        for _lbl in TARGET_LABELS:
            _cfg = BMC_ENS_BEST_PER_LABEL[_lbl]
            if _cfg["alpha"] >= 1.0:
                _cal_bmc[_lbl] = (None, None)
            else:
                _pp, _np2, _ = build_bmc_prototype(
                    _lbl, _cfg["n_shots"], seed=RANDOM_SEED_CAL)
                _cal_bmc[_lbl] = (_pp, _np2)
        print("  BioMedCLIP  : built fresh")

print("  ✅ Prototypes ready\n")


# ----------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------

def _cal_score_one(img_path, label, fv):
    """Score one train_df image with all active ensemble models."""
    wx_pp, wx_np = _cal_wx[label]
    cz_pa, cz_na = _cal_cz[label]
    inc    = LABEL_MODEL_INCLUSION.get(
        label, {"wx": True, "cz": True, "bv": True, "bmc": True})
    scores = {}
    if inc.get("wx",  True):
        scores["wx"]  = _ens_a_score_wxrc(img_path, label, fv, wx_pp, wx_np)
    if inc.get("cz",  True):
        scores["cz"]  = _ens_a_score_czero(img_path, label, cz_pa, cz_na)
    if inc.get("bv",  True):
        scores["bv"]  = _ens_a_score_biovilt(img_path, label, fv)
    if inc.get("bmc", True) and USE_ENSEMBLE_B and label in _cal_bmc:
        scores["bmc"] = _ens_b_score_bmc(img_path, label, fv)
    return scores


def _cal_weight_dict(label):
    """Active model weights for rank fusion."""
    inc = LABEL_MODEL_INCLUSION.get(
        label, {"wx": True, "cz": True, "bv": True, "bmc": True})
    w = {}
    if inc.get("wx",  True): w["wx"]  = WXRC_ENS_WEIGHTS[label]
    if inc.get("cz",  True): w["cz"]  = CZ_ENS_WEIGHTS[label]
    if inc.get("bv",  True): w["bv"]  = BV_ENS_WEIGHTS[label]
    if inc.get("bmc", True) and USE_ENSEMBLE_B \
            and "BMC_ENS_WEIGHTS" in globals():
        w["bmc"] = BMC_ENS_WEIGHTS[label]
    return w


def _cal_fuse_batch(scores_lists, weight_dict):
    """Rank fusion over N samples. scores_lists: {model: list[N]}."""
    n     = len(next(iter(scores_lists.values())))
    fused = np.zeros(n)
    total = sum(weight_dict.get(m, 1.0) for m in scores_lists)
    for model, vals in scores_lists.items():
        w          = weight_dict.get(model, 1.0)
        norm_ranks = rankdata(np.array(vals), method="average") / n
        fused     += w * norm_ranks
    return fused / total


# ----------------------------------------------------------------
# MAIN LOOP — one label at a time
# ----------------------------------------------------------------

ENSEMBLE_THRESHOLDS = {}

for label in TARGET_LABELS:
    print(f"\n{'━'*70}")
    print(f"  Label: {label}")
    print(f"{'━'*70}")

    weight_dict = _cal_weight_dict(label)
    active_models = list(weight_dict.keys())
    print(f"  Active models: {', '.join(active_models)}")

    # ── Sample balanced certain rows from train_df ────────────────
    df_certain = train_df[train_df[label].isin([0, 1])].copy()
    n_pos_avail = int((df_certain[label] == 1).sum())
    n_neg_avail = int((df_certain[label] == 0).sum())
    n_each = min(CAL_N_EACH, n_pos_avail, n_neg_avail)

    if n_each < 20:
        print(f"  ⚠️  Too few certain samples "
              f"(pos={n_pos_avail}, neg={n_neg_avail}) — using 0.5")
        ENSEMBLE_THRESHOLDS[label] = 0.5
        continue

    cal_df = pd.concat([
        df_certain[df_certain[label] == 1].sample(
            n_each, random_state=RANDOM_SEED_CAL),
        df_certain[df_certain[label] == 0].sample(
            n_each, random_state=RANDOM_SEED_CAL),
    ]).sample(frac=1, random_state=RANDOM_SEED_CAL).reset_index(drop=True)

    print(f"  Calibration set : {len(cal_df)} samples "
          f"(pos={n_each}, neg={n_each})")

    # ── Score all calibration samples ────────────────────────────
    top_feats    = top_features_per_label.get(label, [])
    model_scores = {k: [] for k in active_models}
    labs         = []

    for _, row in tqdm(cal_df.iterrows(), total=len(cal_df),
                       desc=f"  Scoring [{label}]", leave=False):
        try:
            clean    = (str(row["Path"])
                        .replace("CheXpert-v1.0-small/", "")
                        .replace("CheXpert-v1.0/", ""))
            img_path = os.path.join(BASE_PATH, clean)
            fv       = get_sample_features(
                row["Path"], label, top_feats, train_df)
            sc = _cal_score_one(img_path, label, fv)
            for k in active_models:
                model_scores[k].append(sc.get(k, 0.0))
            labs.append(int(row[label]))
        except Exception:
            continue

    n_scored = len(labs)
    print(f"  Scored          : {n_scored} / {len(cal_df)}")

    if n_scored < 20:
        print(f"  ⚠️  Too few scored — using 0.5")
        ENSEMBLE_THRESHOLDS[label] = 0.5
        continue

    # ── Rank fusion over the full calibration batch ───────────────
    fused    = _cal_fuse_batch(
        {k: model_scores[k] for k in active_models},
        weight_dict)
    labs_arr = np.array(labs)

    # ── Half-split: find-half → validate ─────────────────────────
    sss = StratifiedShuffleSplit(
        n_splits=1, test_size=0.5, random_state=RANDOM_SEED_CAL)
    find_idx, val_idx = next(sss.split(fused, labs_arr))

    s_f, l_f = fused[find_idx], labs_arr[find_idx]
    s_v, l_v = fused[val_idx],  labs_arr[val_idx]

    # Step 1: scan 61 candidates on find-half
    best_t_find, best_ba_find = 0.5, 0.0
    for t in np.linspace(0.2, 0.8, 61):
        preds = (s_f > t).astype(int)
        if preds.sum() in (0, len(preds)):
            continue
        ba = balanced_accuracy_score(l_f, preds)
        if ba > best_ba_find:
            best_ba_find, best_t_find = ba, t

    # Step 2: also check midpoint and median-pos as extra candidates
    midpoint   = float(
        (s_f[l_f == 1].mean() + s_f[l_f == 0].mean()) / 2
    ) if (l_f == 1).sum() > 0 and (l_f == 0).sum() > 0 else best_t_find
    median_pos = float(np.median(s_f[l_f == 1])) \
                 if (l_f == 1).sum() > 0 else best_t_find

    # Step 3: validate all candidates on held-out val-half
    best_final_t, best_val_ba = best_t_find, 0.0
    for t in sorted({best_t_find, midpoint, median_pos}):
        preds = (s_v > t).astype(int)
        if preds.sum() in (0, len(preds)):
            continue
        ba = balanced_accuracy_score(l_v, preds)
        if ba > best_val_ba:
            best_val_ba, best_final_t = ba, t

    ENSEMBLE_THRESHOLDS[label] = float(best_final_t)

    # ── Report ────────────────────────────────────────────────────
    # AUC on full calibration set (threshold-free sanity check)
    auc = roc_auc_score(labs_arr, fused) \
          if len(np.unique(labs_arr)) > 1 else 0.5

    print(f"  AUC (cal set)   : {auc:.4f}  (threshold-free)")
    print(f"  Find-half BAC   : {best_ba_find:.4f}  @ t={best_t_find:.4f}")
    print(f"  Val-half BAC    : {best_val_ba:.4f}  @ t={best_final_t:.4f}")
    print(f"  ✅ Threshold     : {best_final_t:.4f}")


# ----------------------------------------------------------------
# SUMMARY
# ----------------------------------------------------------------
print("\n" + "=" * 70)
print("  FINAL ENSEMBLE_THRESHOLDS  (train_df calibration)")
print("=" * 70)
print(f"  {'Label':<35} {'Threshold':>10}  {'Active Models'}")
print("  " + "-" * 65)

for _lbl in TARGET_LABELS:
    _t   = ENSEMBLE_THRESHOLDS.get(_lbl, 0.5)
    _inc = LABEL_MODEL_INCLUSION.get(
        _lbl, {"wx": True, "cz": True, "bv": True, "bmc": True})
    _active = [k for k, v in _inc.items() if v]
    print(f"  {_lbl:<35} {_t:>10.4f}  {', '.join(_active)}")

print("\n✅ ENSEMBLE_THRESHOLDS set — relabeling cells will use these.")
print("   Calibrated on train_df certain samples (balanced 50/50).")
print("   Half-split BAC method — same as individual model calibrations.")

# Relabeling the Uncertain data points

In [ ]:
# ================================================================
# Analyze distribution of uncertain (-1) labels in train dataset
# for our 4 target pathologies
# ================================================================

print("="*80)
print("  TRAIN DATASET UNCERTAIN LABEL ANALYSIS")
print("="*80)

train_uncertain_stats = []

for label in TARGET_LABELS:
    if label not in train_df.columns:
        print(f"⚠️  {label} not in train_df columns")
        continue
    
    total_samples = len(train_df)
    uncertain_samples = (train_df[label] == -1).sum()
    positive_samples = (train_df[label] == 1).sum()
    negative_samples = (train_df[label] == 0).sum()
    nan_samples = train_df[label].isna().sum()
    
    uncertain_pct = (uncertain_samples / total_samples) * 100
    
    train_uncertain_stats.append({
        'label': label,
        'total': total_samples,
        'uncertain': uncertain_samples,
        'uncertain_pct': uncertain_pct,
        'positive': positive_samples,
        'negative': negative_samples,
        'nan': nan_samples
    })
    
    print(f"\n{label}:")
    print(f"  Total samples:     {total_samples:,}")
    print(f"  Uncertain (-1):    {uncertain_samples:,} ({uncertain_pct:.2f}%)")
    print(f"  Positive (1):      {positive_samples:,}")
    print(f"  Negative (0):      {negative_samples:,}")
    print(f"  NaN:               {nan_samples:,}")

stats_df = pd.DataFrame(train_uncertain_stats)
stats_df.to_csv('train_uncertain_stats.csv', index=False)
print("\n✅ Saved: train_uncertain_stats.csv")

print("\n" + "="*80)
print("  SUMMARY")
print("="*80)
print(stats_df[['label', 'uncertain', 'uncertain_pct']].to_string(index=False))

In [ ]:
# ================================================================
# ENSEMBLE THRESHOLD CONFIG (Hardcoded from GT-set calibration)
#
# These thresholds were calibrated using the combined GT + train_df
# weighted approach (GT samples weighted 3x).
# Section heading: "ENSEMBLE THRESHOLD CALIBRATION ON GROUND TRUTH DATASET"
#
# Run this cell at the start of any fresh session to avoid re-running
# the full threshold calibration pipeline.
# ================================================================

ENSEMBLE_THRESHOLDS = {
    "Edema":                      0.4950,
    "Atelectasis":                0.4200,
    "Pleural Effusion":           0.4959,
    "Enlarged Cardiomediastinum": 0.4944,
    "Consolidation":              0.4900,
}

print("=" * 65)
print("  ENSEMBLE_THRESHOLDS (hardcoded from GT calibration)")
print("=" * 65)
for lbl, t in ENSEMBLE_THRESHOLDS.items():
    print(f"  {lbl:<35}: {t:.4f}")
print()
print("  Ready — run Ensemble A or Ensemble B relabeling cell next.")


In [ ]:
# ================================================================
# TRAIN SET RELABELING — ENSEMBLE A  (per-label model selection)
#
# Per-label model inclusion from LABEL_MODEL_INCLUSION (Cell 147).
# Models disabled for a label are excluded from rank fusion for
# that label. This applies to both scoring and the fusion weights.
#
# Per-label CSV saved after every label completes (crash-safe).
# Final full CSV written at the end.
#
# OUTPUTS (written incrementally):
#   train_relabeled_ensA_{label}.csv     — state after each label
#   train_relabeling_log_ensA_{label}.csv — per-row log each label
#   train_relabeled_ensembleA.csv        — final full CSV
#   train_relabeling_log_ensembleA.csv   — combined log
#   train_relabeling_stats_ensembleA.csv — per-label statistics
# ================================================================
 
import os, gc, torch, torch.nn.functional as F
import numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from scipy.stats import rankdata
 
RELABEL_SEED     = 0
CHECKPOINT_EVERY = 500
 
if "ENSEMBLE_THRESHOLDS" not in globals() or not ENSEMBLE_THRESHOLDS:
    raise RuntimeError("Run the ENSEMBLE THRESHOLD CONFIG cell first.")
RELABEL_THRESHOLDS = ENSEMBLE_THRESHOLDS
 
print("=" * 70)
print("  TRAIN SET RELABELING — ENSEMBLE A")
print(f"  Per-label model inclusion active.")
print("=" * 70)
print("\\nThresholds:")
for lbl, t in RELABEL_THRESHOLDS.items():
    print(f"  {lbl:<35}: {t:.4f}")
 
def get_train_img_path(path_str):
    clean = (str(path_str)
             .replace("CheXpert-v1.0-small/", "")
             .replace("CheXpert-v1.0/", ""))
    return os.path.join(BASE_PATH, clean)
 
# ── STEP 1: Build prototypes ──────────────────────────────────────
print("\\n" + "="*70)
print(f"  STEP 1: Building Ensemble A prototypes (seed={RELABEL_SEED})")
print("="*70)
 
if ("WXRC_FINAL_PROTO" in globals()
        and globals().get("WXRC_FINAL_N_SHOTS") == WXRC_ENS_N_SHOTS
        and globals().get("WXRC_FINAL_ALPHA")   == WXRC_ENS_ALPHA):
    relabel_wx_protos = WXRC_FINAL_PROTO
    print("  WhyXrayCLIP: reusing WXRC_FINAL_PROTO")
else:
    relabel_wx_protos = {}
    for label in TARGET_LABELS:
        # Only build if this model is active for at least one label
        if not LABEL_MODEL_INCLUSION.get(label, {}).get("wx", True):
            relabel_wx_protos[label] = (None, None)
            print(f"  WhyXrayCLIP [{label}]: skipped (disabled)")
            continue
        pp, np_, meta = build_wxrc_prototype(
            label, WXRC_ENS_N_SHOTS, seed=RELABEL_SEED)
        relabel_wx_protos[label] = (pp, np_)
        status = (f"pos={meta.get('n_pos','?')} neg={meta.get('n_neg','?')}"
                  if pp is not None else "FAILED")
        print(f"  WhyXrayCLIP [{label}]: {status}")
 
if ("CHEXZERO_COMBINED_PROTO" in globals()
        and CHEXZERO_COMBINED_PROTO
        and globals().get("best_shot", -1) == CZ_ENS_N_SHOTS):
    relabel_cz_anchors = CHEXZERO_COMBINED_PROTO
    print("  CheXZero: reusing CHEXZERO_COMBINED_PROTO")
else:
    _saved_ctx = dict(CZERO_CONTEXT_STRATEGY) if "CZERO_CONTEXT_STRATEGY" in globals() else {}
    if "CZERO_CONTEXT_STRATEGY" in globals():
        CZERO_CONTEXT_STRATEGY.update(CZERO_ENS_CONTEXT_STRATEGY)
    relabel_cz_anchors, _ = build_chexzero_prototypes(
        CZ_ENS_N_SHOTS, CZ_ENS_LABEL_WEIGHTS)
    if "CZERO_CONTEXT_STRATEGY" in globals():
        CZERO_CONTEXT_STRATEGY.update(_saved_ctx)
    print("  CheXZero: built fresh")
 
print("  BioViL-T: text_only (alpha=1.0) — no prototypes needed")
 
# ── STEP 2: Relabel ───────────────────────────────────────────────
print("\\n" + "="*70)
print("  STEP 2: Relabeling uncertain rows (Ensemble A)")
print("="*70)
 
train_relabeled_a = train_df.copy()
all_log_rows_a    = []
stats_rows_a      = []
 
for label in TARGET_LABELS:
    uncertain_mask = train_relabeled_a[label] == -1
    uncertain_df   = train_relabeled_a[uncertain_mask].copy()
    n_uncertain    = len(uncertain_df)
 
    # Show active models for this label
    inc = LABEL_MODEL_INCLUSION.get(label, {"wx": True, "cz": True, "bv": True})
    active_models = [k for k, v in inc.items() if v and k in ("wx", "cz", "bv")]
    print(f"\\n  {'='*60}")
    print(f"  Label    : {label}")
    print(f"  Uncertain: {n_uncertain:,}")
    print(f"  Models   : {', '.join(active_models)}")
    print(f"  {'='*60}")
 
    if n_uncertain == 0:
        print(f"  No uncertain rows — skipping")
        # Still save intermediate CSV to keep checkpoints consistent
        _lbl_safe = label.replace(" ", "_")
        train_relabeled_a.to_csv(
            f"train_relabeled_ensA_{_lbl_safe}.csv", index=False)
        print(f"  Saved: train_relabeled_ensA_{_lbl_safe}.csv (no changes)")
        continue
 
    wx_pp, wx_np = relabel_wx_protos[label]
    cz_pa, cz_na = relabel_cz_anchors[label]
    threshold    = RELABEL_THRESHOLDS[label]
 
    # Only collect arrays for active models
    score_arrays = {k: [] for k in active_models}
    valid_indices, log_rows = [], []
 
    for i, (idx, row) in enumerate(tqdm(
            uncertain_df.iterrows(), total=n_uncertain,
            desc=f"  Relabeling A [{label}]")):
        try:
            img_path = get_train_img_path(row["Path"])
            fv       = get_sample_features(
                row["Path"], label,
                top_features_per_label.get(label, []), train_df)
 
            row_scores = {}
            if inc.get("wx", True):
                row_scores["wx"]  = _ens_a_score_wxrc(img_path, label, fv, wx_pp, wx_np)
            if inc.get("cz", True):
                row_scores["cz"]  = _ens_a_score_czero(img_path, label, cz_pa, cz_na)
            if inc.get("bv", True):
                row_scores["bv"]  = _ens_a_score_biovilt(img_path, label, fv)
 
            for k in active_models:
                score_arrays[k].append(row_scores[k])
            valid_indices.append(idx)
 
            log_entry = {
                "label": label, "train_idx": idx, "path": row["Path"],
                "active_models": "|".join(active_models),
            }
            for k, v in row_scores.items():
                log_entry[f"{k}_score"] = round(v, 6)
            log_rows.append(log_entry)
 
        except Exception as e:
            log_rows.append({
                "label": label, "train_idx": idx, "path": row["Path"],
                "active_models": "|".join(active_models),
                "error": str(e),
            })
            continue
 
        if (i + 1) % CHECKPOINT_EVERY == 0:
            _lbl_safe = label.replace(" ", "_")
            pd.DataFrame(log_rows).to_csv(
                f"relabel_a_checkpoint_{_lbl_safe}.csv", index=False)
 
    if len(valid_indices) == 0:
        print(f"  No images could be scored for {label}")
    else:
        # Rank fusion — active models only
        label_w = {k: WXRC_ENS_WEIGHTS[label] if k == "wx"
                      else CZ_ENS_WEIGHTS[label] if k == "cz"
                      else BV_ENS_WEIGHTS[label]
                   for k in active_models}
        fused_scores = _rank_fuse(
            {k: np.array(score_arrays[k]) for k in active_models},
            label_w)
 
        predictions = (fused_scores > threshold).astype(int)
 
        for j, idx in enumerate(valid_indices):
            train_relabeled_a.at[idx, label] = predictions[j]
            log_rows[j]["fused_score"] = round(float(fused_scores[j]), 6)
            log_rows[j]["y_pred"]      = int(predictions[j])
            log_rows[j]["threshold"]   = threshold
 
        n_scored   = len(valid_indices)
        n_pos_pred = int(predictions.sum())
        n_neg_pred = int((predictions == 0).sum())
        n_failed   = n_uncertain - n_scored
 
        orig_pos = int((train_df[label] == 1).sum())
        orig_neg = int((train_df[label] == 0).sum())
        orig_unc = int((train_df[label] == -1).sum())
        new_pos  = int((train_relabeled_a[label] == 1).sum())
        new_neg  = int((train_relabeled_a[label] == 0).sum())
        new_unc  = int((train_relabeled_a[label] == -1).sum())
 
        print(f"  Scored      : {n_scored:,} / {n_uncertain:,}")
        print(f"  → Positive  : {n_pos_pred:,}  ({100*n_pos_pred/max(n_scored,1):.1f}%)")
        print(f"  → Negative  : {n_neg_pred:,}  ({100*n_neg_pred/max(n_scored,1):.1f}%)")
        print(f"  Failed (-1) : {n_failed:,}")
        print(f"  Distribution: pos {orig_pos:,}→{new_pos:,}  "
              f"neg {orig_neg:,}→{new_neg:,}  unc {orig_unc:,}→{new_unc:,}")
 
        stats_rows_a.append({
            "ensemble": "A", "label": label,
            "active_models": "|".join(active_models),
            "original_uncertain": orig_unc,
            "scored": n_scored,
            "relabeled_positive": new_pos - orig_pos,
            "relabeled_negative": new_neg - orig_neg,
            "failed_remain_neg1": still_unc if (still_unc := int((train_relabeled_a[label]==-1).sum())) else 0,
            "threshold": threshold,
            "new_total_positive": new_pos,
            "new_total_negative": new_neg,
        })
 
    all_log_rows_a.extend(log_rows)
 
    # ── Save per-label CSV immediately after this label ───────────
    _lbl_safe = label.replace(" ", "_")
 
    # 1. Full train CSV up to this point
    train_relabeled_a.to_csv(
        f"train_relabeled_ensA_{_lbl_safe}.csv", index=False)
    print(f"  Saved: train_relabeled_ensA_{_lbl_safe}.csv")
 
    # 2. Log for this label only
    pd.DataFrame(log_rows).to_csv(
        f"train_relabeling_log_ensA_{_lbl_safe}.csv", index=False)
    print(f"  Saved: train_relabeling_log_ensA_{_lbl_safe}.csv")
 
    gc.collect()
 
# ── STEP 3: Save final outputs ────────────────────────────────────
print("\\n" + "="*70)
print("  STEP 3: Saving final outputs (Ensemble A)")
print("="*70)
 
train_relabeled_a.to_csv("train_relabeled_ensembleA.csv", index=False)
print(f"  train_relabeled_ensembleA.csv  ({len(train_relabeled_a):,} rows)")
 
log_df_a = pd.DataFrame(all_log_rows_a)
log_df_a.to_csv("train_relabeling_log_ensembleA.csv", index=False)
print(f"  train_relabeling_log_ensembleA.csv  ({len(log_df_a):,} rows)")
 
if stats_rows_a:
    stats_df_a = pd.DataFrame(stats_rows_a)
    stats_df_a.to_csv("train_relabeling_stats_ensembleA.csv", index=False)
    print(f"  train_relabeling_stats_ensembleA.csv")
    print("\\n" + "="*70)
    print("  ENSEMBLE A RELABELING SUMMARY")
    print("="*70)
    print(stats_df_a.to_string(index=False))
 
print("\\n  Remaining -1 values:")
for label in TARGET_LABELS:
    n_left = int((train_relabeled_a[label] == -1).sum())
    print(f"    {label:<35}: {'clean' if n_left==0 else f'{n_left:,} remain'}")
 
print("\\n  train_relabeled_ensembleA.csv ready for Phase 3.")

In [ ]:

TARGET_LABELS = [
    "Edema",
    "Atelectasis",
    "Pleural Effusion",
    "Enlarged Cardiomediastinum",
    "Consolidation", 
]


In [ ]:
# ================================================================
# TRAIN SET RELABELING — ENSEMBLE B  (per-label model selection)
#
# Same structure as Ensemble A relabeling but includes BioMedCLIP(FT)
# for labels where it is enabled in LABEL_MODEL_INCLUSION.
#
# OUTPUTS (written incrementally):
#   train_relabeled_ensB_{label}.csv     — state after each label
#   train_relabeling_log_ensB_{label}.csv — per-row log each label
#   train_relabeled_ensembleB.csv        — final full CSV
#   train_relabeling_log_ensembleB.csv   — combined log
#   train_relabeling_stats_ensembleB.csv — per-label statistics
# ================================================================
 
import os, gc, torch, torch.nn.functional as F
import numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from scipy.stats import rankdata
 
RELABEL_SEED     = 0
CHECKPOINT_EVERY = 500
 
if "ENSEMBLE_THRESHOLDS" not in globals() or not ENSEMBLE_THRESHOLDS:
    raise RuntimeError("Run the ENSEMBLE THRESHOLD CONFIG cell first.")
RELABEL_THRESHOLDS = ENSEMBLE_THRESHOLDS
 
print("=" * 70)
print("  TRAIN SET RELABELING — ENSEMBLE B")
print(f"  Per-label model inclusion active.")
print("=" * 70)
print("\\nThresholds:")
for lbl, t in RELABEL_THRESHOLDS.items():
    print(f"  {lbl:<35}: {t:.4f}")
 
def get_train_img_path(path_str):
    clean = (str(path_str)
             .replace("CheXpert-v1.0-small/", "")
             .replace("CheXpert-v1.0/", ""))
    return os.path.join(BASE_PATH, clean)
 
# ── STEP 1: Build prototypes ──────────────────────────────────────
print("\\n" + "="*70)
print(f"  STEP 1: Building Ensemble B prototypes (seed={RELABEL_SEED})")
print("="*70)
 
# WhyXrayCLIP — reuse from Ensemble A if available
if "relabel_wx_protos" in globals() and relabel_wx_protos:
    print("  WhyXrayCLIP: reusing relabel_wx_protos from Ensemble A")
else:
    if ("WXRC_FINAL_PROTO" in globals()
            and globals().get("WXRC_FINAL_N_SHOTS") == WXRC_ENS_N_SHOTS
            and globals().get("WXRC_FINAL_ALPHA")   == WXRC_ENS_ALPHA):
        relabel_wx_protos = WXRC_FINAL_PROTO
        print("  WhyXrayCLIP: reusing WXRC_FINAL_PROTO")
    else:
        relabel_wx_protos = {}
        for label in TARGET_LABELS:
            if not LABEL_MODEL_INCLUSION.get(label, {}).get("wx", True):
                relabel_wx_protos[label] = (None, None)
                continue
            pp, np_, meta = build_wxrc_prototype(
                label, WXRC_ENS_N_SHOTS, seed=RELABEL_SEED)
            relabel_wx_protos[label] = (pp, np_)
            print(f"  WhyXrayCLIP [{label}]: "
                  f"pos={meta.get('n_pos','?')} neg={meta.get('n_neg','?')}")
 
# CheXZero — reuse from Ensemble A if available
if "relabel_cz_anchors" in globals() and relabel_cz_anchors:
    print("  CheXZero: reusing relabel_cz_anchors from Ensemble A")
else:
    _saved_ctx = dict(CZERO_CONTEXT_STRATEGY) if "CZERO_CONTEXT_STRATEGY" in globals() else {}
    if "CZERO_CONTEXT_STRATEGY" in globals():
        CZERO_CONTEXT_STRATEGY.update(CZERO_ENS_CONTEXT_STRATEGY)
    relabel_cz_anchors, _ = build_chexzero_prototypes(
        CZ_ENS_N_SHOTS, CZ_ENS_LABEL_WEIGHTS)
    if "CZERO_CONTEXT_STRATEGY" in globals():
        CZERO_CONTEXT_STRATEGY.update(_saved_ctx)
    print("  CheXZero: built fresh")
 
print("  BioViL-T: text_only (alpha=1.0) — no prototypes needed")
 
# BioMedCLIP — build only for labels where bmc is active
print("\\n  [BioMedCLIP(FT)] building per-label prototypes...")
relabel_bmc_protos = {}
for label in TARGET_LABELS:
    if not LABEL_MODEL_INCLUSION.get(label, {}).get("bmc", True):
        relabel_bmc_protos[label] = (None, None)
        print(f"    {label}: disabled for this label — skipping")
        continue
    cfg    = BMC_ENS_BEST_PER_LABEL[label]
    n_shot = cfg["n_shots"]
    alpha  = cfg["alpha"]
    print(f"    {label}  [n={n_shot}, α={alpha:.2f}, "
          f"model={cfg['model']}, ctx={cfg['context_strategy']}]", end="")
    if alpha >= 1.0:
        relabel_bmc_protos[label] = (None, None)
        print("  → text_only")
    else:
        pp, np_, meta = build_bmc_prototype(label, n_shot, seed=RELABEL_SEED)
        relabel_bmc_protos[label] = (pp, np_)
        if pp is not None:
            print(f"  ok  pos={meta.get('n_pos','?')} neg={meta.get('n_neg','?')}")
        else:
            print("  FAILED — text_only fallback")
 
_bmc_ens_protos = relabel_bmc_protos   # needed by _ens_b_score_bmc
 
# ── STEP 2: Relabel ───────────────────────────────────────────────
print("\\n" + "="*70)
print("  STEP 2: Relabeling uncertain rows (Ensemble B)")
print("="*70)
 
train_relabeled_b = train_df.copy()
all_log_rows_b    = []
stats_rows_b      = []
 
for label in TARGET_LABELS:
    uncertain_mask = train_relabeled_b[label] == -1
    uncertain_df   = train_relabeled_b[uncertain_mask].copy()
    n_uncertain    = len(uncertain_df)
 
    inc = LABEL_MODEL_INCLUSION.get(
        label, {"wx": True, "cz": True, "bv": True, "bmc": True})
    active_models = [k for k, v in inc.items()
                     if v and k in ("wx", "cz", "bv", "bmc")]
    print(f"\\n  {'='*60}")
    print(f"  Label    : {label}")
    print(f"  Uncertain: {n_uncertain:,}")
    print(f"  Models   : {', '.join(active_models)}")
    print(f"  {'='*60}")
 
    if n_uncertain == 0:
        print(f"  No uncertain rows — skipping")
        _lbl_safe = label.replace(" ", "_")
        train_relabeled_b.to_csv(
            f"train_relabeled_ensB_{_lbl_safe}.csv", index=False)
        print(f"  Saved: train_relabeled_ensB_{_lbl_safe}.csv (no changes)")
        continue
 
    wx_pp, wx_np = relabel_wx_protos[label]
    cz_pa, cz_na = relabel_cz_anchors[label]
    threshold    = RELABEL_THRESHOLDS[label]
 
    score_arrays = {k: [] for k in active_models}
    valid_indices, log_rows = [], []
 
    for i, (idx, row) in enumerate(tqdm(
            uncertain_df.iterrows(), total=n_uncertain,
            desc=f"  Relabeling B [{label}]")):
        try:
            img_path = get_train_img_path(row["Path"])
            fv       = get_sample_features(
                row["Path"], label,
                top_features_per_label.get(label, []), train_df)
 
            row_scores = {}
            if inc.get("wx",  True): row_scores["wx"]  = _ens_a_score_wxrc(img_path, label, fv, wx_pp, wx_np)
            if inc.get("cz",  True): row_scores["cz"]  = _ens_a_score_czero(img_path, label, cz_pa, cz_na)
            if inc.get("bv",  True): row_scores["bv"]  = _ens_a_score_biovilt(img_path, label, fv)
            if inc.get("bmc", True): row_scores["bmc"] = _ens_b_score_bmc(img_path, label, fv)
 
            for k in active_models:
                score_arrays[k].append(row_scores[k])
            valid_indices.append(idx)
 
            log_entry = {
                "label": label, "train_idx": idx, "path": row["Path"],
                "active_models": "|".join(active_models),
            }
            for k, v in row_scores.items():
                log_entry[f"{k}_score"] = round(v, 6)
            log_rows.append(log_entry)
 
        except Exception as e:
            log_rows.append({
                "label": label, "train_idx": idx, "path": row["Path"],
                "active_models": "|".join(active_models), "error": str(e),
            })
            continue
 
        if (i + 1) % CHECKPOINT_EVERY == 0:
            _lbl_safe = label.replace(" ", "_")
            pd.DataFrame(log_rows).to_csv(
                f"relabel_b_checkpoint_{_lbl_safe}.csv", index=False)
 
    if len(valid_indices) == 0:
        print(f"  No images could be scored for {label}")
    else:
        # Rank fusion — active models only, with their weights
        label_w = {}
        for k in active_models:
            if k == "wx":  label_w[k] = WXRC_ENS_WEIGHTS[label]
            elif k == "cz": label_w[k] = CZ_ENS_WEIGHTS[label]
            elif k == "bv": label_w[k] = BV_ENS_WEIGHTS[label]
            elif k == "bmc": label_w[k] = BMC_ENS_WEIGHTS[label]
 
        fused_scores = _rank_fuse(
            {k: np.array(score_arrays[k]) for k in active_models},
            label_w)
 
        predictions = (fused_scores > threshold).astype(int)
 
        for j, idx in enumerate(valid_indices):
            train_relabeled_b.at[idx, label] = predictions[j]
            log_rows[j]["fused_score"] = round(float(fused_scores[j]), 6)
            log_rows[j]["y_pred"]      = int(predictions[j])
            log_rows[j]["threshold"]   = threshold
 
        n_scored   = len(valid_indices)
        n_pos_pred = int(predictions.sum())
        n_neg_pred = int((predictions == 0).sum())
        n_failed   = n_uncertain - n_scored
 
        orig_pos = int((train_df[label] == 1).sum())
        orig_neg = int((train_df[label] == 0).sum())
        orig_unc = int((train_df[label] == -1).sum())
        new_pos  = int((train_relabeled_b[label] == 1).sum())
        new_neg  = int((train_relabeled_b[label] == 0).sum())
        new_unc  = int((train_relabeled_b[label] == -1).sum())
        still_unc = new_unc
 
        print(f"  Scored      : {n_scored:,} / {n_uncertain:,}")
        print(f"  → Positive  : {n_pos_pred:,}  ({100*n_pos_pred/max(n_scored,1):.1f}%)")
        print(f"  → Negative  : {n_neg_pred:,}  ({100*n_neg_pred/max(n_scored,1):.1f}%)")
        print(f"  Failed (-1) : {n_failed:,}")
        print(f"  Distribution: pos {orig_pos:,}→{new_pos:,}  "
              f"neg {orig_neg:,}→{new_neg:,}  unc {orig_unc:,}→{new_unc:,}")
 
        stats_rows_b.append({
            "ensemble": "B", "label": label,
            "active_models": "|".join(active_models),
            "original_uncertain": orig_unc,
            "scored": n_scored,
            "relabeled_positive": new_pos - orig_pos,
            "relabeled_negative": new_neg - orig_neg,
            "failed_remain_neg1": still_unc,
            "threshold": threshold,
            "new_total_positive": new_pos,
            "new_total_negative": new_neg,
        })
 
    all_log_rows_b.extend(log_rows)
 
    # ── Save per-label CSV immediately ────────────────────────────
    _lbl_safe = label.replace(" ", "_")
    train_relabeled_b.to_csv(
        f"train_relabeled_ensB_{_lbl_safe}.csv", index=False)
    print(f"  Saved: train_relabeled_ensB_{_lbl_safe}.csv")
 
    pd.DataFrame(log_rows).to_csv(
        f"train_relabeling_log_ensB_{_lbl_safe}.csv", index=False)
    print(f"  Saved: train_relabeling_log_ensB_{_lbl_safe}.csv")
 
    gc.collect()
 
# ── STEP 3: Save final outputs ────────────────────────────────────
print("\\n" + "="*70)
print("  STEP 3: Saving final outputs (Ensemble B)")
print("="*70)
 
train_relabeled_b.to_csv("train_relabeled_ensembleB.csv", index=False)
print(f"  train_relabeled_ensembleB.csv  ({len(train_relabeled_b):,} rows)")
 
log_df_b = pd.DataFrame(all_log_rows_b)
log_df_b.to_csv("train_relabeling_log_ensembleB.csv", index=False)
print(f"  train_relabeling_log_ensembleB.csv  ({len(log_df_b):,} rows)")
 
if stats_rows_b:
    stats_df_b = pd.DataFrame(stats_rows_b)
    stats_df_b.to_csv("train_relabeling_stats_ensembleB.csv", index=False)
    print(f"  train_relabeling_stats_ensembleB.csv")
    print("\\n" + "="*70)
    print("  ENSEMBLE B RELABELING SUMMARY")
    print("="*70)
    print(stats_df_b.to_string(index=False))
 
print("\\n  Remaining -1 values:")
for label in TARGET_LABELS:
    n_left = int((train_relabeled_b[label] == -1).sum())
    print(f"    {label:<35}: {'clean' if n_left==0 else f'{n_left:,} remain'}")
 
print("\\n  train_relabeled_ensembleB.csv ready for Phase 3.")
print("  Per-label CSVs also saved for crash recovery.")

In [ ]:
relabeled_train_df = pd.read_csv(f"/kaggle/input/datasets/masnoonmuztahid/train-relabeled-updated/train_relabeled_v2.csv")

In [ ]:
# ================================================================
# PHASE 3: Multi-Label DenseNet-121 Training & Evaluation
# Comparing uncertainty labeling strategies:
#   U-Zero | U-One | U-Class | U-VLM (your relabeled df)
#
# Target: All 14 CheXpert pathologies (multi-label)
# Primary evaluation: Cardiomegaly AUC + all 14 labels
# GPU: H100 on Kaggle
#
# Estimated runtime per model on H100:
#   - Dataset prep   : ~2 min
#   - Training       : ~25-35 min (5 epochs, ~223k images)
#   - Evaluation     : ~3 min
#   - Total (4 runs) : ~2.5 - 3 hours
# ================================================================

In [ ]:
# ================================================================
# CELL 1: Imports & Config
# ================================================================
import os, gc, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    balanced_accuracy_score, confusion_matrix,
)
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────
# NOTE ON DATA SPLITS:
#   train.csv → training data (uncertain labels handled by U-VLM pipeline)
#   valid.csv → FINAL TEST SET  ← DO NOT use for validation / early stopping
#   train.csv is split 90/10 internally for train / validation.
BASE_PATH      = '/kaggle/input/chexpert-dataset-ml-project'
RELABELED_PATH = '/kaggle/input/datasets/mirajhasan/training-calibration-relabelled/train_relabeled_ensembleB.csv'
OUTPUT_DIR     = '/kaggle/working/phase3_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Labels ────────────────────────────────────────────────────────
ALL_LABELS = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly',
    'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation',
    'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
    'Pleural Other', 'Fracture', 'Support Devices'
]
# 5 labels relabeled by VLM pipeline
TARGET_LABELS = ['Edema', 'Atelectasis', 'Pleural Effusion',
                 'Enlarged Cardiomediastinum', 'Consolidation']
OTHER_LABELS  = [l for l in ALL_LABELS if l not in TARGET_LABELS]

# ── Improved Training Config ──────────────────────────────────────
IMPROVED_CONFIG = {
    'img_size'       : 224,
    'num_epochs'     : 10,
    'warmup_epochs'  : 2,
    'lr'             : 1e-4,
    'min_lr'         : 1e-6,
    'weight_decay'   : 1e-5,
    'batch_size'     : 64,
    'num_workers'    : 4,
    'mixed_precision': True,
    'patience'       : 5,
    'seed'           : 42,
}

EXP_ORDER = ['U-Ignore', 'U-Zeros', 'U-Ones', 'U-MultiClass', 'U-VLM']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(IMPROVED_CONFIG['seed'])
print('✅ Cell 1 complete — imports, config, device ready.')

In [ ]:
# ================================================================
# CELL 2 (Corrected): Build the 5 Experimental DataFrames
#
# DATA SPLIT STRATEGY:
#   - valid.csv  → FINAL TEST SET (held-out, never touched during training)
#   - train.csv  → split 90/10 into internal train / val
#
# LABELING SCHEME:
#   For ALL strategies, we work on the RAW train_df_orig before applying
#   any strategy-specific transformation. The split happens FIRST, then
#   each strategy's relabeling is applied to the train portion only.
#
#   5 TARGET LABELS  : ['Edema', 'Atelectasis', 'Pleural Effusion',
#                        'Enlarged Cardiomediastinum', 'Consolidation']
#   9 OTHER LABELS   : the remaining 9 of the 14 pathology features
#
#   Strategy rules for the TRAIN split:
#     U-Zero      : TARGET -1 → 0       | OTHER -1 → 0
#     U-One       : TARGET -1 → 1       | OTHER -1 → 0
#     U-Ignore    : DROP rows where ANY TARGET label == -1
#                   (OTHER -1 → 0 for kept rows)
#     U-MultiClass: TARGET keep -1      | OTHER -1 → 0
#     U-VLM       : TARGET already replaced by VLM (pre-split, unavoidable)
#                   residual -1 → 0     | OTHER -1 → 0
#
#   VALIDATION split (used for early stopping / model selection):
#     Drop rows where ANY TARGET label == -1  (ambiguous val labels excluded)
#     ALL remaining -1 → 0  (clean binary labels for evaluation)
#
#   TEST set (valid.csv):
#     ALL -1 → 0  (standard CheXpert evaluation protocol)
#
# NO DATA LEAKAGE: The train/val split is performed on the raw df
#   before any label transformation. Rule-based transforms are
#   deterministic and sample-independent, so order doesn't affect
#   leakage — but split-first is the principled standard.
#   U-VLM relabeling was done on the full train_df before this cell
#   (unavoidable since it was a batch VLM inference job), but since
#   the VLM predicted labels only from *images* (no label statistics),
#   there is no leakage either.
#
# Requires: Cell 1
# ================================================================

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

print('\n' + '='*65)
print('  CELL 2 (Corrected): Building 5 experimental train DataFrames')
print('='*65)

# ── Load raw data ────────────────────────────────────────────────
train_df_orig   = pd.read_csv(f'{BASE_PATH}/train.csv')
test_df         = pd.read_csv(f'{BASE_PATH}/valid.csv')   # FINAL TEST SET — DO NOT USE FOR TRAINING
train_relabeled = pd.read_csv(RELABELED_PATH)             # U-VLM: VLM-relabeled train_df

print(f'  Original train_df  : {train_df_orig.shape}')
print(f'  Relabeled train_df : {train_relabeled.shape}')
print(f'  Test df (valid.csv): {test_df.shape}  ← held-out, final evaluation only')

# ── Define label groups ──────────────────────────────────────────
TARGET_LABELS = ['Edema', 'Atelectasis', 'Pleural Effusion',
                 'Enlarged Cardiomediastinum', 'Consolidation']
OTHER_LABELS  = [l for l in ALL_LABELS if l not in TARGET_LABELS]

print(f'\n  Target labels ({len(TARGET_LABELS)}): {TARGET_LABELS}')
print(f'  Other labels  ({len(OTHER_LABELS)}): {OTHER_LABELS}')

# ── Step 1: Train / val split on RAW train_df (before any relabeling) ──
# Stratify on Cardiomegaly for balanced splits.
strat_col  = 'Cardiomegaly'
strat_vals = train_df_orig[strat_col].fillna(0).replace(-1, 0).astype(int)

train_idx, val_idx = train_test_split(
    train_df_orig.index,
    test_size=0.10,
    random_state=IMPROVED_CONFIG['seed'],
    stratify=strat_vals,
)

raw_train_df = train_df_orig.loc[train_idx].reset_index(drop=True)
raw_val_df   = train_df_orig.loc[val_idx].reset_index(drop=True)

print(f'\n  Raw train split : {len(raw_train_df):,} rows')
print(f'  Raw val   split : {len(raw_val_df):,} rows')

# Also slice the relabeled df with the same train indices
# (train_relabeled must have same length/order as train_df_orig
#  OR be matched by Path column)
# Matching by Path for safety:
relabeled_indexed = train_relabeled.set_index('Path') if 'Path' in train_relabeled.columns else train_relabeled
raw_train_paths   = set(raw_train_df['Path'].values)
raw_val_paths     = set(raw_val_df['Path'].values)

train_relabeled_split = train_relabeled[
    train_relabeled['Path'].isin(raw_train_paths)
].reset_index(drop=True)

print(f'  Relabeled train split: {len(train_relabeled_split):,} rows')


# ── Step 2: Build the VALIDATION df (shared across all strategies) ──
# Rule: drop rows with ANY -1 in TARGET labels (ambiguous val GT excluded)
#       then map ALL remaining -1 → 0

def build_val_df(raw_val):
    df = raw_val.copy()
    # Drop rows with uncertain labels in any target feature
    target_uncertain_mask = (df[TARGET_LABELS] == -1).any(axis=1)
    n_before = len(df)
    df = df[~target_uncertain_mask].reset_index(drop=True)
    n_dropped = n_before - len(df)
    print(f'  Val: dropped {n_dropped} rows with uncertain target labels '
          f'({n_before} → {len(df)})')
    # Clean all remaining -1 to 0 (for all 14 labels)
    df[ALL_LABELS] = df[ALL_LABELS].replace(-1, 0).fillna(0)
    return df

val_df_clean = build_val_df(raw_val_df)
print(f'  Final val split: {len(val_df_clean):,} rows (clean binary labels)')


# ── Step 3: Build TEST df ────────────────────────────────────────
# Standard CheXpert protocol: -1 → 0 across all labels
def build_test_df(df):
    df = df.copy()
    df[ALL_LABELS] = df[ALL_LABELS].replace(-1, 0).fillna(0)
    return df

test_df_clean = build_test_df(test_df)
print(f'  Test df (cleaned): {len(test_df_clean):,} rows')


# ── Step 4: Helper — replace -1 in Other labels with 0 ──────────
def clean_other_labels(df):
    """Map -1 → 0 for the 9 non-target labels. In-place on a copy."""
    df = df.copy()
    for col in OTHER_LABELS:
        if col in df.columns:
            df[col] = df[col].replace(-1, 0).fillna(0)
    return df


# ── Step 5: Build each strategy's TRAIN df ──────────────────────

# -- U-Zero: TARGET -1 → 0, OTHER -1 → 0
def build_uzero(raw_train):
    df = raw_train.copy()
    df[ALL_LABELS] = df[ALL_LABELS].replace(-1, 0).fillna(0)
    return df

# -- U-One: TARGET -1 → 1, OTHER -1 → 0
def build_uone(raw_train):
    df = raw_train.copy()
    # First clean other labels
    for col in OTHER_LABELS:
        if col in df.columns:
            df[col] = df[col].replace(-1, 0).fillna(0)
    # Then set target -1 → 1
    for col in TARGET_LABELS:
        if col in df.columns:
            df[col] = df[col].replace(-1, 1).fillna(0)
    return df

# -- U-Ignore: drop rows with ANY -1 in TARGET labels, OTHER -1 → 0
def build_uignore(raw_train):
    df = raw_train.copy()
    # Drop rows uncertain in any target label
    target_uncertain_mask = (df[TARGET_LABELS] == -1).any(axis=1)
    n_before = len(df)
    df = df[~target_uncertain_mask].reset_index(drop=True)
    n_dropped = n_before - len(df)
    print(f'  U-Ignore: dropped {n_dropped} rows ({n_before} → {len(df)})')
    # Clean other labels
    for col in OTHER_LABELS:
        if col in df.columns:
            df[col] = df[col].replace(-1, 0).fillna(0)
    return df

# -- U-MultiClass: TARGET keeps -1 (3-class signal), OTHER -1 → 0
def build_umulticlass(raw_train):
    df = raw_train.copy()
    # Only clean other labels; keep target -1 for 3-class CE loss
    for col in OTHER_LABELS:
        if col in df.columns:
            df[col] = df[col].replace(-1, 0).fillna(0)
    return df

# -- U-VLM: use VLM-relabeled df for target labels, OTHER -1 → 0
#    Residual -1 in target (rows VLM couldn't score) → 0
def build_uvlm(relabeled_train):
    df = relabeled_train.copy()
    # Clean all: residual -1 → 0 for both target and other
    df[ALL_LABELS] = df[ALL_LABELS].replace(-1, 0).fillna(0)
    return df


print('\n' + '='*65)
print('  Building strategy DataFrames from raw train split...')
print('='*65)

EXPERIMENTS = {
    'U-Zeros'     : build_uzero(raw_train_df),
    'U-Ones'      : build_uone(raw_train_df),
    'U-Ignore'    : build_uignore(raw_train_df),
    'U-MultiClass': build_umulticlass(raw_train_df),
    'U-VLM'       : build_uvlm(train_relabeled_split),
}


# ── Step 6: Diagnostic — show uncertainty stats ──────────────────
def show_uncertainty_stats(df, name):
    print(f'\n  [{name}] shape={df.shape}')
    for col in TARGET_LABELS:
        if col not in df.columns:
            continue
        n_neg1 = (df[col] == -1).sum()
        n_0    = (df[col] == 0).sum()
        n_1    = (df[col] == 1).sum()
        print(f'    {col:<35} : -1={n_neg1:>6}  0={n_0:>6}  1={n_1:>6}')

for name, df in EXPERIMENTS.items():
    show_uncertainty_stats(df, name)

print(f'\n  Validation (shared across all strategies):')
for col in TARGET_LABELS:
    n_0 = (val_df_clean[col] == 0).sum()
    n_1 = (val_df_clean[col] == 1).sum()
    print(f'    {col:<35} : 0={n_0:>6}  1={n_1:>6}  (no -1s)')

print('\n✅ Cell 2 complete — EXPERIMENTS dict + val_df_clean + test_df_clean ready.')
print(f'   Keys: {list(EXPERIMENTS.keys())}')

In [ ]:
# ================================================================
# CELL 3: Shared Helpers — Dataset, Transforms, Loss Functions,
#         Model Definitions, LR Schedule, Train/Evaluate,
#         Metrics, compute_detailed_metrics
#
# SELF-CONTAINED: defines everything needed by Cell 4 (baselines),
# Cell 4b (U-VLM DenseNet), Cell 5 (U-VLM EfficientNet), and the
# cross-architecture comparison cell.
#
# Requires: Cell 1 (imports + IMPROVED_CONFIG + device + ALL_LABELS)
# ================================================================

# ── Path fix ──────────────────────────────────────────────────────
def fix_path(path_str, base=BASE_PATH):
    clean = (str(path_str)
             .replace('CheXpert-v1.0-small/', '')
             .replace('CheXpert-v1.0/', ''))
    return os.path.join(base, clean)

# ── Transforms ───────────────────────────────────────────────────
TRAIN_TRANSFORMS = transforms.Compose([
    transforms.Resize((IMPROVED_CONFIG['img_size'], IMPROVED_CONFIG['img_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
VALID_TRANSFORMS = transforms.Compose([
    transforms.Resize((IMPROVED_CONFIG['img_size'], IMPROVED_CONFIG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── Dataset ───────────────────────────────────────────────────────
class CheXpertDataset(Dataset):
    """
    Returns (image_tensor, label_tensor).
    label_tensor values: 0.0 / 1.0 / -1.0
    (-1.0 only for U-MultiClass rows; all other strategies have cleaned labels.)
    """
    def __init__(self, df, labels, transform=None, base_path=BASE_PATH):
        self.df        = df.reset_index(drop=True)
        self.labels    = labels
        self.transform = transform
        self.base_path = base_path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = fix_path(row['Path'], self.base_path)
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (IMPROVED_CONFIG['img_size'],
                                      IMPROVED_CONFIG['img_size']))
        if self.transform:
            image = self.transform(image)
        label_vals = [
            float(row.get(col, 0)) if pd.notna(row.get(col, 0)) else 0.0
            for col in self.labels
        ]
        return image, torch.tensor(label_vals, dtype=torch.float32)

def get_loaders(train_df, valid_df, labels):
    train_ds = CheXpertDataset(train_df, labels, transform=TRAIN_TRANSFORMS)
    valid_ds = CheXpertDataset(valid_df, labels, transform=VALID_TRANSFORMS)
    train_loader = DataLoader(
        train_ds, batch_size=IMPROVED_CONFIG['batch_size'],
        shuffle=True, num_workers=IMPROVED_CONFIG['num_workers'],
        pin_memory=True, drop_last=True)
    valid_loader = DataLoader(
        valid_ds, batch_size=IMPROVED_CONFIG['batch_size'] * 2,
        shuffle=False, num_workers=IMPROVED_CONFIG['num_workers'],
        pin_memory=True)
    return train_loader, valid_loader

# ── Loss functions ────────────────────────────────────────────────
class MaskedBCELoss(nn.Module):
    """
    BCE with logits that skips positions where target == -1.
    For U-Ignore / U-Zeros / U-Ones / U-VLM there are no -1s, so
    this is identical to plain BCEWithLogitsLoss.
    Masking is kept to guard against any residual -1s.
    """
    def forward(self, logits, targets):
        mask = (targets != -1)
        if mask.sum() == 0:
            return torch.tensor(0.0, requires_grad=True, device=logits.device)
        return F.binary_cross_entropy_with_logits(
            logits[mask], targets[mask], reduction='mean')

class MultiClassLoss(nn.Module):
    """
    For U-MultiClass: model outputs (B, N_LABELS * 3).
    Treats each label as a 3-class problem:
      class 0 = negative  (label  0)
      class 1 = positive  (label  1)
      class 2 = uncertain (label -1)
    """
    def forward(self, logits, targets):
        B, N  = targets.shape          # N = 14
        logits3 = logits.view(B, N, 3) # (B, 14, 3)
        t = targets.long().clone()
        t[t == -1] = 2                 # -1 → class index 2
        return F.cross_entropy(
            logits3.view(B * N, 3), t.view(B * N), reduction='mean')

# ── LR: linear warmup → cosine decay ─────────────────────────────
def get_lr(epoch, config):
    if epoch < config['warmup_epochs']:
        return config['lr'] * (epoch + 1) / config['warmup_epochs']
    progress = ((epoch - config['warmup_epochs']) /
                max(1, config['num_epochs'] - config['warmup_epochs']))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return config['min_lr'] + (config['lr'] - config['min_lr']) * cosine

# ── Train one epoch ───────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, scaler, epoch, config):
    model.train()
    total_loss, n_batches = 0.0, 0
    pbar = tqdm(loader, desc=f'  Epoch {epoch+1:>2} [Train]', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if config['mixed_precision']:
            with autocast():
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = criterion(model(images), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    return total_loss / max(n_batches, 1)

# ── Evaluate ──────────────────────────────────────────────────────
@torch.no_grad()
def evaluate_model(model, loader, labels, is_multiclass=False):
    """
    Returns mean_auroc (float), all_probs (N_samples × N_labels),
    all_targets (N_samples × N_labels).

    is_multiclass=True  → model outputs (B, N*3); extract P(positive)
                          via softmax[:, :, 1].
    is_multiclass=False → model outputs (B, N); apply sigmoid.
    """
    model.eval()
    all_probs, all_targets = [], []
    for images, targets in tqdm(loader, desc='  Evaluating', leave=False):
        images = images.to(device, non_blocking=True)
        with autocast():
            logits = model(images)
        if is_multiclass:
            B = logits.shape[0]
            N = len(labels)
            probs = torch.softmax(logits.view(B, N, 3), dim=-1)[:, :, 1].cpu().numpy()
        else:
            probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_targets.append(targets.numpy())

    all_probs   = np.concatenate(all_probs,   axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    aucs = []
    for i in range(len(labels)):
        y_true = all_targets[:, i]
        if len(np.unique(y_true)) >= 2:
            aucs.append(roc_auc_score(y_true, all_probs[:, i]))
    mean_auroc = float(np.mean(aucs)) if aucs else 0.0
    return mean_auroc, all_probs, all_targets

# ── Per-label detailed metrics (all 6 metrics + CSV-ready) ───────
def compute_detailed_metrics(all_probs, all_targets, labels,
                              experiment_name, arch):
    """
    Computes per-label: AUROC, AUPRC, F1, Sensitivity (Recall),
    Specificity, Balanced Accuracy.
    Threshold is fixed at 0.5 for all threshold-dependent metrics.
    Returns a tidy DataFrame ready to concat and save to CSV.
    """
    rows = []
    for i, label in enumerate(labels):
        y_true = all_targets[:, i]
        y_prob = all_probs[:, i]
        y_pred = (y_prob >= 0.5).astype(int)

        # Skip labels with a single class in validation
        if len(np.unique(y_true)) < 2:
            rows.append({
                'Arch'          : arch,
                'Experiment'    : experiment_name,
                'Label'         : label,
                'AUROC'         : np.nan,
                'AUPRC'         : np.nan,
                'F1'            : np.nan,
                'Sensitivity'   : np.nan,
                'Specificity'   : np.nan,
                'BalancedAcc'   : np.nan,
                'N_Pos'         : int(y_true.sum()),
                'N_Total'       : len(y_true),
            })
            continue

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        rows.append({
            'Arch'          : arch,
            'Experiment'    : experiment_name,
            'Label'         : label,
            'AUROC'         : round(float(roc_auc_score(y_true, y_prob)),             4),
            'AUPRC'         : round(float(average_precision_score(y_true, y_prob)),   4),
            'F1'            : round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
            'Sensitivity'   : round(float(sens),                                      4),
            'Specificity'   : round(float(spec),                                      4),
            'BalancedAcc'   : round(float(balanced_accuracy_score(y_true, y_pred)),   4),
            'N_Pos'         : int(y_true.sum()),
            'N_Total'       : len(y_true),
        })
    return pd.DataFrame(rows)

# ── Print per-label metrics table ────────────────────────────────
def print_metrics_table(detail_df, exp_name, arch):
    cols = ['Label', 'AUROC', 'AUPRC', 'F1', 'Sensitivity', 'Specificity', 'BalancedAcc']
    sub  = detail_df[detail_df['Experiment'] == exp_name][cols].copy()
    # Compute mean row (numeric only)
    mean_row = sub[cols[1:]].mean().round(4).to_dict()
    mean_row['Label'] = 'MEAN'
    sub = pd.concat([sub, pd.DataFrame([mean_row])], ignore_index=True)
    hdr = f'  {arch} | {exp_name} — Per-Label Metrics (threshold=0.5)'
    print(f'\n{"-"*len(hdr)}')
    print(hdr)
    print(f'{"-"*len(hdr)}')
    print(f'  {"Label":<35} {"AUROC":>7} {"AUPRC":>7} {"F1":>7} '
          f'{"Sens":>7} {"Spec":>7} {"BalAcc":>7}')
    print(f'  {"-"*79}')
    for _, row in sub.iterrows():
        auroc = f'{row["AUROC"]:.4f}' if pd.notna(row["AUROC"]) else '   nan'
        auprc = f'{row["AUPRC"]:.4f}' if pd.notna(row["AUPRC"]) else '   nan'
        f1    = f'{row["F1"]:.4f}'    if pd.notna(row["F1"])    else '   nan'
        sens  = f'{row["Sensitivity"]:.4f}' if pd.notna(row["Sensitivity"]) else '   nan'
        spec  = f'{row["Specificity"]:.4f}' if pd.notna(row["Specificity"]) else '   nan'
        balacc= f'{row["BalancedAcc"]:.4f}' if pd.notna(row["BalancedAcc"]) else '   nan'
        marker = ' ◀' if row['Label'] == 'MEAN' else ''
        print(f'  {str(row["Label"]):<35} {auroc:>7} {auprc:>7} {f1:>7} '
              f'{sens:>7} {spec:>7} {balacc:>7}{marker}')

# ── Save results and print pivot summaries ────────────────────────
def save_and_summarise(combined_df, history_rows, arch_name):
    """
    Saves detailed CSV and per-metric pivot CSVs.
    Prints concise pivot tables for all 6 metrics.
    """
    # Detailed CSV
    csv_path = os.path.join(OUTPUT_DIR, f'{arch_name}_metrics_detailed.csv')
    combined_df.to_csv(csv_path, index=False)
    print(f'\n  Saved: {csv_path}')

    # Training history CSV
    hist_df = pd.DataFrame(history_rows)
    hist_path = os.path.join(OUTPUT_DIR, f'{arch_name}_training_history.csv')
    hist_df.to_csv(hist_path, index=False)
    print(f'  Saved: {hist_path}')

    # Pivot tables — one per metric
    metrics = ['AUROC', 'AUPRC', 'F1', 'Sensitivity', 'Specificity', 'BalancedAcc']
    pivots  = {}
    for metric in metrics:
        p = (combined_df
             .pivot_table(index='Label', columns='Experiment',
                          values=metric, aggfunc='first')
             .reindex(columns=[c for c in EXP_ORDER if c in combined_df['Experiment'].unique()]))
        p.loc['MEAN'] = p.mean()
        pivots[metric] = p
        p.to_csv(os.path.join(OUTPUT_DIR,
                              f'{arch_name}_pivot_{metric.lower()}.csv'))

    print(f'\n{"="*65}')
    print(f'  {arch_name.upper()} — SUMMARY (Mean over 14 labels)')
    print(f'{"="*65}')
    for metric in metrics:
        print(f'\n  📊 {metric}:')
        print(pivots[metric].loc['MEAN'].round(4).to_string())

    print(f'\n  📊 FOCUS — 5 VLM-Relabeled Labels (AUROC):')
    focus_labels = [l for l in TARGET_LABELS if l in pivots['AUROC'].index]
    print(pivots['AUROC'].loc[focus_labels].round(4).to_string())

    print(f'\n  📊 FOCUS — 5 VLM-Relabeled Labels (AUPRC):')
    print(pivots['AUPRC'].loc[focus_labels].round(4).to_string())

    return pivots

# ── Model definitions (DenseNet-121) ─────────────────────────────
class DenseNet121MultiLabel(nn.Module):
    """Standard binary multi-label model — N_LABELS outputs.
    Used for: U-Ignore, U-Zeros, U-Ones, U-VLM."""
    def __init__(self, num_classes=14, pretrained=True):
        super().__init__()
        weights         = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        base            = models.densenet121(weights=weights)
        self.features   = base.features
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(base.classifier.in_features, num_classes),
        )
    def forward(self, x):
        return self.classifier(F.relu(self.features(x), inplace=True))

class DenseNet121MultiClass(nn.Module):
    """U-MultiClass variant — N_LABELS * 3 outputs (3 classes per label)."""
    def __init__(self, num_classes=14, pretrained=True):
        super().__init__()
        weights         = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        base            = models.densenet121(weights=weights)
        self.features   = base.features
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(base.classifier.in_features, num_classes * 3),
        )
    def forward(self, x):
        return self.classifier(F.relu(self.features(x), inplace=True))

# ── Model definitions (EfficientNet-B4) ──────────────────────────
class EfficientNetB4MultiLabel(nn.Module):
    """Standard binary multi-label model — N_LABELS outputs.
    Used for: U-Ignore, U-Zeros, U-Ones, U-VLM."""
    def __init__(self, num_classes=14, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B4_Weights.IMAGENET1K_V1 if pretrained else None
        base    = models.efficientnet_b4(weights=weights)
        in_feats = base.classifier[1].in_features
        base.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_feats, num_classes),
        )
        self.model = base
    def forward(self, x):
        return self.model(x)

class EfficientNetB4MultiClass(nn.Module):
    """U-MultiClass variant — N_LABELS * 3 outputs (3 classes per label)."""
    def __init__(self, num_classes=14, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B4_Weights.IMAGENET1K_V1 if pretrained else None
        base    = models.efficientnet_b4(weights=weights)
        in_feats = base.classifier[1].in_features
        base.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_feats, num_classes * 3),
        )
        self.model = base
    def forward(self, x):
        return self.model(x)

print('✅ Cell 3 complete — all helpers, models, and loss functions loaded.')

In [ ]:
# ================================================================
# CELL 4: DenseNet-121 — Baseline Strategies
#         U-Ignore | U-Zeros | U-Ones | U-MultiClass
#
# Training  : Warmup (2 ep) + Cosine LR decay + Early Stopping
# Loss      : MaskedBCELoss for binary strategies
#             MultiClassLoss (3-class CE) for U-MultiClass
# Evaluation: AUROC, AUPRC, F1, Sensitivity, Specificity, BalancedAcc
#             (U-MultiClass uses softmax[:,:,1] → P(positive) for all metrics)
#
# Requires : Cell 1, Cell 2, Cell 3
# Outputs  : densenet121_metrics_detailed.csv
#            densenet121_training_history.csv
#            densenet121_pivot_<metric>.csv  (6 metrics)
# ================================================================

BASELINE_EXPS = ['U-Ignore', 'U-Zeros', 'U-Ones', 'U-MultiClass']
ARCH          = 'densenet121'

dn_detailed_dfs = []
dn_history_rows = []

print('=' * 65)
print(f'  DenseNet-121 — Baselines: {", ".join(BASELINE_EXPS)}')
print(f'  Max epochs : {IMPROVED_CONFIG["num_epochs"]}')
print(f'  Warmup     : {IMPROVED_CONFIG["warmup_epochs"]} epochs')
print(f'  Patience   : {IMPROVED_CONFIG["patience"]} epochs')
print('=' * 65)

for exp_name in BASELINE_EXPS:
    is_mc        = (exp_name == 'U-MultiClass')
    train_df_exp = EXPERIMENTS[exp_name]

    print(f'\n{"="*65}')
    print(f'  [{ARCH}]  EXPERIMENT: {exp_name}')
    print(f'  Train : {len(train_df_exp):,}  |  Val (internal): {len(val_df_clean):,}')
    print(f'  Loss  : {"MultiClassLoss (3-class CE)" if is_mc else "MaskedBCELoss (binary BCE)"}')
    print(f'{"="*65}')

    t_start = time.time()
    train_loader, valid_loader = get_loaders(train_df_exp, val_df_clean, ALL_LABELS)

    model = (DenseNet121MultiClass(num_classes=len(ALL_LABELS), pretrained=True)
             if is_mc
             else DenseNet121MultiLabel(num_classes=len(ALL_LABELS), pretrained=True))
    model     = model.to(device)
    criterion = MultiClassLoss() if is_mc else MaskedBCELoss()
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=IMPROVED_CONFIG['lr'],
                                  weight_decay=IMPROVED_CONFIG['weight_decay'])
    scaler    = GradScaler(enabled=IMPROVED_CONFIG['mixed_precision'])

    best_auc          = 0.0
    epochs_no_improve = 0
    best_model_path   = os.path.join(OUTPUT_DIR,
                                     f'best_{ARCH}_{exp_name.replace("-","_")}.pt')
    history           = []

    for epoch in range(IMPROVED_CONFIG['num_epochs']):
        lr_now = get_lr(epoch, IMPROVED_CONFIG)
        for pg in optimizer.param_groups:
            pg['lr'] = lr_now

        ep_start   = time.time()
        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler, epoch, IMPROVED_CONFIG)
        val_auc, _, _ = evaluate_model(
            model, valid_loader, ALL_LABELS, is_multiclass=is_mc)
        ep_min = (time.time() - ep_start) / 60

        if val_auc > best_auc:
            best_auc          = val_auc
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_model_path)
            ckpt_flag = ' ✅'
        else:
            epochs_no_improve += 1
            ckpt_flag = ''

        history.append({
            'arch'      : ARCH,
            'experiment': exp_name,
            'epoch'     : epoch + 1,
            'train_loss': round(train_loss, 4),
            'val_auroc' : round(val_auc,    4),
            'lr'        : round(lr_now,     7),
        })
        print(f'  Ep {epoch+1:>2}/{IMPROVED_CONFIG["num_epochs"]} | '
              f'Loss: {train_loss:.4f} | AUROC: {val_auc:.4f} | '
              f'Best: {best_auc:.4f} | LR: {lr_now:.2e} | '
              f'{ep_min:.1f}min{ckpt_flag}  '
              f'[patience {epochs_no_improve}/{IMPROVED_CONFIG["patience"]}]')

        if epochs_no_improve >= IMPROVED_CONFIG['patience']:
            print(f'\n  ⏹  Early stopping at epoch {epoch+1}')
            break

    # ── Final evaluation with best checkpoint ─────────────────────
    model.load_state_dict(torch.load(best_model_path))
    _, final_probs, final_targets = evaluate_model(
        model, valid_loader, ALL_LABELS, is_multiclass=is_mc)

    t_total    = (time.time() - t_start) / 60
    detail_df  = compute_detailed_metrics(
        final_probs, final_targets, ALL_LABELS, exp_name, ARCH)
    dn_detailed_dfs.append(detail_df)
    dn_history_rows.extend(history)

    print_metrics_table(detail_df, exp_name, ARCH)
    print(f'\n  ✅ {ARCH}/{exp_name} done in {t_total:.1f}min '
          f'| Best AUROC: {best_auc:.4f}')

    del model, optimizer, scaler, train_loader, valid_loader
    gc.collect()
    torch.cuda.empty_cache()

# ── Final test set evaluation (valid.csv = held-out test set) ─────
print('\n' + '='*65)
print('  DenseNet-121 — FINAL TEST SET EVALUATION (valid.csv)')
print('  NOTE: valid.csv is the held-out test set, not used during training.')
print('='*65)

dn_test_detailed_dfs = []
for exp_name in BASELINE_EXPS:
    is_mc         = (exp_name == 'U-MultiClass')
    best_model_path = os.path.join(OUTPUT_DIR,
                                   f'best_{ARCH}_{exp_name.replace("-","_")}.pt')
    if not os.path.exists(best_model_path):
        print(f'  ⚠️  Checkpoint not found for {exp_name}, skipping.')
        continue
    model = (DenseNet121MultiClass(num_classes=len(ALL_LABELS), pretrained=False)
             if is_mc
             else DenseNet121MultiLabel(num_classes=len(ALL_LABELS), pretrained=False))
    model.load_state_dict(torch.load(best_model_path))
    model = model.to(device)
    _, test_ds = DataLoader(
        CheXpertDataset(test_df_clean, ALL_LABELS, transform=VALID_TRANSFORMS),
        batch_size=IMPROVED_CONFIG['batch_size'] * 2,
        shuffle=False, num_workers=IMPROVED_CONFIG['num_workers'], pin_memory=True), None
    test_loader = DataLoader(
        CheXpertDataset(test_df_clean, ALL_LABELS, transform=VALID_TRANSFORMS),
        batch_size=IMPROVED_CONFIG['batch_size'] * 2,
        shuffle=False, num_workers=IMPROVED_CONFIG['num_workers'], pin_memory=True)
    _, test_probs, test_targets = evaluate_model(model, test_loader, ALL_LABELS, is_multiclass=is_mc)
    test_detail_df = compute_detailed_metrics(test_probs, test_targets, ALL_LABELS,
                                              exp_name + '_TEST', ARCH)
    dn_test_detailed_dfs.append(test_detail_df)
    # Print focus on 5 TARGET_LABELS
    focus = test_detail_df[test_detail_df['Label'].isin(TARGET_LABELS)][
        ['Label', 'AUROC', 'AUPRC', 'F1', 'Sensitivity', 'Specificity', 'BalancedAcc']
    ].set_index('Label')
    print(f'\n  [{ARCH}] {exp_name} — TEST SET (5 VLM Target Labels):')
    print(focus.round(4).to_string())
    del model
    gc.collect(); torch.cuda.empty_cache()

if dn_test_detailed_dfs:
    dn_test_combined = pd.concat(dn_test_detailed_dfs, ignore_index=True)
    dn_test_combined.to_csv(os.path.join(OUTPUT_DIR, 'densenet121_test_metrics_detailed.csv'), index=False)
    print(f'\n  Saved: densenet121_test_metrics_detailed.csv')

# ── Save results ──────────────────────────────────────────────────
dn_combined_df = pd.concat(dn_detailed_dfs, ignore_index=True)
save_and_summarise(dn_combined_df, dn_history_rows, ARCH)

In [ ]:
# ================================================================
# CELL 4b: DenseNet-121 — U-VLM Only
#
# FULLY SELF-CONTAINED: no dependency on Cell 4 (baselines).
# Can be run independently after Cell 1 + Cell 2 + Cell 3.
#
# Training  : Warmup (2 ep) + Cosine LR decay + Early Stopping
# Loss      : MaskedBCELoss (no -1s remain in U-VLM after Cell 2)
# Evaluation: AUROC, AUPRC, F1, Sensitivity, Specificity, BalancedAcc
#
# Outputs  : densenet121_uvlm_metrics_detailed.csv
#            densenet121_uvlm_training_history.csv
#            densenet121_uvlm_pivot_<metric>.csv  (6 metrics)
# ================================================================

import os, gc, time, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    balanced_accuracy_score, confusion_matrix,
)

# ── Standalone constants (no dependency on Cell 4) ───────────────
_ARCH     = 'densenet121'
_EXP_NAME = 'U-VLM'
_OUTPUT_DIR = '/kaggle/working/phase3_results'
os.makedirs(_OUTPUT_DIR, exist_ok=True)

# Re-read IMPROVED_CONFIG, device, ALL_LABELS, etc. from Cell 1 globals
# (they are already in scope since Cell 1 was run first)

print('=' * 65)
print(f'  DenseNet-121 — U-VLM Only (independent cell)')
print(f'  Max epochs : {IMPROVED_CONFIG["num_epochs"]}')
print(f'  Warmup     : {IMPROVED_CONFIG["warmup_epochs"]} epochs')
print(f'  Patience   : {IMPROVED_CONFIG["patience"]} epochs')
print('=' * 65)

train_df_exp = EXPERIMENTS[_EXP_NAME]

print(f'\n{"="*65}')
print(f'  [{_ARCH}]  EXPERIMENT: {_EXP_NAME}')
print(f'  Train : {len(train_df_exp):,}  |  Val (internal): {len(val_df_clean):,}')
print(f'  Loss  : MaskedBCELoss (binary BCE — no -1s in U-VLM)')
print(f'{"="*65}')

t_start = time.time()
train_loader, valid_loader = get_loaders(train_df_exp, val_df_clean, ALL_LABELS)

model     = DenseNet121MultiLabel(num_classes=len(ALL_LABELS), pretrained=True).to(device)
criterion = MaskedBCELoss()
optimizer = torch.optim.AdamW(model.parameters(),
                               lr=IMPROVED_CONFIG['lr'],
                               weight_decay=IMPROVED_CONFIG['weight_decay'])
scaler    = GradScaler(enabled=IMPROVED_CONFIG['mixed_precision'])

best_auc          = 0.0
epochs_no_improve = 0
best_model_path   = os.path.join(_OUTPUT_DIR,
                                  f'best_{_ARCH}_{_EXP_NAME.replace("-","_")}.pt')
history           = []

for epoch in range(IMPROVED_CONFIG['num_epochs']):
    lr_now = get_lr(epoch, IMPROVED_CONFIG)
    for pg in optimizer.param_groups:
        pg['lr'] = lr_now

    ep_start   = time.time()
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, epoch, IMPROVED_CONFIG)
    val_auc, _, _ = evaluate_model(model, valid_loader, ALL_LABELS, is_multiclass=False)
    ep_min = (time.time() - ep_start) / 60

    if val_auc > best_auc:
        best_auc          = val_auc
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        ckpt_flag = ' ✅'
    else:
        epochs_no_improve += 1
        ckpt_flag = ''

    history.append({
        'arch'      : _ARCH,
        'experiment': _EXP_NAME,
        'epoch'     : epoch + 1,
        'train_loss': round(train_loss, 4),
        'val_auroc' : round(val_auc,    4),
        'lr'        : round(lr_now,     7),
    })
    print(f'  Ep {epoch+1:>2}/{IMPROVED_CONFIG["num_epochs"]} | '
          f'Loss: {train_loss:.4f} | AUROC: {val_auc:.4f} | '
          f'Best: {best_auc:.4f} | LR: {lr_now:.2e} | '
          f'{ep_min:.1f}min{ckpt_flag}  '
          f'[patience {epochs_no_improve}/{IMPROVED_CONFIG["patience"]}]')

    if epochs_no_improve >= IMPROVED_CONFIG['patience']:
        print(f'\n  ⏹  Early stopping at epoch {epoch+1}')
        break

# ── Final evaluation with best checkpoint ─────────────────────────
model.load_state_dict(torch.load(best_model_path))
_, final_probs, final_targets = evaluate_model(
    model, valid_loader, ALL_LABELS, is_multiclass=False)

t_total   = (time.time() - t_start) / 60
detail_df = compute_detailed_metrics(
    final_probs, final_targets, ALL_LABELS, _EXP_NAME, _ARCH)

print_metrics_table(detail_df, _EXP_NAME, _ARCH)
print(f'\n  ✅ {_ARCH}/{_EXP_NAME} done in {t_total:.1f}min '
      f'| Best AUROC: {best_auc:.4f}')

# ── Save ──────────────────────────────────────────────────────────
detail_csv = os.path.join(_OUTPUT_DIR, f'{_ARCH}_uvlm_metrics_detailed.csv')
hist_csv   = os.path.join(_OUTPUT_DIR, f'{_ARCH}_uvlm_training_history.csv')
detail_df.to_csv(detail_csv, index=False)
pd.DataFrame(history).to_csv(hist_csv, index=False)
print(f'\n  Saved: {detail_csv}')
print(f'  Saved: {hist_csv}')

# Per-metric pivot CSVs
metrics = ['AUROC', 'AUPRC', 'F1', 'Sensitivity', 'Specificity', 'BalancedAcc']
print(f'\n{"="*65}')
print(f'  DenseNet-121 U-VLM — Per-Label Results')
print(f'{"="*65}')
for metric in metrics:
    print(f'\n  📊 {metric}:')
    col_data = detail_df.set_index('Label')[metric]
    print(col_data.round(4).to_string())
    pivot_path = os.path.join(_OUTPUT_DIR,
                               f'{_ARCH}_uvlm_pivot_{metric.lower()}.csv')
    col_data.to_csv(pivot_path, header=[metric])

# Focus on 5 VLM labels
print(f'\n  📊 FOCUS — 5 VLM-Relabeled Labels:')
focus = detail_df[detail_df['Label'].isin(TARGET_LABELS)][
    ['Label', 'AUROC', 'AUPRC', 'F1', 'Sensitivity', 'Specificity', 'BalancedAcc']
].set_index('Label')
print(focus.round(4).to_string())


# ── Final test set evaluation for U-VLM (DenseNet) ───────────────
print('\n' + '='*65)
print(f'  {_ARCH.upper()} U-VLM — FINAL TEST SET EVALUATION (valid.csv)')
print('='*65)
test_loader_uvlm = DataLoader(
    CheXpertDataset(test_df_clean, ALL_LABELS, transform=VALID_TRANSFORMS),
    batch_size=IMPROVED_CONFIG['batch_size'] * 2,
    shuffle=False, num_workers=IMPROVED_CONFIG['num_workers'], pin_memory=True)
model.load_state_dict(torch.load(best_model_path))
model = model.to(device)
_, test_probs_uvlm, test_targets_uvlm = evaluate_model(model, test_loader_uvlm, ALL_LABELS, is_multiclass=False)
test_detail_uvlm = compute_detailed_metrics(test_probs_uvlm, test_targets_uvlm, ALL_LABELS,
                                            'U-VLM_TEST', _ARCH)
test_focus_uvlm = test_detail_uvlm[test_detail_uvlm['Label'].isin(TARGET_LABELS)][
    ['Label', 'AUROC', 'AUPRC', 'F1', 'Sensitivity', 'Specificity', 'BalancedAcc']
].set_index('Label')
print(f'  [{_ARCH}] U-VLM — TEST SET (5 VLM Target Labels):')
print(test_focus_uvlm.round(4).to_string())
test_detail_uvlm.to_csv(os.path.join(_OUTPUT_DIR, f'{_ARCH}_uvlm_test_metrics_detailed.csv'), index=False)
print(f'  Saved: {_ARCH}_uvlm_test_metrics_detailed.csv')

del model, optimizer, scaler, train_loader, valid_loader
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ================================================================
# CELL 5: EfficientNet-B4 — Baseline Strategies
#         U-Ignore | U-Zeros | U-Ones | U-MultiClass
#
# Training  : Warmup (2 ep) + Cosine LR decay + Early Stopping
# Loss      : MaskedBCELoss for binary strategies
#             MultiClassLoss (3-class CE) for U-MultiClass
# Evaluation: AUROC, AUPRC, F1, Sensitivity, Specificity, BalancedAcc
#             (U-MultiClass uses softmax[:,:,1] → P(positive) for all metrics)
#
# Requires : Cell 1, Cell 2, Cell 3
# Outputs  : efficientnet_b4_metrics_detailed.csv
#            efficientnet_b4_training_history.csv
#            efficientnet_b4_pivot_<metric>.csv  (6 metrics)
# ================================================================

BASELINE_EXPS_EFF = ['U-Ignore', 'U-Zeros', 'U-Ones', 'U-MultiClass']
ARCH_EFF          = 'efficientnet_b4'

eff_detailed_dfs = []
eff_history_rows = []

print('=' * 65)
print(f'  EfficientNet-B4 — Baselines: {", ".join(BASELINE_EXPS_EFF)}')
print(f'  Max epochs : {IMPROVED_CONFIG["num_epochs"]}')
print(f'  Warmup     : {IMPROVED_CONFIG["warmup_epochs"]} epochs')
print(f'  Patience   : {IMPROVED_CONFIG["patience"]} epochs')
print('=' * 65)

for exp_name in BASELINE_EXPS_EFF:
    is_mc        = (exp_name == 'U-MultiClass')
    train_df_exp = EXPERIMENTS[exp_name]

    print(f'\n{"="*65}')
    print(f'  [{ARCH_EFF}]  EXPERIMENT: {exp_name}')
    print(f'  Train : {len(train_df_exp):,}  |  Val (internal): {len(val_df_clean):,}')
    print(f'  Loss  : {"MultiClassLoss (3-class CE)" if is_mc else "MaskedBCELoss (binary BCE)"}')
    print(f'{"="*65}')

    t_start = time.time()
    train_loader, valid_loader = get_loaders(train_df_exp, val_df_clean, ALL_LABELS)

    model = (EfficientNetB4MultiClass(num_classes=len(ALL_LABELS), pretrained=True)
             if is_mc
             else EfficientNetB4MultiLabel(num_classes=len(ALL_LABELS), pretrained=True))
    model     = model.to(device)
    criterion = MultiClassLoss() if is_mc else MaskedBCELoss()
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=IMPROVED_CONFIG['lr'],
                                  weight_decay=IMPROVED_CONFIG['weight_decay'])
    scaler    = GradScaler(enabled=IMPROVED_CONFIG['mixed_precision'])

    best_auc          = 0.0
    epochs_no_improve = 0
    best_model_path   = os.path.join(OUTPUT_DIR,
                                     f'best_{ARCH_EFF}_{exp_name.replace("-","_")}.pt')
    history           = []

    for epoch in range(IMPROVED_CONFIG['num_epochs']):
        lr_now = get_lr(epoch, IMPROVED_CONFIG)
        for pg in optimizer.param_groups:
            pg['lr'] = lr_now

        ep_start   = time.time()
        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler, epoch, IMPROVED_CONFIG)
        val_auc, _, _ = evaluate_model(
            model, valid_loader, ALL_LABELS, is_multiclass=is_mc)
        ep_min = (time.time() - ep_start) / 60

        if val_auc > best_auc:
            best_auc          = val_auc
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_model_path)
            ckpt_flag = ' ✅'
        else:
            epochs_no_improve += 1
            ckpt_flag = ''

        history.append({
            'arch'      : ARCH_EFF,
            'experiment': exp_name,
            'epoch'     : epoch + 1,
            'train_loss': round(train_loss, 4),
            'val_auroc' : round(val_auc,    4),
            'lr'        : round(lr_now,     7),
        })
        print(f'  Ep {epoch+1:>2}/{IMPROVED_CONFIG["num_epochs"]} | '
              f'Loss: {train_loss:.4f} | AUROC: {val_auc:.4f} | '
              f'Best: {best_auc:.4f} | LR: {lr_now:.2e} | '
              f'{ep_min:.1f}min{ckpt_flag}  '
              f'[patience {epochs_no_improve}/{IMPROVED_CONFIG["patience"]}]')

        if epochs_no_improve >= IMPROVED_CONFIG['patience']:
            print(f'\n  ⏹  Early stopping at epoch {epoch+1}')
            break

    # ── Final evaluation with best checkpoint ─────────────────────
    model.load_state_dict(torch.load(best_model_path))
    _, final_probs, final_targets = evaluate_model(
        model, valid_loader, ALL_LABELS, is_multiclass=is_mc)

    t_total   = (time.time() - t_start) / 60
    detail_df = compute_detailed_metrics(
        final_probs, final_targets, ALL_LABELS, exp_name, ARCH_EFF)
    eff_detailed_dfs.append(detail_df)
    eff_history_rows.extend(history)

    print_metrics_table(detail_df, exp_name, ARCH_EFF)
    print(f'\n  ✅ {ARCH_EFF}/{exp_name} done in {t_total:.1f}min '
          f'| Best AUROC: {best_auc:.4f}')

    del model, optimizer, scaler, train_loader, valid_loader
    gc.collect()
    torch.cuda.empty_cache()

# ── Final test set evaluation (valid.csv = held-out test set) ─────
print('\n' + '='*65)
print('  EfficientNet-B4 — FINAL TEST SET EVALUATION (valid.csv)')
print('  NOTE: valid.csv is the held-out test set, not used during training.')
print('='*65)

eff_test_detailed_dfs = []
for exp_name in BASELINE_EXPS_EFF:
    is_mc         = (exp_name == 'U-MultiClass')
    best_model_path = os.path.join(OUTPUT_DIR,
                                   f'best_{ARCH_EFF}_{exp_name.replace("-","_")}.pt')
    if not os.path.exists(best_model_path):
        print(f'  ⚠️  Checkpoint not found for {exp_name}, skipping.')
        continue
    model = (EfficientNetB4MultiClass(num_classes=len(ALL_LABELS), pretrained=False)
             if is_mc
             else EfficientNetB4MultiLabel(num_classes=len(ALL_LABELS), pretrained=False))
    model.load_state_dict(torch.load(best_model_path))
    model = model.to(device)
    test_loader = DataLoader(
        CheXpertDataset(test_df_clean, ALL_LABELS, transform=VALID_TRANSFORMS),
        batch_size=IMPROVED_CONFIG['batch_size'] * 2,
        shuffle=False, num_workers=IMPROVED_CONFIG['num_workers'], pin_memory=True)
    _, test_probs, test_targets = evaluate_model(model, test_loader, ALL_LABELS, is_multiclass=is_mc)
    test_detail_df = compute_detailed_metrics(test_probs, test_targets, ALL_LABELS,
                                              exp_name + '_TEST', ARCH_EFF)
    eff_test_detailed_dfs.append(test_detail_df)
    focus = test_detail_df[test_detail_df['Label'].isin(TARGET_LABELS)][
        ['Label', 'AUROC', 'AUPRC', 'F1', 'Sensitivity', 'Specificity', 'BalancedAcc']
    ].set_index('Label')
    print(f'\n  [{ARCH_EFF}] {exp_name} — TEST SET (5 VLM Target Labels):')
    print(focus.round(4).to_string())
    del model
    gc.collect(); torch.cuda.empty_cache()

if eff_test_detailed_dfs:
    eff_test_combined = pd.concat(eff_test_detailed_dfs, ignore_index=True)
    eff_test_combined.to_csv(os.path.join(OUTPUT_DIR, 'efficientnet_b4_test_metrics_detailed.csv'), index=False)
    print(f'\n  Saved: efficientnet_b4_test_metrics_detailed.csv')

# ── Save results ──────────────────────────────────────────────────
eff_combined_df = pd.concat(eff_detailed_dfs, ignore_index=True)
save_and_summarise(eff_combined_df, eff_history_rows, ARCH_EFF)

In [ ]:
# ================================================================
# CELL 5b: EfficientNet-B4 — U-VLM Only
#
# FULLY SELF-CONTAINED: no dependency on Cell 5 (baselines).
# Can be run independently after Cell 1 + Cell 2 + Cell 3.
#
# Training  : Warmup (2 ep) + Cosine LR decay + Early Stopping
# Loss      : MaskedBCELoss (no -1s remain in U-VLM after Cell 2)
# Evaluation: AUROC, AUPRC, F1, Sensitivity, Specificity, BalancedAcc
#
# Outputs  : efficientnet_b4_uvlm_metrics_detailed.csv
#            efficientnet_b4_uvlm_training_history.csv
#            efficientnet_b4_uvlm_pivot_<metric>.csv  (6 metrics)
# ================================================================

import os, gc, time, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    balanced_accuracy_score, confusion_matrix,
)

_ARCH_EFF  = 'efficientnet_b4'
_EXP_NAME  = 'U-VLM'
_OUTPUT_DIR = '/kaggle/working/phase3_results'
os.makedirs(_OUTPUT_DIR, exist_ok=True)

print('=' * 65)
print(f'  EfficientNet-B4 — U-VLM Only (independent cell)')
print(f'  Max epochs : {IMPROVED_CONFIG["num_epochs"]}')
print(f'  Warmup     : {IMPROVED_CONFIG["warmup_epochs"]} epochs')
print(f'  Patience   : {IMPROVED_CONFIG["patience"]} epochs')
print('=' * 65)

train_df_exp = EXPERIMENTS[_EXP_NAME]

print(f'\n{"="*65}')
print(f'  [{_ARCH_EFF}]  EXPERIMENT: {_EXP_NAME}')
print(f'  Train : {len(train_df_exp):,}  |  Val (internal): {len(val_df_clean):,}')
print(f'  Loss  : MaskedBCELoss (binary BCE — no -1s in U-VLM)')
print(f'{"="*65}')

t_start = time.time()
train_loader, valid_loader = get_loaders(train_df_exp, val_df_clean, ALL_LABELS)

model     = EfficientNetB4MultiLabel(num_classes=len(ALL_LABELS), pretrained=True).to(device)
criterion = MaskedBCELoss()
optimizer = torch.optim.AdamW(model.parameters(),
                               lr=IMPROVED_CONFIG['lr'],
                               weight_decay=IMPROVED_CONFIG['weight_decay'])
scaler    = GradScaler(enabled=IMPROVED_CONFIG['mixed_precision'])

best_auc          = 0.0
epochs_no_improve = 0
best_model_path   = os.path.join(_OUTPUT_DIR,
                                  f'best_{_ARCH_EFF}_{_EXP_NAME.replace("-","_")}.pt')
history           = []

for epoch in range(IMPROVED_CONFIG['num_epochs']):
    lr_now = get_lr(epoch, IMPROVED_CONFIG)
    for pg in optimizer.param_groups:
        pg['lr'] = lr_now

    ep_start   = time.time()
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, epoch, IMPROVED_CONFIG)
    val_auc, _, _ = evaluate_model(model, valid_loader, ALL_LABELS, is_multiclass=False)
    ep_min = (time.time() - ep_start) / 60

    if val_auc > best_auc:
        best_auc          = val_auc
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        ckpt_flag = ' ✅'
    else:
        epochs_no_improve += 1
        ckpt_flag = ''

    history.append({
        'arch'      : _ARCH_EFF,
        'experiment': _EXP_NAME,
        'epoch'     : epoch + 1,
        'train_loss': round(train_loss, 4),
        'val_auroc' : round(val_auc,    4),
        'lr'        : round(lr_now,     7),
    })
    print(f'  Ep {epoch+1:>2}/{IMPROVED_CONFIG["num_epochs"]} | '
          f'Loss: {train_loss:.4f} | AUROC: {val_auc:.4f} | '
          f'Best: {best_auc:.4f} | LR: {lr_now:.2e} | '
          f'{ep_min:.1f}min{ckpt_flag}  '
          f'[patience {epochs_no_improve}/{IMPROVED_CONFIG["patience"]}]')

    if epochs_no_improve >= IMPROVED_CONFIG['patience']:
        print(f'\n  ⏹  Early stopping at epoch {epoch+1}')
        break

# ── Final evaluation with best checkpoint ─────────────────────────
model.load_state_dict(torch.load(best_model_path))
_, final_probs, final_targets = evaluate_model(
    model, valid_loader, ALL_LABELS, is_multiclass=False)

t_total   = (time.time() - t_start) / 60
detail_df = compute_detailed_metrics(
    final_probs, final_targets, ALL_LABELS, _EXP_NAME, _ARCH_EFF)

print_metrics_table(detail_df, _EXP_NAME, _ARCH_EFF)
print(f'\n  ✅ {_ARCH_EFF}/{_EXP_NAME} done in {t_total:.1f}min '
      f'| Best AUROC: {best_auc:.4f}')

# ── Save ──────────────────────────────────────────────────────────
detail_csv = os.path.join(_OUTPUT_DIR, f'{_ARCH_EFF}_uvlm_metrics_detailed.csv')
hist_csv   = os.path.join(_OUTPUT_DIR, f'{_ARCH_EFF}_uvlm_training_history.csv')
detail_df.to_csv(detail_csv, index=False)
pd.DataFrame(history).to_csv(hist_csv, index=False)
print(f'\n  Saved: {detail_csv}')
print(f'  Saved: {hist_csv}')

# Per-metric pivot CSVs
metrics = ['AUROC', 'AUPRC', 'F1', 'Sensitivity', 'Specificity', 'BalancedAcc']
print(f'\n{"="*65}')
print(f'  EfficientNet-B4 U-VLM — Per-Label Results')
print(f'{"="*65}')
for metric in metrics:
    print(f'\n  📊 {metric}:')
    col_data = detail_df.set_index('Label')[metric]
    print(col_data.round(4).to_string())
    pivot_path = os.path.join(_OUTPUT_DIR,
                               f'{_ARCH_EFF}_uvlm_pivot_{metric.lower()}.csv')
    col_data.to_csv(pivot_path, header=[metric])

# Focus on 5 VLM labels
print(f'\n  📊 FOCUS — 5 VLM-Relabeled Labels:')
focus = detail_df[detail_df['Label'].isin(TARGET_LABELS)][
    ['Label', 'AUROC', 'AUPRC', 'F1', 'Sensitivity', 'Specificity', 'BalancedAcc']
].set_index('Label')
print(focus.round(4).to_string())


# ── Final test set evaluation for U-VLM (EfficientNet) ──────────
print('\n' + '='*65)
print(f'  {_ARCH_EFF.upper()} U-VLM — FINAL TEST SET EVALUATION (valid.csv)')
print('='*65)
test_loader_uvlm_eff = DataLoader(
    CheXpertDataset(test_df_clean, ALL_LABELS, transform=VALID_TRANSFORMS),
    batch_size=IMPROVED_CONFIG['batch_size'] * 2,
    shuffle=False, num_workers=IMPROVED_CONFIG['num_workers'], pin_memory=True)
model.load_state_dict(torch.load(best_model_path))
model = model.to(device)
_, test_probs_uvlm_eff, test_targets_uvlm_eff = evaluate_model(
    model, test_loader_uvlm_eff, ALL_LABELS, is_multiclass=False)
test_detail_uvlm_eff = compute_detailed_metrics(
    test_probs_uvlm_eff, test_targets_uvlm_eff, ALL_LABELS, 'U-VLM_TEST', _ARCH_EFF)
test_focus_uvlm_eff = test_detail_uvlm_eff[test_detail_uvlm_eff['Label'].isin(TARGET_LABELS)][
    ['Label', 'AUROC', 'AUPRC', 'F1', 'Sensitivity', 'Specificity', 'BalancedAcc']
].set_index('Label')
print(f'  [{_ARCH_EFF}] U-VLM — TEST SET (5 VLM Target Labels):')
print(test_focus_uvlm_eff.round(4).to_string())
test_detail_uvlm_eff.to_csv(os.path.join(_OUTPUT_DIR, f'{_ARCH_EFF}_uvlm_test_metrics_detailed.csv'), index=False)
print(f'  Saved: {_ARCH_EFF}_uvlm_test_metrics_detailed.csv')

del model, optimizer, scaler, train_loader, valid_loader
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ================================================================
# CELL 6: Cross-Architecture Comparison
#
# Reads saved CSVs only — no in-memory dependency on any training cell.
# Run after Cell 4 + Cell 4b (DenseNet) and Cell 5 + Cell 5b (EfficientNet).
#
# Two result sets are merged:
#   *_metrics_detailed.csv        ← internal val set results (used for model selection)
#   *_test_metrics_detailed.csv   ← FINAL TEST SET results (valid.csv, held-out)
#
# The paper should report the TEST SET numbers.
# The internal val set numbers are for ablation / model selection only.
#
# KEY QUESTION — Which labels to report?
#   The U-VLM pipeline specifically relabeled 5 TARGET_LABELS:
#     Edema, Atelectasis, Pleural Effusion,
#     Enlarged Cardiomediastinum, Consolidation
#   → PRIMARY RESULTS: report metrics on these 5 labels.
#   → SECONDARY (for completeness): mean over all 14 labels.
#
# Outputs : all_archs_val_metrics_detailed.csv   (internal val, model selection)
#           all_archs_test_metrics_detailed.csv  (FINAL RESULTS — cite this)
#           cross_arch_<metric>.csv  (6 metrics, test set, 5 target labels)
# ================================================================
import os
import pandas as pd
import numpy as np

OUTPUT_DIR    = '/kaggle/working/phase3_results'
ALL_LABELS    = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly',
    'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation',
    'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
    'Pleural Other', 'Fracture', 'Support Devices'
]
TARGET_LABELS = ['Edema', 'Atelectasis', 'Pleural Effusion',
                 'Enlarged Cardiomediastinum', 'Consolidation']
EXP_ORDER     = ['U-Ignore', 'U-Zeros', 'U-Ones', 'U-MultiClass', 'U-VLM']
ARCHS         = ['densenet121', 'efficientnet_b4']
METRICS       = ['AUROC', 'AUPRC', 'F1', 'Sensitivity', 'Specificity', 'BalancedAcc']

# ── Helper: load and merge CSVs for a given split ─────────────────
def load_split_csvs(split_suffix):
    """
    split_suffix: ''  for internal val  |  '_test'  for test set
    """
    dfs = []
    for arch in ARCHS:
        for exp_tag, file_tag in [('baselines', f'{arch}{split_suffix}_metrics_detailed'),
                                    ('U-VLM',     f'{arch}_uvlm{split_suffix}_metrics_detailed')]:
            # Strip experiment suffix from experiment column names
            path = os.path.join(OUTPUT_DIR, f'{file_tag}.csv')
            if os.path.exists(path):
                df = pd.read_csv(path)
                # Normalise: remove _TEST suffix from Experiment column so pivots work cleanly
                df['Experiment'] = df['Experiment'].str.replace('_TEST', '', regex=False)
                dfs.append(df)
                print(f'  ✅ Loaded: {file_tag}.csv')
            else:
                print(f'  ⚠️  Missing: {path} — run the training cells first')
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print('='*65)
print('  Loading INTERNAL VAL results (model selection / ablation)')
print('='*65)
val_df_all = load_split_csvs('')
if not val_df_all.empty:
    val_df_all = val_df_all.drop_duplicates(subset=['Arch', 'Experiment', 'Label'], keep='last')
    val_df_all.to_csv(os.path.join(OUTPUT_DIR, 'all_archs_val_metrics_detailed.csv'), index=False)
    print(f'  Merged: {len(val_df_all)} rows → all_archs_val_metrics_detailed.csv')

print('\n' + '='*65)
print('  Loading FINAL TEST SET results (valid.csv — cite these)')
print('='*65)
test_df_all = load_split_csvs('_test')
if not test_df_all.empty:
    test_df_all = test_df_all.drop_duplicates(subset=['Arch', 'Experiment', 'Label'], keep='last')
    test_df_all.to_csv(os.path.join(OUTPUT_DIR, 'all_archs_test_metrics_detailed.csv'), index=False)
    print(f'  Merged: {len(test_df_all)} rows → all_archs_test_metrics_detailed.csv')

# ── Use test set for all final reporting ──────────────────────────
all_df = test_df_all if not test_df_all.empty else val_df_all
split_label = 'TEST SET (valid.csv)' if not test_df_all.empty else 'INTERNAL VAL (fallback)'

# ── Build cross-arch pivot tables — 5 TARGET_LABELS (primary) ────
def make_pivot(df, metric, label_filter=None):
    sub = df[df['Label'].isin(label_filter)] if label_filter else df
    return (sub.groupby(['Arch', 'Experiment'])[metric]
            .mean()
            .reset_index()
            .pivot(index='Arch', columns='Experiment', values=metric)
            .reindex(columns=[c for c in EXP_ORDER if c in sub['Experiment'].unique()]))

print('\n' + '='*65)
print(f'  FINAL RESULTS — {split_label}')
print(f'  PRIMARY: 5 VLM-Relabeled Target Labels')
print('='*65)

pivots_5 = {}
for metric in METRICS:
    p = make_pivot(all_df, metric, label_filter=TARGET_LABELS)
    pivots_5[metric] = p
    p.to_csv(os.path.join(OUTPUT_DIR, f'cross_arch_{metric.lower()}_5labels.csv'))
    print(f'\n  📊 {metric} (5 VLM Target Labels):')
    print(p.round(4).to_string())

print('\n' + '='*65)
print(f'  SECONDARY: Mean over all 14 labels (for completeness)')
print('='*65)

pivots_14 = {}
for metric in METRICS:
    p = make_pivot(all_df, metric)
    pivots_14[metric] = p
    p.to_csv(os.path.join(OUTPUT_DIR, f'cross_arch_{metric.lower()}_14labels.csv'))
    print(f'\n  📊 {metric} (All 14 Labels):')
    print(p.round(4).to_string())

# ── Best overall combination ──────────────────────────────────────
if not pivots_5['AUROC'].empty:
    stacked = pivots_5['AUROC'].stack().reset_index()
    stacked.columns = ['Arch', 'Experiment', 'AUROC']
    stacked['AUPRC'] = pivots_5['AUPRC'].stack().values
    best = stacked.loc[stacked['AUROC'].idxmax()]
    print(f'\n  🏆 Best combination by mean AUROC (5 VLM Target Labels):')
    print(f'     Arch       : {best["Arch"]}')
    print(f'     Strategy   : {best["Experiment"]}')
    print(f'     AUROC      : {best["AUROC"]:.4f}')
    print(f'     AUPRC      : {best["AUPRC"]:.4f}')

# ── U-VLM vs best baseline for each arch ─────────────────────────
print('\n' + '='*65)
print('  U-VLM GAIN vs BEST BASELINE (mean AUROC, 5 VLM Target Labels)')
print('='*65)
focus_df = all_df[all_df['Label'].isin(TARGET_LABELS)]
for arch in ARCHS:
    arch_df = focus_df[focus_df['Arch'] == arch]
    baseline_exps  = [e for e in EXP_ORDER if e != 'U-VLM']
    baseline_auroc = (arch_df[arch_df['Experiment'].isin(baseline_exps)]
                      .groupby('Experiment')['AUROC'].mean())
    vlm_auroc = arch_df[arch_df['Experiment'] == 'U-VLM']['AUROC'].mean()
    if len(baseline_auroc) > 0 and not np.isnan(vlm_auroc):
        best_baseline     = baseline_auroc.idxmax()
        best_baseline_val = baseline_auroc.max()
        delta = vlm_auroc - best_baseline_val
        marker = '▲' if delta > 0 else '▼'
        print(f'\n  {arch}:')
        print(f'    U-VLM AUROC              : {vlm_auroc:.4f}')
        print(f'    Best baseline AUROC      : {best_baseline_val:.4f} ({best_baseline})')
        print(f'    Delta (5 VLM labels)     : {delta:+.4f} {marker}')

print(f'\n  ✅ All CSVs saved → {OUTPUT_DIR}')
print(f'  ⚠️  Cite numbers from: all_archs_test_metrics_detailed.csv ({split_label})')
